# INSANE adaptive motion: one frozen six-training job

Train mars_1–3, validation mars_4–5. Test mars_6–7 absent and reserved.
Adaptive full/gps_only, seeds0/1/2. Original encoder + sigmoid scalar gate.
Code SHA256 `ea0480ffdb853952ef1184283b480646927be865c15e83eae71778abc644560c`. All source/config/test bytes embedded; no main checkout.
Train-only Damped-CV selection precedes validation metrics; original fixed masks/GT points.
RUN_TRAINING=True is explicitly authorized. On any technical failure STOP, no retry.


In [ ]:
from pathlib import Path
import sys
# The user explicitly authorized one private Kaggle adaptive job with six trainings after successful tests.
RUN_TRAINING = True
DATA_ROOT = Path('/kaggle/input')  # one setting; discover the unique pinned bundle in current/legacy Kaggle layouts
CODE_ROOT = Path('/kaggle/working/insane_adaptive_code')
OUT_DIR = Path('/kaggle/working/outputs/insane_adaptive')
LAUNCH_DIR = Path('/kaggle/working/insane_adaptive_launch')
LAUNCH_DIR.mkdir(parents=True,exist_ok=True)


In [ ]:
import hashlib
SOURCES = {'src/insane/__init__.py': '"""Separate INSANE pilot; no Zurich imports, weights, or input data."""\n', 'src/insane/data.py': '"""INSANE adapter: causal weighted IMU integration and mask-first GNSS features.\n\nGT is returned separately. Its quaternion columns are deliberately never read.\nPublished timestamps/ENU positions are already processed; no second offset/calibration.\n"""\nfrom pathlib import Path\nimport hashlib,json,re\nimport numpy as np\nimport pandas as pd\n\nIMU_COLUMNS=[\'t\',\'a_x\',\'a_y\',\'a_z\',\'w_x\',\'w_y\',\'w_z\']\nGPS_COLUMNS=[\'t\',\'lat\',\'long\',\'alt\',\'p_x\',\'p_y\',\'p_z\',\'cov_p_x\',\'cov_p_y\',\'cov_p_z\']\nGT_COLUMNS=[\'t\',\'p_x\',\'p_y\',\'p_z\']\n\ndef sha(path):return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\ndef read_columns(path,columns):\n    d=pd.read_csv(path);d.columns=d.columns.str.strip()\n    if not set(columns)<=set(d.columns):raise ValueError(\'Unexpected schema \'+str(path))\n    x=d[columns].to_numpy(dtype=np.float64)\n    if not np.isfinite(x).all():raise ValueError(\'Nonfinite input \'+str(path))\n    if np.any(np.diff(x[:,0])<=0):raise ValueError(\'Nonmonotonic/duplicate timestamps \'+str(path))\n    return x\n\ndef ecef(lla):\n    lla=np.asarray(lla);lat,lon=np.deg2rad(lla[...,0]),np.deg2rad(lla[...,1]);h=lla[...,2]\n    a=6378137.;e2=6.69437999014e-3;n=a/np.sqrt(1-e2*np.sin(lat)**2)\n    return np.stack([(n+h)*np.cos(lat)*np.cos(lon),(n+h)*np.cos(lat)*np.sin(lon),(n*(1-e2)+h)*np.sin(lat)],axis=-1)\n\ndef enu(lla,reference):\n    lat,lon=np.deg2rad(reference[:2]);s,c=np.sin,np.cos\n    r=np.array([[-s(lon),c(lon),0],[-s(lat)*c(lon),-s(lat)*s(lon),c(lat)],[c(lat)*c(lon),c(lat)*s(lon),s(lat)]])\n    return np.einsum(\'ij,nj->ni\',r,ecef(lla)-ecef(np.array(reference)))\n\ndef load_flight(root,sequence,expected=None):\n    root=Path(root);folder=root/sequence\n    if expected:\n        for item in expected[\'files\']:\n            if sha(root/item[\'path\'])!=item[\'sha256\']:raise ValueError(\'Source hash mismatch \'+item[\'path\'])\n    im=read_columns(folder/\'px4_imu.csv\',IMU_COLUMNS)\n    gps=read_columns(folder/\'px4_gps.csv\',GPS_COLUMNS)\n    gt=read_columns(folder/\'ground_truth_8hz.csv\',GT_COLUMNS)\n    readmes=list((folder/\'archive_metadata\').glob(\'*/README.txt\'))\n    if len(readmes)!=1:raise ValueError(\'Missing unique stream README \'+sequence)\n    m=re.search(r\'\\[Latitude,Longitude,Altitude\\]\\s*=\\s*([\\d.\\-]+),\\s*([\\d.\\-]+),\\s*([\\d.\\-]+)\',readmes[0].read_text())\n    if not m:raise ValueError(\'Missing published ENU origin\')\n    rounded_ref=np.array([float(v) for v in m.groups()])\n    # README rounds to six decimals; use authors\' pinned extract_data.m:89-104.\n    ref=np.array([30.5999294,34.867308,526.594] if sequence.startswith(\'mars_\')\n                 else [46.606867399,14.279121199,484.017])\n    if np.max(np.abs(ref-rounded_ref))>1e-6:raise ValueError(\'Wrong location/calibration reference\')\n    projection_error=float(np.max(np.abs(enu(gps[:,1:4],ref)-gps[:,4:7])))\n    if projection_error>.002:raise ValueError(f\'ENU contract mismatch {sequence}: {projection_error}\')\n    if np.any(gps[:,7:10]<0):raise ValueError(\'Negative GNSS covariance\')\n    # One common rebase, using ordinary GNSS only. No GT alignment.\n    epoch=float(im[0,0]);origin=gps[0,4:7].copy()\n    sensor={\'sequence\':sequence,\'epoch\':epoch,\'imu_t\':im[:,0]-epoch,\'imu\':im[:,1:],\n       \'gps_t\':gps[:,0]-epoch,\'gps_p\':gps[:,4:7]-origin,\'gps_cov\':gps[:,7:10],\n       \'origin_enu\':origin,\'published_enu_reference\':ref,\'projection_error_m\':projection_error}\n    reference={\'t\':gt[:,0]-epoch,\'p\':gt[:,1:]-origin}\n    return sensor,reference\n\ndef integral_at(t,x,q):\n    """Integral of left-held measurements, never a sample after q."""\n    t=np.asarray(t);x=np.asarray(x);q=np.asarray(q)\n    if np.any(q<t[0]) or np.any(q>t[-1]):raise ValueError(\'No sensor extrapolation\')\n    prefix=np.vstack([np.zeros(x.shape[1]),np.cumsum(np.diff(t)[:,None]*x[:-1],axis=0)])\n    j=np.searchsorted(t,q,side=\'right\')-1\n    return prefix[j]+(q-t[j])[:,None]*x[j]\n\ndef aggregate_imu(sensor,period):\n    t=sensor[\'imu_t\'];start=max(t[0],sensor[\'gps_t\'][0])\n    edges=start+np.arange(int(np.floor((t[-1]-start)/period))+1)*period\n    integrated=integral_at(t,sensor[\'imu\'],edges)\n    dt=np.diff(edges);query=edges[1:];values=np.diff(integrated,axis=0)/dt[:,None]\n    indices=np.searchsorted(t,query,side=\'right\')-1\n    counts=np.searchsorted(t,query,side=\'right\')-np.searchsorted(t,edges[:-1],side=\'right\')\n    return {\'t\':query,\'imu\':values,\'dt\':dt,\'imu_source_t\':t[indices],\n        \'imu_age\':query-t[indices],\'imu_count\':counts,\'edges\':edges}\n\ndef fit_scaler(aggregated,train_names):\n    total=sum(float(aggregated[s][\'dt\'].sum()) for s in train_names)\n    mean=sum((aggregated[s][\'imu\']*aggregated[s][\'dt\'][:,None]).sum(0) for s in train_names)/total\n    var=sum(((aggregated[s][\'imu\']-mean)**2*aggregated[s][\'dt\'][:,None]).sum(0) for s in train_names)/total\n    return {\'mean\':mean.tolist(),\'std\':np.maximum(np.sqrt(var),1e-5).tolist(),\'fit_flights\':list(train_names),\'weighted_seconds\':total}\n\ndef blocked(t,intervals):\n    out=np.zeros(len(t),bool)\n    for a,b in intervals:out|=(t>=a)&(t<b)\n    return out\n\ndef build_inputs(sensor,agg,indices,intervals,scaler,cfg):\n    """Function deliberately accepts no reference / GT object."""\n    ids=np.asarray(indices,dtype=int);t=agg[\'t\'][ids]\n    # FINAL native-source mask first. Everything derived from GNSS follows it.\n    legal=~blocked(sensor[\'gps_t\'],intervals)\n    st=sensor[\'gps_t\'][legal];sp=sensor[\'gps_p\'][legal];sc=sensor[\'gps_cov\'][legal]\n    original_ids=np.flatnonzero(legal)\n    j=np.searchsorted(st,t,side=\'right\')-1\n    if np.any(j<0):raise ValueError(\'No prior legal GNSS fix\')\n    age=t-st[j]\n    availability=(~blocked(t,intervals))&(age<=cfg[\'gps_gap_limit_s\'])\n    source_index=original_ids[j]\n    update=np.r_[True,np.diff(source_index)!=0]&availability\n    vel=np.zeros_like(sp)\n    for k in range(len(st)):\n        first=np.searchsorted(st,st[k]-cfg[\'cv_history_s\'],side=\'left\')\n        if k>first and st[k]-st[first]>=1:\n            x=st[first:k+1];x=x-x.mean();y=sp[first:k+1];vel[k]=(x[:,None]*(y-y.mean(0))).sum(0)/np.sum(x*x)\n    velocity=vel[j]\n    delta=np.zeros((len(t),3));previous=j[0]\n    for k in range(1,len(t)):\n        if update[k]:delta[k]=sp[j[k]]-sp[previous];previous=j[k]\n    dt=np.r_[agg[\'dt\'][ids[0]],np.diff(t)]\n    integration_dt=dt.copy()\n    first_out=np.flatnonzero((~availability)&np.r_[True,availability[:-1]])\n    for k in first_out:integration_dt[k]=t[k]-st[j[k]]\n    imu=(agg[\'imu\'][ids]-scaler[\'mean\'])/scaler[\'std\']\n    common=np.column_stack([availability,update,age/60.,dt,agg[\'imu_age\'][ids],\n       agg[\'imu_age\'][ids]>.02,agg[\'imu_count\'][ids]/10.,np.sqrt(sc[j])/10.])\n    gnss=np.column_stack([delta/5.,velocity/5.])\n    assert common.shape[1]==10 and imu.shape[1]==6 and gnss.shape[1]==6\n    assert np.all(st[j]<=t+1e-9) and not blocked(st[j],intervals).any()\n    return {\'t\':t,\'imu\':imu.astype(\'float32\'),\'gnss\':gnss.astype(\'float32\'),\'common\':common.astype(\'float32\'),\n      \'velocity_prior\':velocity.astype(\'float32\'),\'integration_dt\':integration_dt.astype(\'float32\'),\n      \'availability\':availability,\'held\':sp[j],\'source_t\':st[j],\'source_index\':source_index,\n      \'imu_source_t\':agg[\'imu_source_t\'][ids],\'raw_imu\':agg[\'imu\'][ids], \'intervals\':intervals}\n\ndef make_labels(reference,x,spec,cfg):\n    """Only native GT rows are scored. One interpolated anchor is target-only."""\n    select=(reference[\'t\']>=x[\'t\'][0])&(reference[\'t\']<=x[\'t\'][-1])\n    gt_ids=np.flatnonzero(select);t=reference[\'t\'][select];gt=reference[\'p\'][select]\n    index=np.searchsorted(x[\'t\'],t,side=\'right\')-1\n    outage=blocked(t,x[\'intervals\'])\n    anchor=np.zeros_like(gt)\n    for a,b in x[\'intervals\']:\n        take=(t>=a)&(t<b)\n        first=np.searchsorted(x[\'t\'],a)\n        at=float(x[\'source_t\'][first])\n        j=np.searchsorted(reference[\'t\'],at)\n        if j==0 or j==len(reference[\'t\']) or reference[\'t\'][j]-reference[\'t\'][j-1]>cfg[\'gt_gap_limit_s\']:\n            raise ValueError(\'Missing GT anchor support\')\n        anchor[take]=[np.interp(at,reference[\'t\'],reference[\'p\'][:,d]) for d in range(3)]\n    target=gt-anchor\n    return {\'t\':t,\'gt\':gt,\'gt_native_index\':gt_ids,\'token_index\':index,\'partial_dt\':t-x[\'t\'][index],\n            \'outage\':outage,\'target_motion\':target,\'anchor_gt\':anchor}\n\ndef specs_indices(agg,spec):\n    ids=np.flatnonzero((agg[\'t\']>=spec[\'start_s\']-1e-9)&(agg[\'t\']<=spec[\'end_s\']+1e-9))\n    if len(ids)<2:raise ValueError(\'Empty scenario\')\n    return ids\n\ndef episode_inputs(sensor,reference,agg,spec,scaler,cfg):\n    blocks=[] if spec[\'duration_s\']==0 else [[spec[\'onset_s\'],spec[\'onset_s\']+spec[\'duration_s\']]]\n    x=build_inputs(sensor,agg,specs_indices(agg,spec),blocks,scaler,cfg)\n    labels=make_labels(reference,x,spec,cfg)\n    if spec[\'duration_s\'] and not labels[\'outage\'].any():raise ValueError(\'No native reference observations\')\n    return x,labels\n\ndef quality_segments(sensor,reference,cfg):\n    start=max(sensor[\'imu_t\'][0]+cfg[\'token_period_s\'],sensor[\'gps_t\'][0],reference[\'t\'][0])\n    end=min(sensor[\'imu_t\'][-1],sensor[\'gps_t\'][-1],reference[\'t\'][-1])\n    invalid=[]\n    for times,limit in [(sensor[\'imu_t\'],cfg[\'imu_gap_limit_s\']),(sensor[\'gps_t\'],cfg[\'gps_gap_limit_s\']),(reference[\'t\'],cfg[\'gt_gap_limit_s\'])]:\n        for k in np.flatnonzero(np.diff(times)>limit):invalid.append((float(times[k]),float(times[k+1])))\n    segments=[(float(start),float(end))]\n    for a,b in sorted(invalid):\n        split=[]\n        for l,r in segments:\n            if b<=l or a>=r:split.append((l,r))\n            else:\n                if a>l:split.append((l,a))\n                if b<r:split.append((b,r))\n        segments=split\n    return [(a,b) for a,b in segments if b>a]\n', 'src/insane/eskf.py': '"""15-state right-error ESKF, ENU world / ROS body FLU.\n\nNominal p,v,R,ba,bg; delta state [p,v,theta,ba,bg]. Gravity=[0,0,-g].\nWhite-noise densities squared times dt. Joseph position update and attitude reset.\nOnly synthetic validation is authorized as ready: real initialization/calibration\nmust pass readiness; never substitute GT or GNSS course for body yaw.\n"""\nimport numpy as np\nfrom scipy.spatial.transform import Rotation\n\nG=9.80665\n\ndef mm(a,b):\n    # Explicit contraction avoids spurious Accelerate BLAS floating flags on macOS.\n    return np.einsum(\'ij,jk->ik\',a,b)\n\ndef mv(a,b):return np.einsum(\'ij,j->i\',a,b)\n\ndef skew(v):\n    x,y,z=v;return np.array([[0,-z,y],[z,0,-x],[-y,x,0]],float)\n\nclass ESKF:\n    def __init__(self,p=None,v=None,R=None,ba=None,bg=None,lever=None,noise=(.1,.01,.001,.0001)):\n        self.p=np.zeros(3) if p is None else np.array(p,float)\n        self.v=np.zeros(3) if v is None else np.array(v,float)\n        self.R=np.eye(3) if R is None else np.array(R,float)\n        self.ba=np.zeros(3) if ba is None else np.array(ba,float)\n        self.bg=np.zeros(3) if bg is None else np.array(bg,float)\n        self.lever=np.zeros(3) if lever is None else np.array(lever,float)\n        self.P=np.diag([10.]*3+[3.]*3+[.1,.1,np.pi**2]+[.1]*3+[.01]*3)\n        self.noise=noise\n    def predict(self,force,omega,dt):\n        if not (0<dt<=.1):raise ValueError(\'Unsupported integration interval\')\n        f=np.array(force)-self.ba;w=np.array(omega)-self.bg\n        old=self.R.copy();mid=mm(old,Rotation.from_rotvec(w*dt/2).as_matrix())\n        accel=mv(mid,f)+[0,0,-G]\n        self.p+=self.v*dt+.5*accel*dt*dt;self.v+=accel*dt\n        self.R=mm(old,Rotation.from_rotvec(w*dt).as_matrix())\n        F=np.zeros((15,15));F[:3,3:6]=np.eye(3);F[3:6,6:9]=-mm(mid,skew(f));F[3:6,9:12]=-mid\n        F[6:9,6:9]=-skew(w);F[6:9,12:15]=-np.eye(3)\n        phi=np.eye(15)+F*dt+.5*mm(F,F)*dt*dt\n        qa,qg,qba,qbg=np.square(self.noise);Q=np.zeros((15,15))\n        Q[:3,:3]=np.eye(3)*qa*dt**3/3;Q[:3,3:6]=np.eye(3)*qa*dt**2/2;Q[3:6,:3]=Q[:3,3:6].T\n        Q[3:6,3:6]=np.eye(3)*qa*dt;Q[6:9,6:9]=np.eye(3)*qg*dt\n        Q[9:12,9:12]=np.eye(3)*qba*dt;Q[12:15,12:15]=np.eye(3)*qbg*dt\n        self.P=mm(mm(phi,self.P),phi.T)+Q;self.P=(self.P+self.P.T)/2\n    def update_gnss(self,position,covariance):\n        H=np.zeros((3,15));H[:,:3]=np.eye(3);H[:,6:9]=-mm(self.R,skew(self.lever))\n        N=np.diag(covariance);S=mm(mm(H,self.P),H.T)+N\n        K=np.linalg.solve(S,mm(H,self.P)).T\n        error=np.asarray(position)-(self.p+mv(self.R,self.lever));delta=mv(K,error)\n        A=np.eye(15)-mm(K,H);P=mm(mm(A,self.P),A.T)+mm(mm(K,N),K.T)\n        self.p+=delta[:3];self.v+=delta[3:6];self.R=mm(self.R,Rotation.from_rotvec(delta[6:9]).as_matrix())\n        self.ba+=delta[9:12];self.bg+=delta[12:15]\n        reset=np.eye(15);reset[6:9,6:9]-=.5*skew(delta[6:9]);self.P=mm(mm(reset,P),reset.T)\n        self.P=(self.P+self.P.T)/2\n\ndef initialize_from_allowed_history(imu,gnss_t,gnss_p,heading_rad=None,lever=None):\n    if heading_rad is None or lever is None:\n        raise RuntimeError(\'ESKF not ready: independent permitted heading/alignment and GNSS lever arm are required; no GT/course fallback\')\n    imu=np.asarray(imu);f=imu[:,:3].mean(0)\n    if abs(np.linalg.norm(f)-G)>.3 or np.max(imu[:,:3].std(0))>.2:\n        raise RuntimeError(\'Initial quasi-static gravity check failed\')\n    tilt,_=Rotation.align_vectors(np.array([[0,0,G]]),f[None])\n    R=mm(Rotation.from_euler(\'z\',heading_rad).as_matrix(),tilt.as_matrix())\n    t=np.asarray(gnss_t);p=np.asarray(gnss_p);keep=t>=t[-1]-5;t=t[keep];p=p[keep]\n    tc=t-t.mean();v=(tc[:,None]*(p-p.mean(0))).sum(0)/np.sum(tc*tc)\n    return ESKF(p=p[-1]-mv(R,np.asarray(lever)),v=v,R=R,lever=lever)\n\ndef readiness(cfg):\n    return {\'status\':\'not_ready\',\'synthetic_mechanization\':\'tested separately\',\n        \'blockers\':cfg[\'eskf\'][\'blockers\'],\'real_metrics_allowed\':False,\n        \'no_gt_initialization\':True,\'no_course_as_yaw\':True,\'parameter_tuning\':\'none\'}\n', 'src/insane/metrics.py': '"""Native-reference timestamp metrics; no fitted trajectory alignment."""\nimport numpy as np\nimport pandas as pd\n\ndef rmse(error):\n    return float(np.sqrt(np.mean(np.sum(np.asarray(error)**2,axis=-1))))\n\ndef score(spec,x,y,pred,motion,method,inference_ms=None):\n    inside=y[\'outage\'];select=inside if inside.any() else np.ones(len(inside),bool)\n    err=pred-y[\'gt\'];e=err[select]\n    cut=None if spec[\'duration_s\']==0 else spec[\'onset_s\']+spec[\'duration_s\']\n    recovery=np.zeros(len(inside),bool) if cut is None else ((y[\'t\']>=cut)&(y[\'t\']<cut+5))\n    row={\'scenario\':spec[\'id\'],\'flight\':spec[\'flight\'],\'split\':\'validation\',\'method\':method,\'seed\':0 if method in [\'full\',\'gps_only\'] else None,\n       \'duration_s\':spec[\'duration_s\'],\'status\':\'complete\',\'n_native_gt\':int(select.sum()),\n       \'relative_motion_rmse3d_m\':rmse((motion-y[\'target_motion\'])[inside]) if inside.any() else None,\n       \'absolute_rmse3d_m\':rmse(e),\'absolute_rmse_h_m\':rmse(e[:,:2]),\'absolute_rmse_v_m\':rmse(e[:,2:3]),\n       \'final_unavailable_error_m\':float(np.linalg.norm(e[-1])) if inside.any() else None,\n       \'final_relative_motion_error_m\':float(np.linalg.norm((motion-y[\'target_motion\'])[inside][-1])) if inside.any() else None,\n       \'recovery_first_5s_rmse3d_m\':rmse(err[recovery]) if recovery.any() else None,\n       \'first_recovered_fix_error_m\':None,\'inference_ms\':inference_ms,\'reason\':None}\n    if cut is not None:\n        idx=y[\'token_index\'];back=(y[\'t\']>=cut)&(x[\'source_t\'][idx]>=cut)\n        if back.any():row[\'first_recovered_fix_error_m\']=float(np.linalg.norm(err[np.flatnonzero(back)[0]]))\n    return row\n\ndef baselines(x,y):\n    ids=y[\'token_index\'];held=x[\'held\'][ids];available=x[\'availability\'][ids]\n    motion=x[\'velocity_prior\'][ids]*(y[\'t\']-x[\'source_t\'][ids])[:,None]*(~available)[:,None]\n    return {\'held_gnss\':(held.copy(),np.zeros_like(held)),\'constant_velocity_gnss\':(held+motion,motion)}\n\ndef paired(rows):\n    d=pd.DataFrame(rows);out=[]\n    for scenario,g in d[(d.status==\'complete\')&(d.duration_s>0)].groupby(\'scenario\',sort=False):\n        a=g.set_index(\'method\');full=float(a.loc[\'full\',\'relative_motion_rmse3d_m\'])\n        for other in [\'gps_only\',\'constant_velocity_gnss\',\'held_gnss\']:\n            b=float(a.loc[other,\'relative_motion_rmse3d_m\'])\n            out.append({\'scenario\':scenario,\'flight\':a.iloc[0].flight,\'duration_s\':int(a.iloc[0].duration_s),\'comparison\':\'full_minus_\'+other,\n                \'full_rmse_m\':full,\'other_rmse_m\':b,\'difference_m\':full-b,\'difference_percent\':100*(full-b)/b if b>0 else None,\n                \'full_better\':full<b,\'n_native_gt\':int(a.loc[\'full\',\'n_native_gt\'])})\n    return out\n', 'src/insane/model.py': '"""Pilot dual-branch GRU. Same learned skeleton/count for full and gps_only."""\nimport torch\nfrom torch import nn\n\nclass FusionNavINSANE(nn.Module):\n    def __init__(self,variant=\'full\',hidden=32):\n        super().__init__()\n        if variant not in [\'full\',\'gps_only\']:raise ValueError(variant)\n        self.variant=variant\n        self.imu_branch=nn.Sequential(nn.Linear(6,16),nn.Tanh())\n        self.gnss_branch=nn.Sequential(nn.Linear(6,16),nn.Tanh())\n        self.common_branch=nn.Sequential(nn.Linear(10,8),nn.Tanh())\n        self.gru=nn.GRU(40,hidden,batch_first=True)\n        self.velocity_head=nn.Linear(hidden,3)\n        nn.init.zeros_(self.velocity_head.weight);nn.init.zeros_(self.velocity_head.bias)\n    def forward(self,imu,gnss,common,state=None):\n        if self.variant==\'gps_only\':imu=torch.zeros_like(imu)\n        features=torch.cat([self.imu_branch(imu),self.gnss_branch(gnss),self.common_branch(common)],dim=-1)\n        out,state=self.gru(features,state)\n        return self.velocity_head(out),state\n\ndef integrate_motion(dv,prior,dt,availability):\n    velocity=prior+dv\n    total=torch.cumsum(velocity*dt[...,None],dim=1)\n    index=torch.arange(total.shape[1],device=total.device)[None].expand(total.shape[0],-1)\n    last=torch.cummax(torch.where(availability,index,torch.zeros_like(index)),dim=1).values\n    anchor=torch.gather(total,1,last[...,None].expand(-1,-1,3))\n    return torch.where(availability[...,None],torch.zeros_like(total),total-anchor),velocity\n\ndef native_motion(motion,velocity,x,labels):\n    """Causal readout at original GT timestamps. Never interpolate future prediction."""\n    idx=torch.as_tensor(labels[\'token_index\'],device=motion.device,dtype=torch.long)\n    partial=torch.as_tensor(labels[\'partial_dt\'],device=motion.device,dtype=motion.dtype)\n    absent=torch.as_tensor(~x[\'availability\'][labels[\'token_index\']],device=motion.device)\n    return motion[idx]+velocity[idx]*partial[:,None]*absent[:,None]\n', 'src/insane/report.py': '"""Automatic complete pilot report: all episodes retained, no winner selection."""\nfrom pathlib import Path\nimport json\nimport pandas as pd\n\ndef table(frame):\n    headers=list(frame.columns);lines=[\'| \'+\' | \'.join(headers)+\' |\',\'|\'+\'|\'.join([\'---\']*len(headers))+\'|\']\n    for row in frame.itertuples(index=False,name=None):\n        vals=[]\n        for v in row:\n            if pd.isna(v):vals.append(\'—\')\n            elif isinstance(v,float):vals.append(f\'{v:.6f}\')\n            else:vals.append(str(v))\n        lines.append(\'| \'+\' | \'.join(vals)+\' |\')\n    return \'\\n\'.join(lines)\n\ndef generate(out,run_info=None):\n    out=Path(out);d=pd.read_csv(out/\'summary.csv\');pair=pd.read_csv(out/\'paired_differences.csv\');hist=pd.read_csv(out/\'training_history.csv\')\n    cfg=json.loads((out/\'config.json\').read_text());split=json.loads((out/\'splits.json\').read_text())\n    env=json.loads((out/\'environment.json\').read_text());status=json.loads((out/\'status.json\').read_text())\n    readiness=json.loads((out/\'eskf_status.json\').read_text())\n    clean=d[(d.status==\'complete\')&(d.duration_s>0)]\n    means=clean.groupby([\'duration_s\',\'method\'],sort=False).relative_motion_rmse3d_m.mean().unstack(\'method\').reset_index()\n    flights=clean.groupby([\'flight\',\'duration_s\',\'method\'],sort=False).relative_motion_rmse3d_m.mean().unstack(\'method\').reset_index()\n    main=pair[pair.comparison==\'full_minus_gps_only\'];wins=int(main.full_better.sum())\n    sections=[\n      \'# INSANE_PILOT_REPORT\',\n      \'**Status: \'+status[\'status\']+\'. STOP after one seed=0 pilot. No test metrics or additional seeds.**\',\n      \'## Run and frozen protocol\',\n      \'Run identity: `\'+json.dumps(run_info or status.get(\'run_identity\',{}),ensure_ascii=False)+\'`.\',\n      \'Code version: `\'+status[\'code_version\']+\'`. Config/manifests/source snapshot are stored alongside this report. No hyperparameter or dataset change after training.\',\n      \'Train: \'+\', \'.join(split[\'train\'])+\'. Validation: \'+\', \'.join(split[\'validation\'])+\'. Reserved test: \'+\', \'.join(split[\'test_reserved\'])+\' (not loaded by the run). Outdoor_1 was loader smoke only.\',\n      \'## Data / reference contract\',\n      \'Inputs: published PX4 IMU specific force (m/s²) and angular rate (rad/s), ordinary PX4 GNSS ENU position/covariance only. IMU is aggregated with causal time-weighted integration at 20 Hz; dt/sample age/count and all GNSS availability/quality features are shared with gps_only. No GT attitude, RTK, onboard pose, magnetometer, or absolute position is a learned input. Published timestamp corrections are already applied; they are not applied twice.\',\n      \'Scoring uses original rows/timestamps of ground_truth_8hz.csv; the nominal filename does not imply 8 independent observations each second. Reference positions are at the PX4 IMU, produced by the authors from dual RTK and magnetometer geometry. Their timing alignment is postprocessed and GT is not error-free. Ordinary GNSS antenna lever arm is absent from the supplied calibration; its offset/dynamic lever effect remains a limitation. No GT-derived rotation or fitted trajectory alignment is applied.\',\n      \'Source: [INSANE data](https://www.aau.at/en/smart-systems-technologies/control-of-networked-systems/datasets/insane-dataset/), [authors tools](https://github.com/aau-cns/insane_dataset_tools/tree/9a1c8c0fdd195f2d869fff292f2ce5b273c5a03d), [Brommer et al.](https://arxiv.org/abs/2210.09114). License and attribution are included in the private data package.\',\n      \'## Architecture and training\',\n      \'Pilot architecture, not established novelty: IMU 6→16, GNSS displacement/velocity 6→16, shared quality/time 10→8; concat40→GRU32→velocity residual3. Matching full/gps_only parameter count: \'+str(json.loads((out/\'parameter_counts.json\').read_text()))+\'. gps_only zeros only the six normalized sensor values; shared features remain available.\',\n      f"AdamW lr={cfg[\'learning_rate\']}, weight_decay={cfg[\'weight_decay\']}, batch={cfg[\'batch_size\']}, max_epochs={cfg[\'max_epochs\']}, patience={cfg[\'patience\']}, Huber delta={cfg[\'huber_delta_m\']}m. Both methods start from scratch with seed0 and identical frozen augmentation/order. Selection: {cfg[\'validation_criterion\']}; strict decrease, min_delta=0. No test selection.",\n      table(hist),\n      \'## Metric definitions\',\n      \'Primary = sqrt(mean(||(prediction−last_legal_GNSS_position)−(GT(t)−GT(last_legal_fix_time))||²)) inside each artificial outage. The anchor GT is interpolated only for the target/reference; scoring rows are original GT timestamps. The state/position is never initialized from GT. Predictions at native GT times use the most recent causal token plus its velocity over the remaining fraction of dt. Absolute3D/H/V and final unavailable error use raw positions without alignment. CV uses least-squares velocity from the last5s of permitted fixes. No new hidden GNSS values or covariance enter outage features.\',\n      \'Summary values are macro arithmetic means of per-episode RMSE, not pooled-point RMSE. Nested durations/overlapping episodes are dependent. One seed, two validation flights, same vehicle/campaign: no confidence interval, significance or generalization claim.\',\n      \'## All fixed validation scenarios\',\n      table(d[[\'scenario\',\'method\',\'status\',\'n_native_gt\',\'relative_motion_rmse3d_m\',\'absolute_rmse3d_m\',\'absolute_rmse_h_m\',\'absolute_rmse_v_m\',\'final_unavailable_error_m\',\'recovery_first_5s_rmse3d_m\']]),\n      \'## By flight and duration — primary RMSE in metres\',table(flights),\n      \'## By duration — primary RMSE in metres\',table(means),\n      \'## Paired full comparisons\',table(pair),\n      f\'Full is lower than matched gps_only in **{wins}/{len(main)}** outage scenarios. Negative full-minus-other means lower full error. Percent =100×(full−other)/other. No cherry-picking of epochs beyond the predefined early-stopping criterion or of favorable scenarios.\',\n      \'## ESKF readiness\',\n      \'A separate 15-state ESKF (p,v,right attitude error,ba,bg) has synthetic stationary/constant-velocity/turn/irregular-dt/Joseph-update tests. Real-data ESKF is **not ready**: \'+ \'; \'.join(readiness[\'blockers\'])+\'. Its rows are explicitly not_ready with no numeric metrics. Unknown heading is not replaced by GNSS course, GT or an arbitrary confident yaw. No network superiority over a calibrated ESKF is claimed. Noise values in config are synthetic engineering defaults, not fitted flight calibration.\',\n      \'## Runtime / recovery\',\n      \'Environment: `\'+json.dumps(env,ensure_ascii=False)+\'`. Model inference_ms is one measured synchronized forward/readout on that hardware, excludes preprocessing/training and is not a repeated latency benchmark. Held/CV rows share the measured time of their joint calculation; those numbers are not separately timed per-baseline latencies. Recovery is first5s after nominal outage end; first_recovered_fix_error uses the first scored point after an actual returned fix. Decoder returns to legal GNSS on availability; no post-outage trajectory alignment.\',\n      table(d[d.status==\'complete\'][[\'scenario\',\'method\',\'inference_ms\',\'first_recovered_fix_error_m\']]),\n      \'## Conclusion and next action\',\n      \'This pilot estimates whether IMU helps under this fixed small-data protocol. A small difference is not proof of IMU utility. Inspect the complete paired table, duration means and replication across flights; a single-seed validation-selected result cannot establish generalization. **STOP. Do not run more seeds, change metrics, or swap flights automatically.**\',\n      \'Artifacts: config/splits/outage/training manifests, train-only scaler, source hashes, environment, best/last checkpoints, training history, each native-timestamp prediction/reference/mask, summary.csv, paired_differences.csv, preflight.json, eskf_status.json, status.json. All test flights remain reserved.\'\n    ]\n    (out/\'INSANE_PILOT_REPORT.md\').write_text(\'\\n\\n\'.join(sections)+\'\\n\')\n', 'src/insane/run.py': '"""One frozen two-model pilot. No execution at import, no resume or extra seeds."""\nfrom pathlib import Path\nimport hashlib,json,platform,time,traceback,shutil,datetime\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch.nn.utils.rnn import pad_sequence\nfrom .data import load_flight,aggregate_imu,fit_scaler,episode_inputs,sha\nfrom .model import FusionNavINSANE,integrate_motion,native_motion\nfrom .metrics import score,baselines,paired\nfrom .eskf import readiness\nfrom .report import generate\n\ndef write(path,obj):\n    Path(path).write_text(json.dumps(obj,indent=2,allow_nan=False)+\'\\n\')\n\ndef code_version(code_root):\n    hashes={str(p.relative_to(code_root)):sha(p) for base in [\'src/insane\',\'tests/insane\',\'configs/insane\']\n            for p in sorted((Path(code_root)/base).glob(\'*\')) if p.is_file() and p.suffix in [\'.py\',\'.json\']}\n    return hashlib.sha256(json.dumps(hashes,sort_keys=True).encode()).hexdigest(),hashes\n\ndef preflight(data_root,config_dir):\n    config_dir=Path(config_dir);cfg=json.loads((config_dir/\'pilot.json\').read_text())\n    lock=json.loads((config_dir/\'protocol_lock.json\').read_text())\n    for name,h in lock[\'sha256\'].items():\n        if sha(config_dir/name)!=h:raise ValueError(\'Frozen protocol changed: \'+name)\n    split=json.loads((config_dir/\'splits.json\').read_text());manifest=json.loads((config_dir/\'download_manifest.json\').read_text())\n    val=json.loads((config_dir/\'outages.json\').read_text())[\'scenarios\']\n    train=json.loads((config_dir/\'training_episodes.json\').read_text())[\'episodes\']\n    sets=[set(split[k]) for k in [\'train\',\'validation\',\'test_reserved\']]\n    if any(sets[i]&sets[j] for i in range(3) for j in range(i)):raise ValueError(\'Flight split overlap\')\n    if cfg[\'seed\']!=0 or cfg[\'max_epochs\']>30 or cfg[\'patience\']!=5:raise ValueError(\'Unexpected pilot plan\')\n    for rel,item in manifest[\'support_files\'].items():\n        if sha(Path(data_root)/rel)!=item[\'sha256\']:raise ValueError(\'Calibration/reference hash mismatch \'+rel)\n    loaded={};aggs={}\n    # Explicit allowlist: NEVER load test or loader-only flight into the run.\n    for seq in split[\'train\']+split[\'validation\']:\n        loaded[seq]=load_flight(data_root,seq,manifest[\'sequences\'][seq]);aggs[seq]=aggregate_imu(loaded[seq][0],cfg[\'token_period_s\'])\n    scaler=fit_scaler(aggs,split[\'train\'])\n    expected=json.loads((config_dir/\'scaler_preflight.json\').read_text())\n    np.testing.assert_allclose(scaler[\'mean\'],expected[\'mean\'],rtol=1e-8,atol=1e-8)\n    np.testing.assert_allclose(scaler[\'std\'],expected[\'std\'],rtol=1e-8,atol=1e-8)\n    cache={}\n    for spec in val+train:\n        expected_split=\'validation\' if spec in val else \'train\'\n        if spec[\'flight\'] not in split[expected_split]:raise ValueError(\'Wrong episode split\')\n        x,y=episode_inputs(*loaded[spec[\'flight\']],aggs[spec[\'flight\']],spec,scaler,cfg)\n        if hashlib.sha256(np.asarray(x[\'availability\'],np.uint8).tobytes()).hexdigest()!=spec[\'token_mask_sha256\']:raise ValueError(\'Mask changed \'+spec[\'id\'])\n        if hashlib.sha256(np.asarray(y[\'t\'],dtype=\'<f8\').tobytes()).hexdigest()!=spec[\'native_gt_timestamp_sha256\']:raise ValueError(\'Native GT support changed\')\n        if int(y[\'outage\'].sum())!=spec[\'n_native_gt_outage\']:raise ValueError(\'Point count changed\')\n        cache[spec[\'id\']]=(x,y)\n    evidence={\'status\':\'pass\',\'train_flights\':split[\'train\'],\'validation_flights\':split[\'validation\'],\n        \'test_loaded\':False,\'raw_calibration_hashes_pass\':True,\'projection_max_m\':max(s[0][\'projection_error_m\'] for s in loaded.values()),\n        \'validated_training_episodes\':len(train),\'validated_eval_scenarios\':len(val),\'train_only_scaler_pass\':True,\n        \'all_masks_and_native_gt_timestamp_hashes_pass\':True}\n    return cfg,split,val,train,scaler,cache,evidence\n\ndef ten(v,device):return torch.as_tensor(v,device=device,dtype=torch.float32)\n\ndef evaluate_model(model,x,y,device):\n    model.eval()\n    if device==\'cuda\':torch.cuda.synchronize()\n    start=time.perf_counter()\n    with torch.inference_mode():\n        dv,_=model(ten(x[\'imu\'],device)[None],ten(x[\'gnss\'],device)[None],ten(x[\'common\'],device)[None])\n        motion,velocity=integrate_motion(dv,ten(x[\'velocity_prior\'],device)[None],ten(x[\'integration_dt\'],device)[None],torch.as_tensor(x[\'availability\'],device=device)[None])\n        native=native_motion(motion[0],velocity[0],x,y).cpu().numpy().astype(float)\n    if device==\'cuda\':torch.cuda.synchronize()\n    ms=(time.perf_counter()-start)*1000\n    pred=x[\'held\'][y[\'token_index\']]+native\n    return pred,native,ms\n\ndef batch_loss(model,selected,device,cfg):\n    # Padding exists only inside computational batches; loss uses real native GT rows.\n    def stack(key):return pad_sequence([ten(x[key],device) for x,y in selected],batch_first=True)\n    imu,gnss,common=stack(\'imu\'),stack(\'gnss\'),stack(\'common\')\n    available=pad_sequence([torch.as_tensor(x[\'availability\'],device=device) for x,y in selected],batch_first=True,padding_value=True)\n    dv,_=model(imu,gnss,common)\n    motion,velocity=integrate_motion(dv,stack(\'velocity_prior\'),stack(\'integration_dt\'),available)\n    losses=[]\n    for k,(x,y) in enumerate(selected):\n        chosen=torch.as_tensor(y[\'outage\'],device=device)\n        native=native_motion(motion[k],velocity[k],x,y)\n        losses.append(torch.nn.functional.huber_loss(native[chosen],ten(y[\'target_motion\'],device)[chosen],delta=cfg[\'huber_delta_m\']))\n    return torch.stack(losses).mean()\n\ndef execute(data_root,config_dir,out,code_root,run_training=False,require_cuda=True):\n    if not run_training:raise RuntimeError(\'Training gate is false\')\n    if require_cuda and not torch.cuda.is_available():raise RuntimeError(\'Pilot must run on Kaggle GPU, not the Mac\')\n    out=Path(out);code_root=Path(code_root)\n    if out.exists():raise FileExistsError(\'No overwrite/restart/resume of one pilot: \'+str(out))\n    cfg,split,val,train,scaler,cache,checks=preflight(data_root,config_dir)\n    version,hashes=code_version(code_root)\n    out.mkdir(parents=True);(out/\'checkpoints\').mkdir();(out/\'predictions\').mkdir()\n    for name in [\'splits\',\'outages\',\'training_episodes\',\'protocol_lock\',\'download_manifest\',\'inventory\']:\n        shutil.copyfile(Path(config_dir)/(name+\'.json\'),out/(name+\'.json\'))\n    write(out/\'config.json\',cfg);write(out/\'scalers.json\',scaler);write(out/\'preflight.json\',checks)\n    for rel in hashes:\n        dst=out/\'source_snapshot\'/rel;dst.parent.mkdir(parents=True,exist_ok=True);shutil.copyfile(code_root/rel,dst)\n    write(out/\'source_hashes.json\',hashes);write(out/\'eskf_status.json\',readiness(cfg))\n    device=\'cuda\' if torch.cuda.is_available() else \'cpu\'\n    torch.backends.cuda.matmul.allow_tf32=False;torch.backends.cudnn.allow_tf32=False\n    torch.backends.cudnn.benchmark=False;torch.backends.cudnn.deterministic=True\n    env={\'python\':platform.python_version(),\'torch\':torch.__version__,\'numpy\':np.__version__,\'pandas\':pd.__version__,\n         \'device\':device,\'cuda\':torch.version.cuda,\'hardware\':torch.cuda.get_device_name(0) if device==\'cuda\' else platform.processor()}\n    write(out/\'environment.json\',env)\n    status={\'status\':\'running\',\'seed\':0,\'test_used\':False,\'code_version\':version,\'started_utc\':datetime.datetime.now(datetime.timezone.utc).isoformat(),\n       \'run_identity\':{\'kernel\':\'shapok/drone-nav-insane-pilot\',\'version\':\'see submission receipt\'},\'models\':{},\'eskf_status\':\'not_ready\'}\n    write(out/\'status.json\',status);history=[];models={};counts={}\n    try:\n        for variant in cfg[\'methods\']:\n            torch.manual_seed(0);np.random.seed(0);torch.cuda.manual_seed_all(0)\n            model=FusionNavINSANE(variant,cfg[\'hidden\']).to(device);counts[variant]=sum(p.numel() for p in model.parameters())\n            optimizer=torch.optim.AdamW(model.parameters(),lr=cfg[\'learning_rate\'],weight_decay=cfg[\'weight_decay\'])\n            best=float(\'inf\');best_epoch=0;bad=0\n            for epoch in range(1,cfg[\'max_epochs\']+1):\n                start=time.perf_counter();model.train();losses=[];episodes=[s for s in train if s[\'epoch\']==epoch]\n                for k in range(0,len(episodes),cfg[\'batch_size\']):\n                    loss=batch_loss(model,[cache[s[\'id\']] for s in episodes[k:k+cfg[\'batch_size\']]],device,cfg)\n                    if not torch.isfinite(loss):raise ValueError(\'Nonfinite train loss\')\n                    optimizer.zero_grad();loss.backward();torch.nn.utils.clip_grad_norm_(model.parameters(),cfg[\'grad_clip\']);optimizer.step()\n                    losses.append(float(loss.detach().cpu()))\n                val_scores=[]\n                for spec in val:\n                    if not spec[\'duration_s\']:continue\n                    x,y=cache[spec[\'id\']];pred,motion,ms=evaluate_model(model,x,y,device)\n                    val_scores.append(score(spec,x,y,pred,motion,variant)[\'relative_motion_rmse3d_m\'])\n                criterion=float(np.mean(val_scores));improved=criterion<best-cfg[\'min_delta_m\']\n                if improved:\n                    best=criterion;best_epoch=epoch;bad=0\n                    torch.save(model.state_dict(),out/\'checkpoints\'/f\'{variant}_best_seed0.pt\')\n                else:bad+=1\n                history.append({\'method\':variant,\'epoch\':epoch,\'mean_batch_huber_train_loss\':float(np.mean(losses)),\n                    \'validation_macro_relative_rmse3d_m\':criterion,\'selected_best_so_far\':improved,\'epochs_without_improvement\':bad,\n                    \'epoch_wall_seconds\':time.perf_counter()-start})\n                pd.DataFrame(history).to_csv(out/\'training_history.csv\',index=False)\n                status[\'models\'][variant]={\'last_epoch\':epoch,\'best_epoch\':best_epoch,\'best_validation_criterion_m\':best};write(out/\'status.json\',status)\n                print(json.dumps(history[-1]),flush=True)\n                if bad>=cfg[\'patience\']:break\n            torch.save(model.state_dict(),out/\'checkpoints\'/f\'{variant}_last_seed0.pt\')\n            model.load_state_dict(torch.load(out/\'checkpoints\'/f\'{variant}_best_seed0.pt\',map_location=device,weights_only=True));models[variant]=model.eval()\n        if counts[\'full\']!=counts[\'gps_only\']:raise ValueError(\'Parameter count mismatch\')\n        write(out/\'parameter_counts.json\',counts)\n        rows=[]\n        for spec in val:\n            x,y=cache[spec[\'id\']]\n            start=time.perf_counter();base=baselines(x,y);base_ms=(time.perf_counter()-start)*1000\n            predictions={key:(p,m,base_ms) for key,(p,m) in base.items()}\n            for name,model in models.items():predictions[name]=evaluate_model(model,x,y,device)\n            folder=out/\'predictions\'/spec[\'id\'];folder.mkdir()\n            for method,(prediction,motion,ms) in predictions.items():\n                if not np.isfinite(prediction).all():raise ValueError(\'Nonfinite predictions\')\n                rows.append(score(spec,x,y,prediction,motion,method,ms));idx=y[\'token_index\']\n                np.savez_compressed(folder/(method+\'.npz\'),t=y[\'t\'],gt=y[\'gt\'],pred=prediction,motion=motion,\n                  target_motion=y[\'target_motion\'],anchor_gt=y[\'anchor_gt\'],gt_native_index=y[\'gt_native_index\'],outage=y[\'outage\'],\n                  availability=x[\'availability\'][idx],source_t=x[\'source_t\'][idx],source_index=x[\'source_index\'][idx],\n                  held=x[\'held\'][idx],token_t=x[\'t\'][idx],imu_source_t=x[\'imu_source_t\'][idx])\n            empty={k:None for k in rows[-1]};empty.update({\'scenario\':spec[\'id\'],\'flight\':spec[\'flight\'],\'split\':\'validation\',\n              \'method\':\'eskf\',\'duration_s\':spec[\'duration_s\'],\'status\':\'not_ready\',\'reason\':\'; \'.join(cfg[\'eskf\'][\'blockers\'])});rows.append(empty)\n        pd.DataFrame(rows).to_csv(out/\'summary.csv\',index=False);pd.DataFrame(paired(rows)).to_csv(out/\'paired_differences.csv\',index=False)\n        status.update(status=\'complete\',evaluation_rows=len(rows),finished_utc=datetime.datetime.now(datetime.timezone.utc).isoformat(),extra_seeds_started=False)\n        write(out/\'status.json\',status);generate(out)\n        print(\'PILOT COMPLETE. STOP. No test / additional seeds.\',flush=True)\n    except Exception as e:\n        status.update(status=\'failed\',error=str(e));write(out/\'status.json\',status)\n        (out/\'failure_traceback.txt\').write_text(traceback.format_exc());raise\n', 'src/insane_adaptive/__init__.py': '"""One fixed INSANE adaptive-motion experiment; no import-time execution."""\n', 'src/insane_adaptive/baselines.py': '"""Analytic causal exponential damping; tune tau exclusively on train."""\nimport numpy as np\nfrom insane.metrics import rmse\n\ndef damped_cv(x,y,tau):\n    if not np.isfinite(tau) or tau<=0:raise ValueError(\'tau must be positive and finite\')\n    idx=y[\'token_index\'];available=x[\'availability\'][idx]\n    age=np.maximum(0,y[\'t\']-x[\'source_t\'][idx])\n    # Integral of v0 exp(-age/tau) from last legal fix to native timestamp.\n    motion=x[\'velocity_prior\'][idx]*(tau*(-np.expm1(-age/tau)))[:,None]*(~available)[:,None]\n    return x[\'held\'][idx]+motion,motion\n\ndef choose_tau(train_specs,cache,candidates,allowed_flights):\n    if not train_specs or any(s[\'flight\'] not in allowed_flights for s in train_specs):\n        raise ValueError(\'Damping selection accepts train episodes only\')\n    rows=[]\n    for tau in candidates:\n        scores=[]\n        for spec in train_specs:\n            x,y=cache[spec[\'id\']];_,m=damped_cv(x,y,tau)\n            scores.append(rmse((m-y[\'target_motion\'])[y[\'outage\']]))\n        rows.append({\'tau_s\':float(tau),\'train_macro_relative_rmse3d_m\':float(np.mean(scores)),\'episodes\':len(scores)})\n    # Fixed ascending candidate list: lower tau wins exact ties.\n    best=min(rows,key=lambda r:(r[\'train_macro_relative_rmse3d_m\'],r[\'tau_s\']))\n    return best[\'tau_s\'],rows\n', 'src/insane_adaptive/model.py': '"""Unchanged pilot encoder + one sigmoid scalar motion gate (33 parameters)."""\nimport torch\nfrom torch import nn\nfrom insane.model import FusionNavINSANE\n\nclass AdaptiveFusionNav(FusionNavINSANE):\n    def __init__(self,variant=\'adaptive_full\',hidden=32):\n        if variant not in [\'adaptive_full\',\'adaptive_gps_only\']:raise ValueError(variant)\n        super().__init__(variant.removeprefix(\'adaptive_\'),hidden)\n        self.gate_head=nn.Linear(hidden,1)\n        nn.init.zeros_(self.gate_head.weight);nn.init.zeros_(self.gate_head.bias)\n    def forward(self,imu,gnss,common,state=None):\n        if self.variant==\'gps_only\':imu=torch.zeros_like(imu)\n        features=torch.cat([self.imu_branch(imu),self.gnss_branch(gnss),self.common_branch(common)],dim=-1)\n        out,state=self.gru(features,state)\n        return self.velocity_head(out),torch.sigmoid(self.gate_head(out)),state\n\ndef integrate_gated(dv,gate,prior,dt,availability):\n    """Prefix integral, with legal-GNSS anchors only. Zero gate stops, never rewinds.\n\n    This cumsum is algebraically the sequential actual-dt recurrence. At the first\n    unavailable token, the unchanged adapter supplies lag from last legal fix.\n    At GNSS return, the decoder resumes legal-GNSS position; GRU state persists.\n    """\n    velocity=gate*(prior+dv)\n    total=torch.cumsum(velocity*dt[...,None],dim=1)\n    index=torch.arange(total.shape[1],device=total.device)[None].expand(total.shape[0],-1)\n    last=torch.cummax(torch.where(availability,index,torch.zeros_like(index)),dim=1).values\n    anchor=torch.gather(total,1,last[...,None].expand(-1,-1,3))\n    return torch.where(availability[...,None],torch.zeros_like(total),total-anchor),velocity\n', 'src/insane_adaptive/report.py': '"""Descriptive report from every saved episode/seed; no scientific claim selection."""\nfrom pathlib import Path\nimport json\nimport numpy as np\nimport pandas as pd\n\nPRIMARY=\'relative_motion_rmse3d_m\'\n\ndef table(df):\n    def fmt(x):\n        if pd.isna(x):return \'—\'\n        if isinstance(x,(float,np.floating)):return f\'{x:.6f}\'\n        return str(x).replace(\'|\',\'/\')\n    return \'| \'+\' | \'.join(df.columns)+\' |\\n|\'+ \'|\'.join([\'---\']*len(df.columns))+\'|\\n\'+\'\'.join(\'| \'+\' | \'.join(fmt(v) for v in row)+\' |\\n\' for row in df.itertuples(index=False,name=None))\n\ndef aggregate(df,keys):\n    valid=df[df.duration_s>0];model=valid[valid.method.str.startswith(\'adaptive_\')]\n    # First macro-average episodes within seed, then summarize variation of seed means.\n    seed_means=model.groupby(keys+[\'method\',\'seed\'])[PRIMARY].mean().reset_index()\n    neural=seed_means.groupby(keys+[\'method\'])[PRIMARY].agg(mean_m=\'mean\',seed_sd_m=\'std\',n_seeds=\'count\').reset_index()\n    baseline=valid[~valid.method.str.startswith(\'adaptive_\')].groupby(keys+[\'method\'])[PRIMARY].mean().reset_index(name=\'mean_m\')\n    baseline[\'seed_sd_m\']=np.nan;baseline[\'n_seeds\']=0\n    return pd.concat([neural,baseline],ignore_index=True).sort_values(keys+[\'method\']),seed_means\n\ndef generate(out):\n    out=Path(out);d=pd.read_csv(out/\'summary.csv\');pairs=pd.read_csv(out/\'paired_differences.csv\');hist=pd.read_csv(out/\'training_history.csv\')\n    status=json.loads((out/\'status.json\').read_text());cfg=json.loads((out/\'config.json\').read_text());damp=json.loads((out/\'damping_lock.json\').read_text())\n    env=json.loads((out/\'environment.json\').read_text());gates=pd.read_csv(out/\'gate_summary.csv\')\n    by_duration,seeds=aggregate(d,[\'duration_s\']);by_flight,flight_seeds=aggregate(d,[\'flight\',\'duration_s\'])\n    by_duration.to_csv(out/\'duration_summary.csv\',index=False);by_flight.to_csv(out/\'flight_duration_summary.csv\',index=False)\n    seeds.to_csv(out/\'seed_duration_summary.csv\',index=False);flight_seeds.to_csv(out/\'seed_flight_duration_summary.csv\',index=False)\n    imu=pairs[(pairs.method==\'adaptive_full\')&(pairs.reference==\'adaptive_gps_only\')]\n    imu_ep=imu.groupby([\'flight\',\'scenario\',\'duration_s\']).agg(mean_delta_m=(\'difference_m\',\'mean\'),seed_sd_delta_m=(\'difference_m\',\'std\'),wins_of_3=(\'better\',\'sum\')).reset_index()\n    # Percentage aggregates are ratio of macro means; do not average episode percentages.\n    comparison=[]\n    for (flight,duration,method,ref),q in pairs.groupby([\'flight\',\'duration_s\',\'method\',\'reference\'],sort=False):\n        a=float(q.rmse_m.mean());b=float(q.reference_rmse_m.mean())\n        comparison.append({\'flight\':flight,\'duration_s\':duration,\'method\':method,\'reference\':ref,\'mean_rmse_m\':a,\'reference_mean_rmse_m\':b,\n          \'delta_m\':a-b,\'delta_percent\':100*(a-b)/b if b else None,\'wins_episode_seed\':int(q.better.sum()),\'episode_seed_pairs\':len(q)})\n    comparisons=pd.DataFrame(comparison);comparisons.to_csv(out/\'flight_comparisons.csv\',index=False)\n    imu_ep.to_csv(out/\'imu_episode_seed_summary.csv\',index=False)\n    title=\'# INSANE adaptive experiment\\n\\n\'\n    text=title+f"Status **{status[\'status\']}**. Six scheduled trainings, seeds 0/1/2; no test use. Kernel `shapok/drone-nav-insane-adaptive`; exact version/output ID in launch receipt. Code SHA256 `{status[\'code_version\']}`.\\n\\n"\n    text+=\'## Two separate questions\\n\\nA. Does adaptive prediction improve on Held, CV and train-selected Damped-CV? All flight/duration comparisons below use every fixed episode and all three seeds. Lower is better; improvements must be judged separately on each flight.\\n\\n\'\n    text+=table(comparisons[comparisons.reference!=\'adaptive_gps_only\'])\n    text+=\'\\nB. Does IMU improve matched gps_only?\\n\\n\'+table(comparisons[comparisons.reference==\'adaptive_gps_only\'])\n    counts=imu.groupby(\'seed\').better.agg([\'sum\',\'count\']).reset_index();counts.columns=[\'seed\',\'full_wins\',\'outage_episodes\']\n    text+=\'\\n\'+table(counts)+\'\\n\'+table(imu_ep)\n    text+=f"\\nAdaptive full has lower error in {int(imu.better.sum())}/{len(imu)} episode–seed pairs. All-three-seed wins occur in {int((imu_ep.wins_of_3==3).sum())}/9 fixed episodes. Three seeds are repeat trainings on the same two validation flights, not independent flights or evidence of generalization. No practical-effect threshold or significance claim is invented after results.\\n"\n    text+=\'\\n## Every episode and seed, before aggregate tables\\n\\nDeterministic baselines appear once per episode; their seed is blank. Controls are retained, primary outage metric is undefined for them.\\n\\n\'\n    text+=table(d[[\'flight\',\'scenario\',\'method\',\'seed\',\'duration_s\',\'n_native_gt\',PRIMARY,\'absolute_rmse3d_m\',\'absolute_rmse_h_m\',\'absolute_rmse_v_m\',\'final_unavailable_error_m\',\'recovery_first_5s_rmse3d_m\']])\n    text+=\'\\n## Aggregates\\n\\nFirst average episode RMSE within a seed, then report mean and sample SD (ddof=1) of the three seed means. SD is not a confidence interval. Episode durations are nested, episodes can overlap. Overall duration means weight three episodes equally (two mars_4, one mars_5); flight tables make that imbalance explicit. Baselines have no seed SD.\\n\\n\'+table(by_duration)+\'\\n\'+table(by_flight)\n    text+=\'\\n## Gate behavior\\n\\n![Every gate trace](gate_traces.png)\\n\\nEach panel shows all six models on one fixed outage; traces include permitted history and recovery, shaded region is the outage. Gate observations describe prediction control, not a stop detector or causal explanation. Complete token timestamps, gate, residual velocity, integrated velocity/motion and native predictions are in NPZ.\\n\\n\'+table(gates)\n    text+=\'\\n## Damped-CV selection\\n\\n\'+f"Tau = **{damp[\'tau_s\']:g} s**, selected exclusively on 540 frozen train episodes before any adaptive validation metric. Fixed candidates: {cfg[\'damping\'][\'tau_candidates_s\']}. Criterion: train macro relative-motion RMSE3D, smaller tau on ties. v(age)=v_CV exp(-age/tau), displacement=v_CV tau(1−exp(-age/tau)); age starts at last legal fix. This is a bounded baseline calibration, not a neural hyperparameter search.\\n\\n"+table(pd.read_csv(out/\'damping_train_selection.csv\'))\n    text+=\'\\n## Training and checks\\n\\nThe encoder and all original preprocessing/scenario/scaler hashes are preserved. Both variants have 7548 parameters. For each seed, initial tensor hashes and frozen training episode order match. AdamW lr0.001, weight_decay0.0001, batch4, clip1, Huber delta1m, max30 epochs, patience5, strict decrease in fixed validation macro RMSE, selected best checkpoint. Histories include all attempted epochs; same episode prefix is consumed until each model independently early-stops.\\n\\n\'\n    text+=table(pd.DataFrame([{\'model_seed\':k,**v} for k,v in status[\'models\'].items()]))\n    text+=\'\\nPreflight validates source/calibration hashes, ENU projection, train-only scaler, all 540 train and 11 validation scenarios, exact native timestamps and masks. Mandatory original and adaptive tests cover gate0/gate1, irregular dt, state carry, future input perturbation, hidden GNSS mutation, shared inputs and backward gradients. Logs are in launch artifacts.\\n\'\n    text+=\'\\n## Previous pilot — separate seed0 only\\n\\nThese saved results are descriptive context; they are never pooled with new seeds. No ungated seeds1/2 were trained, so a controlled three-seed estimate of the isolated gate effect is unavailable.\\n\\n\'\n    previous=pd.read_csv(out/\'previous_pilot_seed0.csv\');previous=previous[(previous.duration_s>0)&previous.method.isin([\'full\',\'gps_only\'])]\n    text+=table(previous[[\'flight\',\'scenario\',\'method\',\'seed\',PRIMARY]])\n    text+=\'\\n## Metric / data / reference limitations\\n\\nNative GT timestamps are scored without trajectory fitting. Primary is sqrt(mean(norm((pred−last legal GNSS)−(GT(t)−GT(last legal fix time)))²)) inside outage. Only the anchor is interpolated as a target/reference; no interpolated GT samples are counted as observations. Raw absolute3D/H/V, final unavailable error and recovery-first5s are saved. No GT attitude/position or RTK enters the encoder or initialization. Original timestamp corrections are already applied. GT is authors’ dual-RTK/magnetometer-derived PX4 IMU reference; ordinary GNSS antenna lever arm is missing. Filename 8hz does not guarantee 8 independent points/s. Test mars_6/7 remains absent from the Kaggle data.\\n\\n\'\n    text+=\'ESKF remains **not_ready**, with no invented scores: missing ordinary-GNSS lever arm and validated absolute heading initialization. No superiority over INS/EKF is claimed. Validation selects checkpoints and is reused for this follow-up hypothesis after viewing the prior pilot; this is exploratory validation, not untouched confirmatory testing.\\n\\n\'\n    text+=\'Environment: `\'+json.dumps(env)+\'`. Saved inference_ms is synchronized single forward/readout, not repeated benchmark; Held/CV share their joint timing. No training or full evaluation on Mac.\\n\\n**STOP. No test, additional seeds/jobs, parameter changes or automatic next version.**\\n\'\n    (out/\'INSANE_ADAPTIVE_REPORT.md\').write_text(text)\n    methods=f\'\'\'# Methods draft — completed INSANE adaptive experiment\n\nThis draft describes the executed exploratory protocol; it does not establish architectural novelty or guarantee improved navigation.\n\n## Data and split\n\nThe existing INSANE Sensor Data exports and author-provided calibrations were reused. Inputs are `px4_imu.csv` accelerometer and gyroscope (specific force m/s², angular rate rad/s, published PX4 MAVROS body axes) and ordinary `px4_gps.csv` positions/covariance. GT is `ground_truth_8hz.csv` position at the PX4 IMU reference, derived by the dataset authors from dual RTK and magnetometer geometry. No RTK, magnetometer, onboard pose, GT orientation or GT position enters learned inputs or initial state. Train flights are mars_1–3; validation mars_4–5. Test mars_6–7 was neither uploaded nor evaluated. Outdoor_1 was used only in the preceding loader check.\n\nPublic sources: [dataset](https://www.aau.at/en/smart-systems-technologies/control-of-networked-systems/datasets/insane-dataset/), [authors\' tools, pinned revision](https://github.com/aau-cns/insane_dataset_tools/tree/9a1c8c0fdd195f2d869fff292f2ce5b273c5a03d), [dataset publication](https://arxiv.org/abs/2210.09114). Download manifests, calibration hashes and license were retained. No new dataset was selected for this experiment.\n\n## Causal preparation\n\nThe unchanged pilot adapter aggregates native approximately 200Hz IMU to 20Hz by time-weighted integration of past-held measurements, not row dropping. Published timestamp corrections are not applied twice. WGS84 GPS positions are checked against documented ENU coordinates, with one common first-ordinary-GNSS origin per flight, no GT-fitted alignment. The unknown ordinary-GNSS-to-IMU lever arm is not guessed. No GT rotation or additional intrinsic correction is applied to IMU.\n\nThe final outage mask is applied to native GNSS before position deltas, CV velocity, covariance, availability and fix age are computed. CV uses the most recent permitted 5s GNSS history. No hidden GNSS fix enters derived inputs. IMU scalers are fitted on train flights only. The exact previous episode manifests, native GT row support and mask hashes are preserved. All 540 training episodes (18/epoch for 30 epochs) are shared in the same order across seeds/variants; only initial weights vary across seeds. Validation has nine fixed outages (three each of 10/30/60s), plus two GNSS controls, with 20s history and 10s recovery; two onset positions are on mars_4 and one on mars_5. Flights are never concatenated.\n\n## Pilot architecture and integration\n\nIMU6→Linear16+tanh; GNSS displacement/velocity6→Linear16+tanh; shared quality/time10→Linear8+tanh; concatenated40→causal GRU32. The retained residual head maps hidden32→velocity3. The sole architectural addition maps hidden32→scalar sigmoid gate g∈[0,1]. Residual head starts at zero and gate head at zero weight/bias (g=0.5). Each variant has 7548 parameters. gps_only zeros only the six normalized IMU sensor values; availability, fix age, actual dt, IMU timing/count and GNSS quality remain shared. Paired variants begin with identical tensor hashes for each seed.\n\nVelocity is g(t)[v_CV+residual(t)]. Position integrates this velocity by the actual per-step time interval; the first unavailable token includes lag since the last legal fix. Zero gate freezes accumulated displacement rather than resetting it; gate1 reduces to the original decoder. The state is carried through outages. On legal GNSS return the position decoder resumes that permitted fix while GRU state persists. Independent episodes start from zero hidden state with permitted history; no true-state resets occur. At a native GT timestamp, prediction uses the latest past token and its velocity over the remaining partial interval. The gate uses only the causal hidden state, never GT at inference. It is prediction attenuation, not an established drone stop detector.\n\n## Training and checkpoint choice\n\nSix from-scratch trainings were scheduled in one Kaggle job: adaptive_full and adaptive_gps_only for seeds0,1,2. AdamW lr=0.001, weight_decay=0.0001, batch4, gradient clip1, Huber delta1m on native outage relative-motion targets; max30 epochs, patience5. Batching padding contributes no reference measurements or loss terms. The target is GT(t)−GT(last legal fix time); the single anchor may be interpolated for the reference only. No checkpoint from Zurich or the previous INSANE pilot initializes these models.\n\nThe predefined selection criterion is the macro mean of the nine validation episode relative-motion RMSE3D values, strict decrease (min_delta=0). Each model retains its own best checkpoint; paired models see the same training episode prefix until independent early stopping. Actual selected/last epochs are:\n\n\'\'\'+table(pd.DataFrame([{\'model_seed\':k,**v} for k,v in status[\'models\'].items()]))+f\'\'\'\n## Baselines and evaluation\n\nHeld-GNSS retains the last legal fix. CV extrapolates its permitted-history velocity. Damped-CV uses v(age)=v_CV exp(−age/tau), integrated analytically; tau={damp[\'tau_s\']:g}s was selected only on all 540 frozen train episodes from {cfg[\'damping\'][\'tau_candidates_s\']}, minimizing train macro relative-motion RMSE3D before any validation scoring. No adaptive model hyperparameter sweep was performed. ESKF remains not_ready and was not numerically evaluated.\n\nPrimary metric: sqrt(mean(||(prediction−last legal GNSS position)−(GT(t)−GT(last legal fix time))||²)) on original GT rows strictly within each outage. Additional metrics are unaligned absolute3D, horizontal and vertical RMSE, final unavailable error, first5s recovery RMSE and error after actual fix return. All methods use identical native timestamps and masks. Gate traces and inference timing are saved; gates are descriptive, not causal explanations.\n\nReports expose each episode, flight and seed before aggregation. Episode RMSEs are averaged within seed; then mean and sample SD across three seed means are reported. SD is not a confidence interval; seeds are not independent flights. Overall means weight the three episodes equally; per-flight tables are also reported. Baselines are deterministic and not duplicated as independent seed observations. The prior pilot is reported separately as seed0 only; no estimate of an isolated gate effect across three ungated seeds is available.\n\n## Reproducibility and limits\n\nRun status: {status[\'status\']}; code SHA256 `{status[\'code_version\']}`. Environment: `{json.dumps(env)}`. Configuration, source snapshot/hashes, original manifests, scalers, train damping-selection log, initial weight hashes, histories, best/last checkpoints, prediction/reference/mask/gate arrays and all comparisons are saved. Mandatory tests precede training. Actual run identifiers are recorded in the submission/download receipts.\n\nThe two validation flights are a small correlated sample, outage durations overlap, and checkpoints are selected on these same episodes. The adaptive hypothesis follows inspection of the previous validation pilot. Thus results are exploratory and do not establish generalization, significance or physical interpretation of g. Reference postprocessing and absent ordinary-GNSS lever arm limit interpretation. No test results, INS/EKF superiority or proven novelty are claimed. After the six trainings, execution stops.\n\'\'\'\n    (out/\'METHODS_DRAFT.md\').write_text(methods)\n    plot_gates(out)\n\ndef plot_gates(out):\n    import matplotlib\n    matplotlib.use(\'Agg\')\n    import matplotlib.pyplot as plt\n    specs=[s for s in json.loads((out/\'outages.json\').read_text())[\'scenarios\'] if s[\'duration_s\']]\n    fig,axes=plt.subplots(3,3,figsize=(15,10),sharey=True)\n    for ax,s in zip(axes.flat,specs):\n        for method,color in [(\'adaptive_full\',\'tab:blue\'),(\'adaptive_gps_only\',\'tab:orange\')]:\n            for seed,style in zip([0,1,2],[\'-\',\'--\',\':\']):\n                with np.load(out/\'predictions\'/s[\'id\']/f\'{method}_seed{seed}.npz\',allow_pickle=False) as z:\n                    ax.plot(z[\'all_token_t\']-s[\'onset_s\'],z[\'g_token\'],color=color,ls=style,lw=1,label=f\'{method}, seed{seed}\')\n        ax.axvspan(0,s[\'duration_s\'],color=\'grey\',alpha=.12);ax.set_title(s[\'id\']);ax.set_ylim(0,1);ax.set_xlabel(\'Time from outage onset (s)\');ax.set_ylabel(\'g(t)\')\n    handles,labels=axes.flat[0].get_legend_handles_labels();fig.legend(handles,labels,loc=\'lower center\',ncol=3,fontsize=9)\n    fig.suptitle(\'Learned motion attenuation: descriptive gate traces, not stop detection\')\n    fig.tight_layout(rect=[0,.06,1,.96]);fig.savefig(out/\'gate_traces.png\',dpi=160);plt.close(fig)\n', 'src/insane_adaptive/run.py': '"""Exactly six trainings in one job; no retries, test access, or auto next run."""\nfrom pathlib import Path\nimport json,hashlib,shutil,platform,time,datetime,traceback\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch.nn.utils.rnn import pad_sequence\nfrom insane.run import preflight as original_preflight,ten,write\nfrom insane.data import sha\nfrom insane.model import native_motion\nfrom insane.metrics import score,baselines\nfrom .model import AdaptiveFusionNav,integrate_gated\nfrom .baselines import damped_cv,choose_tau\n\nBASES=[\'held_gnss\',\'constant_velocity_gnss\',\'damped_cv\']\n\ndef code_version(root):\n    root=Path(root);hashes={}\n    for base in [\'src/insane\',\'src/insane_adaptive\',\'tests/insane\',\'tests/insane_adaptive\',\'configs/insane\',\'configs/insane_adaptive\']:\n        for p in sorted((root/base).glob(\'*\')):\n            if p.is_file() and p.suffix in [\'.py\',\'.json\',\'.csv\']:hashes[str(p.relative_to(root))]=sha(p)\n    return hashlib.sha256(json.dumps(hashes,sort_keys=True).encode()).hexdigest(),hashes\n\ndef preflight(raw,root):\n    root=Path(root);cfg=json.loads((root/\'configs/insane_adaptive/adaptive.json\').read_text())\n    lock=json.loads((root/\'configs/insane_adaptive/protocol_lock.json\').read_text())\n    for name,h in lock[\'sha256\'].items():\n        if sha(root/name)!=h:raise ValueError(\'Original protocol modified: \'+name)\n    if sha(root/\'configs/insane_adaptive/adaptive.json\')!=lock[\'adaptive_config_sha256\']:raise ValueError(\'Adaptive configuration changed\')\n    assert cfg[\'seeds\']==[0,1,2] and cfg[\'methods\']==[\'adaptive_full\',\'adaptive_gps_only\']\n    original,split,val,train,scaler,cache,evidence=original_preflight(raw,root/\'configs/insane\')\n    for key in [\'learning_rate\',\'weight_decay\',\'batch_size\',\'max_epochs\',\'patience\',\'min_delta_m\',\'huber_delta_m\',\'grad_clip\',\'hidden\',\'validation_criterion\']:\n        assert cfg[key]==original[key],key\n    assert all(not (Path(raw)/seq).exists() for seq in split[\'test_reserved\']) or str(raw).startswith(\'/Users/\')\n    # Local preparation can point to the original folder; no test file is read.\n    evidence.update(adaptive_protocol_lock_pass=True,planned_trainings=6,gate_initial_value=.5)\n    return cfg,split,val,train,scaler,cache,evidence\n\ndef state_hash(model):\n    h=hashlib.sha256()\n    for key,value in model.state_dict().items():h.update(key.encode());h.update(value.detach().cpu().numpy().tobytes())\n    return h.hexdigest()\n\ndef evaluate(model,x,y,device):\n    model.eval()\n    if device==\'cuda\':torch.cuda.synchronize()\n    start=time.perf_counter()\n    with torch.inference_mode():\n        dv,g,_=model(*(ten(x[k],device)[None] for k in [\'imu\',\'gnss\',\'common\']))\n        m,v=integrate_gated(dv,g,ten(x[\'velocity_prior\'],device)[None],ten(x[\'integration_dt\'],device)[None],torch.as_tensor(x[\'availability\'],device=device)[None])\n        native=native_motion(m[0],v[0],x,y).cpu().numpy().astype(float)\n        detail={\'g_token\':g[0,:,0].cpu().numpy(),\'residual_velocity_token\':dv[0].cpu().numpy(),\n                \'motion_token\':m[0].cpu().numpy(),\'velocity_token\':v[0].cpu().numpy()}\n    if device==\'cuda\':torch.cuda.synchronize()\n    return x[\'held\'][y[\'token_index\']]+native,native,(time.perf_counter()-start)*1000,detail\n\ndef batch_loss(model,selected,device,cfg):\n    def stack(key):return pad_sequence([ten(x[key],device) for x,y in selected],batch_first=True)\n    av=pad_sequence([torch.as_tensor(x[\'availability\'],device=device) for x,y in selected],batch_first=True,padding_value=True)\n    dv,g,_=model(stack(\'imu\'),stack(\'gnss\'),stack(\'common\'))\n    m,v=integrate_gated(dv,g,stack(\'velocity_prior\'),stack(\'integration_dt\'),av)\n    losses=[]\n    for k,(x,y) in enumerate(selected):\n        chosen=torch.as_tensor(y[\'outage\'],device=device);p=native_motion(m[k],v[k],x,y)\n        losses.append(torch.nn.functional.huber_loss(p[chosen],ten(y[\'target_motion\'],device)[chosen],delta=cfg[\'huber_delta_m\']))\n    return torch.stack(losses).mean()\n\ndef paired_rows(rows):\n    df=pd.DataFrame(rows);result=[]\n    for scenario,group in df[df.duration_s>0].groupby(\'scenario\',sort=False):\n        base=group[group.method.isin(BASES)].set_index(\'method\')\n        for seed in [0,1,2]:\n            pair=group[group.seed==seed].set_index(\'method\')\n            for method in [\'adaptive_full\',\'adaptive_gps_only\']:\n                own=pair.loc[method]\n                others=BASES+([\'adaptive_gps_only\'] if method==\'adaptive_full\' else [])\n                for other in others:\n                    ref=pair.loc[other] if other.startswith(\'adaptive_\') else base.loc[other]\n                    a=float(own.relative_motion_rmse3d_m);b=float(ref.relative_motion_rmse3d_m)\n                    result.append({\'scenario\':scenario,\'flight\':own.flight,\'duration_s\':int(own.duration_s),\'seed\':seed,\'method\':method,\'reference\':other,\n                       \'rmse_m\':a,\'reference_rmse_m\':b,\'difference_m\':a-b,\'difference_percent\':100*(a-b)/b if b else None,\'better\':a<b})\n    return result\n\ndef save_prediction(folder,method,seed,x,y,p,m,detail):\n    idx=y[\'token_index\'];name=method if seed is None else f\'{method}_seed{seed}\'\n    np.savez_compressed(folder/(name+\'.npz\'),t=y[\'t\'],gt=y[\'gt\'],pred=p,motion=m,target_motion=y[\'target_motion\'],anchor_gt=y[\'anchor_gt\'],\n      gt_native_index=y[\'gt_native_index\'],outage=y[\'outage\'],availability=x[\'availability\'][idx],source_t=x[\'source_t\'][idx],source_index=x[\'source_index\'][idx],\n      held=x[\'held\'][idx],token_t=x[\'t\'][idx],imu_source_t=x[\'imu_source_t\'][idx],\n      all_token_t=x[\'t\'],all_availability=x[\'availability\'],integration_dt=x[\'integration_dt\'],velocity_prior=x[\'velocity_prior\'],token_index=idx,**detail)\n\ndef execute(raw,root,out,run_training=False):\n    if not run_training:raise RuntimeError(\'Training gate false\')\n    if not torch.cuda.is_available():raise RuntimeError(\'All training/evaluation is Kaggle GPU only\')\n    root=Path(root);out=Path(out)\n    if out.exists():raise FileExistsError(\'One job only: outputs already exist\')\n    cfg,split,val,train,scaler,cache,checks=preflight(raw,root)\n    version,hashes=code_version(root);out.mkdir(parents=True);(out/\'checkpoints\').mkdir();(out/\'predictions\').mkdir()\n    status={\'status\':\'running\',\'test_used\':False,\'seeds\':[0,1,2],\'planned_trainings\':6,\'models\':{},\'eskf_status\':\'not_ready\',\n      \'code_version\':version,\'started_utc\':datetime.datetime.now(datetime.timezone.utc).isoformat(),\n      \'run_identity\':{\'kernel\':\'shapok/drone-nav-insane-adaptive\',\'version\':\'see submission receipt\'}}\n    write(out/\'status.json\',status)\n    try:\n        for rel in hashes:\n            dst=out/\'source_snapshot\'/rel;dst.parent.mkdir(parents=True,exist_ok=True);shutil.copyfile(root/rel,dst)\n        write(out/\'source_hashes.json\',hashes);write(out/\'config.json\',cfg);write(out/\'scalers.json\',scaler);write(out/\'preflight.json\',checks)\n        for name in [\'splits\',\'outages\',\'training_episodes\',\'download_manifest\',\'inventory\',\'protocol_lock\']:\n            shutil.copyfile(root/\'configs/insane\'/f\'{name}.json\',out/f\'{name}.json\')\n        shutil.copyfile(root/\'configs/insane_adaptive/protocol_lock.json\',out/\'adaptive_protocol_lock.json\')\n        shutil.copyfile(root/\'configs/insane_adaptive/previous_pilot_summary.csv\',out/\'previous_pilot_seed0.csv\')\n        write(out/\'eskf_status.json\',{\'status\':\'not_ready\',\'blockers\':cfg[\'eskf\'][\'blockers\'],\'evaluated\':False})\n        torch.backends.cuda.matmul.allow_tf32=False;torch.backends.cudnn.allow_tf32=False\n        torch.backends.cudnn.benchmark=False;torch.backends.cudnn.deterministic=True\n        write(out/\'environment.json\',{\'python\':platform.python_version(),\'torch\':torch.__version__,\'numpy\':np.__version__,\'pandas\':pd.__version__,\n              \'device\':\'cuda\',\'cuda\':torch.version.cuda,\'hardware\':torch.cuda.get_device_name(0)})\n        # Only train targets are read here; absolutely no validation metric precedes this lock.\n        train_cache={s[\'id\']:cache[s[\'id\']] for s in train}\n        tau,search=choose_tau(train,train_cache,cfg[\'damping\'][\'tau_candidates_s\'],split[\'train\'])\n        pd.DataFrame(search).to_csv(out/\'damping_train_selection.csv\',index=False)\n        write(out/\'damping_lock.json\',{\'tau_s\':tau,\'selected_on\':\'train only\',\'flights\':split[\'train\'],\'episodes\':len(train),\n           \'validation_metrics_computed_before_lock\':False,\'selected_utc\':datetime.datetime.now(datetime.timezone.utc).isoformat(),\n           \'rule\':cfg[\'damping\'],\'episode_ids_sha256\':hashlib.sha256(json.dumps([s[\'id\'] for s in train]).encode()).hexdigest()})\n        print(\'DAMPING LOCKED ON TRAIN:\',tau,flush=True)\n        models={};history=[];counts={};initial={}\n        for seed in cfg[\'seeds\']:\n            for method in cfg[\'methods\']:\n                key=f\'{method}_seed{seed}\';torch.manual_seed(seed);np.random.seed(seed);torch.cuda.manual_seed_all(seed)\n                model=AdaptiveFusionNav(method,cfg[\'hidden\']).to(\'cuda\');counts[key]=sum(p.numel() for p in model.parameters());initial[key]=state_hash(model)\n                if method==\'adaptive_gps_only\':assert initial[key]==initial[f\'adaptive_full_seed{seed}\']\n                write(out/\'initial_weight_hashes.json\',initial)\n                optimizer=torch.optim.AdamW(model.parameters(),lr=cfg[\'learning_rate\'],weight_decay=cfg[\'weight_decay\'])\n                best=float(\'inf\');best_epoch=0;bad=0\n                for epoch in range(1,cfg[\'max_epochs\']+1):\n                    start=time.perf_counter();model.train();losses=[];episodes=[s for s in train if s[\'epoch\']==epoch]\n                    for k in range(0,len(episodes),cfg[\'batch_size\']):\n                        loss=batch_loss(model,[cache[s[\'id\']] for s in episodes[k:k+cfg[\'batch_size\']]],\'cuda\',cfg)\n                        if not torch.isfinite(loss):raise ValueError(\'Nonfinite train loss\')\n                        optimizer.zero_grad();loss.backward();torch.nn.utils.clip_grad_norm_(model.parameters(),cfg[\'grad_clip\']);optimizer.step();losses.append(float(loss.detach().cpu()))\n                    values=[]\n                    for spec in val:\n                        if not spec[\'duration_s\']:continue\n                        x,y=cache[spec[\'id\']];p,m,_,_=evaluate(model,x,y,\'cuda\');values.append(score(spec,x,y,p,m,method)[\'relative_motion_rmse3d_m\'])\n                    criterion=float(np.mean(values))\n                    if not np.isfinite(criterion):raise ValueError(\'Nonfinite validation criterion\')\n                    improved=criterion<best-cfg[\'min_delta_m\']\n                    if improved:\n                        best=criterion;best_epoch=epoch;bad=0;torch.save(model.state_dict(),out/\'checkpoints\'/f\'{key}_best.pt\')\n                    else:bad+=1\n                    history.append({\'method\':method,\'seed\':seed,\'epoch\':epoch,\'mean_batch_huber_train_loss\':float(np.mean(losses)),\n                        \'validation_macro_relative_rmse3d_m\':criterion,\'selected_best_so_far\':improved,\'epochs_without_improvement\':bad,\n                        \'epoch_wall_seconds\':time.perf_counter()-start,\'episode_order_sha256\':hashlib.sha256(json.dumps([s[\'id\'] for s in episodes]).encode()).hexdigest()})\n                    pd.DataFrame(history).to_csv(out/\'training_history.csv\',index=False)\n                    status[\'models\'][key]={\'last_epoch\':epoch,\'best_epoch\':best_epoch,\'best_validation_criterion_m\':best};write(out/\'status.json\',status)\n                    print(json.dumps(history[-1]),flush=True)\n                    if bad>=cfg[\'patience\']:break\n                torch.save(model.state_dict(),out/\'checkpoints\'/f\'{key}_last.pt\')\n                model.load_state_dict(torch.load(out/\'checkpoints\'/f\'{key}_best.pt\',map_location=\'cuda\',weights_only=True));models[(method,seed)]=model.eval()\n        assert set(counts.values())=={7548};write(out/\'parameter_counts.json\',counts)\n        rows=[];gates=[]\n        for spec in val:\n            x,y=cache[spec[\'id\']];folder=out/\'predictions\'/spec[\'id\'];folder.mkdir()\n            start=time.perf_counter();base=baselines(x,y);base_ms=(time.perf_counter()-start)*1000\n            start=time.perf_counter();dp,dm=damped_cv(x,y,tau);dms=(time.perf_counter()-start)*1000\n            predictions=[(method,None,p,m,base_ms,{}) for method,(p,m) in base.items()]+[(\'damped_cv\',None,dp,dm,dms,{})]\n            for (method,seed),model in models.items():\n                p,m,ms,detail=evaluate(model,x,y,\'cuda\');predictions.append((method,seed,p,m,ms,detail))\n            for method,seed,p,m,ms,detail in predictions:\n                if not np.isfinite(p).all():raise ValueError(\'Nonfinite predictions\')\n                row=score(spec,x,y,p,m,method,ms);row[\'seed\']=seed;rows.append(row)\n                save_prediction(folder,method,seed,x,y,p,m,detail)\n                if detail and spec[\'duration_s\']:\n                    mask=(x[\'t\']>=spec[\'onset_s\'])&(x[\'t\']<spec[\'onset_s\']+spec[\'duration_s\']);g=detail[\'g_token\'][mask]\n                    gates.append({\'scenario\':spec[\'id\'],\'flight\':spec[\'flight\'],\'duration_s\':spec[\'duration_s\'],\'method\':method,\'seed\':seed,\n                      \'g_mean\':float(g.mean()),\'g_min\':float(g.min()),\'g_max\':float(g.max()),\'g_first\':float(g[0]),\'g_last\':float(g[-1])})\n        pd.DataFrame(rows).to_csv(out/\'summary.csv\',index=False);pd.DataFrame(paired_rows(rows)).to_csv(out/\'paired_differences.csv\',index=False)\n        pd.DataFrame(gates).to_csv(out/\'gate_summary.csv\',index=False)\n        status.update(status=\'complete\',completed_trainings=len(models),evaluation_rows=len(rows),finished_utc=datetime.datetime.now(datetime.timezone.utc).isoformat(),additional_jobs_started=False)\n        write(out/\'status.json\',status)\n        from .report import generate\n        generate(out)\n        print(\'ADAPTIVE COMPLETE. STOP: no test, no additional seed/job.\',flush=True)\n    except Exception as e:\n        status.update(status=\'failed\',error=str(e));write(out/\'status.json\',status);(out/\'failure_traceback.txt\').write_text(traceback.format_exc());raise\n', 'tests/insane/test_pipeline.py': "import sys,unittest,json,copy\nfrom pathlib import Path\nimport numpy as np\nimport torch\nfrom scipy.spatial.transform import Rotation\nROOT=Path(__file__).resolve().parents[2];sys.path.insert(0,str(ROOT/'src'))\nfrom insane.data import integral_at,aggregate_imu,build_inputs,fit_scaler,make_labels,enu,quality_segments\nfrom insane.model import FusionNavINSANE,integrate_motion,native_motion\nfrom insane.metrics import baselines,score\nfrom insane.eskf import ESKF,G,initialize_from_allowed_history\n\ndef fixture():\n    t=np.arange(0,30.001,.01);gt=np.arange(0,30.01,.2)\n    sensor={'imu_t':t,'imu':np.column_stack([np.sin(t),t*0,np.ones(len(t))*G,np.zeros((len(t),3))]),\n       'gps_t':gt,'gps_p':np.column_stack([gt,gt*2,gt*0]),'gps_cov':np.ones((len(gt),3))}\n    cfg={'gps_gap_limit_s':1.,'imu_gap_limit_s':.1,'gt_gap_limit_s':2.,'cv_history_s':5.,'token_period_s':.05}\n    agg=aggregate_imu(sensor,.05);scaler={'mean':np.zeros(6),'std':np.ones(6)}\n    return sensor,agg,scaler,cfg\n\nclass CausalityTests(unittest.TestCase):\n    def test_weighted_aggregation_irregular(self):\n        t=np.array([0.,.1,.3,.7,1.]);v=np.column_stack([t,t*2])\n        actual=np.diff(integral_at(t,v,np.array([0.,.2,.6,1.])),axis=0)\n        # [0.6,1]: 0.3*0.1 + 0.7*0.3 = 0.24 (left-held samples).\n        np.testing.assert_allclose(actual,[[.01,.02],[.10,.20],[.24,.48]],atol=1e-12)\n    def test_future_imu_no_prefix_effect(self):\n        s,a,sc,c=fixture();s2=copy.deepcopy(s);s2['imu'][s2['imu_t']>15]+=1e6\n        b=aggregate_imu(s2,.05);take=a['t']<=15\n        np.testing.assert_array_equal(a['imu'][take],b['imu'][take])\n    def test_outage_gps_position_and_covariance_do_not_leak(self):\n        s,a,sc,c=fixture();ids=np.arange(len(a['t']));blocks=[[10.,20.]]\n        x=build_inputs(s,a,ids,blocks,sc,c);s2=copy.deepcopy(s);mask=(s2['gps_t']>=10)&(s2['gps_t']<20)\n        s2['gps_p'][mask]+=99999;s2['gps_cov'][mask]+=888\n        y=build_inputs(s2,a,ids,blocks,sc,c)\n        for k in ['gnss','common','held','velocity_prior','source_t','availability']:np.testing.assert_array_equal(x[k],y[k])\n    def test_no_future_gps(self):\n        s,a,sc,c=fixture();x=build_inputs(s,a,np.arange(len(a['t'])),[],sc,c)\n        self.assertTrue(np.all(x['source_t']<=x['t']))\n        s2=copy.deepcopy(s);s2['gps_p'][s['gps_t']>12]+=10000\n        y=build_inputs(s2,a,np.arange(len(a['t'])),[],sc,c)\n        for k in ['gnss','common','held']:np.testing.assert_array_equal(x[k][x['t']<=12],y[k][y['t']<=12])\n    def test_gt_does_not_enter_inputs(self):\n        import inspect\n        self.assertNotIn('reference',inspect.signature(build_inputs).parameters)\n        self.assertNotIn('gt',inspect.signature(build_inputs).parameters)\n    def test_train_only_scaler(self):\n        s,a,sc,c=fixture();b=copy.deepcopy(a);b['imu']+=1e5\n        n1=fit_scaler({'train':a,'val':b},['train']);b['imu']*=99\n        self.assertEqual(n1,fit_scaler({'train':a,'val':b},['train']))\n    def test_shared_features_not_ablated(self):\n        torch.manual_seed(7);full=FusionNavINSANE('full');gps=FusionNavINSANE('gps_only');gps.load_state_dict(full.state_dict())\n        self.assertEqual(sum(p.numel() for p in full.parameters()),sum(p.numel() for p in gps.parameters()))\n        with torch.no_grad():gps.velocity_head.weight.fill_(.1)\n        imu=torch.randn(1,10,6);gnss=torch.randn(1,10,6);common=torch.randn(1,10,10)\n        a,_=gps(imu,gnss,common);b,_=gps(imu*999,gnss,common);torch.testing.assert_close(a,b)\n        d,_=gps(imu,gnss,common+1);self.assertGreater(float((d-a).abs().max().detach()),1e-5)\n    def test_recurrence_prefix_and_state_carry(self):\n        torch.manual_seed(8);m=FusionNavINSANE();torch.nn.init.normal_(m.velocity_head.weight)\n        args=[torch.randn(1,31,k) for k in [6,6,10]]\n        whole,_=m(*args);prefix,state=m(*(v[:,:13] for v in args));rest,_=m(*(v[:,13:] for v in args),state=state)\n        torch.testing.assert_close(whole,torch.cat([prefix,rest],1),atol=2e-6,rtol=1e-5)\n        alt=[v.clone() for v in args]\n        for v in alt:v[:,14:]+=100\n        perturbed,_=m(*alt);torch.testing.assert_close(whole[:,:14],perturbed[:,:14])\n    def test_zero_head_matches_cv_native_gt(self):\n        s,a,sc,c=fixture();x=build_inputs(s,a,np.arange(len(a['t'])),[[10,20]],sc,c)\n        rt=np.arange(.13,29.9,.125);ref={'t':rt,'p':np.column_stack([rt,rt*2,rt*0])}\n        y=make_labels(ref,x,{'duration_s':10},c)\n        dv=torch.zeros((1,len(x['t']),3));prior=torch.tensor(x['velocity_prior'])[None]\n        motion,velocity=integrate_motion(dv,prior,torch.tensor(x['integration_dt'])[None],torch.tensor(x['availability'])[None])\n        pm=native_motion(motion[0],velocity[0],x,y).numpy();expected=baselines(x,y)['constant_velocity_gnss'][1]\n        np.testing.assert_allclose(pm,expected,atol=1e-4)\n        self.assertEqual(len(y['t']),len(rt[(rt>=x['t'][0])&(rt<=x['t'][-1])]))\n        self.assertTrue(np.all(x['t'][y['token_index']]<=y['t']))\n    def test_no_padding_counted_as_reference(self):\n        s,a,sc,c=fixture();x=build_inputs(s,a,np.arange(100,300),[[8,10]],sc,c)\n        rt=np.arange(0,31,.7);ref={'t':rt,'p':np.column_stack([rt]*3)};y=make_labels(ref,x,{},c)\n        self.assertTrue(np.all(y['t']<=x['t'][-1]));self.assertTrue(np.isin(y['t'],rt).all())\n    def test_projection_anchor(self):\n        ref=np.array([30.5999294,34.867308,526.594]);np.testing.assert_allclose(enu(ref[None],ref),0,atol=1e-9)\n        p=enu(np.array([ref+[0,.00001,0],ref+[.00001,0,0],ref+[0,0,1]]),ref)\n        self.assertGreater(p[0,0],.8);self.assertGreater(p[1,1],1);self.assertAlmostEqual(p[2,2],1,places=6)\n    def test_tiny_backward_smoke(self):\n        torch.manual_seed(0);m=FusionNavINSANE();args=[torch.randn(2,16,k) for k in [6,6,10]]\n        pred,_=m(*args);loss=(pred-1).square().mean();loss.backward()\n        self.assertTrue(all(torch.isfinite(p.grad).all() for p in m.parameters() if p.grad is not None))\n\nclass ESKFTests(unittest.TestCase):\n    def test_stationary_gravity(self):\n        e=ESKF()\n        for _ in range(100):e.predict([0,0,G],[0,0,0],.01)\n        np.testing.assert_allclose(e.p,0,atol=1e-12);np.testing.assert_allclose(e.v,0,atol=1e-12)\n    def test_constant_velocity_and_variable_dt(self):\n        e=ESKF(v=[2,-1,.3]);ts=np.resize([.01,.025,.005,.02],100)\n        for dt in ts:e.predict([0,0,G],[0,0,0],dt)\n        np.testing.assert_allclose(e.p,np.array([2,-1,.3])*sum(ts),atol=1e-10)\n    def test_turn_mechanization(self):\n        e=ESKF(v=[2,0,0]);dt=.005;w=.5\n        for k in range(400):e.predict([0,2*w,G],[0,0,w],dt)\n        t=400*dt;expected=np.array([2/w*np.sin(w*t),2/w*(1-np.cos(w*t)),0])\n        np.testing.assert_allclose(e.p,expected,atol=2e-5)\n        np.testing.assert_allclose(e.R,Rotation.from_euler('z',w*t).as_matrix(),atol=1e-12)\n    def test_covariance_gnss_return_and_bias_states(self):\n        e=ESKF(p=[5,2,1],ba=[.1,0,0],bg=[0,0,.01])\n        for _ in range(100):e.predict([.1,0,G],[0,0,.01],.01)\n        old=np.linalg.norm(e.p);e.update_gnss([0,0,0],[.1,.1,.2])\n        self.assertLess(np.linalg.norm(e.p),old);self.assertGreater(np.linalg.eigvalsh(e.P).min(),-1e-10)\n        np.testing.assert_allclose(e.R.T@e.R,np.eye(3),atol=1e-12)\n    def test_no_course_or_gt_initialization(self):\n        with self.assertRaisesRegex(RuntimeError,'no GT/course fallback'):\n            initialize_from_allowed_history(np.tile([0,0,G,0,0,0],(20,1)),np.arange(20),np.zeros((20,3)))\n    def test_lever_measurement(self):\n        e=ESKF(lever=[1,0,0]);e.update_gnss([1,0,0],[.1]*3);np.testing.assert_allclose(e.p,0,atol=1e-12)\n\nif __name__=='__main__':unittest.main()\n", 'tests/insane_adaptive/test_gate.py': "import sys,unittest,copy\nfrom pathlib import Path\nimport numpy as np\nimport torch\nROOT=Path(__file__).resolve().parents[2]\nsys.path.insert(0,str(ROOT/'src'));sys.path.insert(0,str(ROOT/'tests/insane'))\nfrom test_pipeline import fixture\nfrom insane.data import build_inputs,make_labels\nfrom insane.model import integrate_motion,native_motion\nfrom insane.metrics import baselines\nfrom insane_adaptive.model import AdaptiveFusionNav,integrate_gated\nfrom insane_adaptive.baselines import damped_cv,choose_tau\n\nclass GateTests(unittest.TestCase):\n    def test_one_equals_original_actual_dt(self):\n        torch.manual_seed(0);dv=torch.randn(2,11,3,dtype=torch.float64);prior=torch.randn_like(dv)\n        dt=torch.tensor([[.1,.3,.07,.2,.5,.2,.11,.3,.1,.2,.02]]*2)\n        av=torch.tensor([[True,True,False,False,False,True,True,False,False,False,True]]*2)\n        a=integrate_motion(dv,prior,dt,av);b=integrate_gated(dv,torch.ones(2,11,1),prior,dt,av)\n        for u,v in zip(a,b):torch.testing.assert_close(u,v)\n    def test_zero_stops_without_rewind_and_recovery(self):\n        dv=torch.zeros(1,7,3);prior=torch.ones_like(dv)*2\n        dt=torch.tensor([[.1,.2,.3,.4,.1,.5,.2]])\n        av=torch.tensor([[True,False,False,False,False,True,False]])\n        gate=torch.tensor([[[1.],[1.],[1.],[0.],[0.],[1.],[0.]]])\n        m,v=integrate_gated(dv,gate,prior,dt,av)\n        torch.testing.assert_close(m[0,:,0],torch.tensor([0.,.4,1.,1.,1.,0.,0.]))\n        self.assertTrue(torch.equal(v[0,3:5],torch.zeros(2,3)))\n    def test_all_zero_equals_held(self):\n        m,v=integrate_gated(torch.ones(1,9,3),torch.zeros(1,9,1),torch.ones(1,9,3),torch.ones(1,9),torch.tensor([[True]+[False]*8]))\n        self.assertEqual(float(m.abs().max()),0.);self.assertEqual(float(v.abs().max()),0.)\n    def test_gate_bound_init_matching_weights_and_params(self):\n        for seed in [0,1,2]:\n            torch.manual_seed(seed);a=AdaptiveFusionNav('adaptive_full')\n            torch.manual_seed(seed);b=AdaptiveFusionNav('adaptive_gps_only')\n            self.assertEqual(sum(p.numel() for p in a.parameters()),7548)\n            self.assertEqual(sum(p.numel() for p in b.parameters()),7548)\n            for key,v in a.state_dict().items():torch.testing.assert_close(v,b.state_dict()[key],rtol=0,atol=0)\n            _,g,_=a(torch.randn(1,9,6),torch.randn(1,9,6),torch.randn(1,9,10))\n            self.assertTrue(torch.equal(g,torch.ones_like(g)*.5))\n    def test_recurrence_state_and_future_prefix(self):\n        torch.manual_seed(3);m=AdaptiveFusionNav();torch.nn.init.normal_(m.gate_head.weight);torch.nn.init.normal_(m.velocity_head.weight)\n        args=[torch.randn(1,19,k) for k in [6,6,10]]\n        dv,g,_=m(*args);a,ga,s=m(*(v[:,:8] for v in args));b,gb,_=m(*(v[:,8:] for v in args),state=s)\n        torch.testing.assert_close(dv,torch.cat([a,b],1));torch.testing.assert_close(g,torch.cat([ga,gb],1))\n        alt=[v.clone() for v in args]\n        for v in alt:v[:,8:]+=1e4\n        d2,g2,_=m(*alt);torch.testing.assert_close(dv[:,:8],d2[:,:8]);torch.testing.assert_close(g[:,:8],g2[:,:8])\n    def test_shared_inputs_survive_gps_only(self):\n        m=AdaptiveFusionNav('adaptive_gps_only');torch.nn.init.normal_(m.gate_head.weight)\n        args=[torch.randn(1,12,k) for k in [6,6,10]]\n        _,g,_=m(*args);_,same,_=m(args[0]+999,args[1],args[2]);torch.testing.assert_close(g,same)\n        _,different,_=m(args[0],args[1],args[2]+2)\n        self.assertGreater(float((g-different).abs().max().detach()),1e-5)\n    def test_hidden_gnss_cannot_change_gate_or_motion(self):\n        s,a,sc,c=fixture();ids=np.arange(len(a['t']));x=build_inputs(s,a,ids,[[10,20]],sc,c)\n        s2=copy.deepcopy(s);inside=(s['gps_t']>=10)&(s['gps_t']<20)\n        s2['gps_p'][inside]+=1e6;s2['gps_cov'][inside]+=1e6\n        z=build_inputs(s2,a,ids,[[10,20]],sc,c)\n        m=AdaptiveFusionNav();torch.nn.init.normal_(m.gate_head.weight)\n        def pred(q):\n            dv,g,_=m(*(torch.tensor(q[k])[None] for k in ['imu','gnss','common']))\n            motion,_=integrate_gated(dv,g,torch.tensor(q['velocity_prior'])[None],torch.tensor(q['integration_dt'])[None],torch.tensor(q['availability'])[None])\n            return motion,g\n        p,g=pred(x);p2,g2=pred(z);torch.testing.assert_close(p,p2);torch.testing.assert_close(g,g2)\n    def test_native_readout_gate_one_cv_and_zero_hold(self):\n        s,a,sc,c=fixture();x=build_inputs(s,a,np.arange(len(a['t'])),[[10,20]],sc,c)\n        rt=np.arange(.13,29.9,.125);ref={'t':rt,'p':np.column_stack([rt,rt*2,rt*0])};y=make_labels(ref,x,{},c)\n        for gate in [0,1]:\n            m,v=integrate_gated(torch.zeros(1,len(x['t']),3),torch.full((1,len(x['t']),1),float(gate)),torch.tensor(x['velocity_prior'])[None],torch.tensor(x['integration_dt'])[None],torch.tensor(x['availability'])[None])\n            z=native_motion(m[0],v[0],x,y).numpy();expected=baselines(x,y)['constant_velocity_gnss' if gate else 'held_gnss'][1]\n            np.testing.assert_allclose(z,expected,atol=1e-4)\n    def test_damped_analytic_integral_and_train_guard(self):\n        x={'availability':np.array([False,False]),'source_t':np.array([0.,0.]),'velocity_prior':np.ones((2,3))*2,'held':np.zeros((2,3))}\n        y={'token_index':np.array([0,1]),'t':np.array([1.,10.]),'outage':np.ones(2,bool),'target_motion':np.zeros((2,3))}\n        p,_=damped_cv(x,y,5);np.testing.assert_allclose(p[:,0],10*(1-np.exp(-y['t']/5)))\n        with self.assertRaisesRegex(ValueError,'train episodes only'):choose_tau([{'id':'bad','flight':'mars_4'}],{},[1],['mars_1'])\n        tau,rows=choose_tau([{'id':'ok','flight':'mars_1'}],{'ok':(x,y)},[1,10],['mars_1']);self.assertEqual(tau,1)\n    def test_tiny_backward_gate_and_velocity(self):\n        m=AdaptiveFusionNav();dv,g,_=m(torch.randn(1,8,6),torch.randn(1,8,6),torch.randn(1,8,10))\n        motion,_=integrate_gated(dv,g,torch.ones_like(dv),torch.ones(1,8)*.1,torch.tensor([[True]+[False]*7]))\n        (motion-2).square().mean().backward()\n        self.assertGreater(float(m.gate_head.bias.grad.abs().sum()),0)\n        self.assertGreater(float(m.velocity_head.bias.grad.abs().sum()),0)\n        self.assertTrue(all(torch.isfinite(p.grad).all() for p in m.parameters() if p.grad is not None))\n\nif __name__=='__main__':unittest.main()\n", 'configs/insane/download_manifest.json': '{\n  "source": "https://www.aau.at/en/smart-systems-technologies/control-of-networked-systems/datasets/insane-dataset/",\n  "license": "metadata/LICENSE.txt",\n  "calibration": "calibration/insane_sensor_calib_preprocessed/sensor_calibration.yaml",\n  "author_tools_commit": "9a1c8c0fdd195f2d869fff292f2ce5b273c5a03d",\n  "sequences": {\n    "mars_1": {\n      "sequence": "mars_1",\n      "calibration_set": "MA",\n      "url": "https://cns-data.aau.at/insane-dataset/mars_1_sensors.zip",\n      "retrieved_utc": "2026-09-26T21:18:20.314113+00:00",\n      "archive_bytes": 7924101,\n      "archive_etag": "\\"78e985-5eb5e7e29790c\\"",\n      "http_bytes_received": 1234297,\n      "archive_members": [\n        "mars_1_sensors/",\n        "mars_1_sensors/LICENSE",\n        "mars_1_sensors/README.txt",\n        "mars_1_sensors/ground_truth/",\n        "mars_1_sensors/ground_truth/README.txt",\n        "mars_1_sensors/ground_truth/ground_truth_80hz.csv",\n        "mars_1_sensors/ground_truth/ground_truth_8hz.csv",\n        "mars_1_sensors/ground_truth/px4_imu_data.csv",\n        "mars_1_sensors/ground_truth/px4_mag_data_revised.csv",\n        "mars_1_sensors/ground_truth/rtk_gps1_data_revised.csv",\n        "mars_1_sensors/ground_truth/rtk_gps2_data_revised.csv",\n        "mars_1_sensors/lrf_range.csv",\n        "mars_1_sensors/lsm_imu.csv",\n        "mars_1_sensors/lsm_mag.csv",\n        "mars_1_sensors/px4_baro.csv",\n        "mars_1_sensors/px4_gps.csv",\n        "mars_1_sensors/px4_gps_vel_data.csv",\n        "mars_1_sensors/px4_imu.csv",\n        "mars_1_sensors/px4_mag.csv",\n        "mars_1_sensors/rs_imu.csv",\n        "mars_1_sensors/rs_odom.csv",\n        "mars_1_sensors/rtk_gps1.csv",\n        "mars_1_sensors/rtk_gps2.csv",\n        "mars_1_sensors/time_info.yaml"\n      ],\n      "files": [\n        {\n          "path": "mars_1/px4_imu.csv",\n          "bytes": 2775124,\n          "sha256": "3167b9b3fc8ed5ef37d66e7a843adc26712d55a3c5effa891c8249bf8fd523c2",\n          "archive_member": "mars_1_sensors/px4_imu.csv"\n        },\n        {\n          "path": "mars_1/px4_gps.csv",\n          "bytes": 92751,\n          "sha256": "95903a27fb8033b18ab1cbcf886ba7719de392fa28dfd686d162e6ac04e20237",\n          "archive_member": "mars_1_sensors/px4_gps.csv"\n        },\n        {\n          "path": "mars_1/ground_truth_8hz.csv",\n          "bytes": 67338,\n          "sha256": "aed52b0df3aa45503eb4576e2579312014cbc9d5b4522b7b8540f21bf443cdec",\n          "archive_member": "mars_1_sensors/ground_truth/ground_truth_8hz.csv"\n        },\n        {\n          "path": "mars_1/archive_metadata/mars_1_sensors/README.txt",\n          "bytes": 1599,\n          "sha256": "b00c8ffe9fb2a6763c1c2e1e9b97e44e1ba05f238fe9f1c769a7941c99a364d1",\n          "archive_member": "mars_1_sensors/README.txt"\n        },\n        {\n          "path": "mars_1/archive_metadata/mars_1_sensors/ground_truth/README.txt",\n          "bytes": 433,\n          "sha256": "4d4e53c5da1f7aced7a76ad31192ed562bb0db10b73743b0f9df7facfd161beb",\n          "archive_member": "mars_1_sensors/ground_truth/README.txt"\n        },\n        {\n          "path": "mars_1/archive_metadata/mars_1_sensors/time_info.yaml",\n          "bytes": 239,\n          "sha256": "f21c0eb96c1ef50dfb3688db7be625937b3c1c33dd50bab2e41df7f7bab86bae",\n          "archive_member": "mars_1_sensors/time_info.yaml"\n        }\n      ]\n    },\n    "mars_2": {\n      "sequence": "mars_2",\n      "calibration_set": "MA",\n      "url": "https://cns-data.aau.at/insane-dataset/mars_2_sensors.zip",\n      "retrieved_utc": "2026-09-26T21:18:28.676082+00:00",\n      "archive_bytes": 25667151,\n      "archive_etag": "\\"187a64f-5eb5e841fb2ca\\"",\n      "http_bytes_received": 2023527,\n      "archive_members": [\n        "mars_2_sensors/",\n        "mars_2_sensors/LICENSE",\n        "mars_2_sensors/README.txt",\n        "mars_2_sensors/ground_truth/",\n        "mars_2_sensors/ground_truth/README.txt",\n        "mars_2_sensors/ground_truth/ground_truth_80hz.csv",\n        "mars_2_sensors/ground_truth/ground_truth_8hz.csv",\n        "mars_2_sensors/ground_truth/px4_imu_data.csv",\n        "mars_2_sensors/ground_truth/px4_mag_data_revised.csv",\n        "mars_2_sensors/ground_truth/rtk_gps1_data_revised.csv",\n        "mars_2_sensors/ground_truth/rtk_gps2_data_revised.csv",\n        "mars_2_sensors/lrf_range.csv",\n        "mars_2_sensors/lsm_imu.csv",\n        "mars_2_sensors/lsm_mag.csv",\n        "mars_2_sensors/px4_baro.csv",\n        "mars_2_sensors/px4_gps.csv",\n        "mars_2_sensors/px4_gps_vel_data.csv",\n        "mars_2_sensors/px4_imu.csv",\n        "mars_2_sensors/px4_mag.csv",\n        "mars_2_sensors/rs_imu.csv",\n        "mars_2_sensors/rs_odom.csv",\n        "mars_2_sensors/rtk_gps1.csv",\n        "mars_2_sensors/rtk_gps2.csv",\n        "mars_2_sensors/time_info.yaml"\n      ],\n      "files": [\n        {\n          "path": "mars_2/px4_imu.csv",\n          "bytes": 4576685,\n          "sha256": "3da3a5ce72a6d147725d42c71385a3c59d91205c94f3d3ddea16c8928ae91366",\n          "archive_member": "mars_2_sensors/px4_imu.csv"\n        },\n        {\n          "path": "mars_2/px4_gps.csv",\n          "bytes": 149450,\n          "sha256": "88ee9f64d09defec60e45e2c9cf40a2eb94b485fc9ab7398945ca2f408d9b8c8",\n          "archive_member": "mars_2_sensors/px4_gps.csv"\n        },\n        {\n          "path": "mars_2/ground_truth_8hz.csv",\n          "bytes": 108347,\n          "sha256": "e8aaa874b6f4833e0ab2352651a42ed290b6dcdc105ec871fd12740b1b986126",\n          "archive_member": "mars_2_sensors/ground_truth/ground_truth_8hz.csv"\n        },\n        {\n          "path": "mars_2/archive_metadata/mars_2_sensors/README.txt",\n          "bytes": 1620,\n          "sha256": "483e48455d5544c1fc633a2a18b0c360253eb962f319a4f9229353067d301ec7",\n          "archive_member": "mars_2_sensors/README.txt"\n        },\n        {\n          "path": "mars_2/archive_metadata/mars_2_sensors/ground_truth/README.txt",\n          "bytes": 436,\n          "sha256": "7493d7864f02a1b4c0fb39bac8e15bfce558535ace05189846dc362b7d31764c",\n          "archive_member": "mars_2_sensors/ground_truth/README.txt"\n        },\n        {\n          "path": "mars_2/archive_metadata/mars_2_sensors/time_info.yaml",\n          "bytes": 240,\n          "sha256": "c4bf868cfe1c30f7eca78a09dbc076aaada83ddad40ae8a2200900c4739ca671",\n          "archive_member": "mars_2_sensors/time_info.yaml"\n        }\n      ]\n    },\n    "mars_3": {\n      "sequence": "mars_3",\n      "calibration_set": "MA",\n      "url": "https://cns-data.aau.at/insane-dataset/mars_3_sensors.zip",\n      "retrieved_utc": "2026-09-26T21:18:36.844837+00:00",\n      "archive_bytes": 11511010,\n      "archive_etag": "\\"afa4e2-5eb5fa033deff\\"",\n      "http_bytes_received": 1863724,\n      "archive_members": [\n        "mars_3_sensors/",\n        "mars_3_sensors/LICENSE",\n        "mars_3_sensors/README.txt",\n        "mars_3_sensors/ground_truth/",\n        "mars_3_sensors/ground_truth/README.txt",\n        "mars_3_sensors/ground_truth/ground_truth_80hz.csv",\n        "mars_3_sensors/ground_truth/ground_truth_8hz.csv",\n        "mars_3_sensors/ground_truth/outdoor_gt.csv",\n        "mars_3_sensors/ground_truth/outdoor_gt_interpolate.csv",\n        "mars_3_sensors/ground_truth/px4_imu_data.csv",\n        "mars_3_sensors/ground_truth/px4_mag_data_revised.csv",\n        "mars_3_sensors/ground_truth/rtk_gps1_data_revised.csv",\n        "mars_3_sensors/ground_truth/rtk_gps2_data_revised.csv",\n        "mars_3_sensors/ground_truth/time_offset.yaml",\n        "mars_3_sensors/lrf_range.csv",\n        "mars_3_sensors/lsm_imu.csv",\n        "mars_3_sensors/lsm_mag.csv",\n        "mars_3_sensors/px4_baro.csv",\n        "mars_3_sensors/px4_gps.csv",\n        "mars_3_sensors/px4_gps_vel_data.csv",\n        "mars_3_sensors/px4_imu.csv",\n        "mars_3_sensors/px4_mag.csv",\n        "mars_3_sensors/rs_imu.csv",\n        "mars_3_sensors/rs_odom.csv",\n        "mars_3_sensors/rtk_gps1.csv",\n        "mars_3_sensors/rtk_gps2.csv",\n        "mars_3_sensors/time_info.yaml"\n      ],\n      "files": [\n        {\n          "path": "mars_3/px4_imu.csv",\n          "bytes": 4222129,\n          "sha256": "785cea7a614773d444f5c595cacd86954590a237dc902bd9b58ccf2fc2fc3989",\n          "archive_member": "mars_3_sensors/px4_imu.csv"\n        },\n        {\n          "path": "mars_3/px4_gps.csv",\n          "bytes": 131597,\n          "sha256": "f4b87c2f2dbeafb15bac5a599c2f56404da32980336dd9757a503eee9b8b77ac",\n          "archive_member": "mars_3_sensors/px4_gps.csv"\n        },\n        {\n          "path": "mars_3/ground_truth_8hz.csv",\n          "bytes": 91821,\n          "sha256": "7d8b0a399a79a45146c797d1d619881ac13f6b8d5eea21d7a6568b64a3504c8c",\n          "archive_member": "mars_3_sensors/ground_truth/ground_truth_8hz.csv"\n        },\n        {\n          "path": "mars_3/archive_metadata/mars_3_sensors/README.txt",\n          "bytes": 1606,\n          "sha256": "7229dcc17940964c26339ab993707ccb103e0d1a36edda605806f86e084ebe0f",\n          "archive_member": "mars_3_sensors/README.txt"\n        },\n        {\n          "path": "mars_3/archive_metadata/mars_3_sensors/ground_truth/README.txt",\n          "bytes": 436,\n          "sha256": "fad672fb9910da7fe11cc495aa9ff8c02834f68a31beecb70592f8959f7ee051",\n          "archive_member": "mars_3_sensors/ground_truth/README.txt"\n        },\n        {\n          "path": "mars_3/archive_metadata/mars_3_sensors/time_info.yaml",\n          "bytes": 240,\n          "sha256": "63110bf1da5b3fa26f4f359f34ba37d3d4535047019876d134acc1077992bbb3",\n          "archive_member": "mars_3_sensors/time_info.yaml"\n        }\n      ]\n    },\n    "mars_4": {\n      "sequence": "mars_4",\n      "calibration_set": "MA",\n      "url": "https://cns-data.aau.at/insane-dataset/mars_4_sensors.zip",\n      "retrieved_utc": "2026-09-26T21:18:45.881047+00:00",\n      "archive_bytes": 35820714,\n      "archive_etag": "\\"22294aa-5eb5fa9e95030\\"",\n      "http_bytes_received": 2114554,\n      "archive_members": [\n        "mars_4_sensors/",\n        "mars_4_sensors/LICENSE",\n        "mars_4_sensors/README.txt",\n        "mars_4_sensors/ground_truth/",\n        "mars_4_sensors/ground_truth/README.txt",\n        "mars_4_sensors/ground_truth/ground_truth_80hz.csv",\n        "mars_4_sensors/ground_truth/ground_truth_8hz.csv",\n        "mars_4_sensors/ground_truth/px4_imu_data.csv",\n        "mars_4_sensors/ground_truth/px4_mag_data_revised.csv",\n        "mars_4_sensors/ground_truth/rtk_gps1_data_revised.csv",\n        "mars_4_sensors/ground_truth/rtk_gps2_data_revised.csv",\n        "mars_4_sensors/lrf_range.csv",\n        "mars_4_sensors/lrf_range_data.csv",\n        "mars_4_sensors/lsm9ds1_imu_data.csv",\n        "mars_4_sensors/lsm9ds1_mag_data.csv",\n        "mars_4_sensors/lsm_imu.csv",\n        "mars_4_sensors/lsm_mag.csv",\n        "mars_4_sensors/px4_baro.csv",\n        "mars_4_sensors/px4_baro_data.csv",\n        "mars_4_sensors/px4_gps.csv",\n        "mars_4_sensors/px4_gps_data.csv",\n        "mars_4_sensors/px4_gps_vel_data.csv",\n        "mars_4_sensors/px4_imu.csv",\n        "mars_4_sensors/px4_imu_data.csv",\n        "mars_4_sensors/px4_mag.csv",\n        "mars_4_sensors/px4_mag_data.csv",\n        "mars_4_sensors/rs_imu.csv",\n        "mars_4_sensors/rs_imu_data.csv",\n        "mars_4_sensors/rs_odom.csv",\n        "mars_4_sensors/rs_odom_data.csv",\n        "mars_4_sensors/rtk_gps1.csv",\n        "mars_4_sensors/rtk_gps1_data.csv",\n        "mars_4_sensors/rtk_gps2.csv",\n        "mars_4_sensors/rtk_gps2_data.csv",\n        "mars_4_sensors/time_info.yaml"\n      ],\n      "files": [\n        {\n          "path": "mars_4/px4_imu.csv",\n          "bytes": 4771130,\n          "sha256": "fc329f9010d892e54bd1b3dde98826ad84e1d1de24cd86bc576cbffbca93e83c",\n          "archive_member": "mars_4_sensors/px4_imu.csv"\n        },\n        {\n          "path": "mars_4/px4_gps.csv",\n          "bytes": 158195,\n          "sha256": "e59e75d35f47c575188a2bce6e31b06eed8d86fba2d6c3a7e47986efa577abcd",\n          "archive_member": "mars_4_sensors/px4_gps.csv"\n        },\n        {\n          "path": "mars_4/ground_truth_8hz.csv",\n          "bytes": 114363,\n          "sha256": "9f491575c53060344b5391a6d2729c787aeebc5a62abc71eb409235e919a0955",\n          "archive_member": "mars_4_sensors/ground_truth/ground_truth_8hz.csv"\n        },\n        {\n          "path": "mars_4/archive_metadata/mars_4_sensors/README.txt",\n          "bytes": 1620,\n          "sha256": "192f1eab16cbc6ac1485eab154b7f42492d4fcd43a26ba62f983873e375bba0d",\n          "archive_member": "mars_4_sensors/README.txt"\n        },\n        {\n          "path": "mars_4/archive_metadata/mars_4_sensors/ground_truth/README.txt",\n          "bytes": 436,\n          "sha256": "b25125fbb1e9c7262d1d0eb402b7d938edca3dc76f439f735541f6bab5e6bf39",\n          "archive_member": "mars_4_sensors/ground_truth/README.txt"\n        },\n        {\n          "path": "mars_4/archive_metadata/mars_4_sensors/time_info.yaml",\n          "bytes": 240,\n          "sha256": "862c18680b83e8de64a9b79683656f8a9b115073fb3e20daa527bb858f17115b",\n          "archive_member": "mars_4_sensors/time_info.yaml"\n        }\n      ]\n    },\n    "mars_5": {\n      "sequence": "mars_5",\n      "calibration_set": "MA",\n      "url": "https://cns-data.aau.at/insane-dataset/mars_5_sensors.zip",\n      "retrieved_utc": "2026-09-26T21:18:53.964281+00:00",\n      "archive_bytes": 6755008,\n      "archive_etag": "\\"6712c0-5eb5facdfc9a1\\"",\n      "http_bytes_received": 1145249,\n      "archive_members": [\n        "mars_5_sensors/",\n        "mars_5_sensors/LICENSE",\n        "mars_5_sensors/README.txt",\n        "mars_5_sensors/ground_truth/",\n        "mars_5_sensors/ground_truth/README.txt",\n        "mars_5_sensors/ground_truth/ground_truth_80hz.csv",\n        "mars_5_sensors/ground_truth/ground_truth_8hz.csv",\n        "mars_5_sensors/ground_truth/px4_imu_data.csv",\n        "mars_5_sensors/ground_truth/px4_mag_data_revised.csv",\n        "mars_5_sensors/ground_truth/rtk_gps1_data_revised.csv",\n        "mars_5_sensors/ground_truth/rtk_gps2_data_revised.csv",\n        "mars_5_sensors/lrf_range.csv",\n        "mars_5_sensors/lsm_imu.csv",\n        "mars_5_sensors/lsm_mag.csv",\n        "mars_5_sensors/px4_baro.csv",\n        "mars_5_sensors/px4_gps.csv",\n        "mars_5_sensors/px4_gps_vel_data.csv",\n        "mars_5_sensors/px4_imu.csv",\n        "mars_5_sensors/px4_mag.csv",\n        "mars_5_sensors/rs_imu.csv",\n        "mars_5_sensors/rtk_gps1.csv",\n        "mars_5_sensors/rtk_gps2.csv",\n        "mars_5_sensors/time_info.yaml"\n      ],\n      "files": [\n        {\n          "path": "mars_5/px4_imu.csv",\n          "bytes": 2616975,\n          "sha256": "23f3c2ea5cd88027c795d92de67878fccdb09b7da122064f6342bbee72d41e2e",\n          "archive_member": "mars_5_sensors/px4_imu.csv"\n        },\n        {\n          "path": "mars_5/px4_gps.csv",\n          "bytes": 85259,\n          "sha256": "780a543b9286a464e394c1e108bff2cac168d483a66f0ed66350f2107db0b1c8",\n          "archive_member": "mars_5_sensors/px4_gps.csv"\n        },\n        {\n          "path": "mars_5/ground_truth_8hz.csv",\n          "bytes": 56942,\n          "sha256": "ab13987b88c9554eeb14ce32c773cf151f127ba49d2738f1c5bb4b8df4342798",\n          "archive_member": "mars_5_sensors/ground_truth/ground_truth_8hz.csv"\n        },\n        {\n          "path": "mars_5/archive_metadata/mars_5_sensors/README.txt",\n          "bytes": 1500,\n          "sha256": "3b95469bf352f032793c3f00cdad1a6b6a09a2a44f6cf5052d5d4483d1efe7a8",\n          "archive_member": "mars_5_sensors/README.txt"\n        },\n        {\n          "path": "mars_5/archive_metadata/mars_5_sensors/ground_truth/README.txt",\n          "bytes": 433,\n          "sha256": "abccb09a07dd4676e1b742ac9f3b0d0cfdcf7d373d00f00e99695672b32bf1ed",\n          "archive_member": "mars_5_sensors/ground_truth/README.txt"\n        },\n        {\n          "path": "mars_5/archive_metadata/mars_5_sensors/time_info.yaml",\n          "bytes": 239,\n          "sha256": "dfbee02a70e947c096fd8d7c78330e2462558d5001ca67d34a7b16f99108fa89",\n          "archive_member": "mars_5_sensors/time_info.yaml"\n        }\n      ]\n    },\n    "mars_6": {\n      "sequence": "mars_6",\n      "calibration_set": "MA",\n      "url": "https://cns-data.aau.at/insane-dataset/mars_6_sensors.zip",\n      "retrieved_utc": "2026-09-26T21:19:02.502117+00:00",\n      "archive_bytes": 6763964,\n      "archive_etag": "\\"6735bc-5eb5fadc5662d\\"",\n      "http_bytes_received": 1154462,\n      "archive_members": [\n        "mars_6_sensors/",\n        "mars_6_sensors/LICENSE",\n        "mars_6_sensors/README.txt",\n        "mars_6_sensors/ground_truth/",\n        "mars_6_sensors/ground_truth/README.txt",\n        "mars_6_sensors/ground_truth/ground_truth_80hz.csv",\n        "mars_6_sensors/ground_truth/ground_truth_8hz.csv",\n        "mars_6_sensors/ground_truth/px4_imu_data.csv",\n        "mars_6_sensors/ground_truth/px4_mag_data_revised.csv",\n        "mars_6_sensors/ground_truth/rtk_gps1_data_revised.csv",\n        "mars_6_sensors/ground_truth/rtk_gps2_data_revised.csv",\n        "mars_6_sensors/lrf_range.csv",\n        "mars_6_sensors/lsm_imu.csv",\n        "mars_6_sensors/lsm_mag.csv",\n        "mars_6_sensors/px4_baro.csv",\n        "mars_6_sensors/px4_gps.csv",\n        "mars_6_sensors/px4_gps_vel_data.csv",\n        "mars_6_sensors/px4_imu.csv",\n        "mars_6_sensors/px4_mag.csv",\n        "mars_6_sensors/rtk_gps1.csv",\n        "mars_6_sensors/rtk_gps2.csv",\n        "mars_6_sensors/time_info.yaml"\n      ],\n      "files": [\n        {\n          "path": "mars_6/px4_imu.csv",\n          "bytes": 2611811,\n          "sha256": "250df50f10fa9c5312467eff1936102ca3ce0ca40689ab74507c31261948006d",\n          "archive_member": "mars_6_sensors/px4_imu.csv"\n        },\n        {\n          "path": "mars_6/px4_gps.csv",\n          "bytes": 84864,\n          "sha256": "e01ff6a7728958efc43d87909f4654d57f7aa02dbc473df3283d01b7472c00c3",\n          "archive_member": "mars_6_sensors/px4_gps.csv"\n        },\n        {\n          "path": "mars_6/ground_truth_8hz.csv",\n          "bytes": 64130,\n          "sha256": "0a8af9361df1fcaa8f26eeb04ede387adc2f9daf161ee44ddeba6aeeee9e643f",\n          "archive_member": "mars_6_sensors/ground_truth/ground_truth_8hz.csv"\n        },\n        {\n          "path": "mars_6/archive_metadata/mars_6_sensors/README.txt",\n          "bytes": 1393,\n          "sha256": "ecfd4620cc28c41b855023da449666c070f324220cf134c2367e2b004f1b3ffa",\n          "archive_member": "mars_6_sensors/README.txt"\n        },\n        {\n          "path": "mars_6/archive_metadata/mars_6_sensors/ground_truth/README.txt",\n          "bytes": 433,\n          "sha256": "b8a936ab0d1d2bdb4b278fc1442dd4747decf1feebb9de77b8012e61d5509890",\n          "archive_member": "mars_6_sensors/ground_truth/README.txt"\n        },\n        {\n          "path": "mars_6/archive_metadata/mars_6_sensors/time_info.yaml",\n          "bytes": 239,\n          "sha256": "c4ac7c4ee2e2840d20ab6f4df461f125406cba2e3161e65ce264f7a5df26e9dd",\n          "archive_member": "mars_6_sensors/time_info.yaml"\n        }\n      ]\n    },\n    "mars_7": {\n      "sequence": "mars_7",\n      "calibration_set": "MA",\n      "url": "https://cns-data.aau.at/insane-dataset/mars_7_sensors.zip",\n      "retrieved_utc": "2026-09-26T21:19:10.584389+00:00",\n      "archive_bytes": 6710161,\n      "archive_etag": "\\"666391-5eb5fb3012a2f\\"",\n      "http_bytes_received": 1230137,\n      "archive_members": [\n        "mars_7_sensors/",\n        "mars_7_sensors/LICENSE",\n        "mars_7_sensors/README.txt",\n        "mars_7_sensors/ground_truth/",\n        "mars_7_sensors/ground_truth/README.txt",\n        "mars_7_sensors/ground_truth/ground_truth_80hz.csv",\n        "mars_7_sensors/ground_truth/ground_truth_8hz.csv",\n        "mars_7_sensors/ground_truth/px4_imu_data.csv",\n        "mars_7_sensors/ground_truth/px4_mag_data_revised.csv",\n        "mars_7_sensors/ground_truth/rtk_gps1_data_revised.csv",\n        "mars_7_sensors/ground_truth/rtk_gps2_data_revised.csv",\n        "mars_7_sensors/lrf_range.csv",\n        "mars_7_sensors/lsm_imu.csv",\n        "mars_7_sensors/lsm_mag.csv",\n        "mars_7_sensors/px4_baro.csv",\n        "mars_7_sensors/px4_gps.csv",\n        "mars_7_sensors/px4_gps_vel_data.csv",\n        "mars_7_sensors/px4_imu.csv",\n        "mars_7_sensors/px4_mag.csv",\n        "mars_7_sensors/rs_imu.csv",\n        "mars_7_sensors/rtk_gps1.csv",\n        "mars_7_sensors/rtk_gps2.csv",\n        "mars_7_sensors/time_info.yaml"\n      ],\n      "files": [\n        {\n          "path": "mars_7/px4_imu.csv",\n          "bytes": 2768334,\n          "sha256": "fda18c7e539d156bab617c95be1fde051ce54b8ba159f3c2470119773e914d6a",\n          "archive_member": "mars_7_sensors/px4_imu.csv"\n        },\n        {\n          "path": "mars_7/px4_gps.csv",\n          "bytes": 90952,\n          "sha256": "aed66055034757fdae713b1530224d327f3a13410cbd88e9fba700ec1561fc5a",\n          "archive_member": "mars_7_sensors/px4_gps.csv"\n        },\n        {\n          "path": "mars_7/ground_truth_8hz.csv",\n          "bytes": 64394,\n          "sha256": "919b13d506f3ee1d9640155b11174ce9a27cbf142f53125b6ae61227ec477ea8",\n          "archive_member": "mars_7_sensors/ground_truth/ground_truth_8hz.csv"\n        },\n        {\n          "path": "mars_7/archive_metadata/mars_7_sensors/README.txt",\n          "bytes": 1499,\n          "sha256": "6533ba3f5f3b7923942ebfc3bf2a35153338d08af56969bac8630e9c617e43f6",\n          "archive_member": "mars_7_sensors/README.txt"\n        },\n        {\n          "path": "mars_7/archive_metadata/mars_7_sensors/ground_truth/README.txt",\n          "bytes": 433,\n          "sha256": "cc40a9e4294c9df9d38cea40793c2cf708af779eb744245a2ace4192f9b31260",\n          "archive_member": "mars_7_sensors/ground_truth/README.txt"\n        },\n        {\n          "path": "mars_7/archive_metadata/mars_7_sensors/time_info.yaml",\n          "bytes": 239,\n          "sha256": "18fc7adf3c26408d05a4f80039158a312f0f0f3088b0e8cf22de478f7adacd1a",\n          "archive_member": "mars_7_sensors/time_info.yaml"\n        }\n      ]\n    },\n    "outdoor_1": {\n      "sequence": "outdoor_1",\n      "calibration_set": "KLU1",\n      "url": "https://cns-data.aau.at/insane-dataset/outdoor_1_sensors.zip",\n      "retrieved_utc": "2026-09-26T21:17:54.824912+00:00",\n      "archive_bytes": 18586950,\n      "archive_etag": "\\"11b9d46-5eb5e04d63c70\\"",\n      "http_bytes_received": 3161079,\n      "archive_members": [\n        "outdoor_1_sensors/",\n        "outdoor_1_sensors/LICENSE",\n        "outdoor_1_sensors/README.txt",\n        "outdoor_1_sensors/ground_truth/",\n        "outdoor_1_sensors/ground_truth/README.txt",\n        "outdoor_1_sensors/ground_truth/ground_truth_80hz.csv",\n        "outdoor_1_sensors/ground_truth/ground_truth_8hz.csv",\n        "outdoor_1_sensors/ground_truth/px4_imu_data.csv",\n        "outdoor_1_sensors/ground_truth/px4_mag_data_revised.csv",\n        "outdoor_1_sensors/ground_truth/rtk_gps1_data_revised.csv",\n        "outdoor_1_sensors/ground_truth/rtk_gps2_data_revised.csv",\n        "outdoor_1_sensors/lrf_range.csv",\n        "outdoor_1_sensors/lsm_imu.csv",\n        "outdoor_1_sensors/lsm_mag.csv",\n        "outdoor_1_sensors/px4_baro.csv",\n        "outdoor_1_sensors/px4_gps.csv",\n        "outdoor_1_sensors/px4_gps_vel_data.csv",\n        "outdoor_1_sensors/px4_imu.csv",\n        "outdoor_1_sensors/px4_mag.csv",\n        "outdoor_1_sensors/px4_rpm.csv",\n        "outdoor_1_sensors/rs_imu.csv",\n        "outdoor_1_sensors/rs_odom.csv",\n        "outdoor_1_sensors/rtk_gps1.csv",\n        "outdoor_1_sensors/rtk_gps2.csv",\n        "outdoor_1_sensors/time_info.yaml",\n        "outdoor_1_sensors/uwb_range.csv"\n      ],\n      "files": [\n        {\n          "path": "outdoor_1/px4_imu.csv",\n          "bytes": 7340539,\n          "sha256": "443991ebfc292ef7c235b47ba902535977ef29c36f12db28ba8930de2afd23f3",\n          "archive_member": "outdoor_1_sensors/px4_imu.csv"\n        },\n        {\n          "path": "outdoor_1/px4_gps.csv",\n          "bytes": 229771,\n          "sha256": "5fa34ad2d2d1cb32d751f4a4212ad92a86b17147458439a9e173a278d6ff30fd",\n          "archive_member": "outdoor_1_sensors/px4_gps.csv"\n        },\n        {\n          "path": "outdoor_1/ground_truth_8hz.csv",\n          "bytes": 137381,\n          "sha256": "eaec9108dff9888d73ba6c8afd03444f31bb0168c0890ec6559e13d3ff607ec7",\n          "archive_member": "outdoor_1_sensors/ground_truth/ground_truth_8hz.csv"\n        },\n        {\n          "path": "outdoor_1/archive_metadata/outdoor_1_sensors/README.txt",\n          "bytes": 1868,\n          "sha256": "3cf6381d1489663b17db76c8599aacfbabf09fb06e353b0d5118a9eec70fd6a5",\n          "archive_member": "outdoor_1_sensors/README.txt"\n        },\n        {\n          "path": "outdoor_1/archive_metadata/outdoor_1_sensors/ground_truth/README.txt",\n          "bytes": 436,\n          "sha256": "c9104d95fcf66535148a1174b577ae81d7b37247609ae8f64a62fcb22b7f3c47",\n          "archive_member": "outdoor_1_sensors/ground_truth/README.txt"\n        },\n        {\n          "path": "outdoor_1/archive_metadata/outdoor_1_sensors/time_info.yaml",\n          "bytes": 252,\n          "sha256": "ee16a127ae88622dd301309f53f8694b09383658a94d33a8d1772247749a1aea",\n          "archive_member": "outdoor_1_sensors/time_info.yaml"\n        }\n      ]\n    }\n  },\n  "support_files": {\n    "calibration/source.zip": {\n      "bytes": 9458,\n      "sha256": "cff4fbd099051cf0ff29838f7ab4bc67ff35d70cc6555029e537ca1036c215f2"\n    },\n    "calibration/insane_sensor_calib_preprocessed/LICENSE": {\n      "bytes": 2675,\n      "sha256": "effb36636d7fed3d0898ddde0c2179056728cdea1d91e448745d508daaf44632"\n    },\n    "calibration/insane_sensor_calib_preprocessed/mag_calibration_klu.yaml": {\n      "bytes": 1820,\n      "sha256": "195844768b861295b12580bdc2cf392e8557f877524dc3b4ae1f71b9ecc10faa"\n    },\n    "calibration/insane_sensor_calib_preprocessed/camera_calibration_klu2.yaml": {\n      "bytes": 2932,\n      "sha256": "f5ef855b5877240f06c5be8bd67d4fc969a89687e3bf31f31f71614ee58af634"\n    },\n    "calibration/insane_sensor_calib_preprocessed/sensor_calibration.yaml": {\n      "bytes": 2353,\n      "sha256": "5d390c7465e8fd4388748a2c81f0b70bb83c4600ee5f7cd220fdd41c2649976a"\n    },\n    "calibration/insane_sensor_calib_preprocessed/camera_calibration_klu1.yaml": {\n      "bytes": 2928,\n      "sha256": "dc00c4071137460ea76287a757997fd1f2e277f2b2e2b320d410ac8571a4904a"\n    },\n    "calibration/insane_sensor_calib_preprocessed/mag_calibration_mars.yaml": {\n      "bytes": 1831,\n      "sha256": "c2e46e7eea8cfb5faf8d527a30207b4f04ac87600eef6d90853873b3ac504040"\n    },\n    "calibration/insane_sensor_calib_preprocessed/camera_calibration_mars.yaml": {\n      "bytes": 2911,\n      "sha256": "70d5641ed1341da3d6bb0ae8742da77059938bde2c8815c128b0825ff545a91b"\n    },\n    "metadata/author_tools/LICENSE": {\n      "bytes": 2681,\n      "sha256": "69249100f1b292d0abc99224c3f7afd1d69e627d667f007627c8a0b2e7c4db30"\n    },\n    "metadata/author_tools/README.md": {\n      "bytes": 21282,\n      "sha256": "209f2356eea808d77f228728e52af630be6ec0515f666798ffb572495409473b"\n    },\n    "metadata/author_tools/tools/gps_conversion/gps_init_ref.m": {\n      "bytes": 549,\n      "sha256": "2bec7be9d688c19dbab2bb096fb24fdbe9dfe7ea82d14f377e972d4777c406a6"\n    },\n    "metadata/author_tools/tools/gps_conversion/wgs84ToEnu.m": {\n      "bytes": 197,\n      "sha256": "35cc13bee44066521c64ade9ce491744f2deeec647f98f7d43cab5e5296db3dc"\n    },\n    "metadata/author_tools/tools/gps_conversion/wgs84ToECEF.m": {\n      "bytes": 592,\n      "sha256": "88cc678486ae3217b6d108156d838a45a996d6e47a433d2120e31a536f0c62fa"\n    },\n    "metadata/author_tools/tools/gps_conversion/ECEFToENU.m": {\n      "bytes": 130,\n      "sha256": "408a23c496de1503f2df5dc24a773cdf0d3d3393657c3e977d278d2cd4dcdee7"\n    },\n    "metadata/author_tools/tools/time_alignment/time_reg.m": {\n      "bytes": 1679,\n      "sha256": "3b8b5a2f488e667dbbf05d267b6b48c8fe15181853b6e305d543a2ed41117589"\n    },\n    "metadata/author_tools/tools/time_alignment/smooth_time.m": {\n      "bytes": 587,\n      "sha256": "060798ecf4874e07be3d02d3206517b6edce0b940150a233b0b7da0af4d7b6b3"\n    },\n    "metadata/author_tools/post_scripts/csv2bag.py": {\n      "bytes": 19314,\n      "sha256": "301b93bbff55d394d36fe384d092c1a114c07c6314c8da66c1e34f93d4ddcd14"\n    },\n    "metadata/author_tools/data_extraction/extract_data.m": {\n      "bytes": 62070,\n      "sha256": "eda2ff07201d8d767d499b9753423581286e28d1365503a4f3d6bca9ab1a1e15"\n    },\n    "metadata/author_tools/data_extraction/gps_mag_orientation.m": {\n      "bytes": 32008,\n      "sha256": "b49b571dc19cef5fee85bc12c4b6dca88a8659fa588f19adcdfdf70fd54ca96b"\n    },\n    "metadata/author_tools/ros_topic_tools/read_imu_sensor_msgs.m": {\n      "bytes": 1905,\n      "sha256": "ef671f99e84ac4413499872045dffaa1477f444256c99a41026973e821980d4e"\n    },\n    "metadata/author_tools/ros_topic_tools/read_gps_navsat.m": {\n      "bytes": 1894,\n      "sha256": "fc071d21253a49c8a482e0d415bc059c0558fe5872c072ce221a7d664a662389"\n    },\n    "metadata/LICENSE.txt": {\n      "bytes": 2670,\n      "sha256": "3ee421e00a0d3f9f5e61aaa80fa0a6905dedf025765d219e2e322f11b26923a6"\n    }\n  }\n}\n', 'configs/insane/inventory.json': '{\n  "mars_1": {\n    "split": "train",\n    "native_imu_n": 19591,\n    "native_gps_n": 496,\n    "native_gt_n": 421,\n    "imu_duration_s": 99.9275438785553,\n    "gt_duration_s": 98.62491369247437,\n    "imu_median_dt_s": 0.005001068115234375,\n    "gt_median_dt_s": 0.125,\n    "gt_mean_dt_s": 0.23482122307731992,\n    "projection_check_max_m": 2.0108394949147623e-09,\n    "segments_s": [\n      [\n        1.0865027904510498,\n        99.71141648292542\n      ]\n    ],\n    "token_n": 1998,\n    "epoch_unix_s": 1633587463.564368,\n    "origin_enu_m": [\n      -864.5200976399245,\n      -463.62089390811315,\n      11.666886607570728\n    ]\n  },\n  "mars_2": {\n    "split": "train",\n    "native_imu_n": 32255,\n    "native_gps_n": 807,\n    "native_gt_n": 679,\n    "imu_duration_s": 165.16054224967957,\n    "gt_duration_s": 164.37485480308533,\n    "imu_median_dt_s": 0.005001068115234375,\n    "gt_median_dt_s": 0.125,\n    "gt_mean_dt_s": 0.2424407887951111,\n    "projection_check_max_m": 1.3609735560748959e-09,\n    "segments_s": [\n      [\n        0.7476108074188232,\n        164.9416630268097\n      ]\n    ],\n    "token_n": 3303,\n    "epoch_unix_s": 1633590083.672156,\n    "origin_enu_m": [\n      -868.9421510808531,\n      -458.53198388722586,\n      18.135734247855268\n    ]\n  },\n  "mars_3": {\n    "split": "train",\n    "native_imu_n": 29886,\n    "native_gps_n": 713,\n    "native_gt_n": 576,\n    "imu_duration_s": 153.99937629699707,\n    "gt_duration_s": 143.49987363815308,\n    "imu_median_dt_s": 0.005001068115234375,\n    "gt_median_dt_s": 0.125,\n    "gt_mean_dt_s": 0.24956499763157056,\n    "projection_check_max_m": 1.2965983842150308e-09,\n    "segments_s": [\n      [\n        8.386754989624023,\n        144.18523120880127\n      ]\n    ],\n    "token_n": 3079,\n    "epoch_unix_s": 1633590496.4445417,\n    "origin_enu_m": [\n      -878.082245485878,\n      -458.752766060553,\n      15.055661287901508\n    ]\n  },\n  "mars_4": {\n    "split": "validation",\n    "native_imu_n": 33640,\n    "native_gps_n": 854,\n    "native_gt_n": 714,\n    "imu_duration_s": 174.6750738620758,\n    "gt_duration_s": 172.3748471736908,\n    "imu_median_dt_s": 0.005001068115234375,\n    "gt_median_dt_s": 0.125,\n    "gt_mean_dt_s": 0.24175995396029565,\n    "projection_check_max_m": 1.7737562529873685e-09,\n    "segments_s": [\n      [\n        0.6750426292419434,\n        173.04988980293274\n      ]\n    ],\n    "token_n": 3493,\n    "epoch_unix_s": 1633592509.963704,\n    "origin_enu_m": [\n      -870.0263087335251,\n      -464.9402074098901,\n      16.85416228982942\n    ]\n  },\n  "mars_5": {\n    "split": "validation",\n    "native_imu_n": 18328,\n    "native_gps_n": 462,\n    "native_gt_n": 351,\n    "imu_duration_s": 92.28135204315186,\n    "gt_duration_s": 91.24991726875305,\n    "imu_median_dt_s": 0.005002737045288086,\n    "gt_median_dt_s": 0.125,\n    "gt_mean_dt_s": 0.26071404933929443,\n    "projection_check_max_m": 8.868568102116114e-10,\n    "segments_s": [\n      [\n        0.7301135063171387,\n        91.98003077507019\n      ]\n    ],\n    "token_n": 1845,\n    "epoch_unix_s": 1633612559.871542,\n    "origin_enu_m": [\n      -774.0484531872112,\n      -487.7653528212636,\n      21.733849985430968\n    ]\n  },\n  "mars_6": {\n    "split": "test_reserved",\n    "native_imu_n": 18331,\n    "native_gps_n": 461,\n    "native_gt_n": 395,\n    "imu_duration_s": 92.45312476158142,\n    "gt_duration_s": 91.99991750717163,\n    "imu_median_dt_s": 0.005002021789550781,\n    "gt_median_dt_s": 0.125,\n    "gt_mean_dt_s": 0.23350232869840515,\n    "projection_check_max_m": 1.116003289780565e-09,\n    "segments_s": [\n      [\n        0.2471468448638916,\n        92.24706435203552\n      ]\n    ],\n    "token_n": 1847,\n    "epoch_unix_s": 1633612860.6543448,\n    "origin_enu_m": [\n      -777.3852722076814,\n      -488.1414545898579,\n      13.618487086417868\n    ]\n  },\n  "mars_7": {\n    "split": "test_reserved",\n    "native_imu_n": 19562,\n    "native_gps_n": 491,\n    "native_gt_n": 396,\n    "imu_duration_s": 101.2236270904541,\n    "gt_duration_s": 97.999915599823,\n    "imu_median_dt_s": 0.005002021789550781,\n    "gt_median_dt_s": 0.125,\n    "gt_mean_dt_s": 0.24810105215145062,\n    "projection_check_max_m": 1.1317560222323664e-09,\n    "segments_s": [\n      [\n        1.1591477394104004,\n        99.1590633392334\n      ]\n    ],\n    "token_n": 2024,\n    "epoch_unix_s": 1633615405.3179162,\n    "origin_enu_m": [\n      -775.1125347079229,\n      -488.5631676227523,\n      16.688684311765304\n    ]\n  },\n  "outdoor_1": {\n    "split": "loader_smoke_only",\n    "native_imu_n": 51034,\n    "native_gps_n": 1277,\n    "native_gt_n": 854,\n    "imu_duration_s": 260.1535885334015,\n    "gt_duration_s": 199.37483739852905,\n    "imu_median_dt_s": 0.005000114440917969,\n    "gt_median_dt_s": 0.125,\n    "gt_mean_dt_s": 0.23373368979897896,\n    "projection_check_max_m": 2.0524613120187496e-09,\n    "segments_s": [\n      [\n        1.0399415493011475,\n        200.4147789478302\n      ]\n    ],\n    "token_n": 5203,\n    "epoch_unix_s": 1612483639.8658478,\n    "origin_enu_m": [\n      -2.497667235029635,\n      1.7899762927928753,\n      5.540739218489488\n    ]\n  }\n}\n', 'configs/insane/outages.json': '{\n  "selection": "chronological 60s stride; identical starts across durations/methods; no error-based selection",\n  "scenarios": [\n    {\n      "id": "mars_4_e1_d10",\n      "flight": "mars_4",\n      "split": "validation",\n      "duration_s": 10,\n      "onset_s": 20.8,\n      "start_s": 0.8000000000000007,\n      "end_s": 40.8,\n      "n_native_gt_outage": 29,\n      "n_native_gt_total": 165,\n      "token_mask_sha256": "9f12542780490d612306b1de6383455d9c904bc5b73f8b9cebd020d927653a95",\n      "native_gt_timestamp_sha256": "0096e8a0c2686f64c5c6d9fd0ee48e158b162cdf5f04482beffab5e7dddfc708"\n    },\n    {\n      "id": "mars_4_e1_d30",\n      "flight": "mars_4",\n      "split": "validation",\n      "duration_s": 30,\n      "onset_s": 20.8,\n      "start_s": 0.8000000000000007,\n      "end_s": 60.8,\n      "n_native_gt_outage": 115,\n      "n_native_gt_total": 243,\n      "token_mask_sha256": "ec337caf619f2fc34accfde4fefc63f5f100805a2dc4a30b788f58ddce2c4bc8",\n      "native_gt_timestamp_sha256": "1a456d7880849338a969bee1cafc907f5830311552f153827ed52a6d0bb40f1b"\n    },\n    {\n      "id": "mars_4_e1_d60",\n      "flight": "mars_4",\n      "split": "validation",\n      "duration_s": 60,\n      "onset_s": 20.8,\n      "start_s": 0.8000000000000007,\n      "end_s": 90.8,\n      "n_native_gt_outage": 231,\n      "n_native_gt_total": 373,\n      "token_mask_sha256": "71da0041c5ffc52dc55131d6c02ef9fd40886bf07a3990e099fe3e7675dd9bab",\n      "native_gt_timestamp_sha256": "c3c88e220171c3469d768c513f1e1736ed9a312e3368dcacc4b56aafc3a233bb"\n    },\n    {\n      "id": "mars_4_e2_d10",\n      "flight": "mars_4",\n      "split": "validation",\n      "duration_s": 10,\n      "onset_s": 80.80000000000001,\n      "start_s": 60.80000000000001,\n      "end_s": 100.80000000000001,\n      "n_native_gt_outage": 48,\n      "n_native_gt_total": 166,\n      "token_mask_sha256": "9f12542780490d612306b1de6383455d9c904bc5b73f8b9cebd020d927653a95",\n      "native_gt_timestamp_sha256": "674e228d3f7f9b74cf43ddeea1f9e4d48a0c8e08423bc7468904b8672c918fdc"\n    },\n    {\n      "id": "mars_4_e2_d30",\n      "flight": "mars_4",\n      "split": "validation",\n      "duration_s": 30,\n      "onset_s": 80.80000000000001,\n      "start_s": 60.80000000000001,\n      "end_s": 120.80000000000001,\n      "n_native_gt_outage": 120,\n      "n_native_gt_total": 261,\n      "token_mask_sha256": "ec337caf619f2fc34accfde4fefc63f5f100805a2dc4a30b788f58ddce2c4bc8",\n      "native_gt_timestamp_sha256": "c6fd1d26e1641c735affd02b942fd6423e3635028fb75ab259b59867a7456e2d"\n    },\n    {\n      "id": "mars_4_e2_d60",\n      "flight": "mars_4",\n      "split": "validation",\n      "duration_s": 60,\n      "onset_s": 80.80000000000001,\n      "start_s": 60.80000000000001,\n      "end_s": 150.8,\n      "n_native_gt_outage": 244,\n      "n_native_gt_total": 363,\n      "token_mask_sha256": "71da0041c5ffc52dc55131d6c02ef9fd40886bf07a3990e099fe3e7675dd9bab",\n      "native_gt_timestamp_sha256": "d21fb25730994ba9dd0fab409ed5923bccb0b469469264cec11e177163b75555"\n    },\n    {\n      "id": "mars_4_control",\n      "flight": "mars_4",\n      "split": "validation",\n      "duration_s": 0,\n      "onset_s": null,\n      "start_s": 0.7750426292419433,\n      "end_s": 172.94988980293274,\n      "n_native_gt_outage": 0,\n      "n_native_gt_total": 711,\n      "token_mask_sha256": "1ac185c3c8e4228fe29b8fdcd226aaa6b5142f2305b387e5c073d125ac514b07",\n      "native_gt_timestamp_sha256": "443bdb40391ba0766e21b738856c6141d6b3be4009ed4745df60cb665e043217"\n    },\n    {\n      "id": "mars_5_e1_d10",\n      "flight": "mars_5",\n      "split": "validation",\n      "duration_s": 10,\n      "onset_s": 20.85,\n      "start_s": 0.8500000000000014,\n      "end_s": 40.85,\n      "n_native_gt_outage": 40,\n      "n_native_gt_total": 157,\n      "token_mask_sha256": "21b9e16f0abe800c55f145a18abd0a79a37df3cbda1ac658e7e89003a42e7536",\n      "native_gt_timestamp_sha256": "814e7448723ff4788658aa3318bc3fe9c457c671f81947f7e8e0102270c3f804"\n    },\n    {\n      "id": "mars_5_e1_d30",\n      "flight": "mars_5",\n      "split": "validation",\n      "duration_s": 30,\n      "onset_s": 20.85,\n      "start_s": 0.8500000000000014,\n      "end_s": 60.85,\n      "n_native_gt_outage": 122,\n      "n_native_gt_total": 228,\n      "token_mask_sha256": "3dd74d234f7a60145505caa41ec0613e9d52eb9cbdc78e09d68e05c0d586d85a",\n      "native_gt_timestamp_sha256": "d4a9b6bd638e9772f477ebb8936c26beebb8ab65a625ba9d42515972f9bc9760"\n    },\n    {\n      "id": "mars_5_e1_d60",\n      "flight": "mars_5",\n      "split": "validation",\n      "duration_s": 60,\n      "onset_s": 20.85,\n      "start_s": 0.8500000000000014,\n      "end_s": 90.85,\n      "n_native_gt_outage": 226,\n      "n_native_gt_total": 342,\n      "token_mask_sha256": "c0c2ec8564a28a400ecdd4fe90d4d6b443981b414ff75cc3098c2fd7dc691a8b",\n      "native_gt_timestamp_sha256": "029e53bc9895e27950b0af4e80051eb198603e496273dad3e8f9ae524f31726d"\n    },\n    {\n      "id": "mars_5_control",\n      "flight": "mars_5",\n      "split": "validation",\n      "duration_s": 0,\n      "onset_s": null,\n      "start_s": 0.8301135063171386,\n      "end_s": 91.8800307750702,\n      "n_native_gt_outage": 0,\n      "n_native_gt_total": 348,\n      "token_mask_sha256": "326725b6f95e3179af4cb2c2bea2c84272294746b44682039f98d280dac3a055",\n      "native_gt_timestamp_sha256": "7bdecf694438bba25cd4ddff8f323074e7c279f1de187577772d562e683dae18"\n    }\n  ],\n  "unsupported": []\n}\n', 'configs/insane/pilot.json': '{\n  "pilot_id": "fusionnav_insane_pilot_v1",\n  "seed": 0,\n  "methods": ["full", "gps_only"],\n  "max_epochs": 30,\n  "patience": 5,\n  "min_delta_m": 0.0,\n  "validation_criterion": "macro mean episode relative-motion RMSE3D on fixed validation outages",\n  "batch_size": 4,\n  "learning_rate": 0.001,\n  "weight_decay": 0.0001,\n  "grad_clip": 1.0,\n  "huber_delta_m": 1.0,\n  "hidden": 32,\n  "token_period_s": 0.05,\n  "imu_gap_limit_s": 0.1,\n  "gps_gap_limit_s": 1.0,\n  "gt_gap_limit_s": 2.0,\n  "history_s": 20.0,\n  "recovery_s": 10.0,\n  "duration_s": [10, 30, 60],\n  "validation_onset_stride_s": 60.0,\n  "train_episodes_per_flight_duration_epoch": 2,\n  "cv_history_s": 5.0,\n  "normalization": "IMU dt-weighted mean/std on train flights only; delta/velocity fixed /5; cov standard deviation /10",\n  "imu_frame": "published PX4 MAVROS body axes, ROS FLU; never rotate using GT",\n  "position_frame": "published local ENU, common per-flight first ordinary GNSS origin",\n  "gt_point": "PX4 IMU origin, authors dual RTK plus magnetometer reference",\n  "gnss_point": "ordinary GNSS antenna; its lever arm to PX4 IMU is absent from supplied calibration",\n  "eskf": {\n    "real_data_ready": false,\n    "blockers": ["ordinary GNSS antenna to PX4 IMU lever arm not supplied", "absolute initial yaw unavailable from permitted sensors without validated dynamic alignment"],\n    "sigma_accel_density": 0.1,\n    "sigma_gyro_density": 0.01,\n    "sigma_accel_bias_walk": 0.001,\n    "sigma_gyro_bias_walk": 0.0001,\n    "noise_status": "engineering defaults for synthetic mechanization checks only; not tuned or asserted flight calibration",\n    "gnss_lever_arm_body_m": null,\n    "parameter_tuning": "none"\n  },\n  "test_used": false,\n  "extra_seeds": false\n}\n', 'configs/insane/protocol_lock.json': '{\n  "pilot_id": "fusionnav_insane_pilot_v1",\n  "sha256": {\n    "download_manifest.json": "6723a6bdfc90f5ce608be5cba030509d58a694b976f8b2e675d6f9250b1c029a",\n    "outages.json": "208ef71a355441c2f795155b448cc29626bca974ecb506960c74757fc07eb474",\n    "splits.json": "a2f27d70ec0bae01dd7eb5dabf94994dbedd48898f72b293823549867fa075f4",\n    "training_episodes.json": "84b85d23a0006c2039ec6ee3cb2002a02e11faa1a61e140dde24a95d74e442dd",\n    "scaler_preflight.json": "7549771e7428907e5a89039540b08e007b7aeb37023ccd6d92cd4d37b94ab6dd",\n    "pilot.json": "4718d7a7415e89a5ccffe91bcb202db655174113d86186a328761c43de4c1a24",\n    "inventory.json": "f370ecb2063be698d8a2397c5f713afa787c3d40110fba4302e4dfaba20d8918"\n  },\n  "freeze_stage": "before any training/validation metric",\n  "model_comparisons_used_for_selection": false\n}\n', 'configs/insane/scaler_preflight.json': '{\n  "mean": [\n    0.19879936265108974,\n    0.7300404411080188,\n    9.319951522076051,\n    0.006544336375771426,\n    0.0020933271871612018,\n    0.01063691010039449\n  ],\n  "std": [\n    0.8540083054772117,\n    1.0854608479776087,\n    2.949380398130146,\n    0.22206154869206063,\n    0.2398173302525047,\n    0.2513599340277575\n  ],\n  "fit_flights": [\n    "mars_1",\n    "mars_2",\n    "mars_3"\n  ],\n  "weighted_seconds": 419.0\n}\n', 'configs/insane/splits.json': '{\n  "train": [\n    "mars_1",\n    "mars_2",\n    "mars_3"\n  ],\n  "validation": [\n    "mars_4",\n    "mars_5"\n  ],\n  "test_reserved": [\n    "mars_6",\n    "mars_7"\n  ],\n  "loader_smoke_only": [\n    "outdoor_1"\n  ],\n  "rule": "numerical flight order; distinct recordings; selection by availability/duration only",\n  "test_policy": "hash/schema/timing inventory only; no metrics, scalers, initialization, or training",\n  "same_campaign_warning": "separate flights from same campaign/vehicle, not independent sites"\n}\n', 'configs/insane/training_episodes.json': '{\n  "seed": 0,\n  "episodes": [\n    {\n      "id": "train_ep1_mars_3_d60_0",\n      "epoch": 1,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 28.8,\n      "start_s": 8.8,\n      "end_s": 98.8,\n      "n_native_gt_outage": 236,\n      "n_native_gt_total": 355,\n      "token_mask_sha256": "c0c2ec8564a28a400ecdd4fe90d4d6b443981b414ff75cc3098c2fd7dc691a8b",\n      "native_gt_timestamp_sha256": "1230fb6c58b192e1eaeb759d03037fc352d4b778d4bec6c165d4067831bc6ac3"\n    },\n    {\n      "id": "train_ep1_mars_3_d30_1",\n      "epoch": 1,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 43.6,\n      "start_s": 23.6,\n      "end_s": 83.6,\n      "n_native_gt_outage": 109,\n      "n_native_gt_total": 240,\n      "token_mask_sha256": "3dd74d234f7a60145505caa41ec0613e9d52eb9cbdc78e09d68e05c0d586d85a",\n      "native_gt_timestamp_sha256": "f33e3170d5a6d1dcf6c1646c57131a6bea9c5df1f26114548f3e87ce4b436747"\n    },\n    {\n      "id": "train_ep1_mars_1_d30_0",\n      "epoch": 1,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 34.2,\n      "start_s": 14.200000000000003,\n      "end_s": 74.2,\n      "n_native_gt_outage": 111,\n      "n_native_gt_total": 250,\n      "token_mask_sha256": "b34423e399ec53c009571b199c4b245f2f0d58ef83150e9bf0611fa732314343",\n      "native_gt_timestamp_sha256": "c496b1fc54a89380b2e1287d39a62efaa4ad02a40268eef957ea9fe4854a05ca"\n    },\n    {\n      "id": "train_ep1_mars_1_d60_1",\n      "epoch": 1,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 23.55,\n      "start_s": 3.5500000000000007,\n      "end_s": 93.55,\n      "n_native_gt_outage": 252,\n      "n_native_gt_total": 374,\n      "token_mask_sha256": "c0c2ec8564a28a400ecdd4fe90d4d6b443981b414ff75cc3098c2fd7dc691a8b",\n      "native_gt_timestamp_sha256": "0157562d65b6eede425ee0903189debf01c3551d473d0a4e2c94022642ac6a5c"\n    },\n    {\n      "id": "train_ep1_mars_2_d30_0",\n      "epoch": 1,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 88.9,\n      "start_s": 68.9,\n      "end_s": 128.9,\n      "n_native_gt_outage": 121,\n      "n_native_gt_total": 256,\n      "token_mask_sha256": "ec337caf619f2fc34accfde4fefc63f5f100805a2dc4a30b788f58ddce2c4bc8",\n      "native_gt_timestamp_sha256": "0da94e6ea9870dd06ff5d7ec07a5a35af3baad5685aebd442da49bdbcea684f1"\n    },\n    {\n      "id": "train_ep1_mars_2_d10_1",\n      "epoch": 1,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 143.85,\n      "start_s": 123.85,\n      "end_s": 163.85,\n      "n_native_gt_outage": 35,\n      "n_native_gt_total": 175,\n      "token_mask_sha256": "9f12542780490d612306b1de6383455d9c904bc5b73f8b9cebd020d927653a95",\n      "native_gt_timestamp_sha256": "fb136ae7d8a21c443fc0eea402ad2a887793887f0da1a60774776cb0a75aa28c"\n    },\n    {\n      "id": "train_ep1_mars_2_d30_1",\n      "epoch": 1,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 102.55000000000001,\n      "start_s": 82.55000000000001,\n      "end_s": 142.55,\n      "n_native_gt_outage": 128,\n      "n_native_gt_total": 251,\n      "token_mask_sha256": "3dd74d234f7a60145505caa41ec0613e9d52eb9cbdc78e09d68e05c0d586d85a",\n      "native_gt_timestamp_sha256": "32561ffe0fcce28f21471f7bb4e7c882b9e98187141f52ec288f14180af21129"\n    },\n    {\n      "id": "train_ep1_mars_2_d10_0",\n      "epoch": 1,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 24.400000000000002,\n      "start_s": 4.400000000000002,\n      "end_s": 44.400000000000006,\n      "n_native_gt_outage": 33,\n      "n_native_gt_total": 148,\n      "token_mask_sha256": "860d531b40010f5a1ce4a6d8d4c9b3f3cb34a9b01d0a5e57422ce1faf23b77a3",\n      "native_gt_timestamp_sha256": "b5c6b5370e66bd88018ce792a3d6e0ac57047ac8df8d529d48420f5eeacea55f"\n    },\n    {\n      "id": "train_ep1_mars_2_d60_0",\n      "epoch": 1,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 89.35000000000001,\n      "start_s": 69.35000000000001,\n      "end_s": 159.35000000000002,\n      "n_native_gt_outage": 248,\n      "n_native_gt_total": 368,\n      "token_mask_sha256": "fed530af485dca69fb59f9b1a865da85e17ffd444c3c0d42d5f5017e0315ab5f",\n      "native_gt_timestamp_sha256": "eff2ac44664c9a3a7f776eaa77f5514386050140e7bafdda18123bb5b2169525"\n    },\n    {\n      "id": "train_ep1_mars_3_d10_1",\n      "epoch": 1,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 97.95,\n      "start_s": 77.95,\n      "end_s": 117.95,\n      "n_native_gt_outage": 51,\n      "n_native_gt_total": 165,\n      "token_mask_sha256": "9f12542780490d612306b1de6383455d9c904bc5b73f8b9cebd020d927653a95",\n      "native_gt_timestamp_sha256": "944a872cdb9cf3f59c026ec454b51f2314703a3c5863a29a3030a0beb3a59c30"\n    },\n    {\n      "id": "train_ep1_mars_1_d10_1",\n      "epoch": 1,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 58.0,\n      "start_s": 38.0,\n      "end_s": 78.0,\n      "n_native_gt_outage": 38,\n      "n_native_gt_total": 171,\n      "token_mask_sha256": "860d531b40010f5a1ce4a6d8d4c9b3f3cb34a9b01d0a5e57422ce1faf23b77a3",\n      "native_gt_timestamp_sha256": "e4b64aade149adc198e34773cdb9bb285b96e733cc41e17fa964ebf4645456f9"\n    },\n    {\n      "id": "train_ep1_mars_1_d60_0",\n      "epoch": 1,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 23.75,\n      "start_s": 3.75,\n      "end_s": 93.75,\n      "n_native_gt_outage": 252,\n      "n_native_gt_total": 374,\n      "token_mask_sha256": "c0c2ec8564a28a400ecdd4fe90d4d6b443981b414ff75cc3098c2fd7dc691a8b",\n      "native_gt_timestamp_sha256": "98752201d03a1786fa3050f302e32a6e2b936b0e3e93665b6da36b85c435ab75"\n    },\n    {\n      "id": "train_ep1_mars_3_d60_1",\n      "epoch": 1,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 46.800000000000004,\n      "start_s": 26.800000000000004,\n      "end_s": 116.80000000000001,\n      "n_native_gt_outage": 238,\n      "n_native_gt_total": 353,\n      "token_mask_sha256": "c0c2ec8564a28a400ecdd4fe90d4d6b443981b414ff75cc3098c2fd7dc691a8b",\n      "native_gt_timestamp_sha256": "33951ef5953fe91dc0e80bbaa98d9a74f6c4a4f96580f6fc9fe036079643791e"\n    },\n    {\n      "id": "train_ep1_mars_1_d10_0",\n      "epoch": 1,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 62.95,\n      "start_s": 42.95,\n      "end_s": 82.95,\n      "n_native_gt_outage": 55,\n      "n_native_gt_total": 171,\n      "token_mask_sha256": "21b9e16f0abe800c55f145a18abd0a79a37df3cbda1ac658e7e89003a42e7536",\n      "native_gt_timestamp_sha256": "2b753a2fb54efa141ce16110987e668c9a14ef507ccc7812092452b47b3f4a61"\n    },\n    {\n      "id": "train_ep1_mars_3_d10_0",\n      "epoch": 1,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 111.2,\n      "start_s": 91.2,\n      "end_s": 131.2,\n      "n_native_gt_outage": 35,\n      "n_native_gt_total": 150,\n      "token_mask_sha256": "21b9e16f0abe800c55f145a18abd0a79a37df3cbda1ac658e7e89003a42e7536",\n      "native_gt_timestamp_sha256": "d2661ca9b15db9a0505c655b26bc763716bd5c3d916dad831ee9bf423df3a7aa"\n    },\n    {\n      "id": "train_ep1_mars_2_d60_1",\n      "epoch": 1,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 57.1,\n      "start_s": 37.1,\n      "end_s": 127.1,\n      "n_native_gt_outage": 242,\n      "n_native_gt_total": 379,\n      "token_mask_sha256": "71da0041c5ffc52dc55131d6c02ef9fd40886bf07a3990e099fe3e7675dd9bab",\n      "native_gt_timestamp_sha256": "862a76110356443f3bf002a4461d79e120897d9a1cbefc80e8ff0d0afbf774e8"\n    },\n    {\n      "id": "train_ep1_mars_3_d30_0",\n      "epoch": 1,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 68.65,\n      "start_s": 48.650000000000006,\n      "end_s": 108.65,\n      "n_native_gt_outage": 119,\n      "n_native_gt_total": 238,\n      "token_mask_sha256": "b34423e399ec53c009571b199c4b245f2f0d58ef83150e9bf0611fa732314343",\n      "native_gt_timestamp_sha256": "19d1333b7f2cc02b5e3600a76edaa2324a664cb739ae4292506d526f1ea94ef0"\n    },\n    {\n      "id": "train_ep1_mars_1_d30_1",\n      "epoch": 1,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 35.2,\n      "start_s": 15.200000000000003,\n      "end_s": 75.2,\n      "n_native_gt_outage": 112,\n      "n_native_gt_total": 249,\n      "token_mask_sha256": "b34423e399ec53c009571b199c4b245f2f0d58ef83150e9bf0611fa732314343",\n      "native_gt_timestamp_sha256": "130622e5a701504b032cbcb803348644d3110951497235a88be209c8f6fdedbf"\n    },\n    {\n      "id": "train_ep2_mars_1_d10_0",\n      "epoch": 2,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 58.300000000000004,\n      "start_s": 38.300000000000004,\n      "end_s": 78.30000000000001,\n      "n_native_gt_outage": 40,\n      "n_native_gt_total": 171,\n      "token_mask_sha256": "9f12542780490d612306b1de6383455d9c904bc5b73f8b9cebd020d927653a95",\n      "native_gt_timestamp_sha256": "e4b64aade149adc198e34773cdb9bb285b96e733cc41e17fa964ebf4645456f9"\n    },\n    {\n      "id": "train_ep2_mars_2_d60_0",\n      "epoch": 2,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 88.35000000000001,\n      "start_s": 68.35000000000001,\n      "end_s": 158.35000000000002,\n      "n_native_gt_outage": 252,\n      "n_native_gt_total": 368,\n      "token_mask_sha256": "fed530af485dca69fb59f9b1a865da85e17ffd444c3c0d42d5f5017e0315ab5f",\n      "native_gt_timestamp_sha256": "cea2770c7ebb85d057e16e179036a9130c44c16a03ddf4d4338bcb0d7e491e75"\n    },\n    {\n      "id": "train_ep2_mars_1_d10_1",\n      "epoch": 2,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 35.800000000000004,\n      "start_s": 15.800000000000004,\n      "end_s": 55.800000000000004,\n      "n_native_gt_outage": 37,\n      "n_native_gt_total": 164,\n      "token_mask_sha256": "860d531b40010f5a1ce4a6d8d4c9b3f3cb34a9b01d0a5e57422ce1faf23b77a3",\n      "native_gt_timestamp_sha256": "1994b1e1629362bb0079686a9581694cf13d8ade30cb06bf182d19c722354d89"\n    },\n    {\n      "id": "train_ep2_mars_3_d30_1",\n      "epoch": 2,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 48.7,\n      "start_s": 28.700000000000003,\n      "end_s": 88.7,\n      "n_native_gt_outage": 112,\n      "n_native_gt_total": 236,\n      "token_mask_sha256": "63d833f0b713bb4fb84f25c62b86deb8f26260f7a627284500d089600ceaa5cc",\n      "native_gt_timestamp_sha256": "aa0edf5256b9fdf35a393aed5cc7a45750cf7d4bf327b3729f0e9b9e4dd809a8"\n    },\n    {\n      "id": "train_ep2_mars_1_d30_1",\n      "epoch": 2,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 21.5,\n      "start_s": 1.5,\n      "end_s": 61.5,\n      "n_native_gt_outage": 129,\n      "n_native_gt_total": 244,\n      "token_mask_sha256": "ec337caf619f2fc34accfde4fefc63f5f100805a2dc4a30b788f58ddce2c4bc8",\n      "native_gt_timestamp_sha256": "69a7d9aced6a08f2c7e22c5023d4d8f7666e68df335f2b9583a211d1c85bab3d"\n    },\n    {\n      "id": "train_ep2_mars_2_d10_1",\n      "epoch": 2,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 67.45,\n      "start_s": 47.45,\n      "end_s": 87.45,\n      "n_native_gt_outage": 46,\n      "n_native_gt_total": 163,\n      "token_mask_sha256": "5683213f4c537fb13be867ad7c9689f774ac7da78020c8a9f35e859a9dc2a57e",\n      "native_gt_timestamp_sha256": "efff129cb623b8d88bca4303f352021222fee1e0b38f88990575808ea8c976e1"\n    },\n    {\n      "id": "train_ep2_mars_3_d60_0",\n      "epoch": 2,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 41.5,\n      "start_s": 21.5,\n      "end_s": 111.5,\n      "n_native_gt_outage": 230,\n      "n_native_gt_total": 357,\n      "token_mask_sha256": "944469817cb1d463be9ba8e22d7c037bf6b358337512a5973f0eedd13fdd691e",\n      "native_gt_timestamp_sha256": "500f9ddff17484ccceac03744f2b6c9ca8c4b8cc1841e0b07f88d5b40e39d2ac"\n    },\n    {\n      "id": "train_ep2_mars_3_d10_0",\n      "epoch": 2,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 120.60000000000001,\n      "start_s": 100.60000000000001,\n      "end_s": 140.60000000000002,\n      "n_native_gt_outage": 31,\n      "n_native_gt_total": 172,\n      "token_mask_sha256": "21b9e16f0abe800c55f145a18abd0a79a37df3cbda1ac658e7e89003a42e7536",\n      "native_gt_timestamp_sha256": "6f9e1d981556bfcfd9d825d5dd5e10412b6ccd8a91db9f5533fb335dd7441a08"\n    },\n    {\n      "id": "train_ep2_mars_2_d30_0",\n      "epoch": 2,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 102.55000000000001,\n      "start_s": 82.55000000000001,\n      "end_s": 142.55,\n      "n_native_gt_outage": 128,\n      "n_native_gt_total": 251,\n      "token_mask_sha256": "3dd74d234f7a60145505caa41ec0613e9d52eb9cbdc78e09d68e05c0d586d85a",\n      "native_gt_timestamp_sha256": "32561ffe0fcce28f21471f7bb4e7c882b9e98187141f52ec288f14180af21129"\n    },\n    {\n      "id": "train_ep2_mars_2_d30_1",\n      "epoch": 2,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 92.55000000000001,\n      "start_s": 72.55000000000001,\n      "end_s": 132.55,\n      "n_native_gt_outage": 118,\n      "n_native_gt_total": 251,\n      "token_mask_sha256": "3dd74d234f7a60145505caa41ec0613e9d52eb9cbdc78e09d68e05c0d586d85a",\n      "native_gt_timestamp_sha256": "00d9859e4915be0173ef982a16cb667f8a91bcddc116e755a804bf1e90207ed0"\n    },\n    {\n      "id": "train_ep2_mars_2_d60_1",\n      "epoch": 2,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 67.85000000000001,\n      "start_s": 47.85000000000001,\n      "end_s": 137.85000000000002,\n      "n_native_gt_outage": 259,\n      "n_native_gt_total": 379,\n      "token_mask_sha256": "944469817cb1d463be9ba8e22d7c037bf6b358337512a5973f0eedd13fdd691e",\n      "native_gt_timestamp_sha256": "242d41b6089d8210b528a92fcd07cbd355b3a305151729755ee381fb188f3b75"\n    },\n    {\n      "id": "train_ep2_mars_3_d10_1",\n      "epoch": 2,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 112.05000000000001,\n      "start_s": 92.05000000000001,\n      "end_s": 132.05,\n      "n_native_gt_outage": 37,\n      "n_native_gt_total": 150,\n      "token_mask_sha256": "860d531b40010f5a1ce4a6d8d4c9b3f3cb34a9b01d0a5e57422ce1faf23b77a3",\n      "native_gt_timestamp_sha256": "4caeb068d43a80a5ebae63c9af40ff74a45a0465c61b6840301dfa3d19ba6817"\n    },\n    {\n      "id": "train_ep2_mars_3_d30_0",\n      "epoch": 2,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 67.8,\n      "start_s": 47.8,\n      "end_s": 107.8,\n      "n_native_gt_outage": 118,\n      "n_native_gt_total": 240,\n      "token_mask_sha256": "3dd74d234f7a60145505caa41ec0613e9d52eb9cbdc78e09d68e05c0d586d85a",\n      "native_gt_timestamp_sha256": "a28ef1ba7dd6ab167f6a3cfba72eb11d33e8d3da49a81fd75811c9d0bd1e2a73"\n    },\n    {\n      "id": "train_ep2_mars_1_d60_1",\n      "epoch": 2,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 28.700000000000003,\n      "start_s": 8.700000000000003,\n      "end_s": 98.7,\n      "n_native_gt_outage": 237,\n      "n_native_gt_total": 376,\n      "token_mask_sha256": "71da0041c5ffc52dc55131d6c02ef9fd40886bf07a3990e099fe3e7675dd9bab",\n      "native_gt_timestamp_sha256": "420f6f6ab3ca0352637c33077a6cdc06008cc472dc225a104c876112c89ebef3"\n    },\n    {\n      "id": "train_ep2_mars_2_d10_0",\n      "epoch": 2,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 62.650000000000006,\n      "start_s": 42.650000000000006,\n      "end_s": 82.65,\n      "n_native_gt_outage": 49,\n      "n_native_gt_total": 163,\n      "token_mask_sha256": "9f12542780490d612306b1de6383455d9c904bc5b73f8b9cebd020d927653a95",\n      "native_gt_timestamp_sha256": "a1e5b85065c6748507d5a18d8ffb23a8747dcc8a10bb7f8db4eb4321fd42e326"\n    },\n    {\n      "id": "train_ep2_mars_1_d30_0",\n      "epoch": 2,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 42.300000000000004,\n      "start_s": 22.300000000000004,\n      "end_s": 82.30000000000001,\n      "n_native_gt_outage": 133,\n      "n_native_gt_total": 250,\n      "token_mask_sha256": "ec337caf619f2fc34accfde4fefc63f5f100805a2dc4a30b788f58ddce2c4bc8",\n      "native_gt_timestamp_sha256": "5336f1bc12d15a6a4b477666ffce1231d271a40473d6d80fae07b5ac34e3f481"\n    },\n    {\n      "id": "train_ep2_mars_1_d60_0",\n      "epoch": 2,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 26.650000000000002,\n      "start_s": 6.650000000000002,\n      "end_s": 96.65,\n      "n_native_gt_outage": 244,\n      "n_native_gt_total": 379,\n      "token_mask_sha256": "944469817cb1d463be9ba8e22d7c037bf6b358337512a5973f0eedd13fdd691e",\n      "native_gt_timestamp_sha256": "e859a36cb4070754220e79cafcccf9b15d247d7e09181d9173dfc4ec294efa1b"\n    },\n    {\n      "id": "train_ep2_mars_3_d60_1",\n      "epoch": 2,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 39.800000000000004,\n      "start_s": 19.800000000000004,\n      "end_s": 109.80000000000001,\n      "n_native_gt_outage": 231,\n      "n_native_gt_total": 356,\n      "token_mask_sha256": "478ce3b3d3dce9522789ea95341f5ae713e75348c57aff32b390220cd55f9144",\n      "native_gt_timestamp_sha256": "0302fe7b44b0f80d43ae33e42de45a8a718db7a2ddba554fd12da3847dcc8b85"\n    },\n    {\n      "id": "train_ep3_mars_2_d30_1",\n      "epoch": 3,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 45.1,\n      "start_s": 25.1,\n      "end_s": 85.1,\n      "n_native_gt_outage": 130,\n      "n_native_gt_total": 234,\n      "token_mask_sha256": "3dd74d234f7a60145505caa41ec0613e9d52eb9cbdc78e09d68e05c0d586d85a",\n      "native_gt_timestamp_sha256": "1e8025e5971fecd1c83b5b67a4f97b2ac9eb82c487954e06b7f5e9118e2af55f"\n    },\n    {\n      "id": "train_ep3_mars_3_d60_0",\n      "epoch": 3,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 31.55,\n      "start_s": 11.55,\n      "end_s": 101.55,\n      "n_native_gt_outage": 239,\n      "n_native_gt_total": 347,\n      "token_mask_sha256": "c0c2ec8564a28a400ecdd4fe90d4d6b443981b414ff75cc3098c2fd7dc691a8b",\n      "native_gt_timestamp_sha256": "3ba68f67cebffa01a89b3d239a7c20380d66da43182617df2dfb169fb0339c3f"\n    },\n    {\n      "id": "train_ep3_mars_2_d30_0",\n      "epoch": 3,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 63.85,\n      "start_s": 43.85,\n      "end_s": 103.85,\n      "n_native_gt_outage": 136,\n      "n_native_gt_total": 254,\n      "token_mask_sha256": "63d833f0b713bb4fb84f25c62b86deb8f26260f7a627284500d089600ceaa5cc",\n      "native_gt_timestamp_sha256": "48a9f20f4659fbb80bb519c1af89ac43e139cc9d3c4cf4c142900a328cdfd46f"\n    },\n    {\n      "id": "train_ep3_mars_2_d60_0",\n      "epoch": 3,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 47.1,\n      "start_s": 27.1,\n      "end_s": 117.1,\n      "n_native_gt_outage": 255,\n      "n_native_gt_total": 359,\n      "token_mask_sha256": "71da0041c5ffc52dc55131d6c02ef9fd40886bf07a3990e099fe3e7675dd9bab",\n      "native_gt_timestamp_sha256": "e5a875a625d795a8c9fa1ec8670e824a3e783c16534c8d1e8dcfd716d31fef46"\n    },\n    {\n      "id": "train_ep3_mars_1_d30_0",\n      "epoch": 3,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 59.550000000000004,\n      "start_s": 39.550000000000004,\n      "end_s": 99.55000000000001,\n      "n_native_gt_outage": 126,\n      "n_native_gt_total": 261,\n      "token_mask_sha256": "3dd74d234f7a60145505caa41ec0613e9d52eb9cbdc78e09d68e05c0d586d85a",\n      "native_gt_timestamp_sha256": "5186ee9d9529c43f8419bd3d819b5666ba1b83de076904e7cc56403e26fff09f"\n    },\n    {\n      "id": "train_ep3_mars_1_d60_1",\n      "epoch": 3,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 23.85,\n      "start_s": 3.8500000000000014,\n      "end_s": 93.85,\n      "n_native_gt_outage": 252,\n      "n_native_gt_total": 374,\n      "token_mask_sha256": "944469817cb1d463be9ba8e22d7c037bf6b358337512a5973f0eedd13fdd691e",\n      "native_gt_timestamp_sha256": "3798810fabe03a4037a8cb7fd5416d08b8e8feb222862f3fb5d867825f4f7025"\n    },\n    {\n      "id": "train_ep3_mars_3_d30_0",\n      "epoch": 3,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 79.25,\n      "start_s": 59.25,\n      "end_s": 119.25,\n      "n_native_gt_outage": 128,\n      "n_native_gt_total": 236,\n      "token_mask_sha256": "63d833f0b713bb4fb84f25c62b86deb8f26260f7a627284500d089600ceaa5cc",\n      "native_gt_timestamp_sha256": "3f31289e34203be30c2f34d79ce9b41b3da143e4f7e9e57c1df2b553fc920b0a"\n    },\n    {\n      "id": "train_ep3_mars_1_d10_1",\n      "epoch": 3,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 64.35000000000001,\n      "start_s": 44.35000000000001,\n      "end_s": 84.35000000000001,\n      "n_native_gt_outage": 56,\n      "n_native_gt_total": 170,\n      "token_mask_sha256": "21b9e16f0abe800c55f145a18abd0a79a37df3cbda1ac658e7e89003a42e7536",\n      "native_gt_timestamp_sha256": "51f6c8117929e0bc68de1c568e1823361869f36d41fbfbbd6c7e9073075b3037"\n    },\n    {\n      "id": "train_ep3_mars_3_d10_1",\n      "epoch": 3,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 56.2,\n      "start_s": 36.2,\n      "end_s": 76.2,\n      "n_native_gt_outage": 34,\n      "n_native_gt_total": 149,\n      "token_mask_sha256": "21b9e16f0abe800c55f145a18abd0a79a37df3cbda1ac658e7e89003a42e7536",\n      "native_gt_timestamp_sha256": "d55f1beb8fdc13bc07f5452d21d175fb7303022866abd7a3b09eafafbc74efde"\n    },\n    {\n      "id": "train_ep3_mars_1_d10_0",\n      "epoch": 3,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 61.400000000000006,\n      "start_s": 41.400000000000006,\n      "end_s": 81.4,\n      "n_native_gt_outage": 51,\n      "n_native_gt_total": 171,\n      "token_mask_sha256": "860d531b40010f5a1ce4a6d8d4c9b3f3cb34a9b01d0a5e57422ce1faf23b77a3",\n      "native_gt_timestamp_sha256": "19976081c1f5dd7640bbe2ac5c8eef4ab24c707ca8bcb734f72db33844a75b10"\n    },\n    {\n      "id": "train_ep3_mars_2_d60_1",\n      "epoch": 3,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 34.050000000000004,\n      "start_s": 14.050000000000004,\n      "end_s": 104.05000000000001,\n      "n_native_gt_outage": 253,\n      "n_native_gt_total": 361,\n      "token_mask_sha256": "944469817cb1d463be9ba8e22d7c037bf6b358337512a5973f0eedd13fdd691e",\n      "native_gt_timestamp_sha256": "484ffbcef0f23b9e2ed987c85b787cd905839fea2a6045b999b5bddbd0c1ec1e"\n    },\n    {\n      "id": "train_ep3_mars_2_d10_0",\n      "epoch": 3,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 64.75,\n      "start_s": 44.75,\n      "end_s": 84.75,\n      "n_native_gt_outage": 47,\n      "n_native_gt_total": 165,\n      "token_mask_sha256": "860d531b40010f5a1ce4a6d8d4c9b3f3cb34a9b01d0a5e57422ce1faf23b77a3",\n      "native_gt_timestamp_sha256": "93cd7b7dfdfa9543c30f5db864828039b1ba256bd65dcab5056252cd71e9a976"\n    },\n    {\n      "id": "train_ep3_mars_3_d60_1",\n      "epoch": 3,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 49.300000000000004,\n      "start_s": 29.300000000000004,\n      "end_s": 119.30000000000001,\n      "n_native_gt_outage": 241,\n      "n_native_gt_total": 354,\n      "token_mask_sha256": "71da0041c5ffc52dc55131d6c02ef9fd40886bf07a3990e099fe3e7675dd9bab",\n      "native_gt_timestamp_sha256": "0a80d89c19e10c7c39a7ffb71f1dc1ee94de18aab2f61a4b24f526638d7952e6"\n    },\n    {\n      "id": "train_ep3_mars_1_d30_1",\n      "epoch": 3,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 30.650000000000002,\n      "start_s": 10.650000000000002,\n      "end_s": 70.65,\n      "n_native_gt_outage": 111,\n      "n_native_gt_total": 245,\n      "token_mask_sha256": "63d833f0b713bb4fb84f25c62b86deb8f26260f7a627284500d089600ceaa5cc",\n      "native_gt_timestamp_sha256": "a1b91fd00aa235ec24b15ef2911d65a0a10d7e4261a97057aae32ba800d47ff2"\n    },\n    {\n      "id": "train_ep3_mars_2_d10_1",\n      "epoch": 3,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 47.900000000000006,\n      "start_s": 27.900000000000006,\n      "end_s": 67.9,\n      "n_native_gt_outage": 44,\n      "n_native_gt_total": 159,\n      "token_mask_sha256": "9f12542780490d612306b1de6383455d9c904bc5b73f8b9cebd020d927653a95",\n      "native_gt_timestamp_sha256": "d9b6feea078b9fc60e492cf683a0ecd8ef01f3970017699cd84ef3dc87d74be5"\n    },\n    {\n      "id": "train_ep3_mars_3_d30_1",\n      "epoch": 3,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 87.75,\n      "start_s": 67.75,\n      "end_s": 127.75,\n      "n_native_gt_outage": 117,\n      "n_native_gt_total": 237,\n      "token_mask_sha256": "ec337caf619f2fc34accfde4fefc63f5f100805a2dc4a30b788f58ddce2c4bc8",\n      "native_gt_timestamp_sha256": "324a561c1f1791284d9b0341a0773a061b7629e3dc4d6d178cd884d7da62b127"\n    },\n    {\n      "id": "train_ep3_mars_3_d10_0",\n      "epoch": 3,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 88.30000000000001,\n      "start_s": 68.30000000000001,\n      "end_s": 108.30000000000001,\n      "n_native_gt_outage": 33,\n      "n_native_gt_total": 168,\n      "token_mask_sha256": "54d1f8b7a9217c3e8206fec44aa90fc97fdb3e89370798f9369d21bcb530fdd3",\n      "native_gt_timestamp_sha256": "681d6beee152ea2ba5db49c80d4d5c6c4b787f4d5f73ce8d268befacf8e224b7"\n    },\n    {\n      "id": "train_ep3_mars_1_d60_0",\n      "epoch": 3,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 25.950000000000003,\n      "start_s": 5.950000000000003,\n      "end_s": 95.95,\n      "n_native_gt_outage": 248,\n      "n_native_gt_total": 379,\n      "token_mask_sha256": "c0c2ec8564a28a400ecdd4fe90d4d6b443981b414ff75cc3098c2fd7dc691a8b",\n      "native_gt_timestamp_sha256": "0fcf5a915dcecf559c3e49f03df9139a71191e14c9956fecf88691f1143cb307"\n    },\n    {\n      "id": "train_ep4_mars_2_d30_1",\n      "epoch": 4,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 103.4,\n      "start_s": 83.4,\n      "end_s": 143.4,\n      "n_native_gt_outage": 125,\n      "n_native_gt_total": 249,\n      "token_mask_sha256": "b34423e399ec53c009571b199c4b245f2f0d58ef83150e9bf0611fa732314343",\n      "native_gt_timestamp_sha256": "ecd583d5ea1c858e1e706c03c4972d31d638ded3bfaa45699709c12126e6a1ef"\n    },\n    {\n      "id": "train_ep4_mars_2_d10_1",\n      "epoch": 4,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 89.10000000000001,\n      "start_s": 69.10000000000001,\n      "end_s": 109.10000000000001,\n      "n_native_gt_outage": 52,\n      "n_native_gt_total": 165,\n      "token_mask_sha256": "9f12542780490d612306b1de6383455d9c904bc5b73f8b9cebd020d927653a95",\n      "native_gt_timestamp_sha256": "e394a99d3f8123d85a99285e7a2c23ae2cd068a96524162f4eaf1b20063d3836"\n    },\n    {\n      "id": "train_ep4_mars_3_d60_1",\n      "epoch": 4,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 68.7,\n      "start_s": 48.7,\n      "end_s": 138.7,\n      "n_native_gt_outage": 237,\n      "n_native_gt_total": 355,\n      "token_mask_sha256": "944469817cb1d463be9ba8e22d7c037bf6b358337512a5973f0eedd13fdd691e",\n      "native_gt_timestamp_sha256": "667d6a08654c556389d09660992e99155d929a6d198b23326a9996929ab5feec"\n    },\n    {\n      "id": "train_ep4_mars_3_d30_0",\n      "epoch": 4,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 36.4,\n      "start_s": 16.4,\n      "end_s": 76.4,\n      "n_native_gt_outage": 113,\n      "n_native_gt_total": 222,\n      "token_mask_sha256": "3dd74d234f7a60145505caa41ec0613e9d52eb9cbdc78e09d68e05c0d586d85a",\n      "native_gt_timestamp_sha256": "31e6a6145d5c4b4e4a8ba5d5dc6c56a3f9351f4eba4bfbb60c2fe76542e32ac7"\n    },\n    {\n      "id": "train_ep4_mars_2_d30_0",\n      "epoch": 4,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 70.8,\n      "start_s": 50.8,\n      "end_s": 110.8,\n      "n_native_gt_outage": 124,\n      "n_native_gt_total": 241,\n      "token_mask_sha256": "b34423e399ec53c009571b199c4b245f2f0d58ef83150e9bf0611fa732314343",\n      "native_gt_timestamp_sha256": "4d47d31c91afb7d7698a6e8d2c3bd3c4f107ef8646daeabd93d1022ddfbb445a"\n    },\n    {\n      "id": "train_ep4_mars_1_d10_1",\n      "epoch": 4,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 38.85,\n      "start_s": 18.85,\n      "end_s": 58.85,\n      "n_native_gt_outage": 48,\n      "n_native_gt_total": 163,\n      "token_mask_sha256": "54d1f8b7a9217c3e8206fec44aa90fc97fdb3e89370798f9369d21bcb530fdd3",\n      "native_gt_timestamp_sha256": "fa96b8eb29cdccb1f7aff89629fa4c7783a9f08d9d7225eaf8b4ed780ae61562"\n    },\n    {\n      "id": "train_ep4_mars_1_d30_1",\n      "epoch": 4,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 35.15,\n      "start_s": 15.149999999999999,\n      "end_s": 75.15,\n      "n_native_gt_outage": 112,\n      "n_native_gt_total": 249,\n      "token_mask_sha256": "3dd74d234f7a60145505caa41ec0613e9d52eb9cbdc78e09d68e05c0d586d85a",\n      "native_gt_timestamp_sha256": "130622e5a701504b032cbcb803348644d3110951497235a88be209c8f6fdedbf"\n    },\n    {\n      "id": "train_ep4_mars_2_d10_0",\n      "epoch": 4,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 65.95,\n      "start_s": 45.95,\n      "end_s": 85.95,\n      "n_native_gt_outage": 47,\n      "n_native_gt_total": 165,\n      "token_mask_sha256": "860d531b40010f5a1ce4a6d8d4c9b3f3cb34a9b01d0a5e57422ce1faf23b77a3",\n      "native_gt_timestamp_sha256": "7b06e0490f4ffc5665fcd2901b52808906d6a2ffb707bea5864be0539a747fb0"\n    },\n    {\n      "id": "train_ep4_mars_1_d10_0",\n      "epoch": 4,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 70.25,\n      "start_s": 50.25,\n      "end_s": 90.25,\n      "n_native_gt_outage": 43,\n      "n_native_gt_total": 158,\n      "token_mask_sha256": "54d1f8b7a9217c3e8206fec44aa90fc97fdb3e89370798f9369d21bcb530fdd3",\n      "native_gt_timestamp_sha256": "ea7f78458bca1895abef67d688345af11b1c157c7622e245cc489419c0c6767d"\n    },\n    {\n      "id": "train_ep4_mars_3_d60_0",\n      "epoch": 4,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 71.8,\n      "start_s": 51.8,\n      "end_s": 141.8,\n      "n_native_gt_outage": 234,\n      "n_native_gt_total": 364,\n      "token_mask_sha256": "c0c2ec8564a28a400ecdd4fe90d4d6b443981b414ff75cc3098c2fd7dc691a8b",\n      "native_gt_timestamp_sha256": "79c27b0a09cb045dfadbcbb508e3b7a8358879bfbe0d9ad23b3789416cdf06bf"\n    },\n    {\n      "id": "train_ep4_mars_1_d30_0",\n      "epoch": 4,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 38.85,\n      "start_s": 18.85,\n      "end_s": 78.85,\n      "n_native_gt_outage": 122,\n      "n_native_gt_total": 251,\n      "token_mask_sha256": "63d833f0b713bb4fb84f25c62b86deb8f26260f7a627284500d089600ceaa5cc",\n      "native_gt_timestamp_sha256": "dc0e7e41603cbd26786b96b3220974411cf7a713ff966d0911a396623694f239"\n    },\n    {\n      "id": "train_ep4_mars_1_d60_0",\n      "epoch": 4,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 23.75,\n      "start_s": 3.75,\n      "end_s": 93.75,\n      "n_native_gt_outage": 252,\n      "n_native_gt_total": 374,\n      "token_mask_sha256": "c0c2ec8564a28a400ecdd4fe90d4d6b443981b414ff75cc3098c2fd7dc691a8b",\n      "native_gt_timestamp_sha256": "98752201d03a1786fa3050f302e32a6e2b936b0e3e93665b6da36b85c435ab75"\n    },\n    {\n      "id": "train_ep4_mars_2_d60_1",\n      "epoch": 4,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 69.55,\n      "start_s": 49.55,\n      "end_s": 139.55,\n      "n_native_gt_outage": 253,\n      "n_native_gt_total": 375,\n      "token_mask_sha256": "c0c2ec8564a28a400ecdd4fe90d4d6b443981b414ff75cc3098c2fd7dc691a8b",\n      "native_gt_timestamp_sha256": "1d5f4216f1546a9f4e4588c23345947e0225303bba8adb7d9ae32f30df5fabba"\n    },\n    {\n      "id": "train_ep4_mars_1_d60_1",\n      "epoch": 4,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 23.200000000000003,\n      "start_s": 3.200000000000003,\n      "end_s": 93.2,\n      "n_native_gt_outage": 251,\n      "n_native_gt_total": 373,\n      "token_mask_sha256": "fed530af485dca69fb59f9b1a865da85e17ffd444c3c0d42d5f5017e0315ab5f",\n      "native_gt_timestamp_sha256": "d585fc160290d7dc3bd2dfb3b7609e4f6d85ceae060ce92b417caccd5974888b"\n    },\n    {\n      "id": "train_ep4_mars_3_d10_1",\n      "epoch": 4,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 107.5,\n      "start_s": 87.5,\n      "end_s": 127.5,\n      "n_native_gt_outage": 37,\n      "n_native_gt_total": 153,\n      "token_mask_sha256": "54d1f8b7a9217c3e8206fec44aa90fc97fdb3e89370798f9369d21bcb530fdd3",\n      "native_gt_timestamp_sha256": "d5371dd8da5546428fb0f1704507727919bbedcb23fdfca90c324da820c5ea83"\n    },\n    {\n      "id": "train_ep4_mars_3_d10_0",\n      "epoch": 4,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 92.25,\n      "start_s": 72.25,\n      "end_s": 112.25,\n      "n_native_gt_outage": 33,\n      "n_native_gt_total": 168,\n      "token_mask_sha256": "54d1f8b7a9217c3e8206fec44aa90fc97fdb3e89370798f9369d21bcb530fdd3",\n      "native_gt_timestamp_sha256": "ee6d7bb48c6ed4d67fdbe94a8c25cf4ef52a7c8605d9c22c347ee3c2fc3bc995"\n    },\n    {\n      "id": "train_ep4_mars_3_d30_1",\n      "epoch": 4,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 59.45,\n      "start_s": 39.45,\n      "end_s": 99.45,\n      "n_native_gt_outage": 119,\n      "n_native_gt_total": 233,\n      "token_mask_sha256": "b34423e399ec53c009571b199c4b245f2f0d58ef83150e9bf0611fa732314343",\n      "native_gt_timestamp_sha256": "ca00c997def57ec48ec3a0e9b3818b70f70699005cf1552607f4257d6f46fe88"\n    },\n    {\n      "id": "train_ep4_mars_2_d60_0",\n      "epoch": 4,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 41.550000000000004,\n      "start_s": 21.550000000000004,\n      "end_s": 111.55000000000001,\n      "n_native_gt_outage": 252,\n      "n_native_gt_total": 356,\n      "token_mask_sha256": "c0c2ec8564a28a400ecdd4fe90d4d6b443981b414ff75cc3098c2fd7dc691a8b",\n      "native_gt_timestamp_sha256": "1d72c48f16f7bca0d7c8677a2a5443324f9e758489ea3307796ce1078439e0d5"\n    },\n    {\n      "id": "train_ep5_mars_1_d30_1",\n      "epoch": 5,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 49.45,\n      "start_s": 29.450000000000003,\n      "end_s": 89.45,\n      "n_native_gt_outage": 120,\n      "n_native_gt_total": 239,\n      "token_mask_sha256": "63d833f0b713bb4fb84f25c62b86deb8f26260f7a627284500d089600ceaa5cc",\n      "native_gt_timestamp_sha256": "126bb356e69ac0dc6d48aed200731da82cdd53a3b55875883077ca8b99ad333d"\n    },\n    {\n      "id": "train_ep5_mars_2_d30_1",\n      "epoch": 5,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 22.400000000000002,\n      "start_s": 2.400000000000002,\n      "end_s": 62.400000000000006,\n      "n_native_gt_outage": 114,\n      "n_native_gt_total": 229,\n      "token_mask_sha256": "b34423e399ec53c009571b199c4b245f2f0d58ef83150e9bf0611fa732314343",\n      "native_gt_timestamp_sha256": "fcaa2e08190b90e471d0ad16c4666e19ff69a1351ca678959bcb3b88b0e9d294"\n    },\n    {\n      "id": "train_ep5_mars_2_d60_1",\n      "epoch": 5,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 35.7,\n      "start_s": 15.700000000000003,\n      "end_s": 105.7,\n      "n_native_gt_outage": 252,\n      "n_native_gt_total": 361,\n      "token_mask_sha256": "71da0041c5ffc52dc55131d6c02ef9fd40886bf07a3990e099fe3e7675dd9bab",\n      "native_gt_timestamp_sha256": "6be97febe8386bfdaac13b4ef23ac3fc657b9161417f45d5d7a63c90c02ae55f"\n    },\n    {\n      "id": "train_ep5_mars_1_d10_0",\n      "epoch": 5,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 33.550000000000004,\n      "start_s": 13.550000000000004,\n      "end_s": 53.550000000000004,\n      "n_native_gt_outage": 34,\n      "n_native_gt_total": 164,\n      "token_mask_sha256": "21b9e16f0abe800c55f145a18abd0a79a37df3cbda1ac658e7e89003a42e7536",\n      "native_gt_timestamp_sha256": "a0592506c7ed2f6faa8f60304620c58fce3d64b749bc29db6abc1d7045e51a15"\n    },\n    {\n      "id": "train_ep5_mars_1_d60_1",\n      "epoch": 5,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 24.5,\n      "start_s": 4.5,\n      "end_s": 94.5,\n      "n_native_gt_outage": 252,\n      "n_native_gt_total": 378,\n      "token_mask_sha256": "71da0041c5ffc52dc55131d6c02ef9fd40886bf07a3990e099fe3e7675dd9bab",\n      "native_gt_timestamp_sha256": "38f983129bf109a2a8bd9c00847697a2e8ac311795a77f54a3ea563434927b95"\n    },\n    {\n      "id": "train_ep5_mars_3_d60_0",\n      "epoch": 5,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 28.5,\n      "start_s": 8.5,\n      "end_s": 98.5,\n      "n_native_gt_outage": 237,\n      "n_native_gt_total": 355,\n      "token_mask_sha256": "944469817cb1d463be9ba8e22d7c037bf6b358337512a5973f0eedd13fdd691e",\n      "native_gt_timestamp_sha256": "1230fb6c58b192e1eaeb759d03037fc352d4b778d4bec6c165d4067831bc6ac3"\n    },\n    {\n      "id": "train_ep5_mars_1_d60_0",\n      "epoch": 5,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 25.55,\n      "start_s": 5.550000000000001,\n      "end_s": 95.55,\n      "n_native_gt_outage": 248,\n      "n_native_gt_total": 379,\n      "token_mask_sha256": "c0c2ec8564a28a400ecdd4fe90d4d6b443981b414ff75cc3098c2fd7dc691a8b",\n      "native_gt_timestamp_sha256": "3997f3d3c002cd4290baf8c40b3a1c517989498c806f36552b08a06e714d193a"\n    },\n    {\n      "id": "train_ep5_mars_3_d10_1",\n      "epoch": 5,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 89.25,\n      "start_s": 69.25,\n      "end_s": 109.25,\n      "n_native_gt_outage": 32,\n      "n_native_gt_total": 168,\n      "token_mask_sha256": "860d531b40010f5a1ce4a6d8d4c9b3f3cb34a9b01d0a5e57422ce1faf23b77a3",\n      "native_gt_timestamp_sha256": "1c6fe9bafad7cf25bfe9b06cfc22f0202eaee31599c24f159a8c6c0cf1663f0e"\n    },\n    {\n      "id": "train_ep5_mars_3_d10_0",\n      "epoch": 5,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 92.4,\n      "start_s": 72.4,\n      "end_s": 112.4,\n      "n_native_gt_outage": 32,\n      "n_native_gt_total": 168,\n      "token_mask_sha256": "ea945fe6cde424b8878a4a763925347368d146d2fe42a95bb983fd8eb5bdd4c3",\n      "native_gt_timestamp_sha256": "ec32bfc4d00a6f5ae2676aade3dea76c56b3d9b14956464a3999001376e5b7f9"\n    },\n    {\n      "id": "train_ep5_mars_2_d30_0",\n      "epoch": 5,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 77.65,\n      "start_s": 57.650000000000006,\n      "end_s": 117.65,\n      "n_native_gt_outage": 125,\n      "n_native_gt_total": 240,\n      "token_mask_sha256": "63d833f0b713bb4fb84f25c62b86deb8f26260f7a627284500d089600ceaa5cc",\n      "native_gt_timestamp_sha256": "752330610463871fece39b2a7e8138bac529cb008bb84080ccdf6f364c90a8de"\n    },\n    {\n      "id": "train_ep5_mars_1_d30_0",\n      "epoch": 5,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 58.7,\n      "start_s": 38.7,\n      "end_s": 98.7,\n      "n_native_gt_outage": 124,\n      "n_native_gt_total": 259,\n      "token_mask_sha256": "ec337caf619f2fc34accfde4fefc63f5f100805a2dc4a30b788f58ddce2c4bc8",\n      "native_gt_timestamp_sha256": "c647ca8bca5adc11e86452d47349aaf16e4fd1e3d6176fb2b6ad0574c19041e7"\n    },\n    {\n      "id": "train_ep5_mars_3_d30_1",\n      "epoch": 5,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 38.800000000000004,\n      "start_s": 18.800000000000004,\n      "end_s": 78.80000000000001,\n      "n_native_gt_outage": 111,\n      "n_native_gt_total": 230,\n      "token_mask_sha256": "b34423e399ec53c009571b199c4b245f2f0d58ef83150e9bf0611fa732314343",\n      "native_gt_timestamp_sha256": "1d5e75abc9017a461473221f5cc245cc00ae6ac3d57625f4e4d5fa90707fedcf"\n    },\n    {\n      "id": "train_ep5_mars_3_d30_0",\n      "epoch": 5,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 52.050000000000004,\n      "start_s": 32.050000000000004,\n      "end_s": 92.05000000000001,\n      "n_native_gt_outage": 121,\n      "n_native_gt_total": 236,\n      "token_mask_sha256": "63d833f0b713bb4fb84f25c62b86deb8f26260f7a627284500d089600ceaa5cc",\n      "native_gt_timestamp_sha256": "aca66175395fbb108dae8e2b62a6ac93573f21649331fe895a3cd7205b68f586"\n    },\n    {\n      "id": "train_ep5_mars_3_d60_1",\n      "epoch": 5,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 51.0,\n      "start_s": 31.0,\n      "end_s": 121.0,\n      "n_native_gt_outage": 238,\n      "n_native_gt_total": 357,\n      "token_mask_sha256": "c0c2ec8564a28a400ecdd4fe90d4d6b443981b414ff75cc3098c2fd7dc691a8b",\n      "native_gt_timestamp_sha256": "6d3e45bbbf81bacb88419555c89c6b97e5c525cee55a52c56cfc15e715827b03"\n    },\n    {\n      "id": "train_ep5_mars_2_d10_0",\n      "epoch": 5,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 99.55000000000001,\n      "start_s": 79.55000000000001,\n      "end_s": 119.55000000000001,\n      "n_native_gt_outage": 35,\n      "n_native_gt_total": 157,\n      "token_mask_sha256": "860d531b40010f5a1ce4a6d8d4c9b3f3cb34a9b01d0a5e57422ce1faf23b77a3",\n      "native_gt_timestamp_sha256": "3ca05e5670472e52545a4e58414a903e8967f4412806cc87fbda9666b890171f"\n    },\n    {\n      "id": "train_ep5_mars_2_d10_1",\n      "epoch": 5,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 127.25,\n      "start_s": 107.25,\n      "end_s": 147.25,\n      "n_native_gt_outage": 36,\n      "n_native_gt_total": 163,\n      "token_mask_sha256": "54d1f8b7a9217c3e8206fec44aa90fc97fdb3e89370798f9369d21bcb530fdd3",\n      "native_gt_timestamp_sha256": "3eb387bf644a933469998d5046305d9fa2b5fd8fef6afd2d1a9ccc7d18e1339b"\n    },\n    {\n      "id": "train_ep5_mars_1_d10_1",\n      "epoch": 5,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 40.400000000000006,\n      "start_s": 20.400000000000006,\n      "end_s": 60.400000000000006,\n      "n_native_gt_outage": 49,\n      "n_native_gt_total": 161,\n      "token_mask_sha256": "860d531b40010f5a1ce4a6d8d4c9b3f3cb34a9b01d0a5e57422ce1faf23b77a3",\n      "native_gt_timestamp_sha256": "04faed01450196e01a54bbae54c8d9c0d280bee6334d71a55cb562f6e5f7755c"\n    },\n    {\n      "id": "train_ep5_mars_2_d60_0",\n      "epoch": 5,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 50.300000000000004,\n      "start_s": 30.300000000000004,\n      "end_s": 120.30000000000001,\n      "n_native_gt_outage": 244,\n      "n_native_gt_total": 360,\n      "token_mask_sha256": "c0c2ec8564a28a400ecdd4fe90d4d6b443981b414ff75cc3098c2fd7dc691a8b",\n      "native_gt_timestamp_sha256": "d7cfcd2d444c57258efee9d3b457e391220f3f93c6bbe78289fbddd50a049170"\n    },\n    {\n      "id": "train_ep6_mars_1_d30_1",\n      "epoch": 6,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 49.650000000000006,\n      "start_s": 29.650000000000006,\n      "end_s": 89.65,\n      "n_native_gt_outage": 119,\n      "n_native_gt_total": 237,\n      "token_mask_sha256": "63d833f0b713bb4fb84f25c62b86deb8f26260f7a627284500d089600ceaa5cc",\n      "native_gt_timestamp_sha256": "3fa819c7b5111f2eb5d8a0985a5c6aa0ca33d304ea4019b9d31f69e050eb768b"\n    },\n    {\n      "id": "train_ep6_mars_2_d30_1",\n      "epoch": 6,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 100.30000000000001,\n      "start_s": 80.30000000000001,\n      "end_s": 140.3,\n      "n_native_gt_outage": 128,\n      "n_native_gt_total": 252,\n      "token_mask_sha256": "ec337caf619f2fc34accfde4fefc63f5f100805a2dc4a30b788f58ddce2c4bc8",\n      "native_gt_timestamp_sha256": "15727df8ca76aa90a898315959128dbe0a447b52a5618b5ce45187775403a310"\n    },\n    {\n      "id": "train_ep6_mars_2_d10_0",\n      "epoch": 6,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 82.30000000000001,\n      "start_s": 62.30000000000001,\n      "end_s": 102.30000000000001,\n      "n_native_gt_outage": 47,\n      "n_native_gt_total": 172,\n      "token_mask_sha256": "9f12542780490d612306b1de6383455d9c904bc5b73f8b9cebd020d927653a95",\n      "native_gt_timestamp_sha256": "48b85f6da59cfe9da080d0fcfcf3cc8f2ba6404aca3da77269646ab4c7fa01d9"\n    },\n    {\n      "id": "train_ep6_mars_3_d10_0",\n      "epoch": 6,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 43.75,\n      "start_s": 23.75,\n      "end_s": 63.75,\n      "n_native_gt_outage": 38,\n      "n_native_gt_total": 154,\n      "token_mask_sha256": "9f12542780490d612306b1de6383455d9c904bc5b73f8b9cebd020d927653a95",\n      "native_gt_timestamp_sha256": "f7c861322f46794dedf151097f9dc315b5b310a6639b151d16ba11bd07979eaa"\n    },\n    {\n      "id": "train_ep6_mars_3_d30_1",\n      "epoch": 6,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 45.5,\n      "start_s": 25.5,\n      "end_s": 85.5,\n      "n_native_gt_outage": 108,\n      "n_native_gt_total": 237,\n      "token_mask_sha256": "63d833f0b713bb4fb84f25c62b86deb8f26260f7a627284500d089600ceaa5cc",\n      "native_gt_timestamp_sha256": "c68f4931ee38f50387820984c79d4f88ac697f6876b5a75157b3c0672af82e27"\n    },\n    {\n      "id": "train_ep6_mars_1_d60_0",\n      "epoch": 6,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 25.05,\n      "start_s": 5.050000000000001,\n      "end_s": 95.05,\n      "n_native_gt_outage": 249,\n      "n_native_gt_total": 379,\n      "token_mask_sha256": "944469817cb1d463be9ba8e22d7c037bf6b358337512a5973f0eedd13fdd691e",\n      "native_gt_timestamp_sha256": "e03b2eb60b031f7dad8ad8abdfb5a5266990d411e8725c6838d367bfd1a9a488"\n    },\n    {\n      "id": "train_ep6_mars_2_d30_0",\n      "epoch": 6,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 120.60000000000001,\n      "start_s": 100.60000000000001,\n      "end_s": 160.60000000000002,\n      "n_native_gt_outage": 125,\n      "n_native_gt_total": 246,\n      "token_mask_sha256": "63d833f0b713bb4fb84f25c62b86deb8f26260f7a627284500d089600ceaa5cc",\n      "native_gt_timestamp_sha256": "aae74438549e23353a2c56884a6fd2e7089538fd5dded9cfabf7d1ac4fdfce5f"\n    },\n    {\n      "id": "train_ep6_mars_1_d30_0",\n      "epoch": 6,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 37.050000000000004,\n      "start_s": 17.050000000000004,\n      "end_s": 77.05000000000001,\n      "n_native_gt_outage": 116,\n      "n_native_gt_total": 252,\n      "token_mask_sha256": "63d833f0b713bb4fb84f25c62b86deb8f26260f7a627284500d089600ceaa5cc",\n      "native_gt_timestamp_sha256": "ddc46aaae68e11dc9ef96d60cbd042894b67fd440eb392fe5e104e086a637cbe"\n    },\n    {\n      "id": "train_ep6_mars_2_d60_0",\n      "epoch": 6,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 46.550000000000004,\n      "start_s": 26.550000000000004,\n      "end_s": 116.55000000000001,\n      "n_native_gt_outage": 254,\n      "n_native_gt_total": 356,\n      "token_mask_sha256": "c0c2ec8564a28a400ecdd4fe90d4d6b443981b414ff75cc3098c2fd7dc691a8b",\n      "native_gt_timestamp_sha256": "e45b4dcb6583f5496511824fa30de4e8c4c9caa3db5ffa89395dc32cff3bed0e"\n    },\n    {\n      "id": "train_ep6_mars_3_d30_0",\n      "epoch": 6,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 89.30000000000001,\n      "start_s": 69.30000000000001,\n      "end_s": 129.3,\n      "n_native_gt_outage": 116,\n      "n_native_gt_total": 234,\n      "token_mask_sha256": "63d833f0b713bb4fb84f25c62b86deb8f26260f7a627284500d089600ceaa5cc",\n      "native_gt_timestamp_sha256": "c25815307a389a5bb9a7f07b288e5224640bb37ee61aba40a9417b19f4998015"\n    },\n    {\n      "id": "train_ep6_mars_3_d60_0",\n      "epoch": 6,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 64.55,\n      "start_s": 44.55,\n      "end_s": 134.55,\n      "n_native_gt_outage": 236,\n      "n_native_gt_total": 345,\n      "token_mask_sha256": "71da0041c5ffc52dc55131d6c02ef9fd40886bf07a3990e099fe3e7675dd9bab",\n      "native_gt_timestamp_sha256": "0c3c18fc1b6acbcf05a3f80542e53f061c772967835f67919824f8e42fa2f8cf"\n    },\n    {\n      "id": "train_ep6_mars_2_d60_1",\n      "epoch": 6,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 24.05,\n      "start_s": 4.050000000000001,\n      "end_s": 94.05,\n      "n_native_gt_outage": 233,\n      "n_native_gt_total": 367,\n      "token_mask_sha256": "944469817cb1d463be9ba8e22d7c037bf6b358337512a5973f0eedd13fdd691e",\n      "native_gt_timestamp_sha256": "8865f8abb321f5f5d4652641c9dd9c5b69fe5504e947bfaaca95f8e3c7ab812d"\n    },\n    {\n      "id": "train_ep6_mars_1_d10_0",\n      "epoch": 6,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 29.35,\n      "start_s": 9.350000000000001,\n      "end_s": 49.35,\n      "n_native_gt_outage": 31,\n      "n_native_gt_total": 164,\n      "token_mask_sha256": "21b9e16f0abe800c55f145a18abd0a79a37df3cbda1ac658e7e89003a42e7536",\n      "native_gt_timestamp_sha256": "2dfe5c976ba7e6af2496effb0abf3d05c0eed4ab0ed646122d1932cbb8319061"\n    },\n    {\n      "id": "train_ep6_mars_2_d10_1",\n      "epoch": 6,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 75.5,\n      "start_s": 55.5,\n      "end_s": 95.5,\n      "n_native_gt_outage": 36,\n      "n_native_gt_total": 172,\n      "token_mask_sha256": "9f12542780490d612306b1de6383455d9c904bc5b73f8b9cebd020d927653a95",\n      "native_gt_timestamp_sha256": "efc64ec7ce618899b3de45701fd04832c7151fc46f0a76ea8aaafb3982343a6f"\n    },\n    {\n      "id": "train_ep6_mars_3_d10_1",\n      "epoch": 6,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 97.7,\n      "start_s": 77.7,\n      "end_s": 117.7,\n      "n_native_gt_outage": 49,\n      "n_native_gt_total": 165,\n      "token_mask_sha256": "54d1f8b7a9217c3e8206fec44aa90fc97fdb3e89370798f9369d21bcb530fdd3",\n      "native_gt_timestamp_sha256": "29534963dfdd5f42e3b2df67678c271c9f6bd8468fdd019f4e240d3122ac88e6"\n    },\n    {\n      "id": "train_ep6_mars_1_d10_1",\n      "epoch": 6,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 63.050000000000004,\n      "start_s": 43.050000000000004,\n      "end_s": 83.05000000000001,\n      "n_native_gt_outage": 56,\n      "n_native_gt_total": 170,\n      "token_mask_sha256": "54d1f8b7a9217c3e8206fec44aa90fc97fdb3e89370798f9369d21bcb530fdd3",\n      "native_gt_timestamp_sha256": "0055b63140378f667468d9eb75dcb4e85721f0837a718dd97492d39077930101"\n    },\n    {\n      "id": "train_ep6_mars_1_d60_1",\n      "epoch": 6,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 23.1,\n      "start_s": 3.1000000000000014,\n      "end_s": 93.1,\n      "n_native_gt_outage": 251,\n      "n_native_gt_total": 373,\n      "token_mask_sha256": "71da0041c5ffc52dc55131d6c02ef9fd40886bf07a3990e099fe3e7675dd9bab",\n      "native_gt_timestamp_sha256": "d585fc160290d7dc3bd2dfb3b7609e4f6d85ceae060ce92b417caccd5974888b"\n    },\n    {\n      "id": "train_ep6_mars_3_d60_1",\n      "epoch": 6,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 52.75,\n      "start_s": 32.75,\n      "end_s": 122.75,\n      "n_native_gt_outage": 240,\n      "n_native_gt_total": 352,\n      "token_mask_sha256": "c0c2ec8564a28a400ecdd4fe90d4d6b443981b414ff75cc3098c2fd7dc691a8b",\n      "native_gt_timestamp_sha256": "b6ed6e8b6f9a0e0759269986eafea04f902dbd1c5584fed195e246076d28ed6f"\n    },\n    {\n      "id": "train_ep7_mars_3_d60_0",\n      "epoch": 7,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 40.650000000000006,\n      "start_s": 20.650000000000006,\n      "end_s": 110.65,\n      "n_native_gt_outage": 233,\n      "n_native_gt_total": 356,\n      "token_mask_sha256": "fed530af485dca69fb59f9b1a865da85e17ffd444c3c0d42d5f5017e0315ab5f",\n      "native_gt_timestamp_sha256": "c2a300712228b005574c65aba92e5e49fded661a13233905f8043597153109b6"\n    },\n    {\n      "id": "train_ep7_mars_1_d30_0",\n      "epoch": 7,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 36.050000000000004,\n      "start_s": 16.050000000000004,\n      "end_s": 76.05000000000001,\n      "n_native_gt_outage": 114,\n      "n_native_gt_total": 250,\n      "token_mask_sha256": "63d833f0b713bb4fb84f25c62b86deb8f26260f7a627284500d089600ceaa5cc",\n      "native_gt_timestamp_sha256": "cefc241f53b507ade4fd46a6b2eb664f3af70447c8f3d5a1aa8216034b5fce22"\n    },\n    {\n      "id": "train_ep7_mars_3_d30_1",\n      "epoch": 7,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 67.2,\n      "start_s": 47.2,\n      "end_s": 107.2,\n      "n_native_gt_outage": 120,\n      "n_native_gt_total": 238,\n      "token_mask_sha256": "58551dc289b726e34524d3b1aa8536a6218d289b11cbfe5b61c49f96ebb802fd",\n      "native_gt_timestamp_sha256": "647b9fb6c3ee07f657186b6ee24b467df06b163d5122aba91657f3665a78fd1a"\n    },\n    {\n      "id": "train_ep7_mars_2_d10_1",\n      "epoch": 7,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 23.650000000000002,\n      "start_s": 3.650000000000002,\n      "end_s": 43.650000000000006,\n      "n_native_gt_outage": 33,\n      "n_native_gt_total": 146,\n      "token_mask_sha256": "54d1f8b7a9217c3e8206fec44aa90fc97fdb3e89370798f9369d21bcb530fdd3",\n      "native_gt_timestamp_sha256": "9077d2832919db1de695d9ed6bd46bce7c43b5adab1e9c77f421100f99465af2"\n    },\n    {\n      "id": "train_ep7_mars_3_d60_1",\n      "epoch": 7,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 50.95,\n      "start_s": 30.950000000000003,\n      "end_s": 120.95,\n      "n_native_gt_outage": 238,\n      "n_native_gt_total": 357,\n      "token_mask_sha256": "71da0041c5ffc52dc55131d6c02ef9fd40886bf07a3990e099fe3e7675dd9bab",\n      "native_gt_timestamp_sha256": "6d3e45bbbf81bacb88419555c89c6b97e5c525cee55a52c56cfc15e715827b03"\n    },\n    {\n      "id": "train_ep7_mars_1_d10_0",\n      "epoch": 7,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 69.3,\n      "start_s": 49.3,\n      "end_s": 89.3,\n      "n_native_gt_outage": 45,\n      "n_native_gt_total": 159,\n      "token_mask_sha256": "9f12542780490d612306b1de6383455d9c904bc5b73f8b9cebd020d927653a95",\n      "native_gt_timestamp_sha256": "de74eb0698b0b1cb4be3dc5f7fa64ae1d99711f7d4e868f1f21f93ba9e114ff7"\n    },\n    {\n      "id": "train_ep7_mars_2_d30_0",\n      "epoch": 7,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 87.45,\n      "start_s": 67.45,\n      "end_s": 127.45,\n      "n_native_gt_outage": 123,\n      "n_native_gt_total": 259,\n      "token_mask_sha256": "63d833f0b713bb4fb84f25c62b86deb8f26260f7a627284500d089600ceaa5cc",\n      "native_gt_timestamp_sha256": "e2213ae14a28234a93f4d1d667636d26b692cbce9faec4d0c3034e7204fb82fa"\n    },\n    {\n      "id": "train_ep7_mars_2_d10_0",\n      "epoch": 7,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 48.45,\n      "start_s": 28.450000000000003,\n      "end_s": 68.45,\n      "n_native_gt_outage": 45,\n      "n_native_gt_total": 160,\n      "token_mask_sha256": "54d1f8b7a9217c3e8206fec44aa90fc97fdb3e89370798f9369d21bcb530fdd3",\n      "native_gt_timestamp_sha256": "2c34264b07418fd3681738ca8d30b19bf9a70b9ffc8630a1a25dd39fa58d2cd4"\n    },\n    {\n      "id": "train_ep7_mars_2_d30_1",\n      "epoch": 7,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 119.60000000000001,\n      "start_s": 99.60000000000001,\n      "end_s": 159.60000000000002,\n      "n_native_gt_outage": 129,\n      "n_native_gt_total": 242,\n      "token_mask_sha256": "63d833f0b713bb4fb84f25c62b86deb8f26260f7a627284500d089600ceaa5cc",\n      "native_gt_timestamp_sha256": "2299d859b1bcf65d44247a5503f314c90725655f451bd27332ec19a6abc12979"\n    },\n    {\n      "id": "train_ep7_mars_1_d60_1",\n      "epoch": 7,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 27.6,\n      "start_s": 7.600000000000001,\n      "end_s": 97.6,\n      "n_native_gt_outage": 240,\n      "n_native_gt_total": 379,\n      "token_mask_sha256": "fed530af485dca69fb59f9b1a865da85e17ffd444c3c0d42d5f5017e0315ab5f",\n      "native_gt_timestamp_sha256": "c2c314f9ec15963205a251c6774f126948d60f1de5ced77754a4ed91ffb228f9"\n    },\n    {\n      "id": "train_ep7_mars_3_d10_1",\n      "epoch": 7,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 106.45,\n      "start_s": 86.45,\n      "end_s": 126.45,\n      "n_native_gt_outage": 38,\n      "n_native_gt_total": 153,\n      "token_mask_sha256": "860d531b40010f5a1ce4a6d8d4c9b3f3cb34a9b01d0a5e57422ce1faf23b77a3",\n      "native_gt_timestamp_sha256": "f6aa3c77830ad72f110a4cf8174bbd1e52e88f9117f5b1f65862ac0104517af3"\n    },\n    {\n      "id": "train_ep7_mars_3_d30_0",\n      "epoch": 7,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 28.5,\n      "start_s": 8.5,\n      "end_s": 68.5,\n      "n_native_gt_outage": 118,\n      "n_native_gt_total": 236,\n      "token_mask_sha256": "63d833f0b713bb4fb84f25c62b86deb8f26260f7a627284500d089600ceaa5cc",\n      "native_gt_timestamp_sha256": "4eb716f51e0da85861d7413426baee0fc0a6113ba081ec8c021a1f2dcf3a1833"\n    },\n    {\n      "id": "train_ep7_mars_1_d60_0",\n      "epoch": 7,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 24.85,\n      "start_s": 4.850000000000001,\n      "end_s": 94.85,\n      "n_native_gt_outage": 250,\n      "n_native_gt_total": 379,\n      "token_mask_sha256": "944469817cb1d463be9ba8e22d7c037bf6b358337512a5973f0eedd13fdd691e",\n      "native_gt_timestamp_sha256": "5b1516e732717cf09eec260662f8c92d7f8901fbe1a5f720fe19efffb6a71261"\n    },\n    {\n      "id": "train_ep7_mars_3_d10_0",\n      "epoch": 7,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 106.30000000000001,\n      "start_s": 86.30000000000001,\n      "end_s": 126.30000000000001,\n      "n_native_gt_outage": 40,\n      "n_native_gt_total": 152,\n      "token_mask_sha256": "54d1f8b7a9217c3e8206fec44aa90fc97fdb3e89370798f9369d21bcb530fdd3",\n      "native_gt_timestamp_sha256": "98e483e0339b6a2e62dbba4e5a4705ab0c88cb8959bb1036b933eb234454de40"\n    },\n    {\n      "id": "train_ep7_mars_2_d60_0",\n      "epoch": 7,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 48.800000000000004,\n      "start_s": 28.800000000000004,\n      "end_s": 118.80000000000001,\n      "n_native_gt_outage": 251,\n      "n_native_gt_total": 358,\n      "token_mask_sha256": "944469817cb1d463be9ba8e22d7c037bf6b358337512a5973f0eedd13fdd691e",\n      "native_gt_timestamp_sha256": "f8a8426a4606a72d2b5e033640daf636c4f33e59a5624785b71b21537589bfb2"\n    },\n    {\n      "id": "train_ep7_mars_2_d60_1",\n      "epoch": 7,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 25.200000000000003,\n      "start_s": 5.200000000000003,\n      "end_s": 95.2,\n      "n_native_gt_outage": 235,\n      "n_native_gt_total": 367,\n      "token_mask_sha256": "fed530af485dca69fb59f9b1a865da85e17ffd444c3c0d42d5f5017e0315ab5f",\n      "native_gt_timestamp_sha256": "95717a9023c77e36b2687956fe3a71aaaef050c0b6bbf4ae0ab2eb44d798574d"\n    },\n    {\n      "id": "train_ep7_mars_1_d30_1",\n      "epoch": 7,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 51.75,\n      "start_s": 31.75,\n      "end_s": 91.75,\n      "n_native_gt_outage": 121,\n      "n_native_gt_total": 238,\n      "token_mask_sha256": "3dd74d234f7a60145505caa41ec0613e9d52eb9cbdc78e09d68e05c0d586d85a",\n      "native_gt_timestamp_sha256": "47e02129d2512fd6bede88f36187a5a83207047ac7e4f34c9da5ce64e3e1333a"\n    },\n    {\n      "id": "train_ep7_mars_1_d10_1",\n      "epoch": 7,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 24.25,\n      "start_s": 4.25,\n      "end_s": 44.25,\n      "n_native_gt_outage": 50,\n      "n_native_gt_total": 164,\n      "token_mask_sha256": "54d1f8b7a9217c3e8206fec44aa90fc97fdb3e89370798f9369d21bcb530fdd3",\n      "native_gt_timestamp_sha256": "04f5682ab0517867dc4c39a20cd7d9f47dfd7f4389c5f8e2ff9b1acae49641d0"\n    },\n    {\n      "id": "train_ep8_mars_2_d10_0",\n      "epoch": 8,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 52.5,\n      "start_s": 32.5,\n      "end_s": 72.5,\n      "n_native_gt_outage": 34,\n      "n_native_gt_total": 167,\n      "token_mask_sha256": "9f12542780490d612306b1de6383455d9c904bc5b73f8b9cebd020d927653a95",\n      "native_gt_timestamp_sha256": "d999ab685f373cca688a943a17cf75797f920d4488a4d66c3dc209176935c595"\n    },\n    {\n      "id": "train_ep8_mars_2_d60_1",\n      "epoch": 8,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 26.55,\n      "start_s": 6.550000000000001,\n      "end_s": 96.55,\n      "n_native_gt_outage": 235,\n      "n_native_gt_total": 363,\n      "token_mask_sha256": "c0c2ec8564a28a400ecdd4fe90d4d6b443981b414ff75cc3098c2fd7dc691a8b",\n      "native_gt_timestamp_sha256": "cbb2488dac8463ca3b4628491f71a08a46d0f0251e15895aeb0935ec8c107389"\n    },\n    {\n      "id": "train_ep8_mars_1_d60_1",\n      "epoch": 8,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 25.700000000000003,\n      "start_s": 5.700000000000003,\n      "end_s": 95.7,\n      "n_native_gt_outage": 248,\n      "n_native_gt_total": 378,\n      "token_mask_sha256": "71da0041c5ffc52dc55131d6c02ef9fd40886bf07a3990e099fe3e7675dd9bab",\n      "native_gt_timestamp_sha256": "a02cf94feb3226f4048f59afd4f904fa8d7b33895f52c0fbe12e77a26a2c3d98"\n    },\n    {\n      "id": "train_ep8_mars_2_d30_1",\n      "epoch": 8,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 83.05000000000001,\n      "start_s": 63.05000000000001,\n      "end_s": 123.05000000000001,\n      "n_native_gt_outage": 122,\n      "n_native_gt_total": 249,\n      "token_mask_sha256": "ec337caf619f2fc34accfde4fefc63f5f100805a2dc4a30b788f58ddce2c4bc8",\n      "native_gt_timestamp_sha256": "260a617d26dd9e657897dbb9095da161a79c7fd098c62ed151b74233678ba79d"\n    },\n    {\n      "id": "train_ep8_mars_3_d10_0",\n      "epoch": 8,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 90.15,\n      "start_s": 70.15,\n      "end_s": 110.15,\n      "n_native_gt_outage": 32,\n      "n_native_gt_total": 168,\n      "token_mask_sha256": "9f12542780490d612306b1de6383455d9c904bc5b73f8b9cebd020d927653a95",\n      "native_gt_timestamp_sha256": "4fc13b78b56e720df3b88dd0f123aa8a018793f723df147effa716accb18eb24"\n    },\n    {\n      "id": "train_ep8_mars_3_d30_1",\n      "epoch": 8,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 60.300000000000004,\n      "start_s": 40.300000000000004,\n      "end_s": 100.30000000000001,\n      "n_native_gt_outage": 120,\n      "n_native_gt_total": 231,\n      "token_mask_sha256": "63d833f0b713bb4fb84f25c62b86deb8f26260f7a627284500d089600ceaa5cc",\n      "native_gt_timestamp_sha256": "6ca377d07f5050cc139f4c2486102e4ef21978801a9a65cc471baf6f8fd2cf85"\n    },\n    {\n      "id": "train_ep8_mars_3_d60_0",\n      "epoch": 8,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 65.60000000000001,\n      "start_s": 45.60000000000001,\n      "end_s": 135.60000000000002,\n      "n_native_gt_outage": 235,\n      "n_native_gt_total": 349,\n      "token_mask_sha256": "c0c2ec8564a28a400ecdd4fe90d4d6b443981b414ff75cc3098c2fd7dc691a8b",\n      "native_gt_timestamp_sha256": "90e0f8a779041e8c7b31e67f18b28a01d3ea1da403c8812ded6a7dbc104ba6c8"\n    },\n    {\n      "id": "train_ep8_mars_1_d30_0",\n      "epoch": 8,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 45.650000000000006,\n      "start_s": 25.650000000000006,\n      "end_s": 85.65,\n      "n_native_gt_outage": 132,\n      "n_native_gt_total": 248,\n      "token_mask_sha256": "63d833f0b713bb4fb84f25c62b86deb8f26260f7a627284500d089600ceaa5cc",\n      "native_gt_timestamp_sha256": "c224cd9865f4dc7fb82d80e533e411d72e3f92a594f548ddd0246079aa9348b4"\n    },\n    {\n      "id": "train_ep8_mars_3_d30_0",\n      "epoch": 8,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 88.10000000000001,\n      "start_s": 68.10000000000001,\n      "end_s": 128.10000000000002,\n      "n_native_gt_outage": 117,\n      "n_native_gt_total": 235,\n      "token_mask_sha256": "63d833f0b713bb4fb84f25c62b86deb8f26260f7a627284500d089600ceaa5cc",\n      "native_gt_timestamp_sha256": "deddf041f887057d3d00a6fc1353e1b2c79009dd3f834c021f28149e1e6ffce3"\n    },\n    {\n      "id": "train_ep8_mars_3_d60_1",\n      "epoch": 8,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 53.2,\n      "start_s": 33.2,\n      "end_s": 123.2,\n      "n_native_gt_outage": 238,\n      "n_native_gt_total": 352,\n      "token_mask_sha256": "478ce3b3d3dce9522789ea95341f5ae713e75348c57aff32b390220cd55f9144",\n      "native_gt_timestamp_sha256": "be284ff5a564611137e64f67d7f8feaac81f80b2f2ebc582caf7b6c34d5285c6"\n    },\n    {\n      "id": "train_ep8_mars_2_d10_1",\n      "epoch": 8,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 77.5,\n      "start_s": 57.5,\n      "end_s": 97.5,\n      "n_native_gt_outage": 34,\n      "n_native_gt_total": 173,\n      "token_mask_sha256": "9f12542780490d612306b1de6383455d9c904bc5b73f8b9cebd020d927653a95",\n      "native_gt_timestamp_sha256": "6977d58ede9a0d9d37d1b40d3c728a460654b272c2be41bf567a6b5b1b8bd7cc"\n    },\n    {\n      "id": "train_ep8_mars_2_d30_0",\n      "epoch": 8,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 41.900000000000006,\n      "start_s": 21.900000000000006,\n      "end_s": 81.9,\n      "n_native_gt_outage": 129,\n      "n_native_gt_total": 232,\n      "token_mask_sha256": "3dd74d234f7a60145505caa41ec0613e9d52eb9cbdc78e09d68e05c0d586d85a",\n      "native_gt_timestamp_sha256": "efa41ae462fd4a11a55c5615f93e42ba243d6216bc8cf5e977bc6d8d42f6545e"\n    },\n    {\n      "id": "train_ep8_mars_1_d60_0",\n      "epoch": 8,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 29.05,\n      "start_s": 9.05,\n      "end_s": 99.05,\n      "n_native_gt_outage": 239,\n      "n_native_gt_total": 376,\n      "token_mask_sha256": "944469817cb1d463be9ba8e22d7c037bf6b358337512a5973f0eedd13fdd691e",\n      "native_gt_timestamp_sha256": "61e00a2c5f4803a407ff148605c128735d485a8d2935c5d9acd8348d0b635b61"\n    },\n    {\n      "id": "train_ep8_mars_2_d60_0",\n      "epoch": 8,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 22.5,\n      "start_s": 2.5,\n      "end_s": 92.5,\n      "n_native_gt_outage": 231,\n      "n_native_gt_total": 361,\n      "token_mask_sha256": "71da0041c5ffc52dc55131d6c02ef9fd40886bf07a3990e099fe3e7675dd9bab",\n      "native_gt_timestamp_sha256": "95b44d81f9d0e5653e17322e92516ba03851ba1c5c471cfffe8f839070174ba6"\n    },\n    {\n      "id": "train_ep8_mars_1_d10_1",\n      "epoch": 8,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 47.6,\n      "start_s": 27.6,\n      "end_s": 67.6,\n      "n_native_gt_outage": 37,\n      "n_native_gt_total": 153,\n      "token_mask_sha256": "860d531b40010f5a1ce4a6d8d4c9b3f3cb34a9b01d0a5e57422ce1faf23b77a3",\n      "native_gt_timestamp_sha256": "53f26db65670b2cebc2493142d218624e82cbe4b767545bf9c456aaf70f41fea"\n    },\n    {\n      "id": "train_ep8_mars_1_d10_0",\n      "epoch": 8,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 54.900000000000006,\n      "start_s": 34.900000000000006,\n      "end_s": 74.9,\n      "n_native_gt_outage": 34,\n      "n_native_gt_total": 168,\n      "token_mask_sha256": "9f12542780490d612306b1de6383455d9c904bc5b73f8b9cebd020d927653a95",\n      "native_gt_timestamp_sha256": "560a6a7a725897fc526c3d6536772bb63ab5765a153ff2f3e040d064788c5f92"\n    },\n    {\n      "id": "train_ep8_mars_1_d30_1",\n      "epoch": 8,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 54.25,\n      "start_s": 34.25,\n      "end_s": 94.25,\n      "n_native_gt_outage": 122,\n      "n_native_gt_total": 245,\n      "token_mask_sha256": "63d833f0b713bb4fb84f25c62b86deb8f26260f7a627284500d089600ceaa5cc",\n      "native_gt_timestamp_sha256": "a521c13db2b821303f922f580be1ccf69188cab54034852ba19bcb0ee685e129"\n    },\n    {\n      "id": "train_ep8_mars_3_d10_1",\n      "epoch": 8,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 92.0,\n      "start_s": 72.0,\n      "end_s": 112.0,\n      "n_native_gt_outage": 34,\n      "n_native_gt_total": 168,\n      "token_mask_sha256": "860d531b40010f5a1ce4a6d8d4c9b3f3cb34a9b01d0a5e57422ce1faf23b77a3",\n      "native_gt_timestamp_sha256": "41cfcc2844dd1787b0f902a4520dcda8988a7b9ff8e9e10dd36b6e87312968f7"\n    },\n    {\n      "id": "train_ep9_mars_2_d60_1",\n      "epoch": 9,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 79.65,\n      "start_s": 59.650000000000006,\n      "end_s": 149.65,\n      "n_native_gt_outage": 250,\n      "n_native_gt_total": 370,\n      "token_mask_sha256": "71da0041c5ffc52dc55131d6c02ef9fd40886bf07a3990e099fe3e7675dd9bab",\n      "native_gt_timestamp_sha256": "77a2766aa54b6e4440c1a9d8010225d2ea6a18451f3103f88d4f5ab7bdc30d30"\n    },\n    {\n      "id": "train_ep9_mars_1_d30_0",\n      "epoch": 9,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 44.400000000000006,\n      "start_s": 24.400000000000006,\n      "end_s": 84.4,\n      "n_native_gt_outage": 135,\n      "n_native_gt_total": 252,\n      "token_mask_sha256": "b34423e399ec53c009571b199c4b245f2f0d58ef83150e9bf0611fa732314343",\n      "native_gt_timestamp_sha256": "50f609d350048b63634fb00d66ac4b4079cbb2def42e7acf0b2a764bff35fc6e"\n    },\n    {\n      "id": "train_ep9_mars_3_d30_1",\n      "epoch": 9,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 93.35000000000001,\n      "start_s": 73.35000000000001,\n      "end_s": 133.35000000000002,\n      "n_native_gt_outage": 118,\n      "n_native_gt_total": 237,\n      "token_mask_sha256": "ec337caf619f2fc34accfde4fefc63f5f100805a2dc4a30b788f58ddce2c4bc8",\n      "native_gt_timestamp_sha256": "917004e5d6aba3bfde3e840d6bcfc4aedb455c7358ef9f57b9ca89f5e3f8f5a7"\n    },\n    {\n      "id": "train_ep9_mars_3_d10_1",\n      "epoch": 9,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 91.10000000000001,\n      "start_s": 71.10000000000001,\n      "end_s": 111.10000000000001,\n      "n_native_gt_outage": 33,\n      "n_native_gt_total": 167,\n      "token_mask_sha256": "54d1f8b7a9217c3e8206fec44aa90fc97fdb3e89370798f9369d21bcb530fdd3",\n      "native_gt_timestamp_sha256": "0c3035c3ea9a552ee1c06ab0d3a162274cc398a07270f029977878f72ac8c4fb"\n    },\n    {\n      "id": "train_ep9_mars_3_d60_0",\n      "epoch": 9,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 70.2,\n      "start_s": 50.2,\n      "end_s": 140.2,\n      "n_native_gt_outage": 235,\n      "n_native_gt_total": 362,\n      "token_mask_sha256": "c0c2ec8564a28a400ecdd4fe90d4d6b443981b414ff75cc3098c2fd7dc691a8b",\n      "native_gt_timestamp_sha256": "b0928685124bef62003407a8768dceb7aaede8839eb3f3f29b0d72659c0dfb77"\n    },\n    {\n      "id": "train_ep9_mars_1_d60_0",\n      "epoch": 9,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 27.6,\n      "start_s": 7.600000000000001,\n      "end_s": 97.6,\n      "n_native_gt_outage": 240,\n      "n_native_gt_total": 379,\n      "token_mask_sha256": "fed530af485dca69fb59f9b1a865da85e17ffd444c3c0d42d5f5017e0315ab5f",\n      "native_gt_timestamp_sha256": "c2c314f9ec15963205a251c6774f126948d60f1de5ced77754a4ed91ffb228f9"\n    },\n    {\n      "id": "train_ep9_mars_1_d10_1",\n      "epoch": 9,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 54.0,\n      "start_s": 34.0,\n      "end_s": 74.0,\n      "n_native_gt_outage": 32,\n      "n_native_gt_total": 167,\n      "token_mask_sha256": "860d531b40010f5a1ce4a6d8d4c9b3f3cb34a9b01d0a5e57422ce1faf23b77a3",\n      "native_gt_timestamp_sha256": "9c1f7713875d505e89e59821f78b1ecf996d492979cb90359f1d4947e228488a"\n    },\n    {\n      "id": "train_ep9_mars_3_d30_0",\n      "epoch": 9,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 39.150000000000006,\n      "start_s": 19.150000000000006,\n      "end_s": 79.15,\n      "n_native_gt_outage": 114,\n      "n_native_gt_total": 229,\n      "token_mask_sha256": "ec337caf619f2fc34accfde4fefc63f5f100805a2dc4a30b788f58ddce2c4bc8",\n      "native_gt_timestamp_sha256": "5966c2fa8c844256e20b6e81bd1038e53c9af3469a4984a76c2c59a4f3e3f85f"\n    },\n    {\n      "id": "train_ep9_mars_2_d60_0",\n      "epoch": 9,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 38.800000000000004,\n      "start_s": 18.800000000000004,\n      "end_s": 108.80000000000001,\n      "n_native_gt_outage": 253,\n      "n_native_gt_total": 360,\n      "token_mask_sha256": "fed530af485dca69fb59f9b1a865da85e17ffd444c3c0d42d5f5017e0315ab5f",\n      "native_gt_timestamp_sha256": "495e76b4f72593687e7c6c4c1d4ecaa1ef62697e81e1602cb3baeaeb0cc23811"\n    },\n    {\n      "id": "train_ep9_mars_1_d60_1",\n      "epoch": 9,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 24.950000000000003,\n      "start_s": 4.950000000000003,\n      "end_s": 94.95,\n      "n_native_gt_outage": 250,\n      "n_native_gt_total": 379,\n      "token_mask_sha256": "c0c2ec8564a28a400ecdd4fe90d4d6b443981b414ff75cc3098c2fd7dc691a8b",\n      "native_gt_timestamp_sha256": "5b1516e732717cf09eec260662f8c92d7f8901fbe1a5f720fe19efffb6a71261"\n    },\n    {\n      "id": "train_ep9_mars_3_d60_1",\n      "epoch": 9,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 70.75,\n      "start_s": 50.75,\n      "end_s": 140.75,\n      "n_native_gt_outage": 236,\n      "n_native_gt_total": 362,\n      "token_mask_sha256": "71da0041c5ffc52dc55131d6c02ef9fd40886bf07a3990e099fe3e7675dd9bab",\n      "native_gt_timestamp_sha256": "4ec4884660ba64565bfa7d3cd8abc7ced3285883d1305da36d9e2f32a59f347a"\n    },\n    {\n      "id": "train_ep9_mars_2_d10_1",\n      "epoch": 9,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 23.25,\n      "start_s": 3.25,\n      "end_s": 43.25,\n      "n_native_gt_outage": 31,\n      "n_native_gt_total": 146,\n      "token_mask_sha256": "54d1f8b7a9217c3e8206fec44aa90fc97fdb3e89370798f9369d21bcb530fdd3",\n      "native_gt_timestamp_sha256": "9077d2832919db1de695d9ed6bd46bce7c43b5adab1e9c77f421100f99465af2"\n    },\n    {\n      "id": "train_ep9_mars_3_d10_0",\n      "epoch": 9,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 36.35,\n      "start_s": 16.35,\n      "end_s": 56.35,\n      "n_native_gt_outage": 41,\n      "n_native_gt_total": 154,\n      "token_mask_sha256": "21b9e16f0abe800c55f145a18abd0a79a37df3cbda1ac658e7e89003a42e7536",\n      "native_gt_timestamp_sha256": "75550ed559fc9228303dff4a6a15e7cf4babee7623ebaeba4829f8268b09fdbe"\n    },\n    {\n      "id": "train_ep9_mars_2_d30_1",\n      "epoch": 9,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 63.300000000000004,\n      "start_s": 43.300000000000004,\n      "end_s": 103.30000000000001,\n      "n_native_gt_outage": 132,\n      "n_native_gt_total": 255,\n      "token_mask_sha256": "2100feebce9686fec0ee7cc1b5a2e4958b681c407208eec9e7526b6ac135655b",\n      "native_gt_timestamp_sha256": "b637756eaffab3ca23932cb931db0a1ddca8c91ebaea5e5e9220e1dcfd2dd94d"\n    },\n    {\n      "id": "train_ep9_mars_2_d30_0",\n      "epoch": 9,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 43.550000000000004,\n      "start_s": 23.550000000000004,\n      "end_s": 83.55000000000001,\n      "n_native_gt_outage": 130,\n      "n_native_gt_total": 231,\n      "token_mask_sha256": "b34423e399ec53c009571b199c4b245f2f0d58ef83150e9bf0611fa732314343",\n      "native_gt_timestamp_sha256": "b77cca3837a87aa0c95bbe85c7198e260596359b994bcce67d68106f3644d0f9"\n    },\n    {\n      "id": "train_ep9_mars_1_d10_0",\n      "epoch": 9,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 51.75,\n      "start_s": 31.75,\n      "end_s": 71.75,\n      "n_native_gt_outage": 31,\n      "n_native_gt_total": 165,\n      "token_mask_sha256": "ea945fe6cde424b8878a4a763925347368d146d2fe42a95bb983fd8eb5bdd4c3",\n      "native_gt_timestamp_sha256": "26ae6436e235c86d99627c612093b8d5fd842ecbfc507d5df480de991c3a38ba"\n    },\n    {\n      "id": "train_ep9_mars_1_d30_1",\n      "epoch": 9,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 38.0,\n      "start_s": 18.0,\n      "end_s": 78.0,\n      "n_native_gt_outage": 120,\n      "n_native_gt_total": 252,\n      "token_mask_sha256": "b34423e399ec53c009571b199c4b245f2f0d58ef83150e9bf0611fa732314343",\n      "native_gt_timestamp_sha256": "daa14dea3de0a8a622f7ddebf3ee4c15bb7df1b6b228d5a3089e697ca824d0f2"\n    },\n    {\n      "id": "train_ep9_mars_2_d10_0",\n      "epoch": 9,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 106.75,\n      "start_s": 86.75,\n      "end_s": 126.75,\n      "n_native_gt_outage": 32,\n      "n_native_gt_total": 181,\n      "token_mask_sha256": "21b9e16f0abe800c55f145a18abd0a79a37df3cbda1ac658e7e89003a42e7536",\n      "native_gt_timestamp_sha256": "de36a4e91025a68ee7f10e545ded9337600c8660a9c528b917a31e8ba7d84918"\n    },\n    {\n      "id": "train_ep10_mars_3_d30_0",\n      "epoch": 10,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 41.2,\n      "start_s": 21.200000000000003,\n      "end_s": 81.2,\n      "n_native_gt_outage": 113,\n      "n_native_gt_total": 238,\n      "token_mask_sha256": "58551dc289b726e34524d3b1aa8536a6218d289b11cbfe5b61c49f96ebb802fd",\n      "native_gt_timestamp_sha256": "ba90a983c83360e81f3119da9a70754417405b745a118262f28cabf765011374"\n    },\n    {\n      "id": "train_ep10_mars_3_d10_1",\n      "epoch": 10,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 34.45,\n      "start_s": 14.450000000000003,\n      "end_s": 54.45,\n      "n_native_gt_outage": 40,\n      "n_native_gt_total": 153,\n      "token_mask_sha256": "54d1f8b7a9217c3e8206fec44aa90fc97fdb3e89370798f9369d21bcb530fdd3",\n      "native_gt_timestamp_sha256": "700545fe55ecafc98763ab83aa6adb92f1eec2879ac7c29591da718efaee6f1c"\n    },\n    {\n      "id": "train_ep10_mars_2_d10_0",\n      "epoch": 10,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 97.9,\n      "start_s": 77.9,\n      "end_s": 117.9,\n      "n_native_gt_outage": 37,\n      "n_native_gt_total": 158,\n      "token_mask_sha256": "9f12542780490d612306b1de6383455d9c904bc5b73f8b9cebd020d927653a95",\n      "native_gt_timestamp_sha256": "a6d4c08a5dac958a334f8ff9f3929db9d17c7c735d47522f1e8da15085666a34"\n    },\n    {\n      "id": "train_ep10_mars_3_d60_1",\n      "epoch": 10,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 45.75,\n      "start_s": 25.75,\n      "end_s": 115.75,\n      "n_native_gt_outage": 236,\n      "n_native_gt_total": 355,\n      "token_mask_sha256": "71da0041c5ffc52dc55131d6c02ef9fd40886bf07a3990e099fe3e7675dd9bab",\n      "native_gt_timestamp_sha256": "ac39429927d2c6c180ed9db7cb5c1a6aab99f8ed2c3a3021d2fc708b88f89282"\n    },\n    {\n      "id": "train_ep10_mars_1_d60_1",\n      "epoch": 10,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 24.25,\n      "start_s": 4.25,\n      "end_s": 94.25,\n      "n_native_gt_outage": 252,\n      "n_native_gt_total": 377,\n      "token_mask_sha256": "944469817cb1d463be9ba8e22d7c037bf6b358337512a5973f0eedd13fdd691e",\n      "native_gt_timestamp_sha256": "68b58e23d83bf183de7c8764cf307d534364ebfbfae692e3dddb2412d81e82fb"\n    },\n    {\n      "id": "train_ep10_mars_2_d30_1",\n      "epoch": 10,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 72.55,\n      "start_s": 52.55,\n      "end_s": 112.55,\n      "n_native_gt_outage": 123,\n      "n_native_gt_total": 242,\n      "token_mask_sha256": "3dd74d234f7a60145505caa41ec0613e9d52eb9cbdc78e09d68e05c0d586d85a",\n      "native_gt_timestamp_sha256": "307407c18ef0982a079432234ff67f6b10d3d37905e6bf8fc6a515f77a853fab"\n    },\n    {\n      "id": "train_ep10_mars_3_d30_1",\n      "epoch": 10,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 49.300000000000004,\n      "start_s": 29.300000000000004,\n      "end_s": 89.30000000000001,\n      "n_native_gt_outage": 113,\n      "n_native_gt_total": 238,\n      "token_mask_sha256": "63d833f0b713bb4fb84f25c62b86deb8f26260f7a627284500d089600ceaa5cc",\n      "native_gt_timestamp_sha256": "3a96113b493f89a20e1386b580ed42899ff3fa510bab3d38b03e9db086e9b995"\n    },\n    {\n      "id": "train_ep10_mars_1_d60_0",\n      "epoch": 10,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 29.25,\n      "start_s": 9.25,\n      "end_s": 99.25,\n      "n_native_gt_outage": 240,\n      "n_native_gt_total": 375,\n      "token_mask_sha256": "944469817cb1d463be9ba8e22d7c037bf6b358337512a5973f0eedd13fdd691e",\n      "native_gt_timestamp_sha256": "3eec85600a74ea72993dfd710e238e88a0993cd52cecdbc2cf2c23daaca23f7a"\n    },\n    {\n      "id": "train_ep10_mars_2_d10_1",\n      "epoch": 10,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 21.900000000000002,\n      "start_s": 1.9000000000000021,\n      "end_s": 41.900000000000006,\n      "n_native_gt_outage": 31,\n      "n_native_gt_total": 149,\n      "token_mask_sha256": "9f12542780490d612306b1de6383455d9c904bc5b73f8b9cebd020d927653a95",\n      "native_gt_timestamp_sha256": "90a0aff1823006eed529f18ca81f16410a9d738b5cefe2a9c65f7c4a0114ceef"\n    },\n    {\n      "id": "train_ep10_mars_2_d60_0",\n      "epoch": 10,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 56.1,\n      "start_s": 36.1,\n      "end_s": 126.1,\n      "n_native_gt_outage": 241,\n      "n_native_gt_total": 376,\n      "token_mask_sha256": "71da0041c5ffc52dc55131d6c02ef9fd40886bf07a3990e099fe3e7675dd9bab",\n      "native_gt_timestamp_sha256": "f46cb770092beeb9d0638e5b55184481d72523dd1f7223793d23ef9599e54dd1"\n    },\n    {\n      "id": "train_ep10_mars_1_d10_0",\n      "epoch": 10,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 28.55,\n      "start_s": 8.55,\n      "end_s": 48.55,\n      "n_native_gt_outage": 31,\n      "n_native_gt_total": 166,\n      "token_mask_sha256": "21b9e16f0abe800c55f145a18abd0a79a37df3cbda1ac658e7e89003a42e7536",\n      "native_gt_timestamp_sha256": "4ce61298686fd23d5c221a68447f79ca4a299ad24c102d3cf340e138cc83a185"\n    },\n    {\n      "id": "train_ep10_mars_1_d10_1",\n      "epoch": 10,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 23.650000000000002,\n      "start_s": 3.650000000000002,\n      "end_s": 43.650000000000006,\n      "n_native_gt_outage": 47,\n      "n_native_gt_total": 164,\n      "token_mask_sha256": "54d1f8b7a9217c3e8206fec44aa90fc97fdb3e89370798f9369d21bcb530fdd3",\n      "native_gt_timestamp_sha256": "d570616332d124c37fc235077dbc28797264b3df1a6a28bd4b8298f54afc682e"\n    },\n    {\n      "id": "train_ep10_mars_1_d30_0",\n      "epoch": 10,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 49.1,\n      "start_s": 29.1,\n      "end_s": 89.1,\n      "n_native_gt_outage": 122,\n      "n_native_gt_total": 240,\n      "token_mask_sha256": "ec337caf619f2fc34accfde4fefc63f5f100805a2dc4a30b788f58ddce2c4bc8",\n      "native_gt_timestamp_sha256": "0da0b40aea40ab1deefebd62e9f6308ba0d04a0136bf53b6bc28227166c91f37"\n    },\n    {\n      "id": "train_ep10_mars_2_d30_0",\n      "epoch": 10,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 56.95,\n      "start_s": 36.95,\n      "end_s": 96.95,\n      "n_native_gt_outage": 119,\n      "n_native_gt_total": 251,\n      "token_mask_sha256": "3dd74d234f7a60145505caa41ec0613e9d52eb9cbdc78e09d68e05c0d586d85a",\n      "native_gt_timestamp_sha256": "00850110dff8da08899607f6e571c7c66dab4b8e4a18ea8c5b0d82fd0c263626"\n    },\n    {\n      "id": "train_ep10_mars_2_d60_1",\n      "epoch": 10,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 56.75,\n      "start_s": 36.75,\n      "end_s": 126.75,\n      "n_native_gt_outage": 242,\n      "n_native_gt_total": 381,\n      "token_mask_sha256": "c0c2ec8564a28a400ecdd4fe90d4d6b443981b414ff75cc3098c2fd7dc691a8b",\n      "native_gt_timestamp_sha256": "950c6d361ad78159fee1f52cd6b8c017fc75a46ee65d325d5e96e70e3738bb24"\n    },\n    {\n      "id": "train_ep10_mars_1_d30_1",\n      "epoch": 10,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 59.45,\n      "start_s": 39.45,\n      "end_s": 99.45,\n      "n_native_gt_outage": 126,\n      "n_native_gt_total": 261,\n      "token_mask_sha256": "63d833f0b713bb4fb84f25c62b86deb8f26260f7a627284500d089600ceaa5cc",\n      "native_gt_timestamp_sha256": "5186ee9d9529c43f8419bd3d819b5666ba1b83de076904e7cc56403e26fff09f"\n    },\n    {\n      "id": "train_ep10_mars_3_d60_0",\n      "epoch": 10,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 33.6,\n      "start_s": 13.600000000000001,\n      "end_s": 103.6,\n      "n_native_gt_outage": 236,\n      "n_native_gt_total": 345,\n      "token_mask_sha256": "c0c2ec8564a28a400ecdd4fe90d4d6b443981b414ff75cc3098c2fd7dc691a8b",\n      "native_gt_timestamp_sha256": "1ff2fbb8c3a0082bad021090dbd4813aee2780840a911ecaf93769181038aa4c"\n    },\n    {\n      "id": "train_ep10_mars_3_d10_0",\n      "epoch": 10,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 34.75,\n      "start_s": 14.75,\n      "end_s": 54.75,\n      "n_native_gt_outage": 41,\n      "n_native_gt_total": 151,\n      "token_mask_sha256": "21b9e16f0abe800c55f145a18abd0a79a37df3cbda1ac658e7e89003a42e7536",\n      "native_gt_timestamp_sha256": "e0bef69cfa9d1fcbef324d3e3bbff576f8452d48b5526b36069ed873d46e7125"\n    },\n    {\n      "id": "train_ep11_mars_2_d10_0",\n      "epoch": 11,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 137.70000000000002,\n      "start_s": 117.70000000000002,\n      "end_s": 157.70000000000002,\n      "n_native_gt_outage": 38,\n      "n_native_gt_total": 167,\n      "token_mask_sha256": "21b9e16f0abe800c55f145a18abd0a79a37df3cbda1ac658e7e89003a42e7536",\n      "native_gt_timestamp_sha256": "dd9e2b2f43a60564cf59c3ad045d6516d7339bfdbda9c3758547d75bc21a5dd9"\n    },\n    {\n      "id": "train_ep11_mars_1_d30_0",\n      "epoch": 11,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 32.9,\n      "start_s": 12.899999999999999,\n      "end_s": 72.9,\n      "n_native_gt_outage": 111,\n      "n_native_gt_total": 250,\n      "token_mask_sha256": "ec337caf619f2fc34accfde4fefc63f5f100805a2dc4a30b788f58ddce2c4bc8",\n      "native_gt_timestamp_sha256": "ab863b0343fb59565844e566fa77ee825ac8ff20cd45b33b2a461fdef93aa0b3"\n    },\n    {\n      "id": "train_ep11_mars_1_d60_0",\n      "epoch": 11,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 25.8,\n      "start_s": 5.800000000000001,\n      "end_s": 95.8,\n      "n_native_gt_outage": 248,\n      "n_native_gt_total": 379,\n      "token_mask_sha256": "fed530af485dca69fb59f9b1a865da85e17ffd444c3c0d42d5f5017e0315ab5f",\n      "native_gt_timestamp_sha256": "1aabd525baeb05fdd16d0d4a2743a73fbaccc8596649059d190869a8f3232377"\n    },\n    {\n      "id": "train_ep11_mars_1_d30_1",\n      "epoch": 11,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 29.55,\n      "start_s": 9.55,\n      "end_s": 69.55,\n      "n_native_gt_outage": 112,\n      "n_native_gt_total": 241,\n      "token_mask_sha256": "3dd74d234f7a60145505caa41ec0613e9d52eb9cbdc78e09d68e05c0d586d85a",\n      "native_gt_timestamp_sha256": "81bb4a742017c9ee77efbae507441070da32fdd1f168c3f3cedcb4a34e2ffbb0"\n    },\n    {\n      "id": "train_ep11_mars_2_d30_1",\n      "epoch": 11,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 100.30000000000001,\n      "start_s": 80.30000000000001,\n      "end_s": 140.3,\n      "n_native_gt_outage": 128,\n      "n_native_gt_total": 252,\n      "token_mask_sha256": "ec337caf619f2fc34accfde4fefc63f5f100805a2dc4a30b788f58ddce2c4bc8",\n      "native_gt_timestamp_sha256": "15727df8ca76aa90a898315959128dbe0a447b52a5618b5ce45187775403a310"\n    },\n    {\n      "id": "train_ep11_mars_1_d10_0",\n      "epoch": 11,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 50.400000000000006,\n      "start_s": 30.400000000000006,\n      "end_s": 70.4,\n      "n_native_gt_outage": 32,\n      "n_native_gt_total": 161,\n      "token_mask_sha256": "860d531b40010f5a1ce4a6d8d4c9b3f3cb34a9b01d0a5e57422ce1faf23b77a3",\n      "native_gt_timestamp_sha256": "74e6b0d1a28427f6d34ee326f7fb7afb2e7f6be8774b82bf5654935781210457"\n    },\n    {\n      "id": "train_ep11_mars_3_d60_0",\n      "epoch": 11,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 38.2,\n      "start_s": 18.200000000000003,\n      "end_s": 108.2,\n      "n_native_gt_outage": 232,\n      "n_native_gt_total": 356,\n      "token_mask_sha256": "c0c2ec8564a28a400ecdd4fe90d4d6b443981b414ff75cc3098c2fd7dc691a8b",\n      "native_gt_timestamp_sha256": "30cef3a0eeded22470a2fb73f24aaea76f31d61401840e1c12af128d25d6d5a2"\n    },\n    {\n      "id": "train_ep11_mars_2_d10_1",\n      "epoch": 11,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 35.95,\n      "start_s": 15.950000000000003,\n      "end_s": 55.95,\n      "n_native_gt_outage": 33,\n      "n_native_gt_total": 151,\n      "token_mask_sha256": "ea945fe6cde424b8878a4a763925347368d146d2fe42a95bb983fd8eb5bdd4c3",\n      "native_gt_timestamp_sha256": "5c56960fe420fb2a4ad8634616cc6f5092ff9a2a5ceeb822f4ed79c2f0d37cc4"\n    },\n    {\n      "id": "train_ep11_mars_2_d60_1",\n      "epoch": 11,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 70.3,\n      "start_s": 50.3,\n      "end_s": 140.3,\n      "n_native_gt_outage": 251,\n      "n_native_gt_total": 371,\n      "token_mask_sha256": "71da0041c5ffc52dc55131d6c02ef9fd40886bf07a3990e099fe3e7675dd9bab",\n      "native_gt_timestamp_sha256": "75d1db608330f2c4018278b567e29671351497073c07a988e1871384de7a8460"\n    },\n    {\n      "id": "train_ep11_mars_1_d60_1",\n      "epoch": 11,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 27.150000000000002,\n      "start_s": 7.150000000000002,\n      "end_s": 97.15,\n      "n_native_gt_outage": 243,\n      "n_native_gt_total": 379,\n      "token_mask_sha256": "c0c2ec8564a28a400ecdd4fe90d4d6b443981b414ff75cc3098c2fd7dc691a8b",\n      "native_gt_timestamp_sha256": "b4fd08d47fd9be54a7f33e50410e4361a817b1463cf4237103aafaf4711a5841"\n    },\n    {\n      "id": "train_ep11_mars_3_d30_0",\n      "epoch": 11,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 35.35,\n      "start_s": 15.350000000000001,\n      "end_s": 75.35,\n      "n_native_gt_outage": 115,\n      "n_native_gt_total": 221,\n      "token_mask_sha256": "ec337caf619f2fc34accfde4fefc63f5f100805a2dc4a30b788f58ddce2c4bc8",\n      "native_gt_timestamp_sha256": "7912cb91468626cfc85d083746f2e859a4066b72a5ca11dc56ee3b17b4f8b222"\n    },\n    {\n      "id": "train_ep11_mars_2_d60_0",\n      "epoch": 11,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 68.4,\n      "start_s": 48.400000000000006,\n      "end_s": 138.4,\n      "n_native_gt_outage": 256,\n      "n_native_gt_total": 379,\n      "token_mask_sha256": "fed530af485dca69fb59f9b1a865da85e17ffd444c3c0d42d5f5017e0315ab5f",\n      "native_gt_timestamp_sha256": "2acb533985d9c97675002cee560d6599f6c4b85ac59c9a3c300e12ff40ba2f6e"\n    },\n    {\n      "id": "train_ep11_mars_3_d10_0",\n      "epoch": 11,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 38.85,\n      "start_s": 18.85,\n      "end_s": 58.85,\n      "n_native_gt_outage": 40,\n      "n_native_gt_total": 157,\n      "token_mask_sha256": "860d531b40010f5a1ce4a6d8d4c9b3f3cb34a9b01d0a5e57422ce1faf23b77a3",\n      "native_gt_timestamp_sha256": "c5befe9a6781b8af9125f5681f9a18e8b05328bb5b34b10c1eb9a825c7ae6ef2"\n    },\n    {\n      "id": "train_ep11_mars_3_d60_1",\n      "epoch": 11,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 52.550000000000004,\n      "start_s": 32.550000000000004,\n      "end_s": 122.55000000000001,\n      "n_native_gt_outage": 240,\n      "n_native_gt_total": 352,\n      "token_mask_sha256": "71da0041c5ffc52dc55131d6c02ef9fd40886bf07a3990e099fe3e7675dd9bab",\n      "native_gt_timestamp_sha256": "ee1d430920eea11e93d6c36d100a60579ec739a8c22740e43cc7da72243a46e7"\n    },\n    {\n      "id": "train_ep11_mars_2_d30_0",\n      "epoch": 11,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 40.95,\n      "start_s": 20.950000000000003,\n      "end_s": 80.95,\n      "n_native_gt_outage": 129,\n      "n_native_gt_total": 233,\n      "token_mask_sha256": "65e37395e153a7b3efe8de1dd41646a64c5f33d6d16eb16b954eac4a73120341",\n      "native_gt_timestamp_sha256": "23a463ccea1227f769923e886550f5fa9f79b32ad7bd42849f240cc29084a060"\n    },\n    {\n      "id": "train_ep11_mars_3_d10_1",\n      "epoch": 11,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 50.650000000000006,\n      "start_s": 30.650000000000006,\n      "end_s": 70.65,\n      "n_native_gt_outage": 36,\n      "n_native_gt_total": 152,\n      "token_mask_sha256": "860d531b40010f5a1ce4a6d8d4c9b3f3cb34a9b01d0a5e57422ce1faf23b77a3",\n      "native_gt_timestamp_sha256": "9abd13a9413ff2058afd57112d03ea647e54417900e1ce67beb0fe2d56a012f0"\n    },\n    {\n      "id": "train_ep11_mars_3_d30_1",\n      "epoch": 11,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 61.050000000000004,\n      "start_s": 41.050000000000004,\n      "end_s": 101.05000000000001,\n      "n_native_gt_outage": 119,\n      "n_native_gt_total": 230,\n      "token_mask_sha256": "b34423e399ec53c009571b199c4b245f2f0d58ef83150e9bf0611fa732314343",\n      "native_gt_timestamp_sha256": "51c24c8415284d04fd768b5de0a2f1db64e6558e5c2652062f30fbe03d633da8"\n    },\n    {\n      "id": "train_ep11_mars_1_d10_1",\n      "epoch": 11,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 71.85000000000001,\n      "start_s": 51.85000000000001,\n      "end_s": 91.85000000000001,\n      "n_native_gt_outage": 35,\n      "n_native_gt_total": 157,\n      "token_mask_sha256": "54d1f8b7a9217c3e8206fec44aa90fc97fdb3e89370798f9369d21bcb530fdd3",\n      "native_gt_timestamp_sha256": "f8c47558355fd0f9b65910be19c3e315a3178fc66ae63d94dac343240dbd9c01"\n    },\n    {\n      "id": "train_ep12_mars_3_d30_1",\n      "epoch": 12,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 52.25,\n      "start_s": 32.25,\n      "end_s": 92.25,\n      "n_native_gt_outage": 123,\n      "n_native_gt_total": 235,\n      "token_mask_sha256": "63d833f0b713bb4fb84f25c62b86deb8f26260f7a627284500d089600ceaa5cc",\n      "native_gt_timestamp_sha256": "be9664364be24a494909cc14708f52f129cfb02ac29ee067fc7877cfe0726f8f"\n    },\n    {\n      "id": "train_ep12_mars_3_d60_1",\n      "epoch": 12,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 40.35,\n      "start_s": 20.35,\n      "end_s": 110.35,\n      "n_native_gt_outage": 232,\n      "n_native_gt_total": 358,\n      "token_mask_sha256": "71da0041c5ffc52dc55131d6c02ef9fd40886bf07a3990e099fe3e7675dd9bab",\n      "native_gt_timestamp_sha256": "c9873aa4389c4e8a068feeb9a10787f9edd04af79921c705bfd5601ca7bd0e26"\n    },\n    {\n      "id": "train_ep12_mars_3_d60_0",\n      "epoch": 12,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 72.95,\n      "start_s": 52.95,\n      "end_s": 142.95,\n      "n_native_gt_outage": 235,\n      "n_native_gt_total": 364,\n      "token_mask_sha256": "71da0041c5ffc52dc55131d6c02ef9fd40886bf07a3990e099fe3e7675dd9bab",\n      "native_gt_timestamp_sha256": "941f6ba376d31a47e03e8ec9c89387515a1e2df832c4b37d096f73aa954781e4"\n    },\n    {\n      "id": "train_ep12_mars_2_d60_0",\n      "epoch": 12,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 47.550000000000004,\n      "start_s": 27.550000000000004,\n      "end_s": 117.55000000000001,\n      "n_native_gt_outage": 253,\n      "n_native_gt_total": 359,\n      "token_mask_sha256": "c0c2ec8564a28a400ecdd4fe90d4d6b443981b414ff75cc3098c2fd7dc691a8b",\n      "native_gt_timestamp_sha256": "e5a875a625d795a8c9fa1ec8670e824a3e783c16534c8d1e8dcfd716d31fef46"\n    },\n    {\n      "id": "train_ep12_mars_2_d10_1",\n      "epoch": 12,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 118.60000000000001,\n      "start_s": 98.60000000000001,\n      "end_s": 138.60000000000002,\n      "n_native_gt_outage": 57,\n      "n_native_gt_total": 162,\n      "token_mask_sha256": "860d531b40010f5a1ce4a6d8d4c9b3f3cb34a9b01d0a5e57422ce1faf23b77a3",\n      "native_gt_timestamp_sha256": "60115ee1c19653ffbf7e25505d3ee8324056cb8eb40135ac31bafe6f8165e8ca"\n    },\n    {\n      "id": "train_ep12_mars_2_d30_0",\n      "epoch": 12,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 99.05000000000001,\n      "start_s": 79.05000000000001,\n      "end_s": 139.05,\n      "n_native_gt_outage": 128,\n      "n_native_gt_total": 251,\n      "token_mask_sha256": "63d833f0b713bb4fb84f25c62b86deb8f26260f7a627284500d089600ceaa5cc",\n      "native_gt_timestamp_sha256": "3c1beb3611aeb3e2eb43a43527002c9274faf48a3fba4e35a62d65cf88213b85"\n    },\n    {\n      "id": "train_ep12_mars_1_d10_0",\n      "epoch": 12,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 26.55,\n      "start_s": 6.550000000000001,\n      "end_s": 46.55,\n      "n_native_gt_outage": 41,\n      "n_native_gt_total": 164,\n      "token_mask_sha256": "21b9e16f0abe800c55f145a18abd0a79a37df3cbda1ac658e7e89003a42e7536",\n      "native_gt_timestamp_sha256": "77cfe2a91a76fd0d56dd88f1ed85f7f5737ee740c62473f035934bf3d2badb83"\n    },\n    {\n      "id": "train_ep12_mars_2_d60_1",\n      "epoch": 12,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 71.05,\n      "start_s": 51.05,\n      "end_s": 141.05,\n      "n_native_gt_outage": 250,\n      "n_native_gt_total": 367,\n      "token_mask_sha256": "944469817cb1d463be9ba8e22d7c037bf6b358337512a5973f0eedd13fdd691e",\n      "native_gt_timestamp_sha256": "fef88ffe8a7329535bcf90d8055d57f48ba6208f4e85272ba99f1f6e5415d691"\n    },\n    {\n      "id": "train_ep12_mars_3_d10_1",\n      "epoch": 12,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 79.55000000000001,\n      "start_s": 59.55000000000001,\n      "end_s": 99.55000000000001,\n      "n_native_gt_outage": 43,\n      "n_native_gt_total": 153,\n      "token_mask_sha256": "9f12542780490d612306b1de6383455d9c904bc5b73f8b9cebd020d927653a95",\n      "native_gt_timestamp_sha256": "cde545b722029f2274bd69b3f205f7f62ed7b0fc5ceef152eb7be11eae1384f6"\n    },\n    {\n      "id": "train_ep12_mars_2_d10_0",\n      "epoch": 12,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 59.550000000000004,\n      "start_s": 39.550000000000004,\n      "end_s": 79.55000000000001,\n      "n_native_gt_outage": 44,\n      "n_native_gt_total": 163,\n      "token_mask_sha256": "860d531b40010f5a1ce4a6d8d4c9b3f3cb34a9b01d0a5e57422ce1faf23b77a3",\n      "native_gt_timestamp_sha256": "190708bedf6e5c459dadf09d2270617c53d2770c5d8a389cafe01d2f1ee37618"\n    },\n    {\n      "id": "train_ep12_mars_1_d60_0",\n      "epoch": 12,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 23.150000000000002,\n      "start_s": 3.150000000000002,\n      "end_s": 93.15,\n      "n_native_gt_outage": 251,\n      "n_native_gt_total": 373,\n      "token_mask_sha256": "c0c2ec8564a28a400ecdd4fe90d4d6b443981b414ff75cc3098c2fd7dc691a8b",\n      "native_gt_timestamp_sha256": "d585fc160290d7dc3bd2dfb3b7609e4f6d85ceae060ce92b417caccd5974888b"\n    },\n    {\n      "id": "train_ep12_mars_1_d10_1",\n      "epoch": 12,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 32.85,\n      "start_s": 12.850000000000001,\n      "end_s": 52.85,\n      "n_native_gt_outage": 31,\n      "n_native_gt_total": 165,\n      "token_mask_sha256": "54d1f8b7a9217c3e8206fec44aa90fc97fdb3e89370798f9369d21bcb530fdd3",\n      "native_gt_timestamp_sha256": "72c515ed7cba956390c78764681c6d69874c416b91f1e4240e5c5937f9e6235f"\n    },\n    {\n      "id": "train_ep12_mars_1_d60_1",\n      "epoch": 12,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 26.35,\n      "start_s": 6.350000000000001,\n      "end_s": 96.35,\n      "n_native_gt_outage": 245,\n      "n_native_gt_total": 379,\n      "token_mask_sha256": "c0c2ec8564a28a400ecdd4fe90d4d6b443981b414ff75cc3098c2fd7dc691a8b",\n      "native_gt_timestamp_sha256": "fcf471b8cbeda11e1b96ff4c56ad50cf725919636b11848629aea3498a84ce2a"\n    },\n    {\n      "id": "train_ep12_mars_1_d30_0",\n      "epoch": 12,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 53.95,\n      "start_s": 33.95,\n      "end_s": 93.95,\n      "n_native_gt_outage": 124,\n      "n_native_gt_total": 245,\n      "token_mask_sha256": "3dd74d234f7a60145505caa41ec0613e9d52eb9cbdc78e09d68e05c0d586d85a",\n      "native_gt_timestamp_sha256": "2e8f89882a0dafa2076c0e8c45e139ec3f6e5e0f18937f7291f694c61ff409d7"\n    },\n    {\n      "id": "train_ep12_mars_3_d30_0",\n      "epoch": 12,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 75.35000000000001,\n      "start_s": 55.35000000000001,\n      "end_s": 115.35000000000001,\n      "n_native_gt_outage": 127,\n      "n_native_gt_total": 238,\n      "token_mask_sha256": "ec337caf619f2fc34accfde4fefc63f5f100805a2dc4a30b788f58ddce2c4bc8",\n      "native_gt_timestamp_sha256": "587faf405e74c494125a21ee672bb23dd609db6f767c90e60cb4ac6613064790"\n    },\n    {\n      "id": "train_ep12_mars_2_d30_1",\n      "epoch": 12,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 24.0,\n      "start_s": 4.0,\n      "end_s": 64.0,\n      "n_native_gt_outage": 115,\n      "n_native_gt_total": 231,\n      "token_mask_sha256": "b34423e399ec53c009571b199c4b245f2f0d58ef83150e9bf0611fa732314343",\n      "native_gt_timestamp_sha256": "ce99315a8fa41bda7155eeae83eeb51c22daa2535c96156b3ca970df68e60c39"\n    },\n    {\n      "id": "train_ep12_mars_1_d30_1",\n      "epoch": 12,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 25.200000000000003,\n      "start_s": 5.200000000000003,\n      "end_s": 65.2,\n      "n_native_gt_outage": 126,\n      "n_native_gt_total": 242,\n      "token_mask_sha256": "b34423e399ec53c009571b199c4b245f2f0d58ef83150e9bf0611fa732314343",\n      "native_gt_timestamp_sha256": "f1a9df48954ff1a784b3a1151f88e185262b66449383adff19ca1053e120ccd2"\n    },\n    {\n      "id": "train_ep12_mars_3_d10_0",\n      "epoch": 12,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 37.5,\n      "start_s": 17.5,\n      "end_s": 57.5,\n      "n_native_gt_outage": 39,\n      "n_native_gt_total": 155,\n      "token_mask_sha256": "9f12542780490d612306b1de6383455d9c904bc5b73f8b9cebd020d927653a95",\n      "native_gt_timestamp_sha256": "416d1df6ce2d887009b659de54247579fd648b15db8a924a8892bb4977ebcc62"\n    },\n    {\n      "id": "train_ep13_mars_1_d30_1",\n      "epoch": 13,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 40.1,\n      "start_s": 20.1,\n      "end_s": 80.1,\n      "n_native_gt_outage": 127,\n      "n_native_gt_total": 251,\n      "token_mask_sha256": "ec337caf619f2fc34accfde4fefc63f5f100805a2dc4a30b788f58ddce2c4bc8",\n      "native_gt_timestamp_sha256": "5949c7751c99db984fce5e4392be92e892a711562c6d58a3fc582bd2375dc462"\n    },\n    {\n      "id": "train_ep13_mars_3_d60_0",\n      "epoch": 13,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 57.0,\n      "start_s": 37.0,\n      "end_s": 127.0,\n      "n_native_gt_outage": 238,\n      "n_native_gt_total": 351,\n      "token_mask_sha256": "c0c2ec8564a28a400ecdd4fe90d4d6b443981b414ff75cc3098c2fd7dc691a8b",\n      "native_gt_timestamp_sha256": "cd360b37191dcc0eb023375896e5cd30cbdbe687df038a29ff6ed96c05945a71"\n    },\n    {\n      "id": "train_ep13_mars_2_d60_0",\n      "epoch": 13,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 24.0,\n      "start_s": 4.0,\n      "end_s": 94.0,\n      "n_native_gt_outage": 233,\n      "n_native_gt_total": 367,\n      "token_mask_sha256": "fed530af485dca69fb59f9b1a865da85e17ffd444c3c0d42d5f5017e0315ab5f",\n      "native_gt_timestamp_sha256": "8865f8abb321f5f5d4652641c9dd9c5b69fe5504e947bfaaca95f8e3c7ab812d"\n    },\n    {\n      "id": "train_ep13_mars_2_d30_0",\n      "epoch": 13,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 100.95,\n      "start_s": 80.95,\n      "end_s": 140.95,\n      "n_native_gt_outage": 125,\n      "n_native_gt_total": 249,\n      "token_mask_sha256": "58551dc289b726e34524d3b1aa8536a6218d289b11cbfe5b61c49f96ebb802fd",\n      "native_gt_timestamp_sha256": "ec1101ae71a3f09545e0cd61e7efa5becd76496688131a454de725ae9cd938ba"\n    },\n    {\n      "id": "train_ep13_mars_1_d60_1",\n      "epoch": 13,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 26.1,\n      "start_s": 6.100000000000001,\n      "end_s": 96.1,\n      "n_native_gt_outage": 246,\n      "n_native_gt_total": 379,\n      "token_mask_sha256": "71da0041c5ffc52dc55131d6c02ef9fd40886bf07a3990e099fe3e7675dd9bab",\n      "native_gt_timestamp_sha256": "b010f72eae44eb1cf177fdf59adb1ce6d45486f26c1d8f388b3ef3c4c00ee1c9"\n    },\n    {\n      "id": "train_ep13_mars_3_d10_1",\n      "epoch": 13,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 78.35000000000001,\n      "start_s": 58.35000000000001,\n      "end_s": 98.35000000000001,\n      "n_native_gt_outage": 46,\n      "n_native_gt_total": 154,\n      "token_mask_sha256": "9f12542780490d612306b1de6383455d9c904bc5b73f8b9cebd020d927653a95",\n      "native_gt_timestamp_sha256": "68085d5dd7b164f1faabbae5e5bb670e29ee5346db99e5e24d9ef0b9741b6643"\n    },\n    {\n      "id": "train_ep13_mars_3_d60_1",\n      "epoch": 13,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 67.65,\n      "start_s": 47.650000000000006,\n      "end_s": 137.65,\n      "n_native_gt_outage": 238,\n      "n_native_gt_total": 354,\n      "token_mask_sha256": "fed530af485dca69fb59f9b1a865da85e17ffd444c3c0d42d5f5017e0315ab5f",\n      "native_gt_timestamp_sha256": "6eeadc001294f5445f4c5186a277774470c7b142aabf3bec6afda6d879ffe1a2"\n    },\n    {\n      "id": "train_ep13_mars_3_d30_1",\n      "epoch": 13,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 93.45,\n      "start_s": 73.45,\n      "end_s": 133.45,\n      "n_native_gt_outage": 117,\n      "n_native_gt_total": 236,\n      "token_mask_sha256": "b34423e399ec53c009571b199c4b245f2f0d58ef83150e9bf0611fa732314343",\n      "native_gt_timestamp_sha256": "85f33e6fefa549c9c4b4894c276e9315e290b4956900d7f135b25623a64551ec"\n    },\n    {\n      "id": "train_ep13_mars_2_d60_1",\n      "epoch": 13,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 60.550000000000004,\n      "start_s": 40.550000000000004,\n      "end_s": 130.55,\n      "n_native_gt_outage": 244,\n      "n_native_gt_total": 379,\n      "token_mask_sha256": "c0c2ec8564a28a400ecdd4fe90d4d6b443981b414ff75cc3098c2fd7dc691a8b",\n      "native_gt_timestamp_sha256": "9e5028463b6748cc661f9428050d449457d26d32c798da20f86c0d033497ed42"\n    },\n    {\n      "id": "train_ep13_mars_1_d30_0",\n      "epoch": 13,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 46.75,\n      "start_s": 26.75,\n      "end_s": 86.75,\n      "n_native_gt_outage": 128,\n      "n_native_gt_total": 244,\n      "token_mask_sha256": "3dd74d234f7a60145505caa41ec0613e9d52eb9cbdc78e09d68e05c0d586d85a",\n      "native_gt_timestamp_sha256": "dc6dcf652db17b111c3f362c641ce0276cde6db7df2fcc84d6695d0f16315c2a"\n    },\n    {\n      "id": "train_ep13_mars_1_d60_0",\n      "epoch": 13,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 24.3,\n      "start_s": 4.300000000000001,\n      "end_s": 94.3,\n      "n_native_gt_outage": 252,\n      "n_native_gt_total": 377,\n      "token_mask_sha256": "71da0041c5ffc52dc55131d6c02ef9fd40886bf07a3990e099fe3e7675dd9bab",\n      "native_gt_timestamp_sha256": "68b58e23d83bf183de7c8764cf307d534364ebfbfae692e3dddb2412d81e82fb"\n    },\n    {\n      "id": "train_ep13_mars_1_d10_1",\n      "epoch": 13,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 54.400000000000006,\n      "start_s": 34.400000000000006,\n      "end_s": 74.4,\n      "n_native_gt_outage": 32,\n      "n_native_gt_total": 168,\n      "token_mask_sha256": "860d531b40010f5a1ce4a6d8d4c9b3f3cb34a9b01d0a5e57422ce1faf23b77a3",\n      "native_gt_timestamp_sha256": "560a6a7a725897fc526c3d6536772bb63ab5765a153ff2f3e040d064788c5f92"\n    },\n    {\n      "id": "train_ep13_mars_1_d10_0",\n      "epoch": 13,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 47.150000000000006,\n      "start_s": 27.150000000000006,\n      "end_s": 67.15,\n      "n_native_gt_outage": 37,\n      "n_native_gt_total": 153,\n      "token_mask_sha256": "ea945fe6cde424b8878a4a763925347368d146d2fe42a95bb983fd8eb5bdd4c3",\n      "native_gt_timestamp_sha256": "4825a005aa84df42807907e9b06c913692e3b338c0a53b4b7a7c662c6521ef3f"\n    },\n    {\n      "id": "train_ep13_mars_2_d30_1",\n      "epoch": 13,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 30.950000000000003,\n      "start_s": 10.950000000000003,\n      "end_s": 70.95,\n      "n_native_gt_outage": 120,\n      "n_native_gt_total": 237,\n      "token_mask_sha256": "3dd74d234f7a60145505caa41ec0613e9d52eb9cbdc78e09d68e05c0d586d85a",\n      "native_gt_timestamp_sha256": "33dc70df486e868677abb94b396d0353002ef55b9c7ab4e33b3481f21f6a9f93"\n    },\n    {\n      "id": "train_ep13_mars_3_d30_0",\n      "epoch": 13,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 63.25,\n      "start_s": 43.25,\n      "end_s": 103.25,\n      "n_native_gt_outage": 118,\n      "n_native_gt_total": 229,\n      "token_mask_sha256": "b34423e399ec53c009571b199c4b245f2f0d58ef83150e9bf0611fa732314343",\n      "native_gt_timestamp_sha256": "e33660aa6eb4a36fa363ce9b41eba364fa2385bd2b323ad404b6bd884dacb8d9"\n    },\n    {\n      "id": "train_ep13_mars_2_d10_1",\n      "epoch": 13,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 42.25,\n      "start_s": 22.25,\n      "end_s": 62.25,\n      "n_native_gt_outage": 46,\n      "n_native_gt_total": 149,\n      "token_mask_sha256": "9f12542780490d612306b1de6383455d9c904bc5b73f8b9cebd020d927653a95",\n      "native_gt_timestamp_sha256": "63f235831e3db410ad9a7eec81cd4064851e3f4df549fcd5fbdaa2686df51161"\n    },\n    {\n      "id": "train_ep13_mars_2_d10_0",\n      "epoch": 13,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 37.1,\n      "start_s": 17.1,\n      "end_s": 57.1,\n      "n_native_gt_outage": 33,\n      "n_native_gt_total": 151,\n      "token_mask_sha256": "21b9e16f0abe800c55f145a18abd0a79a37df3cbda1ac658e7e89003a42e7536",\n      "native_gt_timestamp_sha256": "b1c0f74a6862c884ff1386e1fca1652525f0c3b8cfc7b1282ab9a4af0eb324b3"\n    },\n    {\n      "id": "train_ep13_mars_3_d10_0",\n      "epoch": 13,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 123.95,\n      "start_s": 103.95,\n      "end_s": 143.95,\n      "n_native_gt_outage": 31,\n      "n_native_gt_total": 170,\n      "token_mask_sha256": "9f12542780490d612306b1de6383455d9c904bc5b73f8b9cebd020d927653a95",\n      "native_gt_timestamp_sha256": "38a841678b51349a2e55613a944b92da387e531e6e0f43f6712be6d469462af2"\n    },\n    {\n      "id": "train_ep14_mars_3_d10_0",\n      "epoch": 14,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 55.1,\n      "start_s": 35.1,\n      "end_s": 75.1,\n      "n_native_gt_outage": 35,\n      "n_native_gt_total": 149,\n      "token_mask_sha256": "54d1f8b7a9217c3e8206fec44aa90fc97fdb3e89370798f9369d21bcb530fdd3",\n      "native_gt_timestamp_sha256": "c942b40ca6a093cc66e52982f596c103e8b312b949c27c54a337f593042a0a5c"\n    },\n    {\n      "id": "train_ep14_mars_2_d10_1",\n      "epoch": 14,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 116.45,\n      "start_s": 96.45,\n      "end_s": 136.45,\n      "n_native_gt_outage": 58,\n      "n_native_gt_total": 161,\n      "token_mask_sha256": "54d1f8b7a9217c3e8206fec44aa90fc97fdb3e89370798f9369d21bcb530fdd3",\n      "native_gt_timestamp_sha256": "43d84221bb2085cd312392cfacaea0350b11a16c677712acd9b78563815c9925"\n    },\n    {\n      "id": "train_ep14_mars_2_d30_0",\n      "epoch": 14,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 56.6,\n      "start_s": 36.6,\n      "end_s": 96.6,\n      "n_native_gt_outage": 119,\n      "n_native_gt_total": 253,\n      "token_mask_sha256": "b34423e399ec53c009571b199c4b245f2f0d58ef83150e9bf0611fa732314343",\n      "native_gt_timestamp_sha256": "c6096319ddd1462802c5df8eb5bfe96c459ceef6b78c040da0740ced5f1fcd84"\n    },\n    {\n      "id": "train_ep14_mars_1_d10_0",\n      "epoch": 14,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 28.55,\n      "start_s": 8.55,\n      "end_s": 48.55,\n      "n_native_gt_outage": 31,\n      "n_native_gt_total": 166,\n      "token_mask_sha256": "21b9e16f0abe800c55f145a18abd0a79a37df3cbda1ac658e7e89003a42e7536",\n      "native_gt_timestamp_sha256": "4ce61298686fd23d5c221a68447f79ca4a299ad24c102d3cf340e138cc83a185"\n    },\n    {\n      "id": "train_ep14_mars_3_d10_1",\n      "epoch": 14,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 53.150000000000006,\n      "start_s": 33.150000000000006,\n      "end_s": 73.15,\n      "n_native_gt_outage": 36,\n      "n_native_gt_total": 147,\n      "token_mask_sha256": "9f12542780490d612306b1de6383455d9c904bc5b73f8b9cebd020d927653a95",\n      "native_gt_timestamp_sha256": "a9784b256ad7b125a3fc05cb73b207edfcfebafbce744aae461762ebe21d923a"\n    },\n    {\n      "id": "train_ep14_mars_2_d30_1",\n      "epoch": 14,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 56.85,\n      "start_s": 36.85,\n      "end_s": 96.85,\n      "n_native_gt_outage": 120,\n      "n_native_gt_total": 252,\n      "token_mask_sha256": "63d833f0b713bb4fb84f25c62b86deb8f26260f7a627284500d089600ceaa5cc",\n      "native_gt_timestamp_sha256": "cf3944676cb1b5ba23ef62a5e171c3d2b013d048e0495fc65ed2dc44e5d6000c"\n    },\n    {\n      "id": "train_ep14_mars_2_d10_0",\n      "epoch": 14,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 90.25,\n      "start_s": 70.25,\n      "end_s": 110.25,\n      "n_native_gt_outage": 48,\n      "n_native_gt_total": 159,\n      "token_mask_sha256": "54d1f8b7a9217c3e8206fec44aa90fc97fdb3e89370798f9369d21bcb530fdd3",\n      "native_gt_timestamp_sha256": "9edf1c57935bda9b90214530e15df98792ca5b0769347e1abb32cef5b341d952"\n    },\n    {\n      "id": "train_ep14_mars_2_d60_1",\n      "epoch": 14,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 63.75,\n      "start_s": 43.75,\n      "end_s": 133.75,\n      "n_native_gt_outage": 253,\n      "n_native_gt_total": 381,\n      "token_mask_sha256": "c0c2ec8564a28a400ecdd4fe90d4d6b443981b414ff75cc3098c2fd7dc691a8b",\n      "native_gt_timestamp_sha256": "15303773e964a532650098b08fc3efa41f540c0d65b9f7f613b7165b344d1aab"\n    },\n    {\n      "id": "train_ep14_mars_3_d60_1",\n      "epoch": 14,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 48.85,\n      "start_s": 28.85,\n      "end_s": 118.85,\n      "n_native_gt_outage": 239,\n      "n_native_gt_total": 353,\n      "token_mask_sha256": "fed530af485dca69fb59f9b1a865da85e17ffd444c3c0d42d5f5017e0315ab5f",\n      "native_gt_timestamp_sha256": "dbdcce50b38b454b5cc5b14aab3a0943799836f2b2e87d3997ed38c213b78cdd"\n    },\n    {\n      "id": "train_ep14_mars_3_d30_0",\n      "epoch": 14,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 78.80000000000001,\n      "start_s": 58.80000000000001,\n      "end_s": 118.80000000000001,\n      "n_native_gt_outage": 126,\n      "n_native_gt_total": 236,\n      "token_mask_sha256": "3dd74d234f7a60145505caa41ec0613e9d52eb9cbdc78e09d68e05c0d586d85a",\n      "native_gt_timestamp_sha256": "85479f22f5f4b4b22b8f3719f4d63d4dd127bb4bd4ddc501d573e17b7a6cfc14"\n    },\n    {\n      "id": "train_ep14_mars_1_d30_0",\n      "epoch": 14,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 34.85,\n      "start_s": 14.850000000000001,\n      "end_s": 74.85,\n      "n_native_gt_outage": 114,\n      "n_native_gt_total": 249,\n      "token_mask_sha256": "63d833f0b713bb4fb84f25c62b86deb8f26260f7a627284500d089600ceaa5cc",\n      "native_gt_timestamp_sha256": "49caf97f19ee787c43bcd91c2663c365d7e788b4712f276bf1a09abb6867079a"\n    },\n    {\n      "id": "train_ep14_mars_1_d60_0",\n      "epoch": 14,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 23.55,\n      "start_s": 3.5500000000000007,\n      "end_s": 93.55,\n      "n_native_gt_outage": 252,\n      "n_native_gt_total": 374,\n      "token_mask_sha256": "c0c2ec8564a28a400ecdd4fe90d4d6b443981b414ff75cc3098c2fd7dc691a8b",\n      "native_gt_timestamp_sha256": "0157562d65b6eede425ee0903189debf01c3551d473d0a4e2c94022642ac6a5c"\n    },\n    {\n      "id": "train_ep14_mars_3_d60_0",\n      "epoch": 14,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 47.150000000000006,\n      "start_s": 27.150000000000006,\n      "end_s": 117.15,\n      "n_native_gt_outage": 238,\n      "n_native_gt_total": 355,\n      "token_mask_sha256": "71da0041c5ffc52dc55131d6c02ef9fd40886bf07a3990e099fe3e7675dd9bab",\n      "native_gt_timestamp_sha256": "ba45a77af133d717699c005b13feeba65bcb208ffd6ef437f5e23d1a1e3f1fee"\n    },\n    {\n      "id": "train_ep14_mars_1_d60_1",\n      "epoch": 14,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 29.55,\n      "start_s": 9.55,\n      "end_s": 99.55,\n      "n_native_gt_outage": 238,\n      "n_native_gt_total": 375,\n      "token_mask_sha256": "c0c2ec8564a28a400ecdd4fe90d4d6b443981b414ff75cc3098c2fd7dc691a8b",\n      "native_gt_timestamp_sha256": "3eec85600a74ea72993dfd710e238e88a0993cd52cecdbc2cf2c23daaca23f7a"\n    },\n    {\n      "id": "train_ep14_mars_2_d60_0",\n      "epoch": 14,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 48.0,\n      "start_s": 28.0,\n      "end_s": 118.0,\n      "n_native_gt_outage": 253,\n      "n_native_gt_total": 359,\n      "token_mask_sha256": "fed530af485dca69fb59f9b1a865da85e17ffd444c3c0d42d5f5017e0315ab5f",\n      "native_gt_timestamp_sha256": "0c843d16e6a74a300244601fc47c11e54e83d185de5b9d5a6075902c753686c3"\n    },\n    {\n      "id": "train_ep14_mars_1_d30_1",\n      "epoch": 14,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 56.95,\n      "start_s": 36.95,\n      "end_s": 96.95,\n      "n_native_gt_outage": 125,\n      "n_native_gt_total": 256,\n      "token_mask_sha256": "3dd74d234f7a60145505caa41ec0613e9d52eb9cbdc78e09d68e05c0d586d85a",\n      "native_gt_timestamp_sha256": "4ac6e6d13210c0966d7ed6cf7ad8dc5ae9883d7204619866f8b7083194a900d5"\n    },\n    {\n      "id": "train_ep14_mars_3_d30_1",\n      "epoch": 14,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 70.7,\n      "start_s": 50.7,\n      "end_s": 110.7,\n      "n_native_gt_outage": 120,\n      "n_native_gt_total": 238,\n      "token_mask_sha256": "63d833f0b713bb4fb84f25c62b86deb8f26260f7a627284500d089600ceaa5cc",\n      "native_gt_timestamp_sha256": "c8fb796507d0a16e60059cc93d51990326d1c17e2e0c503ba76e24db95105d02"\n    },\n    {\n      "id": "train_ep14_mars_1_d10_1",\n      "epoch": 14,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 72.0,\n      "start_s": 52.0,\n      "end_s": 92.0,\n      "n_native_gt_outage": 35,\n      "n_native_gt_total": 156,\n      "token_mask_sha256": "860d531b40010f5a1ce4a6d8d4c9b3f3cb34a9b01d0a5e57422ce1faf23b77a3",\n      "native_gt_timestamp_sha256": "104a8c0e0035d0452c32f9d3597a8e76dc720c9d80f2bb3afd6b8ef02ee351c0"\n    },\n    {\n      "id": "train_ep15_mars_2_d30_0",\n      "epoch": 15,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 75.10000000000001,\n      "start_s": 55.10000000000001,\n      "end_s": 115.10000000000001,\n      "n_native_gt_outage": 123,\n      "n_native_gt_total": 241,\n      "token_mask_sha256": "ec337caf619f2fc34accfde4fefc63f5f100805a2dc4a30b788f58ddce2c4bc8",\n      "native_gt_timestamp_sha256": "c5ba613f90cceef058f0eb402384ad866aefe649d5e482fe21ea185235edd855"\n    },\n    {\n      "id": "train_ep15_mars_3_d30_0",\n      "epoch": 15,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 77.35000000000001,\n      "start_s": 57.35000000000001,\n      "end_s": 117.35000000000001,\n      "n_native_gt_outage": 129,\n      "n_native_gt_total": 238,\n      "token_mask_sha256": "ec337caf619f2fc34accfde4fefc63f5f100805a2dc4a30b788f58ddce2c4bc8",\n      "native_gt_timestamp_sha256": "41f469c5261484aec947ddb6e812c47deccaa6a772850b4dd9cdc59b008a1408"\n    },\n    {\n      "id": "train_ep15_mars_1_d10_0",\n      "epoch": 15,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 49.45,\n      "start_s": 29.450000000000003,\n      "end_s": 69.45,\n      "n_native_gt_outage": 32,\n      "n_native_gt_total": 157,\n      "token_mask_sha256": "54d1f8b7a9217c3e8206fec44aa90fc97fdb3e89370798f9369d21bcb530fdd3",\n      "native_gt_timestamp_sha256": "6577ef249de873adccb9f187df8080caedaa8c55f45419812fdfd1ea9eab5e53"\n    },\n    {\n      "id": "train_ep15_mars_2_d60_1",\n      "epoch": 15,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 65.35000000000001,\n      "start_s": 45.35000000000001,\n      "end_s": 135.35000000000002,\n      "n_native_gt_outage": 257,\n      "n_native_gt_total": 378,\n      "token_mask_sha256": "c0c2ec8564a28a400ecdd4fe90d4d6b443981b414ff75cc3098c2fd7dc691a8b",\n      "native_gt_timestamp_sha256": "0035f0bfb776fc4784378e17ceda09d9412ced38ac645284eedc683b4fa18c4d"\n    },\n    {\n      "id": "train_ep15_mars_1_d10_1",\n      "epoch": 15,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 39.25,\n      "start_s": 19.25,\n      "end_s": 59.25,\n      "n_native_gt_outage": 49,\n      "n_native_gt_total": 163,\n      "token_mask_sha256": "54d1f8b7a9217c3e8206fec44aa90fc97fdb3e89370798f9369d21bcb530fdd3",\n      "native_gt_timestamp_sha256": "fa96b8eb29cdccb1f7aff89629fa4c7783a9f08d9d7225eaf8b4ed780ae61562"\n    },\n    {\n      "id": "train_ep15_mars_1_d30_0",\n      "epoch": 15,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 29.55,\n      "start_s": 9.55,\n      "end_s": 69.55,\n      "n_native_gt_outage": 112,\n      "n_native_gt_total": 241,\n      "token_mask_sha256": "3dd74d234f7a60145505caa41ec0613e9d52eb9cbdc78e09d68e05c0d586d85a",\n      "native_gt_timestamp_sha256": "81bb4a742017c9ee77efbae507441070da32fdd1f168c3f3cedcb4a34e2ffbb0"\n    },\n    {\n      "id": "train_ep15_mars_1_d30_1",\n      "epoch": 15,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 34.050000000000004,\n      "start_s": 14.050000000000004,\n      "end_s": 74.05000000000001,\n      "n_native_gt_outage": 112,\n      "n_native_gt_total": 249,\n      "token_mask_sha256": "63d833f0b713bb4fb84f25c62b86deb8f26260f7a627284500d089600ceaa5cc",\n      "native_gt_timestamp_sha256": "46bd66ce3c1df6237c263e1b1963960007c46b73b655be39205bca8bfc94277d"\n    },\n    {\n      "id": "train_ep15_mars_3_d30_1",\n      "epoch": 15,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 101.9,\n      "start_s": 81.9,\n      "end_s": 141.9,\n      "n_native_gt_outage": 115,\n      "n_native_gt_total": 244,\n      "token_mask_sha256": "63d833f0b713bb4fb84f25c62b86deb8f26260f7a627284500d089600ceaa5cc",\n      "native_gt_timestamp_sha256": "20098e45ad0906b863be7dc4b38e574229c50abc31826fef42b2e8fc2b63258e"\n    },\n    {\n      "id": "train_ep15_mars_3_d60_0",\n      "epoch": 15,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 52.35,\n      "start_s": 32.35,\n      "end_s": 122.35,\n      "n_native_gt_outage": 239,\n      "n_native_gt_total": 352,\n      "token_mask_sha256": "71da0041c5ffc52dc55131d6c02ef9fd40886bf07a3990e099fe3e7675dd9bab",\n      "native_gt_timestamp_sha256": "ee1d430920eea11e93d6c36d100a60579ec739a8c22740e43cc7da72243a46e7"\n    },\n    {\n      "id": "train_ep15_mars_1_d60_1",\n      "epoch": 15,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 23.650000000000002,\n      "start_s": 3.650000000000002,\n      "end_s": 93.65,\n      "n_native_gt_outage": 252,\n      "n_native_gt_total": 374,\n      "token_mask_sha256": "944469817cb1d463be9ba8e22d7c037bf6b358337512a5973f0eedd13fdd691e",\n      "native_gt_timestamp_sha256": "1df850c74df78c4ad07727d6f137055822d716208f2dc00f703adec174267cd8"\n    },\n    {\n      "id": "train_ep15_mars_2_d10_1",\n      "epoch": 15,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 125.05000000000001,\n      "start_s": 105.05000000000001,\n      "end_s": 145.05,\n      "n_native_gt_outage": 41,\n      "n_native_gt_total": 163,\n      "token_mask_sha256": "54d1f8b7a9217c3e8206fec44aa90fc97fdb3e89370798f9369d21bcb530fdd3",\n      "native_gt_timestamp_sha256": "a1946e31fb0115c77bff9e40c0f9056274e3179c79141c8607db90c77eaaa6df"\n    },\n    {\n      "id": "train_ep15_mars_2_d10_0",\n      "epoch": 15,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 73.95,\n      "start_s": 53.95,\n      "end_s": 93.95,\n      "n_native_gt_outage": 33,\n      "n_native_gt_total": 171,\n      "token_mask_sha256": "21b9e16f0abe800c55f145a18abd0a79a37df3cbda1ac658e7e89003a42e7536",\n      "native_gt_timestamp_sha256": "5c376ac84d7e1b7c2e6f8d183ae73f06ef6ce2d6f9a3f9eadcebc27a484c3776"\n    },\n    {\n      "id": "train_ep15_mars_3_d10_1",\n      "epoch": 15,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 105.80000000000001,\n      "start_s": 85.80000000000001,\n      "end_s": 125.80000000000001,\n      "n_native_gt_outage": 40,\n      "n_native_gt_total": 150,\n      "token_mask_sha256": "21b9e16f0abe800c55f145a18abd0a79a37df3cbda1ac658e7e89003a42e7536",\n      "native_gt_timestamp_sha256": "0dffc7ae216f8cc895e4cf3027863d8e5a28de5e1cf85d48519be5aa48cf1ee8"\n    },\n    {\n      "id": "train_ep15_mars_2_d60_0",\n      "epoch": 15,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 24.5,\n      "start_s": 4.5,\n      "end_s": 94.5,\n      "n_native_gt_outage": 234,\n      "n_native_gt_total": 365,\n      "token_mask_sha256": "71da0041c5ffc52dc55131d6c02ef9fd40886bf07a3990e099fe3e7675dd9bab",\n      "native_gt_timestamp_sha256": "14d6b47ff2a4ac860c97da09efb85844632f650cab9edeb23071ad29ed4e5962"\n    },\n    {\n      "id": "train_ep15_mars_3_d10_0",\n      "epoch": 15,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 82.80000000000001,\n      "start_s": 62.80000000000001,\n      "end_s": 102.80000000000001,\n      "n_native_gt_outage": 36,\n      "n_native_gt_total": 153,\n      "token_mask_sha256": "ea945fe6cde424b8878a4a763925347368d146d2fe42a95bb983fd8eb5bdd4c3",\n      "native_gt_timestamp_sha256": "cecb250f085a722a563de3493be483ce66feb74f227965d80b1cd1b73b3fa0b1"\n    },\n    {\n      "id": "train_ep15_mars_3_d60_1",\n      "epoch": 15,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 65.45,\n      "start_s": 45.45,\n      "end_s": 135.45,\n      "n_native_gt_outage": 236,\n      "n_native_gt_total": 349,\n      "token_mask_sha256": "fed530af485dca69fb59f9b1a865da85e17ffd444c3c0d42d5f5017e0315ab5f",\n      "native_gt_timestamp_sha256": "21b137c3aed99f7e2512944eebefb5d9a35fa6c39df9b1f371d8f8695452dc25"\n    },\n    {\n      "id": "train_ep15_mars_2_d30_1",\n      "epoch": 15,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 43.050000000000004,\n      "start_s": 23.050000000000004,\n      "end_s": 83.05000000000001,\n      "n_native_gt_outage": 129,\n      "n_native_gt_total": 232,\n      "token_mask_sha256": "ec337caf619f2fc34accfde4fefc63f5f100805a2dc4a30b788f58ddce2c4bc8",\n      "native_gt_timestamp_sha256": "ebbbd3a911a0d5e1c0be25d85a7324f3090397f7cae30a8dfbb2e8efda61445c"\n    },\n    {\n      "id": "train_ep15_mars_1_d60_0",\n      "epoch": 15,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 21.650000000000002,\n      "start_s": 1.6500000000000021,\n      "end_s": 91.65,\n      "n_native_gt_outage": 252,\n      "n_native_gt_total": 371,\n      "token_mask_sha256": "944469817cb1d463be9ba8e22d7c037bf6b358337512a5973f0eedd13fdd691e",\n      "native_gt_timestamp_sha256": "2c48084aa530bd4648f6ced1eaf1426323a094d7414da3c977bfa0fb89a029c4"\n    },\n    {\n      "id": "train_ep16_mars_2_d10_0",\n      "epoch": 16,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 47.45,\n      "start_s": 27.450000000000003,\n      "end_s": 67.45,\n      "n_native_gt_outage": 46,\n      "n_native_gt_total": 157,\n      "token_mask_sha256": "9f12542780490d612306b1de6383455d9c904bc5b73f8b9cebd020d927653a95",\n      "native_gt_timestamp_sha256": "04b6bf9dcd632e0516f8935d1010fc48bc10811399b74778a495c470da4d07d0"\n    },\n    {\n      "id": "train_ep16_mars_3_d10_1",\n      "epoch": 16,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 59.2,\n      "start_s": 39.2,\n      "end_s": 79.2,\n      "n_native_gt_outage": 35,\n      "n_native_gt_total": 154,\n      "token_mask_sha256": "21b9e16f0abe800c55f145a18abd0a79a37df3cbda1ac658e7e89003a42e7536",\n      "native_gt_timestamp_sha256": "5f908e879ac825a98d09c1b52d4aa194b417e095ab828b4abf40a89e49b6efb6"\n    },\n    {\n      "id": "train_ep16_mars_1_d60_1",\n      "epoch": 16,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 28.700000000000003,\n      "start_s": 8.700000000000003,\n      "end_s": 98.7,\n      "n_native_gt_outage": 237,\n      "n_native_gt_total": 376,\n      "token_mask_sha256": "71da0041c5ffc52dc55131d6c02ef9fd40886bf07a3990e099fe3e7675dd9bab",\n      "native_gt_timestamp_sha256": "420f6f6ab3ca0352637c33077a6cdc06008cc472dc225a104c876112c89ebef3"\n    },\n    {\n      "id": "train_ep16_mars_2_d10_1",\n      "epoch": 16,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 84.60000000000001,\n      "start_s": 64.60000000000001,\n      "end_s": 104.60000000000001,\n      "n_native_gt_outage": 52,\n      "n_native_gt_total": 172,\n      "token_mask_sha256": "860d531b40010f5a1ce4a6d8d4c9b3f3cb34a9b01d0a5e57422ce1faf23b77a3",\n      "native_gt_timestamp_sha256": "ceec06567a4c47f39ca3a14e9011fe10eda91cacbeb4e45a9545470899a93d1e"\n    },\n    {\n      "id": "train_ep16_mars_3_d60_0",\n      "epoch": 16,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 53.85,\n      "start_s": 33.85,\n      "end_s": 123.85,\n      "n_native_gt_outage": 240,\n      "n_native_gt_total": 352,\n      "token_mask_sha256": "fed530af485dca69fb59f9b1a865da85e17ffd444c3c0d42d5f5017e0315ab5f",\n      "native_gt_timestamp_sha256": "b40104e9391fc977c0aa11aa47df800b1a1f704c3edfc6f7cb765ca09ac99101"\n    },\n    {\n      "id": "train_ep16_mars_1_d30_1",\n      "epoch": 16,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 34.4,\n      "start_s": 14.399999999999999,\n      "end_s": 74.4,\n      "n_native_gt_outage": 112,\n      "n_native_gt_total": 252,\n      "token_mask_sha256": "b34423e399ec53c009571b199c4b245f2f0d58ef83150e9bf0611fa732314343",\n      "native_gt_timestamp_sha256": "229ff4d31d8f57bd964adada77614af3c1732c9eae1b1b5efad4a86c3cb1b9b2"\n    },\n    {\n      "id": "train_ep16_mars_3_d10_0",\n      "epoch": 16,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 71.7,\n      "start_s": 51.7,\n      "end_s": 91.7,\n      "n_native_gt_outage": 50,\n      "n_native_gt_total": 156,\n      "token_mask_sha256": "9f12542780490d612306b1de6383455d9c904bc5b73f8b9cebd020d927653a95",\n      "native_gt_timestamp_sha256": "a4c9522c4d8d28337d7f3441593124dab0c38f00717476113345c22660496b1b"\n    },\n    {\n      "id": "train_ep16_mars_2_d30_0",\n      "epoch": 16,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 48.050000000000004,\n      "start_s": 28.050000000000004,\n      "end_s": 88.05000000000001,\n      "n_native_gt_outage": 127,\n      "n_native_gt_total": 236,\n      "token_mask_sha256": "ec337caf619f2fc34accfde4fefc63f5f100805a2dc4a30b788f58ddce2c4bc8",\n      "native_gt_timestamp_sha256": "370120639847c0662931b7828474a53481f0c7622755de2a2ca9794389ee16bf"\n    },\n    {\n      "id": "train_ep16_mars_2_d60_0",\n      "epoch": 16,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 51.550000000000004,\n      "start_s": 31.550000000000004,\n      "end_s": 121.55000000000001,\n      "n_native_gt_outage": 241,\n      "n_native_gt_total": 364,\n      "token_mask_sha256": "fed530af485dca69fb59f9b1a865da85e17ffd444c3c0d42d5f5017e0315ab5f",\n      "native_gt_timestamp_sha256": "44a7a1e1ca411bfadc98accbcc66d492cdba6540ce81a79fadffd31a555e444f"\n    },\n    {\n      "id": "train_ep16_mars_1_d10_0",\n      "epoch": 16,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 27.75,\n      "start_s": 7.75,\n      "end_s": 47.75,\n      "n_native_gt_outage": 35,\n      "n_native_gt_total": 164,\n      "token_mask_sha256": "21b9e16f0abe800c55f145a18abd0a79a37df3cbda1ac658e7e89003a42e7536",\n      "native_gt_timestamp_sha256": "cec7cfc353d41195e3bb5a190452f20f7863473daee966a47b8da49bf2ff201c"\n    },\n    {\n      "id": "train_ep16_mars_3_d30_1",\n      "epoch": 16,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 37.9,\n      "start_s": 17.9,\n      "end_s": 77.9,\n      "n_native_gt_outage": 114,\n      "n_native_gt_total": 226,\n      "token_mask_sha256": "63d833f0b713bb4fb84f25c62b86deb8f26260f7a627284500d089600ceaa5cc",\n      "native_gt_timestamp_sha256": "8956c93ba907252c89dad57514d729638d58e513d03631feefd624c5ba2d63cd"\n    },\n    {\n      "id": "train_ep16_mars_1_d30_0",\n      "epoch": 16,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 30.400000000000002,\n      "start_s": 10.400000000000002,\n      "end_s": 70.4,\n      "n_native_gt_outage": 113,\n      "n_native_gt_total": 243,\n      "token_mask_sha256": "b34423e399ec53c009571b199c4b245f2f0d58ef83150e9bf0611fa732314343",\n      "native_gt_timestamp_sha256": "2727b974fd5db1c437b0cfb4bf3db62c35e4197c9dfa383c96fd5ae73d9c0a0a"\n    },\n    {\n      "id": "train_ep16_mars_3_d60_1",\n      "epoch": 16,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 66.45,\n      "start_s": 46.45,\n      "end_s": 136.45,\n      "n_native_gt_outage": 238,\n      "n_native_gt_total": 350,\n      "token_mask_sha256": "fed530af485dca69fb59f9b1a865da85e17ffd444c3c0d42d5f5017e0315ab5f",\n      "native_gt_timestamp_sha256": "5137c6036468105d509d6303c3d57942462c2ae28fcf6407208576838662dfda"\n    },\n    {\n      "id": "train_ep16_mars_2_d30_1",\n      "epoch": 16,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 48.150000000000006,\n      "start_s": 28.150000000000006,\n      "end_s": 88.15,\n      "n_native_gt_outage": 128,\n      "n_native_gt_total": 236,\n      "token_mask_sha256": "b34423e399ec53c009571b199c4b245f2f0d58ef83150e9bf0611fa732314343",\n      "native_gt_timestamp_sha256": "370120639847c0662931b7828474a53481f0c7622755de2a2ca9794389ee16bf"\n    },\n    {\n      "id": "train_ep16_mars_1_d60_0",\n      "epoch": 16,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 23.55,\n      "start_s": 3.5500000000000007,\n      "end_s": 93.55,\n      "n_native_gt_outage": 252,\n      "n_native_gt_total": 374,\n      "token_mask_sha256": "c0c2ec8564a28a400ecdd4fe90d4d6b443981b414ff75cc3098c2fd7dc691a8b",\n      "native_gt_timestamp_sha256": "0157562d65b6eede425ee0903189debf01c3551d473d0a4e2c94022642ac6a5c"\n    },\n    {\n      "id": "train_ep16_mars_3_d30_0",\n      "epoch": 16,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 88.25,\n      "start_s": 68.25,\n      "end_s": 128.25,\n      "n_native_gt_outage": 118,\n      "n_native_gt_total": 235,\n      "token_mask_sha256": "b34423e399ec53c009571b199c4b245f2f0d58ef83150e9bf0611fa732314343",\n      "native_gt_timestamp_sha256": "deddf041f887057d3d00a6fc1353e1b2c79009dd3f834c021f28149e1e6ffce3"\n    },\n    {\n      "id": "train_ep16_mars_2_d60_1",\n      "epoch": 16,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 36.800000000000004,\n      "start_s": 16.800000000000004,\n      "end_s": 106.80000000000001,\n      "n_native_gt_outage": 252,\n      "n_native_gt_total": 359,\n      "token_mask_sha256": "fed530af485dca69fb59f9b1a865da85e17ffd444c3c0d42d5f5017e0315ab5f",\n      "native_gt_timestamp_sha256": "ad1b5b619d1aee30a3350b72e7cc94c1c963fb53828c50db4d74e43b3472202e"\n    },\n    {\n      "id": "train_ep16_mars_1_d10_1",\n      "epoch": 16,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 44.25,\n      "start_s": 24.25,\n      "end_s": 64.25,\n      "n_native_gt_outage": 48,\n      "n_native_gt_total": 161,\n      "token_mask_sha256": "54d1f8b7a9217c3e8206fec44aa90fc97fdb3e89370798f9369d21bcb530fdd3",\n      "native_gt_timestamp_sha256": "0fdd081dc4a0eecf5a21222d84172e30106d33b14ab3e5ef966def86c3297af8"\n    },\n    {\n      "id": "train_ep17_mars_1_d10_0",\n      "epoch": 17,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 56.300000000000004,\n      "start_s": 36.300000000000004,\n      "end_s": 76.30000000000001,\n      "n_native_gt_outage": 32,\n      "n_native_gt_total": 168,\n      "token_mask_sha256": "9f12542780490d612306b1de6383455d9c904bc5b73f8b9cebd020d927653a95",\n      "native_gt_timestamp_sha256": "7091fb7638f86f51bf514ab9e478bc21c314315639e8331737302108c8fc7967"\n    },\n    {\n      "id": "train_ep17_mars_3_d60_1",\n      "epoch": 17,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 31.150000000000002,\n      "start_s": 11.150000000000002,\n      "end_s": 101.15,\n      "n_native_gt_outage": 238,\n      "n_native_gt_total": 349,\n      "token_mask_sha256": "71da0041c5ffc52dc55131d6c02ef9fd40886bf07a3990e099fe3e7675dd9bab",\n      "native_gt_timestamp_sha256": "560af584814e54aa4fa580005cf5c642d1ea84b33c293e77416269da4a1d6cd5"\n    },\n    {\n      "id": "train_ep17_mars_1_d60_1",\n      "epoch": 17,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 24.450000000000003,\n      "start_s": 4.450000000000003,\n      "end_s": 94.45,\n      "n_native_gt_outage": 252,\n      "n_native_gt_total": 378,\n      "token_mask_sha256": "944469817cb1d463be9ba8e22d7c037bf6b358337512a5973f0eedd13fdd691e",\n      "native_gt_timestamp_sha256": "38f983129bf109a2a8bd9c00847697a2e8ac311795a77f54a3ea563434927b95"\n    },\n    {\n      "id": "train_ep17_mars_1_d30_1",\n      "epoch": 17,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 33.7,\n      "start_s": 13.700000000000003,\n      "end_s": 73.7,\n      "n_native_gt_outage": 114,\n      "n_native_gt_total": 249,\n      "token_mask_sha256": "ec337caf619f2fc34accfde4fefc63f5f100805a2dc4a30b788f58ddce2c4bc8",\n      "native_gt_timestamp_sha256": "bad34a4a8a86632cd29c300813ea731a4c4610e89095d6952aa660148230ff57"\n    },\n    {\n      "id": "train_ep17_mars_1_d10_1",\n      "epoch": 17,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 37.4,\n      "start_s": 17.4,\n      "end_s": 57.4,\n      "n_native_gt_outage": 45,\n      "n_native_gt_total": 162,\n      "token_mask_sha256": "860d531b40010f5a1ce4a6d8d4c9b3f3cb34a9b01d0a5e57422ce1faf23b77a3",\n      "native_gt_timestamp_sha256": "e5afbd322833ef23e6317f45d9fa2d44f196b85f5f62822e7bb36546bd8acc5a"\n    },\n    {\n      "id": "train_ep17_mars_1_d30_0",\n      "epoch": 17,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 45.050000000000004,\n      "start_s": 25.050000000000004,\n      "end_s": 85.05000000000001,\n      "n_native_gt_outage": 131,\n      "n_native_gt_total": 249,\n      "token_mask_sha256": "63d833f0b713bb4fb84f25c62b86deb8f26260f7a627284500d089600ceaa5cc",\n      "native_gt_timestamp_sha256": "7aa67889376aeb7fb90a536cc7a7618bb5886b2300ae052dcbac5c6a33532289"\n    },\n    {\n      "id": "train_ep17_mars_3_d30_1",\n      "epoch": 17,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 47.800000000000004,\n      "start_s": 27.800000000000004,\n      "end_s": 87.80000000000001,\n      "n_native_gt_outage": 110,\n      "n_native_gt_total": 237,\n      "token_mask_sha256": "3dd74d234f7a60145505caa41ec0613e9d52eb9cbdc78e09d68e05c0d586d85a",\n      "native_gt_timestamp_sha256": "446efa8bd1c4a46174c9628c404c0b8fd0d760ecc4a475c261564b2b25743aa9"\n    },\n    {\n      "id": "train_ep17_mars_2_d60_0",\n      "epoch": 17,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 51.2,\n      "start_s": 31.200000000000003,\n      "end_s": 121.2,\n      "n_native_gt_outage": 241,\n      "n_native_gt_total": 364,\n      "token_mask_sha256": "944469817cb1d463be9ba8e22d7c037bf6b358337512a5973f0eedd13fdd691e",\n      "native_gt_timestamp_sha256": "e4128c9c705ce199694577ed986db242f9dd33158a863d7d59d596ed1b5a25e2"\n    },\n    {\n      "id": "train_ep17_mars_2_d60_1",\n      "epoch": 17,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 51.45,\n      "start_s": 31.450000000000003,\n      "end_s": 121.45,\n      "n_native_gt_outage": 242,\n      "n_native_gt_total": 364,\n      "token_mask_sha256": "71da0041c5ffc52dc55131d6c02ef9fd40886bf07a3990e099fe3e7675dd9bab",\n      "native_gt_timestamp_sha256": "caad3132705385853bbd2201702784f1e9930691b8ebfd6656458215c45243fa"\n    },\n    {\n      "id": "train_ep17_mars_3_d10_0",\n      "epoch": 17,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 105.85000000000001,\n      "start_s": 85.85000000000001,\n      "end_s": 125.85000000000001,\n      "n_native_gt_outage": 41,\n      "n_native_gt_total": 150,\n      "token_mask_sha256": "00670280358ee61951a804e979e4299c5c00d28c945549d9f9c8aaf60bb28906",\n      "native_gt_timestamp_sha256": "0dffc7ae216f8cc895e4cf3027863d8e5a28de5e1cf85d48519be5aa48cf1ee8"\n    },\n    {\n      "id": "train_ep17_mars_3_d10_1",\n      "epoch": 17,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 42.400000000000006,\n      "start_s": 22.400000000000006,\n      "end_s": 62.400000000000006,\n      "n_native_gt_outage": 40,\n      "n_native_gt_total": 156,\n      "token_mask_sha256": "21b9e16f0abe800c55f145a18abd0a79a37df3cbda1ac658e7e89003a42e7536",\n      "native_gt_timestamp_sha256": "a5bd1f921c23917bafaba542d53b887ca201eb012cb8a3ebb3e410e967696d64"\n    },\n    {\n      "id": "train_ep17_mars_2_d30_1",\n      "epoch": 17,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 46.2,\n      "start_s": 26.200000000000003,\n      "end_s": 86.2,\n      "n_native_gt_outage": 130,\n      "n_native_gt_total": 233,\n      "token_mask_sha256": "422603b9cb8e3d617b8e1b3beca5e4b341ed33ab6539387e0cb29d31cf8f0ba5",\n      "native_gt_timestamp_sha256": "4f8f7f329d60964ae2bfdf58a7bd210c1cc670eb498e5fa0d474a20b2aab8b08"\n    },\n    {\n      "id": "train_ep17_mars_2_d30_0",\n      "epoch": 17,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 118.25,\n      "start_s": 98.25,\n      "end_s": 158.25,\n      "n_native_gt_outage": 128,\n      "n_native_gt_total": 238,\n      "token_mask_sha256": "ec337caf619f2fc34accfde4fefc63f5f100805a2dc4a30b788f58ddce2c4bc8",\n      "native_gt_timestamp_sha256": "e492303fceb630750df50ba0885eb3fab451897217ef1f17006945172c7efb27"\n    },\n    {\n      "id": "train_ep17_mars_1_d60_0",\n      "epoch": 17,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 23.900000000000002,\n      "start_s": 3.900000000000002,\n      "end_s": 93.9,\n      "n_native_gt_outage": 252,\n      "n_native_gt_total": 374,\n      "token_mask_sha256": "71da0041c5ffc52dc55131d6c02ef9fd40886bf07a3990e099fe3e7675dd9bab",\n      "native_gt_timestamp_sha256": "3798810fabe03a4037a8cb7fd5416d08b8e8feb222862f3fb5d867825f4f7025"\n    },\n    {\n      "id": "train_ep17_mars_3_d30_0",\n      "epoch": 17,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 35.5,\n      "start_s": 15.5,\n      "end_s": 75.5,\n      "n_native_gt_outage": 116,\n      "n_native_gt_total": 221,\n      "token_mask_sha256": "63d833f0b713bb4fb84f25c62b86deb8f26260f7a627284500d089600ceaa5cc",\n      "native_gt_timestamp_sha256": "7912cb91468626cfc85d083746f2e859a4066b72a5ca11dc56ee3b17b4f8b222"\n    },\n    {\n      "id": "train_ep17_mars_2_d10_0",\n      "epoch": 17,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 57.550000000000004,\n      "start_s": 37.550000000000004,\n      "end_s": 77.55000000000001,\n      "n_native_gt_outage": 39,\n      "n_native_gt_total": 164,\n      "token_mask_sha256": "860d531b40010f5a1ce4a6d8d4c9b3f3cb34a9b01d0a5e57422ce1faf23b77a3",\n      "native_gt_timestamp_sha256": "a951d2c14bfee5252da94cc4ae8f3785e5b485f2e6f4eee1fc0a66ef5e280377"\n    },\n    {\n      "id": "train_ep17_mars_2_d10_1",\n      "epoch": 17,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 74.95,\n      "start_s": 54.95,\n      "end_s": 94.95,\n      "n_native_gt_outage": 34,\n      "n_native_gt_total": 171,\n      "token_mask_sha256": "21b9e16f0abe800c55f145a18abd0a79a37df3cbda1ac658e7e89003a42e7536",\n      "native_gt_timestamp_sha256": "f2fe2093cc11557600d220e93d7ab2301129700679c7f079369bf61c1105584e"\n    },\n    {\n      "id": "train_ep17_mars_3_d60_0",\n      "epoch": 17,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 43.95,\n      "start_s": 23.950000000000003,\n      "end_s": 113.95,\n      "n_native_gt_outage": 230,\n      "n_native_gt_total": 356,\n      "token_mask_sha256": "57b2040310b429bf3d0a62f6a12fd890c5659c8afbadea54748f8f4504f5bae7",\n      "native_gt_timestamp_sha256": "62489ffd584c41129dca7acbcc03864a86db21b36c1fa93c0150d3a27d7af3cc"\n    },\n    {\n      "id": "train_ep18_mars_1_d30_1",\n      "epoch": 18,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 45.300000000000004,\n      "start_s": 25.300000000000004,\n      "end_s": 85.30000000000001,\n      "n_native_gt_outage": 133,\n      "n_native_gt_total": 248,\n      "token_mask_sha256": "ec337caf619f2fc34accfde4fefc63f5f100805a2dc4a30b788f58ddce2c4bc8",\n      "native_gt_timestamp_sha256": "13037e61057beb3ce39d8c19c84f99a2baf3fff7d70f27f5209b91c270ca700d"\n    },\n    {\n      "id": "train_ep18_mars_3_d10_0",\n      "epoch": 18,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 64.2,\n      "start_s": 44.2,\n      "end_s": 84.2,\n      "n_native_gt_outage": 33,\n      "n_native_gt_total": 158,\n      "token_mask_sha256": "ea945fe6cde424b8878a4a763925347368d146d2fe42a95bb983fd8eb5bdd4c3",\n      "native_gt_timestamp_sha256": "95882388ed7f06a7614800df9fe66aae0e73488b35b0dcc9f7239a8e5c285326"\n    },\n    {\n      "id": "train_ep18_mars_2_d10_0",\n      "epoch": 18,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 28.75,\n      "start_s": 8.75,\n      "end_s": 48.75,\n      "n_native_gt_outage": 38,\n      "n_native_gt_total": 148,\n      "token_mask_sha256": "21b9e16f0abe800c55f145a18abd0a79a37df3cbda1ac658e7e89003a42e7536",\n      "native_gt_timestamp_sha256": "5b7516b00266285bbfe1f1b1b37b28017fcf92f5fce9e6062673a1605e5184fc"\n    },\n    {\n      "id": "train_ep18_mars_1_d10_0",\n      "epoch": 18,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 47.0,\n      "start_s": 27.0,\n      "end_s": 67.0,\n      "n_native_gt_outage": 38,\n      "n_native_gt_total": 153,\n      "token_mask_sha256": "00670280358ee61951a804e979e4299c5c00d28c945549d9f9c8aaf60bb28906",\n      "native_gt_timestamp_sha256": "e2f3068192c3e13cc74227373edca69966b2d63606c0b68f09a4cb0efa985f7c"\n    },\n    {\n      "id": "train_ep18_mars_3_d30_1",\n      "epoch": 18,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 66.7,\n      "start_s": 46.7,\n      "end_s": 106.7,\n      "n_native_gt_outage": 118,\n      "n_native_gt_total": 238,\n      "token_mask_sha256": "63d833f0b713bb4fb84f25c62b86deb8f26260f7a627284500d089600ceaa5cc",\n      "native_gt_timestamp_sha256": "cc20a33d507071a24216fabb0fe444f3f5b5e7addb901f226a61fb1ea6d7a5a4"\n    },\n    {\n      "id": "train_ep18_mars_2_d30_0",\n      "epoch": 18,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 115.25,\n      "start_s": 95.25,\n      "end_s": 155.25,\n      "n_native_gt_outage": 129,\n      "n_native_gt_total": 233,\n      "token_mask_sha256": "ec337caf619f2fc34accfde4fefc63f5f100805a2dc4a30b788f58ddce2c4bc8",\n      "native_gt_timestamp_sha256": "df4c1494d18246db8afa1d46680d168715b86783ebb4a10314172e89df4287c6"\n    },\n    {\n      "id": "train_ep18_mars_1_d60_0",\n      "epoch": 18,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 29.6,\n      "start_s": 9.600000000000001,\n      "end_s": 99.6,\n      "n_native_gt_outage": 237,\n      "n_native_gt_total": 375,\n      "token_mask_sha256": "fed530af485dca69fb59f9b1a865da85e17ffd444c3c0d42d5f5017e0315ab5f",\n      "native_gt_timestamp_sha256": "3eec85600a74ea72993dfd710e238e88a0993cd52cecdbc2cf2c23daaca23f7a"\n    },\n    {\n      "id": "train_ep18_mars_3_d30_0",\n      "epoch": 18,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 71.9,\n      "start_s": 51.900000000000006,\n      "end_s": 111.9,\n      "n_native_gt_outage": 120,\n      "n_native_gt_total": 237,\n      "token_mask_sha256": "ec337caf619f2fc34accfde4fefc63f5f100805a2dc4a30b788f58ddce2c4bc8",\n      "native_gt_timestamp_sha256": "dee5f33725c851b58edb6c6f810ad20d524992f6716b2a30e4923ed959cccbb7"\n    },\n    {\n      "id": "train_ep18_mars_3_d60_1",\n      "epoch": 18,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 61.1,\n      "start_s": 41.1,\n      "end_s": 131.1,\n      "n_native_gt_outage": 237,\n      "n_native_gt_total": 348,\n      "token_mask_sha256": "944469817cb1d463be9ba8e22d7c037bf6b358337512a5973f0eedd13fdd691e",\n      "native_gt_timestamp_sha256": "531c76f4c87e44fc4031e4217e5b932f0c0f601aa6e4ace505f024b09e377df9"\n    },\n    {\n      "id": "train_ep18_mars_2_d60_1",\n      "epoch": 18,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 73.0,\n      "start_s": 53.0,\n      "end_s": 143.0,\n      "n_native_gt_outage": 249,\n      "n_native_gt_total": 369,\n      "token_mask_sha256": "fed530af485dca69fb59f9b1a865da85e17ffd444c3c0d42d5f5017e0315ab5f",\n      "native_gt_timestamp_sha256": "99cbb5af6d13ed89b14d52ede81b47363ae493eee05c823af208e6bcf3393442"\n    },\n    {\n      "id": "train_ep18_mars_3_d60_0",\n      "epoch": 18,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 63.95,\n      "start_s": 43.95,\n      "end_s": 133.95,\n      "n_native_gt_outage": 238,\n      "n_native_gt_total": 343,\n      "token_mask_sha256": "71da0041c5ffc52dc55131d6c02ef9fd40886bf07a3990e099fe3e7675dd9bab",\n      "native_gt_timestamp_sha256": "3ee4598d38a5c072dc8b6277df103eeee0ac5d9ea6f8f5a3bd2761127866147d"\n    },\n    {\n      "id": "train_ep18_mars_1_d30_0",\n      "epoch": 18,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 35.800000000000004,\n      "start_s": 15.800000000000004,\n      "end_s": 75.80000000000001,\n      "n_native_gt_outage": 114,\n      "n_native_gt_total": 252,\n      "token_mask_sha256": "b34423e399ec53c009571b199c4b245f2f0d58ef83150e9bf0611fa732314343",\n      "native_gt_timestamp_sha256": "fbd91b5dfad97babaf4eb91f2dfd488df15cfe9f7242e7173c10adc0208612ba"\n    },\n    {\n      "id": "train_ep18_mars_3_d10_1",\n      "epoch": 18,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 38.0,\n      "start_s": 18.0,\n      "end_s": 58.0,\n      "n_native_gt_outage": 41,\n      "n_native_gt_total": 154,\n      "token_mask_sha256": "21b9e16f0abe800c55f145a18abd0a79a37df3cbda1ac658e7e89003a42e7536",\n      "native_gt_timestamp_sha256": "f446f638afa34f16d35e06bbf26e1d17cfadde97acfb998c56da86cacd8b6db0"\n    },\n    {\n      "id": "train_ep18_mars_1_d60_1",\n      "epoch": 18,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 29.400000000000002,\n      "start_s": 9.400000000000002,\n      "end_s": 99.4,\n      "n_native_gt_outage": 239,\n      "n_native_gt_total": 375,\n      "token_mask_sha256": "fed530af485dca69fb59f9b1a865da85e17ffd444c3c0d42d5f5017e0315ab5f",\n      "native_gt_timestamp_sha256": "3eec85600a74ea72993dfd710e238e88a0993cd52cecdbc2cf2c23daaca23f7a"\n    },\n    {\n      "id": "train_ep18_mars_2_d10_1",\n      "epoch": 18,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 101.5,\n      "start_s": 81.5,\n      "end_s": 121.5,\n      "n_native_gt_outage": 36,\n      "n_native_gt_total": 166,\n      "token_mask_sha256": "21b9e16f0abe800c55f145a18abd0a79a37df3cbda1ac658e7e89003a42e7536",\n      "native_gt_timestamp_sha256": "55697d504e6da99eb26098710c56ae42e8e6b0eb7f1218e81cdcd948a29a4cb4"\n    },\n    {\n      "id": "train_ep18_mars_2_d30_1",\n      "epoch": 18,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 35.95,\n      "start_s": 15.950000000000003,\n      "end_s": 75.95,\n      "n_native_gt_outage": 116,\n      "n_native_gt_total": 234,\n      "token_mask_sha256": "3dd74d234f7a60145505caa41ec0613e9d52eb9cbdc78e09d68e05c0d586d85a",\n      "native_gt_timestamp_sha256": "97ee9d308e42e29b5705bcdc523c7a21224b6e17b487dc32ef82322ff9b43e33"\n    },\n    {\n      "id": "train_ep18_mars_2_d60_0",\n      "epoch": 18,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 25.5,\n      "start_s": 5.5,\n      "end_s": 95.5,\n      "n_native_gt_outage": 234,\n      "n_native_gt_total": 364,\n      "token_mask_sha256": "71da0041c5ffc52dc55131d6c02ef9fd40886bf07a3990e099fe3e7675dd9bab",\n      "native_gt_timestamp_sha256": "b2283151effbb4b19244317c2038363d8dad5b3e1ea96098644204e456178d4f"\n    },\n    {\n      "id": "train_ep18_mars_1_d10_1",\n      "epoch": 18,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 39.550000000000004,\n      "start_s": 19.550000000000004,\n      "end_s": 59.550000000000004,\n      "n_native_gt_outage": 51,\n      "n_native_gt_total": 161,\n      "token_mask_sha256": "21b9e16f0abe800c55f145a18abd0a79a37df3cbda1ac658e7e89003a42e7536",\n      "native_gt_timestamp_sha256": "3dd8a138542e8a7a9518e86b59d22198dda35e178f22967f36dc41ad40f30394"\n    },\n    {\n      "id": "train_ep19_mars_1_d60_0",\n      "epoch": 19,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 27.700000000000003,\n      "start_s": 7.700000000000003,\n      "end_s": 97.7,\n      "n_native_gt_outage": 240,\n      "n_native_gt_total": 379,\n      "token_mask_sha256": "71da0041c5ffc52dc55131d6c02ef9fd40886bf07a3990e099fe3e7675dd9bab",\n      "native_gt_timestamp_sha256": "c2c314f9ec15963205a251c6774f126948d60f1de5ced77754a4ed91ffb228f9"\n    },\n    {\n      "id": "train_ep19_mars_2_d60_1",\n      "epoch": 19,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 88.0,\n      "start_s": 68.0,\n      "end_s": 158.0,\n      "n_native_gt_outage": 251,\n      "n_native_gt_total": 367,\n      "token_mask_sha256": "944469817cb1d463be9ba8e22d7c037bf6b358337512a5973f0eedd13fdd691e",\n      "native_gt_timestamp_sha256": "c2f5ed681f12be6c3aa9b7caab302860b35f83f63447abbff124aec4335b03b5"\n    },\n    {\n      "id": "train_ep19_mars_3_d30_0",\n      "epoch": 19,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 85.95,\n      "start_s": 65.95,\n      "end_s": 125.95,\n      "n_native_gt_outage": 118,\n      "n_native_gt_total": 235,\n      "token_mask_sha256": "2100feebce9686fec0ee7cc1b5a2e4958b681c407208eec9e7526b6ac135655b",\n      "native_gt_timestamp_sha256": "1ccacaa4d78bf6b9ae787f4ff73cd4fc2eef3e3b61825be8b2b6372150a02c39"\n    },\n    {\n      "id": "train_ep19_mars_1_d60_1",\n      "epoch": 19,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 23.8,\n      "start_s": 3.8000000000000007,\n      "end_s": 93.8,\n      "n_native_gt_outage": 252,\n      "n_native_gt_total": 374,\n      "token_mask_sha256": "fed530af485dca69fb59f9b1a865da85e17ffd444c3c0d42d5f5017e0315ab5f",\n      "native_gt_timestamp_sha256": "98752201d03a1786fa3050f302e32a6e2b936b0e3e93665b6da36b85c435ab75"\n    },\n    {\n      "id": "train_ep19_mars_3_d10_1",\n      "epoch": 19,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 75.5,\n      "start_s": 55.5,\n      "end_s": 95.5,\n      "n_native_gt_outage": 52,\n      "n_native_gt_total": 154,\n      "token_mask_sha256": "54d1f8b7a9217c3e8206fec44aa90fc97fdb3e89370798f9369d21bcb530fdd3",\n      "native_gt_timestamp_sha256": "4f2a2df6b01effa8ed5a2a4308a7aba4c92375e99c6bc61245cdf23f995f5dd8"\n    },\n    {\n      "id": "train_ep19_mars_2_d30_1",\n      "epoch": 19,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 45.050000000000004,\n      "start_s": 25.050000000000004,\n      "end_s": 85.05000000000001,\n      "n_native_gt_outage": 130,\n      "n_native_gt_total": 234,\n      "token_mask_sha256": "ec337caf619f2fc34accfde4fefc63f5f100805a2dc4a30b788f58ddce2c4bc8",\n      "native_gt_timestamp_sha256": "1e8025e5971fecd1c83b5b67a4f97b2ac9eb82c487954e06b7f5e9118e2af55f"\n    },\n    {\n      "id": "train_ep19_mars_1_d30_0",\n      "epoch": 19,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 30.650000000000002,\n      "start_s": 10.650000000000002,\n      "end_s": 70.65,\n      "n_native_gt_outage": 111,\n      "n_native_gt_total": 245,\n      "token_mask_sha256": "63d833f0b713bb4fb84f25c62b86deb8f26260f7a627284500d089600ceaa5cc",\n      "native_gt_timestamp_sha256": "a1b91fd00aa235ec24b15ef2911d65a0a10d7e4261a97057aae32ba800d47ff2"\n    },\n    {\n      "id": "train_ep19_mars_2_d10_0",\n      "epoch": 19,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 132.75,\n      "start_s": 112.75,\n      "end_s": 152.75,\n      "n_native_gt_outage": 36,\n      "n_native_gt_total": 162,\n      "token_mask_sha256": "00670280358ee61951a804e979e4299c5c00d28c945549d9f9c8aaf60bb28906",\n      "native_gt_timestamp_sha256": "57e9121c1b6a677955dfe00beabef3cf0f587dcac6ff22aafd7353ca953cf156"\n    },\n    {\n      "id": "train_ep19_mars_1_d30_1",\n      "epoch": 19,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 26.1,\n      "start_s": 6.100000000000001,\n      "end_s": 66.1,\n      "n_native_gt_outage": 121,\n      "n_native_gt_total": 239,\n      "token_mask_sha256": "ec337caf619f2fc34accfde4fefc63f5f100805a2dc4a30b788f58ddce2c4bc8",\n      "native_gt_timestamp_sha256": "0bb70a160d86f2c7faaadb8ff476d4f48615ec41b5fbcf825f43ef42e2410593"\n    },\n    {\n      "id": "train_ep19_mars_1_d10_0",\n      "epoch": 19,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 60.400000000000006,\n      "start_s": 40.400000000000006,\n      "end_s": 80.4,\n      "n_native_gt_outage": 48,\n      "n_native_gt_total": 171,\n      "token_mask_sha256": "860d531b40010f5a1ce4a6d8d4c9b3f3cb34a9b01d0a5e57422ce1faf23b77a3",\n      "native_gt_timestamp_sha256": "0bfd0fcb17f402d98aab407e90d4babf6172cc2dc94b2f35e42045d0996a533f"\n    },\n    {\n      "id": "train_ep19_mars_1_d10_1",\n      "epoch": 19,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 51.95,\n      "start_s": 31.950000000000003,\n      "end_s": 71.95,\n      "n_native_gt_outage": 31,\n      "n_native_gt_total": 165,\n      "token_mask_sha256": "21b9e16f0abe800c55f145a18abd0a79a37df3cbda1ac658e7e89003a42e7536",\n      "native_gt_timestamp_sha256": "9895e9f103ab9946beb41a09bd75d83b666220ca539e421ae8dcd85fbfc23a21"\n    },\n    {\n      "id": "train_ep19_mars_2_d10_1",\n      "epoch": 19,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 130.45000000000002,\n      "start_s": 110.45000000000002,\n      "end_s": 150.45000000000002,\n      "n_native_gt_outage": 36,\n      "n_native_gt_total": 162,\n      "token_mask_sha256": "9f12542780490d612306b1de6383455d9c904bc5b73f8b9cebd020d927653a95",\n      "native_gt_timestamp_sha256": "6f79bb277c454e1e9d68fb938ac3b76f037acc6f7229ebe74bca99191df0d080"\n    },\n    {\n      "id": "train_ep19_mars_2_d60_0",\n      "epoch": 19,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 26.8,\n      "start_s": 6.800000000000001,\n      "end_s": 96.8,\n      "n_native_gt_outage": 235,\n      "n_native_gt_total": 361,\n      "token_mask_sha256": "fed530af485dca69fb59f9b1a865da85e17ffd444c3c0d42d5f5017e0315ab5f",\n      "native_gt_timestamp_sha256": "2e84e5ebf71ef63be79e614868cf37ea5e46cf6c15e6b7bc284f2d00ebaff818"\n    },\n    {\n      "id": "train_ep19_mars_3_d30_1",\n      "epoch": 19,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 37.1,\n      "start_s": 17.1,\n      "end_s": 77.1,\n      "n_native_gt_outage": 115,\n      "n_native_gt_total": 223,\n      "token_mask_sha256": "63d833f0b713bb4fb84f25c62b86deb8f26260f7a627284500d089600ceaa5cc",\n      "native_gt_timestamp_sha256": "144d0c4ac1003207432414f812926609be037275cbd43008622267e597dee055"\n    },\n    {\n      "id": "train_ep19_mars_2_d30_0",\n      "epoch": 19,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 55.95,\n      "start_s": 35.95,\n      "end_s": 95.95,\n      "n_native_gt_outage": 118,\n      "n_native_gt_total": 251,\n      "token_mask_sha256": "3dd74d234f7a60145505caa41ec0613e9d52eb9cbdc78e09d68e05c0d586d85a",\n      "native_gt_timestamp_sha256": "05087b9fb498427075791284c0c04d594e7cf3cb0c06cb09b5b3c49c9f6b2895"\n    },\n    {\n      "id": "train_ep19_mars_3_d60_0",\n      "epoch": 19,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 59.050000000000004,\n      "start_s": 39.050000000000004,\n      "end_s": 129.05,\n      "n_native_gt_outage": 235,\n      "n_native_gt_total": 348,\n      "token_mask_sha256": "fed530af485dca69fb59f9b1a865da85e17ffd444c3c0d42d5f5017e0315ab5f",\n      "native_gt_timestamp_sha256": "fabc97f29bd58b031c3c70cc8aed10cc82c6c01d543b30b86a73234ae7e5c556"\n    },\n    {\n      "id": "train_ep19_mars_3_d10_0",\n      "epoch": 19,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 60.25,\n      "start_s": 40.25,\n      "end_s": 80.25,\n      "n_native_gt_outage": 35,\n      "n_native_gt_total": 158,\n      "token_mask_sha256": "860d531b40010f5a1ce4a6d8d4c9b3f3cb34a9b01d0a5e57422ce1faf23b77a3",\n      "native_gt_timestamp_sha256": "b03bb15c526d7f02bcc5013a8fd3c13fcf2a311cafd0c97a5c49c8fc5bd860c2"\n    },\n    {\n      "id": "train_ep19_mars_3_d60_1",\n      "epoch": 19,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 44.35,\n      "start_s": 24.35,\n      "end_s": 114.35,\n      "n_native_gt_outage": 231,\n      "n_native_gt_total": 355,\n      "token_mask_sha256": "71da0041c5ffc52dc55131d6c02ef9fd40886bf07a3990e099fe3e7675dd9bab",\n      "native_gt_timestamp_sha256": "765ca2ed0e7f84f30ace3159a832ed607719d42ac471d80ae9a80c9124bf42fc"\n    },\n    {\n      "id": "train_ep20_mars_2_d60_0",\n      "epoch": 20,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 34.75,\n      "start_s": 14.75,\n      "end_s": 104.75,\n      "n_native_gt_outage": 251,\n      "n_native_gt_total": 362,\n      "token_mask_sha256": "c0c2ec8564a28a400ecdd4fe90d4d6b443981b414ff75cc3098c2fd7dc691a8b",\n      "native_gt_timestamp_sha256": "ebca127c4b893ddbc6a45484112ceccab146cb915bf2484fbbcf5db3ff2f6144"\n    },\n    {\n      "id": "train_ep20_mars_1_d10_1",\n      "epoch": 20,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 78.2,\n      "start_s": 58.2,\n      "end_s": 98.2,\n      "n_native_gt_outage": 37,\n      "n_native_gt_total": 179,\n      "token_mask_sha256": "860d531b40010f5a1ce4a6d8d4c9b3f3cb34a9b01d0a5e57422ce1faf23b77a3",\n      "native_gt_timestamp_sha256": "7d21fe2d18a287f84ef5159e48d45245162796daca68074529469d6be0c8caf7"\n    },\n    {\n      "id": "train_ep20_mars_1_d30_0",\n      "epoch": 20,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 36.050000000000004,\n      "start_s": 16.050000000000004,\n      "end_s": 76.05000000000001,\n      "n_native_gt_outage": 114,\n      "n_native_gt_total": 250,\n      "token_mask_sha256": "63d833f0b713bb4fb84f25c62b86deb8f26260f7a627284500d089600ceaa5cc",\n      "native_gt_timestamp_sha256": "cefc241f53b507ade4fd46a6b2eb664f3af70447c8f3d5a1aa8216034b5fce22"\n    },\n    {\n      "id": "train_ep20_mars_2_d10_1",\n      "epoch": 20,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 72.10000000000001,\n      "start_s": 52.10000000000001,\n      "end_s": 92.10000000000001,\n      "n_native_gt_outage": 36,\n      "n_native_gt_total": 165,\n      "token_mask_sha256": "9f12542780490d612306b1de6383455d9c904bc5b73f8b9cebd020d927653a95",\n      "native_gt_timestamp_sha256": "76200c0eb305d08dfe5a9a64502350a37550f058b7b6c043031d51655dfc461f"\n    },\n    {\n      "id": "train_ep20_mars_3_d30_1",\n      "epoch": 20,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 76.60000000000001,\n      "start_s": 56.60000000000001,\n      "end_s": 116.60000000000001,\n      "n_native_gt_outage": 131,\n      "n_native_gt_total": 236,\n      "token_mask_sha256": "3dd74d234f7a60145505caa41ec0613e9d52eb9cbdc78e09d68e05c0d586d85a",\n      "native_gt_timestamp_sha256": "73d27a06d503b2561e5e6e6956bc08ddaf63ec8d278651b5dd207cfcb72e5eb9"\n    },\n    {\n      "id": "train_ep20_mars_1_d60_1",\n      "epoch": 20,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 26.75,\n      "start_s": 6.75,\n      "end_s": 96.75,\n      "n_native_gt_outage": 244,\n      "n_native_gt_total": 378,\n      "token_mask_sha256": "c0c2ec8564a28a400ecdd4fe90d4d6b443981b414ff75cc3098c2fd7dc691a8b",\n      "native_gt_timestamp_sha256": "f82a1c0b6818a6abd4060f5ed9166386a4773f421d349bbfa6655ec72ca36ff1"\n    },\n    {\n      "id": "train_ep20_mars_1_d60_0",\n      "epoch": 20,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 21.900000000000002,\n      "start_s": 1.9000000000000021,\n      "end_s": 91.9,\n      "n_native_gt_outage": 250,\n      "n_native_gt_total": 371,\n      "token_mask_sha256": "71da0041c5ffc52dc55131d6c02ef9fd40886bf07a3990e099fe3e7675dd9bab",\n      "native_gt_timestamp_sha256": "2c48084aa530bd4648f6ced1eaf1426323a094d7414da3c977bfa0fb89a029c4"\n    },\n    {\n      "id": "train_ep20_mars_3_d10_0",\n      "epoch": 20,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 116.55000000000001,\n      "start_s": 96.55000000000001,\n      "end_s": 136.55,\n      "n_native_gt_outage": 35,\n      "n_native_gt_total": 158,\n      "token_mask_sha256": "9f12542780490d612306b1de6383455d9c904bc5b73f8b9cebd020d927653a95",\n      "native_gt_timestamp_sha256": "9ccfb7f547432521db5041269d45ecbc58902099e718509d13472a3431829a97"\n    },\n    {\n      "id": "train_ep20_mars_2_d10_0",\n      "epoch": 20,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 86.05000000000001,\n      "start_s": 66.05000000000001,\n      "end_s": 106.05000000000001,\n      "n_native_gt_outage": 54,\n      "n_native_gt_total": 172,\n      "token_mask_sha256": "54d1f8b7a9217c3e8206fec44aa90fc97fdb3e89370798f9369d21bcb530fdd3",\n      "native_gt_timestamp_sha256": "a57ee30a6d063df9aaafb79758b7314fc0b1525c50cd1b07253a8638226862ee"\n    },\n    {\n      "id": "train_ep20_mars_2_d30_0",\n      "epoch": 20,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 69.8,\n      "start_s": 49.8,\n      "end_s": 109.8,\n      "n_native_gt_outage": 127,\n      "n_native_gt_total": 246,\n      "token_mask_sha256": "b34423e399ec53c009571b199c4b245f2f0d58ef83150e9bf0611fa732314343",\n      "native_gt_timestamp_sha256": "a5ead2ab134f495a46f9f4f1e64d0e89a15e730a2712f142b490c48779df2821"\n    },\n    {\n      "id": "train_ep20_mars_3_d60_1",\n      "epoch": 20,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 73.60000000000001,\n      "start_s": 53.60000000000001,\n      "end_s": 143.60000000000002,\n      "n_native_gt_outage": 235,\n      "n_native_gt_total": 364,\n      "token_mask_sha256": "478ce3b3d3dce9522789ea95341f5ae713e75348c57aff32b390220cd55f9144",\n      "native_gt_timestamp_sha256": "941f6ba376d31a47e03e8ec9c89387515a1e2df832c4b37d096f73aa954781e4"\n    },\n    {\n      "id": "train_ep20_mars_3_d10_1",\n      "epoch": 20,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 81.65,\n      "start_s": 61.650000000000006,\n      "end_s": 101.65,\n      "n_native_gt_outage": 36,\n      "n_native_gt_total": 153,\n      "token_mask_sha256": "54d1f8b7a9217c3e8206fec44aa90fc97fdb3e89370798f9369d21bcb530fdd3",\n      "native_gt_timestamp_sha256": "df195db6b4093d3e8bc74095eb1934dce2a6ea2b052fb0e9bb3e8d40caf26a8b"\n    },\n    {\n      "id": "train_ep20_mars_2_d30_1",\n      "epoch": 20,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 51.800000000000004,\n      "start_s": 31.800000000000004,\n      "end_s": 91.80000000000001,\n      "n_native_gt_outage": 118,\n      "n_native_gt_total": 246,\n      "token_mask_sha256": "b34423e399ec53c009571b199c4b245f2f0d58ef83150e9bf0611fa732314343",\n      "native_gt_timestamp_sha256": "bba7f69704da43cb409e13d1944694e0afc9e40bf96bc427153fdbf2d5c4b05d"\n    },\n    {\n      "id": "train_ep20_mars_2_d60_1",\n      "epoch": 20,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 42.400000000000006,\n      "start_s": 22.400000000000006,\n      "end_s": 112.4,\n      "n_native_gt_outage": 254,\n      "n_native_gt_total": 357,\n      "token_mask_sha256": "fed530af485dca69fb59f9b1a865da85e17ffd444c3c0d42d5f5017e0315ab5f",\n      "native_gt_timestamp_sha256": "7a39ee99296661db9fd4838596337d3996d730f9b1bb68d4905d72112da0ac7a"\n    },\n    {\n      "id": "train_ep20_mars_3_d30_0",\n      "epoch": 20,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 52.0,\n      "start_s": 32.0,\n      "end_s": 92.0,\n      "n_native_gt_outage": 121,\n      "n_native_gt_total": 236,\n      "token_mask_sha256": "b34423e399ec53c009571b199c4b245f2f0d58ef83150e9bf0611fa732314343",\n      "native_gt_timestamp_sha256": "aca66175395fbb108dae8e2b62a6ac93573f21649331fe895a3cd7205b68f586"\n    },\n    {\n      "id": "train_ep20_mars_3_d60_0",\n      "epoch": 20,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 60.35,\n      "start_s": 40.35,\n      "end_s": 130.35,\n      "n_native_gt_outage": 236,\n      "n_native_gt_total": 349,\n      "token_mask_sha256": "71da0041c5ffc52dc55131d6c02ef9fd40886bf07a3990e099fe3e7675dd9bab",\n      "native_gt_timestamp_sha256": "625a466174084882cea56604d59e4fb809cf0e2266e9b849006858300d8aed6d"\n    },\n    {\n      "id": "train_ep20_mars_1_d30_1",\n      "epoch": 20,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 26.0,\n      "start_s": 6.0,\n      "end_s": 66.0,\n      "n_native_gt_outage": 121,\n      "n_native_gt_total": 240,\n      "token_mask_sha256": "b34423e399ec53c009571b199c4b245f2f0d58ef83150e9bf0611fa732314343",\n      "native_gt_timestamp_sha256": "c2e1c2124b339494126a42d4f7dc553fbf44045b1f549fb48dc73c77ab7ca9f1"\n    },\n    {\n      "id": "train_ep20_mars_1_d10_0",\n      "epoch": 20,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 27.35,\n      "start_s": 7.350000000000001,\n      "end_s": 47.35,\n      "n_native_gt_outage": 36,\n      "n_native_gt_total": 165,\n      "token_mask_sha256": "21b9e16f0abe800c55f145a18abd0a79a37df3cbda1ac658e7e89003a42e7536",\n      "native_gt_timestamp_sha256": "cb6e19ebba4eaa7c36b1f026e6fe52ba58a0e9557dcaebce6d274c76e7ee56d9"\n    },\n    {\n      "id": "train_ep21_mars_3_d10_0",\n      "epoch": 21,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 31.5,\n      "start_s": 11.5,\n      "end_s": 51.5,\n      "n_native_gt_outage": 40,\n      "n_native_gt_total": 159,\n      "token_mask_sha256": "9f12542780490d612306b1de6383455d9c904bc5b73f8b9cebd020d927653a95",\n      "native_gt_timestamp_sha256": "c21afabfeb44307151266140996bf7373b3dd612a493640eb58a9dd928387c26"\n    },\n    {\n      "id": "train_ep21_mars_2_d30_0",\n      "epoch": 21,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 93.60000000000001,\n      "start_s": 73.60000000000001,\n      "end_s": 133.60000000000002,\n      "n_native_gt_outage": 117,\n      "n_native_gt_total": 251,\n      "token_mask_sha256": "b34423e399ec53c009571b199c4b245f2f0d58ef83150e9bf0611fa732314343",\n      "native_gt_timestamp_sha256": "0f5ebaa0613d8d04f0193c1e8fafe9b7c83da5be44716bbaca83aeb3ab556838"\n    },\n    {\n      "id": "train_ep21_mars_1_d60_1",\n      "epoch": 21,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 28.150000000000002,\n      "start_s": 8.150000000000002,\n      "end_s": 98.15,\n      "n_native_gt_outage": 240,\n      "n_native_gt_total": 379,\n      "token_mask_sha256": "c0c2ec8564a28a400ecdd4fe90d4d6b443981b414ff75cc3098c2fd7dc691a8b",\n      "native_gt_timestamp_sha256": "cb6d73ce0bbee40977262189869b15661e94498297675e640588cd4544d12b0a"\n    },\n    {\n      "id": "train_ep21_mars_3_d60_1",\n      "epoch": 21,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 49.85,\n      "start_s": 29.85,\n      "end_s": 119.85,\n      "n_native_gt_outage": 238,\n      "n_native_gt_total": 356,\n      "token_mask_sha256": "fed530af485dca69fb59f9b1a865da85e17ffd444c3c0d42d5f5017e0315ab5f",\n      "native_gt_timestamp_sha256": "1388c7bf3afce55dc3c5713962eb9f4553012a9bd0935b326fcfa85bfd7b0e61"\n    },\n    {\n      "id": "train_ep21_mars_1_d60_0",\n      "epoch": 21,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 26.75,\n      "start_s": 6.75,\n      "end_s": 96.75,\n      "n_native_gt_outage": 244,\n      "n_native_gt_total": 378,\n      "token_mask_sha256": "c0c2ec8564a28a400ecdd4fe90d4d6b443981b414ff75cc3098c2fd7dc691a8b",\n      "native_gt_timestamp_sha256": "f82a1c0b6818a6abd4060f5ed9166386a4773f421d349bbfa6655ec72ca36ff1"\n    },\n    {\n      "id": "train_ep21_mars_2_d10_0",\n      "epoch": 21,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 84.65,\n      "start_s": 64.65,\n      "end_s": 104.65,\n      "n_native_gt_outage": 52,\n      "n_native_gt_total": 172,\n      "token_mask_sha256": "54d1f8b7a9217c3e8206fec44aa90fc97fdb3e89370798f9369d21bcb530fdd3",\n      "native_gt_timestamp_sha256": "ceec06567a4c47f39ca3a14e9011fe10eda91cacbeb4e45a9545470899a93d1e"\n    },\n    {\n      "id": "train_ep21_mars_1_d30_0",\n      "epoch": 21,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 53.300000000000004,\n      "start_s": 33.300000000000004,\n      "end_s": 93.30000000000001,\n      "n_native_gt_outage": 122,\n      "n_native_gt_total": 242,\n      "token_mask_sha256": "ec337caf619f2fc34accfde4fefc63f5f100805a2dc4a30b788f58ddce2c4bc8",\n      "native_gt_timestamp_sha256": "c62d1ff55e1e7bef0f09e52ee5db4e7394e5490e29a2d721b68ac224c54f96eb"\n    },\n    {\n      "id": "train_ep21_mars_2_d30_1",\n      "epoch": 21,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 26.450000000000003,\n      "start_s": 6.450000000000003,\n      "end_s": 66.45,\n      "n_native_gt_outage": 117,\n      "n_native_gt_total": 229,\n      "token_mask_sha256": "63d833f0b713bb4fb84f25c62b86deb8f26260f7a627284500d089600ceaa5cc",\n      "native_gt_timestamp_sha256": "c602222ad3bcf20e91ef8380171bf1f7f3ff23d52d5a77c9665ad494afbfdce4"\n    },\n    {\n      "id": "train_ep21_mars_2_d60_0",\n      "epoch": 21,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 88.4,\n      "start_s": 68.4,\n      "end_s": 158.4,\n      "n_native_gt_outage": 253,\n      "n_native_gt_total": 368,\n      "token_mask_sha256": "944469817cb1d463be9ba8e22d7c037bf6b358337512a5973f0eedd13fdd691e",\n      "native_gt_timestamp_sha256": "c014654b7c4f607f3e5b13383daa6d45c9a2b872c45dc0410f40b042e1d39ff9"\n    },\n    {\n      "id": "train_ep21_mars_3_d30_1",\n      "epoch": 21,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 89.9,\n      "start_s": 69.9,\n      "end_s": 129.9,\n      "n_native_gt_outage": 119,\n      "n_native_gt_total": 237,\n      "token_mask_sha256": "63d833f0b713bb4fb84f25c62b86deb8f26260f7a627284500d089600ceaa5cc",\n      "native_gt_timestamp_sha256": "566ea04f9a729820082dac4c8362755a524b79a0fa08d3095fa797de6361a4a7"\n    },\n    {\n      "id": "train_ep21_mars_3_d60_0",\n      "epoch": 21,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 43.550000000000004,\n      "start_s": 23.550000000000004,\n      "end_s": 113.55000000000001,\n      "n_native_gt_outage": 230,\n      "n_native_gt_total": 356,\n      "token_mask_sha256": "71da0041c5ffc52dc55131d6c02ef9fd40886bf07a3990e099fe3e7675dd9bab",\n      "native_gt_timestamp_sha256": "492116f20349df3548f26085a825e4b255f2651a357aad4b0359f97162782e00"\n    },\n    {\n      "id": "train_ep21_mars_3_d30_0",\n      "epoch": 21,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 29.25,\n      "start_s": 9.25,\n      "end_s": 69.25,\n      "n_native_gt_outage": 118,\n      "n_native_gt_total": 238,\n      "token_mask_sha256": "b34423e399ec53c009571b199c4b245f2f0d58ef83150e9bf0611fa732314343",\n      "native_gt_timestamp_sha256": "c9a4d2c4625614752bb9517d341b42e19c2e2a27556bc51501262cb4eec51256"\n    },\n    {\n      "id": "train_ep21_mars_1_d10_1",\n      "epoch": 21,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 37.6,\n      "start_s": 17.6,\n      "end_s": 57.6,\n      "n_native_gt_outage": 44,\n      "n_native_gt_total": 162,\n      "token_mask_sha256": "860d531b40010f5a1ce4a6d8d4c9b3f3cb34a9b01d0a5e57422ce1faf23b77a3",\n      "native_gt_timestamp_sha256": "0e341f2e83e3e0d2b7cf3470a2a310a968c2030641ea1d567adc8ea406394475"\n    },\n    {\n      "id": "train_ep21_mars_2_d10_1",\n      "epoch": 21,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 61.800000000000004,\n      "start_s": 41.800000000000004,\n      "end_s": 81.80000000000001,\n      "n_native_gt_outage": 48,\n      "n_native_gt_total": 163,\n      "token_mask_sha256": "54d1f8b7a9217c3e8206fec44aa90fc97fdb3e89370798f9369d21bcb530fdd3",\n      "native_gt_timestamp_sha256": "8656ed1849f3d8c4cb690d1b43f6a4dcc16af9291a6254be6987263bb4cec85c"\n    },\n    {\n      "id": "train_ep21_mars_3_d10_1",\n      "epoch": 21,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 57.800000000000004,\n      "start_s": 37.800000000000004,\n      "end_s": 77.80000000000001,\n      "n_native_gt_outage": 34,\n      "n_native_gt_total": 150,\n      "token_mask_sha256": "21b9e16f0abe800c55f145a18abd0a79a37df3cbda1ac658e7e89003a42e7536",\n      "native_gt_timestamp_sha256": "6def97c2e0ceaa94823da86244c3fde8e6db8ea3ca24eeef3286b9eb370241b6"\n    },\n    {\n      "id": "train_ep21_mars_1_d10_0",\n      "epoch": 21,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 30.85,\n      "start_s": 10.850000000000001,\n      "end_s": 50.85,\n      "n_native_gt_outage": 31,\n      "n_native_gt_total": 166,\n      "token_mask_sha256": "bd368bef35525fa77168e8ce0433a5bdf8d7c81095da65febd051dcdb9e712cb",\n      "native_gt_timestamp_sha256": "c88008ae65b0c4afff61755e73a482d4cd07b7e044487b5218e5fe45b96b2fd2"\n    },\n    {\n      "id": "train_ep21_mars_1_d30_1",\n      "epoch": 21,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 39.45,\n      "start_s": 19.450000000000003,\n      "end_s": 79.45,\n      "n_native_gt_outage": 126,\n      "n_native_gt_total": 250,\n      "token_mask_sha256": "63d833f0b713bb4fb84f25c62b86deb8f26260f7a627284500d089600ceaa5cc",\n      "native_gt_timestamp_sha256": "3706007844abbb2f04c3489f9e215e6ea4a1893b85bcae04b4c414bff3bbb9f4"\n    },\n    {\n      "id": "train_ep21_mars_2_d60_1",\n      "epoch": 21,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 47.400000000000006,\n      "start_s": 27.400000000000006,\n      "end_s": 117.4,\n      "n_native_gt_outage": 253,\n      "n_native_gt_total": 359,\n      "token_mask_sha256": "fed530af485dca69fb59f9b1a865da85e17ffd444c3c0d42d5f5017e0315ab5f",\n      "native_gt_timestamp_sha256": "e5a875a625d795a8c9fa1ec8670e824a3e783c16534c8d1e8dcfd716d31fef46"\n    },\n    {\n      "id": "train_ep22_mars_3_d30_1",\n      "epoch": 22,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 56.400000000000006,\n      "start_s": 36.400000000000006,\n      "end_s": 96.4,\n      "n_native_gt_outage": 118,\n      "n_native_gt_total": 234,\n      "token_mask_sha256": "3dd74d234f7a60145505caa41ec0613e9d52eb9cbdc78e09d68e05c0d586d85a",\n      "native_gt_timestamp_sha256": "8f821a0247ae0e75cbab9f99d97c6416742ffc07d51d11f92512d815d04f7208"\n    },\n    {\n      "id": "train_ep22_mars_1_d60_0",\n      "epoch": 22,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 21.450000000000003,\n      "start_s": 1.4500000000000028,\n      "end_s": 91.45,\n      "n_native_gt_outage": 252,\n      "n_native_gt_total": 370,\n      "token_mask_sha256": "944469817cb1d463be9ba8e22d7c037bf6b358337512a5973f0eedd13fdd691e",\n      "native_gt_timestamp_sha256": "79d9d171b363fb4e34197e9924fe35bb1f1cd07fbe369457561076682451ef75"\n    },\n    {\n      "id": "train_ep22_mars_2_d10_1",\n      "epoch": 22,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 42.85,\n      "start_s": 22.85,\n      "end_s": 62.85,\n      "n_native_gt_outage": 47,\n      "n_native_gt_total": 149,\n      "token_mask_sha256": "54d1f8b7a9217c3e8206fec44aa90fc97fdb3e89370798f9369d21bcb530fdd3",\n      "native_gt_timestamp_sha256": "d13145a43644485d4ea78afc1e0a6bed76927edbeb2e8b690c7fb98391ac0ead"\n    },\n    {\n      "id": "train_ep22_mars_2_d30_0",\n      "epoch": 22,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 25.450000000000003,\n      "start_s": 5.450000000000003,\n      "end_s": 65.45,\n      "n_native_gt_outage": 115,\n      "n_native_gt_total": 231,\n      "token_mask_sha256": "63d833f0b713bb4fb84f25c62b86deb8f26260f7a627284500d089600ceaa5cc",\n      "native_gt_timestamp_sha256": "e7b7797684f8b964183414c5002bfb5848a6effcda089e71d3a3e88cf23d539b"\n    },\n    {\n      "id": "train_ep22_mars_2_d60_1",\n      "epoch": 22,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 92.4,\n      "start_s": 72.4,\n      "end_s": 162.4,\n      "n_native_gt_outage": 239,\n      "n_native_gt_total": 376,\n      "token_mask_sha256": "944469817cb1d463be9ba8e22d7c037bf6b358337512a5973f0eedd13fdd691e",\n      "native_gt_timestamp_sha256": "84924dd28f5479f7dfa5cc5d998d4209a0aa9025af6996114913f59b3d5e9f9f"\n    },\n    {\n      "id": "train_ep22_mars_1_d60_1",\n      "epoch": 22,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 28.5,\n      "start_s": 8.5,\n      "end_s": 98.5,\n      "n_native_gt_outage": 237,\n      "n_native_gt_total": 376,\n      "token_mask_sha256": "71da0041c5ffc52dc55131d6c02ef9fd40886bf07a3990e099fe3e7675dd9bab",\n      "native_gt_timestamp_sha256": "0c0e2ed65519d1f591135387fc836a6492f8323a644570c9fe43d8b6bfda828d"\n    },\n    {\n      "id": "train_ep22_mars_2_d30_1",\n      "epoch": 22,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 86.05000000000001,\n      "start_s": 66.05000000000001,\n      "end_s": 126.05000000000001,\n      "n_native_gt_outage": 124,\n      "n_native_gt_total": 261,\n      "token_mask_sha256": "63d833f0b713bb4fb84f25c62b86deb8f26260f7a627284500d089600ceaa5cc",\n      "native_gt_timestamp_sha256": "03b6a3e49431fb103f17bd8292955bee7a3b5aad66f8a3e9158e99e38eee00fc"\n    },\n    {\n      "id": "train_ep22_mars_3_d60_1",\n      "epoch": 22,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 56.550000000000004,\n      "start_s": 36.550000000000004,\n      "end_s": 126.55000000000001,\n      "n_native_gt_outage": 236,\n      "n_native_gt_total": 352,\n      "token_mask_sha256": "71da0041c5ffc52dc55131d6c02ef9fd40886bf07a3990e099fe3e7675dd9bab",\n      "native_gt_timestamp_sha256": "f0c2c4c6b118edc711bc7708f06ea4aa6c3d4fb238c0438d06e155eeb62adee0"\n    },\n    {\n      "id": "train_ep22_mars_1_d10_0",\n      "epoch": 22,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 63.1,\n      "start_s": 43.1,\n      "end_s": 83.1,\n      "n_native_gt_outage": 56,\n      "n_native_gt_total": 170,\n      "token_mask_sha256": "9f12542780490d612306b1de6383455d9c904bc5b73f8b9cebd020d927653a95",\n      "native_gt_timestamp_sha256": "c572d44433a185aa13825e07d675812d34e8c5a0d011c2bebcbbcf784700d111"\n    },\n    {\n      "id": "train_ep22_mars_1_d30_0",\n      "epoch": 22,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 50.25,\n      "start_s": 30.25,\n      "end_s": 90.25,\n      "n_native_gt_outage": 123,\n      "n_native_gt_total": 239,\n      "token_mask_sha256": "63d833f0b713bb4fb84f25c62b86deb8f26260f7a627284500d089600ceaa5cc",\n      "native_gt_timestamp_sha256": "cdc38f65194d81ec9421f07837e106cdb29cd9d9cfcae2398af4c56037b050a8"\n    },\n    {\n      "id": "train_ep22_mars_3_d30_0",\n      "epoch": 22,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 79.65,\n      "start_s": 59.650000000000006,\n      "end_s": 119.65,\n      "n_native_gt_outage": 125,\n      "n_native_gt_total": 238,\n      "token_mask_sha256": "63d833f0b713bb4fb84f25c62b86deb8f26260f7a627284500d089600ceaa5cc",\n      "native_gt_timestamp_sha256": "db1aede55a56b244b39b6014e66aebfa8b0b45552a46d6eeebde6519f50c1c20"\n    },\n    {\n      "id": "train_ep22_mars_3_d10_1",\n      "epoch": 22,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 111.05000000000001,\n      "start_s": 91.05000000000001,\n      "end_s": 131.05,\n      "n_native_gt_outage": 36,\n      "n_native_gt_total": 151,\n      "token_mask_sha256": "860d531b40010f5a1ce4a6d8d4c9b3f3cb34a9b01d0a5e57422ce1faf23b77a3",\n      "native_gt_timestamp_sha256": "564dfc82aab2c8ffd48376e9624f87ac24f5a898c83d9382bb72a2e1f7199a36"\n    },\n    {\n      "id": "train_ep22_mars_2_d60_0",\n      "epoch": 22,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 46.7,\n      "start_s": 26.700000000000003,\n      "end_s": 116.7,\n      "n_native_gt_outage": 255,\n      "n_native_gt_total": 356,\n      "token_mask_sha256": "71da0041c5ffc52dc55131d6c02ef9fd40886bf07a3990e099fe3e7675dd9bab",\n      "native_gt_timestamp_sha256": "92f8d486d30385f0a2ad5f7bfcf3943bbbd3bf84a90a1e48bc0139b995ae0425"\n    },\n    {\n      "id": "train_ep22_mars_3_d10_0",\n      "epoch": 22,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 74.05,\n      "start_s": 54.05,\n      "end_s": 94.05,\n      "n_native_gt_outage": 50,\n      "n_native_gt_total": 156,\n      "token_mask_sha256": "54d1f8b7a9217c3e8206fec44aa90fc97fdb3e89370798f9369d21bcb530fdd3",\n      "native_gt_timestamp_sha256": "ffefcf832bac9eee5d009ff665eb0e16fb98fe2e2564149e91617e3b3ba6d9a9"\n    },\n    {\n      "id": "train_ep22_mars_2_d10_0",\n      "epoch": 22,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 135.6,\n      "start_s": 115.6,\n      "end_s": 155.6,\n      "n_native_gt_outage": 36,\n      "n_native_gt_total": 165,\n      "token_mask_sha256": "54d1f8b7a9217c3e8206fec44aa90fc97fdb3e89370798f9369d21bcb530fdd3",\n      "native_gt_timestamp_sha256": "9d24f13a7ed83eae16a35a72d025862f34b68ca320664f946c9379fa16641db3"\n    },\n    {\n      "id": "train_ep22_mars_1_d10_1",\n      "epoch": 22,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 64.35000000000001,\n      "start_s": 44.35000000000001,\n      "end_s": 84.35000000000001,\n      "n_native_gt_outage": 56,\n      "n_native_gt_total": 170,\n      "token_mask_sha256": "21b9e16f0abe800c55f145a18abd0a79a37df3cbda1ac658e7e89003a42e7536",\n      "native_gt_timestamp_sha256": "51f6c8117929e0bc68de1c568e1823361869f36d41fbfbbd6c7e9073075b3037"\n    },\n    {\n      "id": "train_ep22_mars_3_d60_0",\n      "epoch": 22,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 66.35000000000001,\n      "start_s": 46.35000000000001,\n      "end_s": 136.35000000000002,\n      "n_native_gt_outage": 238,\n      "n_native_gt_total": 350,\n      "token_mask_sha256": "71da0041c5ffc52dc55131d6c02ef9fd40886bf07a3990e099fe3e7675dd9bab",\n      "native_gt_timestamp_sha256": "24e2f3a380f3d038a3feaee426e81ec95a2037e9da4c7fd397282b2667f49bae"\n    },\n    {\n      "id": "train_ep22_mars_1_d30_1",\n      "epoch": 22,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 28.950000000000003,\n      "start_s": 8.950000000000003,\n      "end_s": 68.95,\n      "n_native_gt_outage": 115,\n      "n_native_gt_total": 238,\n      "token_mask_sha256": "3dd74d234f7a60145505caa41ec0613e9d52eb9cbdc78e09d68e05c0d586d85a",\n      "native_gt_timestamp_sha256": "1bc0b95407448398c55efbdf2951d7467b8d225d2e24cbb723fb05dffee4cbf1"\n    },\n    {\n      "id": "train_ep23_mars_3_d10_1",\n      "epoch": 23,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 95.95,\n      "start_s": 75.95,\n      "end_s": 115.95,\n      "n_native_gt_outage": 42,\n      "n_native_gt_total": 168,\n      "token_mask_sha256": "9f12542780490d612306b1de6383455d9c904bc5b73f8b9cebd020d927653a95",\n      "native_gt_timestamp_sha256": "936a42701e064455027afe54668f27296308e7fdea5605879e962d1a031d72f4"\n    },\n    {\n      "id": "train_ep23_mars_2_d30_0",\n      "epoch": 23,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 47.7,\n      "start_s": 27.700000000000003,\n      "end_s": 87.7,\n      "n_native_gt_outage": 129,\n      "n_native_gt_total": 238,\n      "token_mask_sha256": "3dd74d234f7a60145505caa41ec0613e9d52eb9cbdc78e09d68e05c0d586d85a",\n      "native_gt_timestamp_sha256": "0d99aad1812ff39d6a35bbef7d825263a654134386cc2b0385654aa4101e1885"\n    },\n    {\n      "id": "train_ep23_mars_2_d10_0",\n      "epoch": 23,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 138.95000000000002,\n      "start_s": 118.95000000000002,\n      "end_s": 158.95000000000002,\n      "n_native_gt_outage": 36,\n      "n_native_gt_total": 171,\n      "token_mask_sha256": "860d531b40010f5a1ce4a6d8d4c9b3f3cb34a9b01d0a5e57422ce1faf23b77a3",\n      "native_gt_timestamp_sha256": "8090e8772502213c195c3ecb5df9387f72678bd778ca22cc891b64a4f94ec0a5"\n    },\n    {\n      "id": "train_ep23_mars_1_d60_1",\n      "epoch": 23,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 29.05,\n      "start_s": 9.05,\n      "end_s": 99.05,\n      "n_native_gt_outage": 239,\n      "n_native_gt_total": 376,\n      "token_mask_sha256": "944469817cb1d463be9ba8e22d7c037bf6b358337512a5973f0eedd13fdd691e",\n      "native_gt_timestamp_sha256": "61e00a2c5f4803a407ff148605c128735d485a8d2935c5d9acd8348d0b635b61"\n    },\n    {\n      "id": "train_ep23_mars_1_d30_0",\n      "epoch": 23,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 50.150000000000006,\n      "start_s": 30.150000000000006,\n      "end_s": 90.15,\n      "n_native_gt_outage": 123,\n      "n_native_gt_total": 238,\n      "token_mask_sha256": "3dd74d234f7a60145505caa41ec0613e9d52eb9cbdc78e09d68e05c0d586d85a",\n      "native_gt_timestamp_sha256": "8e6ad34d14788c2ff36ed32c36756c455c3bd3eced81d056067e88a8cd042631"\n    },\n    {\n      "id": "train_ep23_mars_3_d10_0",\n      "epoch": 23,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 80.35000000000001,\n      "start_s": 60.35000000000001,\n      "end_s": 100.35000000000001,\n      "n_native_gt_outage": 40,\n      "n_native_gt_total": 152,\n      "token_mask_sha256": "9f12542780490d612306b1de6383455d9c904bc5b73f8b9cebd020d927653a95",\n      "native_gt_timestamp_sha256": "139ba967e7d115e235491fa7188868909d554c268d8c3444c7377261e2f8e906"\n    },\n    {\n      "id": "train_ep23_mars_3_d60_0",\n      "epoch": 23,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 67.45,\n      "start_s": 47.45,\n      "end_s": 137.45,\n      "n_native_gt_outage": 238,\n      "n_native_gt_total": 354,\n      "token_mask_sha256": "fed530af485dca69fb59f9b1a865da85e17ffd444c3c0d42d5f5017e0315ab5f",\n      "native_gt_timestamp_sha256": "86dec7367e1a9d827129b8ad5c6189691c478c7bd50b7fcfe0269e7c07b1eacb"\n    },\n    {\n      "id": "train_ep23_mars_2_d30_1",\n      "epoch": 23,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 45.0,\n      "start_s": 25.0,\n      "end_s": 85.0,\n      "n_native_gt_outage": 130,\n      "n_native_gt_total": 234,\n      "token_mask_sha256": "63d833f0b713bb4fb84f25c62b86deb8f26260f7a627284500d089600ceaa5cc",\n      "native_gt_timestamp_sha256": "1e8025e5971fecd1c83b5b67a4f97b2ac9eb82c487954e06b7f5e9118e2af55f"\n    },\n    {\n      "id": "train_ep23_mars_1_d10_0",\n      "epoch": 23,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 59.75,\n      "start_s": 39.75,\n      "end_s": 79.75,\n      "n_native_gt_outage": 46,\n      "n_native_gt_total": 170,\n      "token_mask_sha256": "21b9e16f0abe800c55f145a18abd0a79a37df3cbda1ac658e7e89003a42e7536",\n      "native_gt_timestamp_sha256": "d3a638bdd89390421f9db938bd42867ada3b8473115a11fb39210576cb43d4e0"\n    },\n    {\n      "id": "train_ep23_mars_1_d60_0",\n      "epoch": 23,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 27.0,\n      "start_s": 7.0,\n      "end_s": 97.0,\n      "n_native_gt_outage": 244,\n      "n_native_gt_total": 379,\n      "token_mask_sha256": "fed530af485dca69fb59f9b1a865da85e17ffd444c3c0d42d5f5017e0315ab5f",\n      "native_gt_timestamp_sha256": "c7a4b3f058e6922abec303808e00f9289401c8519a126cf79aba18d017fbf211"\n    },\n    {\n      "id": "train_ep23_mars_3_d30_0",\n      "epoch": 23,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 39.900000000000006,\n      "start_s": 19.900000000000006,\n      "end_s": 79.9,\n      "n_native_gt_outage": 111,\n      "n_native_gt_total": 233,\n      "token_mask_sha256": "63d833f0b713bb4fb84f25c62b86deb8f26260f7a627284500d089600ceaa5cc",\n      "native_gt_timestamp_sha256": "3abc7a0b420defe9192f57fdf4d676aaa531f8d9b899649dd636dfd01ee27715"\n    },\n    {\n      "id": "train_ep23_mars_3_d60_1",\n      "epoch": 23,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 58.300000000000004,\n      "start_s": 38.300000000000004,\n      "end_s": 128.3,\n      "n_native_gt_outage": 239,\n      "n_native_gt_total": 349,\n      "token_mask_sha256": "944469817cb1d463be9ba8e22d7c037bf6b358337512a5973f0eedd13fdd691e",\n      "native_gt_timestamp_sha256": "331293b1c9e8c38c07fbf9623366d61b6ee72bcc1264928e16d01002d902bb30"\n    },\n    {\n      "id": "train_ep23_mars_1_d10_1",\n      "epoch": 23,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 64.15,\n      "start_s": 44.150000000000006,\n      "end_s": 84.15,\n      "n_native_gt_outage": 56,\n      "n_native_gt_total": 170,\n      "token_mask_sha256": "21b9e16f0abe800c55f145a18abd0a79a37df3cbda1ac658e7e89003a42e7536",\n      "native_gt_timestamp_sha256": "f394606ac5acd25029772f419db4205be4d5298b6db62d0899dfc2ef1267f7f2"\n    },\n    {\n      "id": "train_ep23_mars_2_d10_1",\n      "epoch": 23,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 142.0,\n      "start_s": 122.0,\n      "end_s": 162.0,\n      "n_native_gt_outage": 38,\n      "n_native_gt_total": 175,\n      "token_mask_sha256": "54d1f8b7a9217c3e8206fec44aa90fc97fdb3e89370798f9369d21bcb530fdd3",\n      "native_gt_timestamp_sha256": "abac9afc9346e89d44024980e348daccc25c49a2fcf85bbd5f1986e62441231f"\n    },\n    {\n      "id": "train_ep23_mars_3_d30_1",\n      "epoch": 23,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 83.0,\n      "start_s": 63.0,\n      "end_s": 123.0,\n      "n_native_gt_outage": 118,\n      "n_native_gt_total": 239,\n      "token_mask_sha256": "3dd74d234f7a60145505caa41ec0613e9d52eb9cbdc78e09d68e05c0d586d85a",\n      "native_gt_timestamp_sha256": "cd2399ed5863b907ee58e66553b180adec819929c78cb76d6c9b56db8ca6bb93"\n    },\n    {\n      "id": "train_ep23_mars_1_d30_1",\n      "epoch": 23,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 52.1,\n      "start_s": 32.1,\n      "end_s": 92.1,\n      "n_native_gt_outage": 121,\n      "n_native_gt_total": 237,\n      "token_mask_sha256": "ec337caf619f2fc34accfde4fefc63f5f100805a2dc4a30b788f58ddce2c4bc8",\n      "native_gt_timestamp_sha256": "3f3438c2c88b90682fe2b22a5c19aca6782587342d71497fc47af6b6c25dae40"\n    },\n    {\n      "id": "train_ep23_mars_2_d60_0",\n      "epoch": 23,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 38.300000000000004,\n      "start_s": 18.300000000000004,\n      "end_s": 108.30000000000001,\n      "n_native_gt_outage": 251,\n      "n_native_gt_total": 358,\n      "token_mask_sha256": "71da0041c5ffc52dc55131d6c02ef9fd40886bf07a3990e099fe3e7675dd9bab",\n      "native_gt_timestamp_sha256": "9337a2b8ae988af7bbf274b80add228240af906b432e431b30e9a41a45266a9a"\n    },\n    {\n      "id": "train_ep23_mars_2_d60_1",\n      "epoch": 23,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 69.75,\n      "start_s": 49.75,\n      "end_s": 139.75,\n      "n_native_gt_outage": 252,\n      "n_native_gt_total": 373,\n      "token_mask_sha256": "c0c2ec8564a28a400ecdd4fe90d4d6b443981b414ff75cc3098c2fd7dc691a8b",\n      "native_gt_timestamp_sha256": "410e81863e2cdb4660e3103a6c3f75c86505cb77c2eeb3da60f488f60f91bfe5"\n    },\n    {\n      "id": "train_ep24_mars_1_d10_0",\n      "epoch": 24,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 22.650000000000002,\n      "start_s": 2.650000000000002,\n      "end_s": 42.650000000000006,\n      "n_native_gt_outage": 47,\n      "n_native_gt_total": 163,\n      "token_mask_sha256": "54d1f8b7a9217c3e8206fec44aa90fc97fdb3e89370798f9369d21bcb530fdd3",\n      "native_gt_timestamp_sha256": "8beeb49cc0b4bcfb3950a7f6b9467ff3bbb5e05d33129399f154af30e179dd10"\n    },\n    {\n      "id": "train_ep24_mars_3_d30_0",\n      "epoch": 24,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 33.800000000000004,\n      "start_s": 13.800000000000004,\n      "end_s": 73.80000000000001,\n      "n_native_gt_outage": 115,\n      "n_native_gt_total": 226,\n      "token_mask_sha256": "3dd74d234f7a60145505caa41ec0613e9d52eb9cbdc78e09d68e05c0d586d85a",\n      "native_gt_timestamp_sha256": "7c04a1565e0f64f0e68f5f27bfbb60c960de401a03e18f5627462f3f91b21cb1"\n    },\n    {\n      "id": "train_ep24_mars_3_d60_0",\n      "epoch": 24,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 42.800000000000004,\n      "start_s": 22.800000000000004,\n      "end_s": 112.80000000000001,\n      "n_native_gt_outage": 230,\n      "n_native_gt_total": 358,\n      "token_mask_sha256": "c0c2ec8564a28a400ecdd4fe90d4d6b443981b414ff75cc3098c2fd7dc691a8b",\n      "native_gt_timestamp_sha256": "fc1f02e7e32d646350f5f1337ce9dc5e73659f7951019ab0e1f9e14bbed64125"\n    },\n    {\n      "id": "train_ep24_mars_3_d10_1",\n      "epoch": 24,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 60.150000000000006,\n      "start_s": 40.150000000000006,\n      "end_s": 80.15,\n      "n_native_gt_outage": 34,\n      "n_native_gt_total": 157,\n      "token_mask_sha256": "9f12542780490d612306b1de6383455d9c904bc5b73f8b9cebd020d927653a95",\n      "native_gt_timestamp_sha256": "fc87e3e68ba3e039685213e3222a0564d493df13af02332183f6bd6a8e71de16"\n    },\n    {\n      "id": "train_ep24_mars_3_d30_1",\n      "epoch": 24,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 86.60000000000001,\n      "start_s": 66.60000000000001,\n      "end_s": 126.60000000000001,\n      "n_native_gt_outage": 117,\n      "n_native_gt_total": 237,\n      "token_mask_sha256": "3dd74d234f7a60145505caa41ec0613e9d52eb9cbdc78e09d68e05c0d586d85a",\n      "native_gt_timestamp_sha256": "f45ee8f8cc6640eb233ff75c6e738091748ef457b8b602c9695a8c887406d0fd"\n    },\n    {\n      "id": "train_ep24_mars_1_d60_1",\n      "epoch": 24,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 29.450000000000003,\n      "start_s": 9.450000000000003,\n      "end_s": 99.45,\n      "n_native_gt_outage": 239,\n      "n_native_gt_total": 375,\n      "token_mask_sha256": "944469817cb1d463be9ba8e22d7c037bf6b358337512a5973f0eedd13fdd691e",\n      "native_gt_timestamp_sha256": "3eec85600a74ea72993dfd710e238e88a0993cd52cecdbc2cf2c23daaca23f7a"\n    },\n    {\n      "id": "train_ep24_mars_3_d10_0",\n      "epoch": 24,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 120.65,\n      "start_s": 100.65,\n      "end_s": 140.65,\n      "n_native_gt_outage": 31,\n      "n_native_gt_total": 172,\n      "token_mask_sha256": "860d531b40010f5a1ce4a6d8d4c9b3f3cb34a9b01d0a5e57422ce1faf23b77a3",\n      "native_gt_timestamp_sha256": "6f9e1d981556bfcfd9d825d5dd5e10412b6ccd8a91db9f5533fb335dd7441a08"\n    },\n    {\n      "id": "train_ep24_mars_3_d60_1",\n      "epoch": 24,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 44.800000000000004,\n      "start_s": 24.800000000000004,\n      "end_s": 114.80000000000001,\n      "n_native_gt_outage": 232,\n      "n_native_gt_total": 356,\n      "token_mask_sha256": "c0c2ec8564a28a400ecdd4fe90d4d6b443981b414ff75cc3098c2fd7dc691a8b",\n      "native_gt_timestamp_sha256": "9cf8809b12e2a99a90dc5715ce97e956111d329da4931eacb55b61bcfb73830f"\n    },\n    {\n      "id": "train_ep24_mars_2_d10_1",\n      "epoch": 24,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 126.80000000000001,\n      "start_s": 106.80000000000001,\n      "end_s": 146.8,\n      "n_native_gt_outage": 34,\n      "n_native_gt_total": 160,\n      "token_mask_sha256": "860d531b40010f5a1ce4a6d8d4c9b3f3cb34a9b01d0a5e57422ce1faf23b77a3",\n      "native_gt_timestamp_sha256": "2b439c93d6b320a8d39ad274b461be272ab1540ab346e0747ce0f1b3a85cc906"\n    },\n    {\n      "id": "train_ep24_mars_1_d10_1",\n      "epoch": 24,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 54.75,\n      "start_s": 34.75,\n      "end_s": 74.75,\n      "n_native_gt_outage": 34,\n      "n_native_gt_total": 168,\n      "token_mask_sha256": "21b9e16f0abe800c55f145a18abd0a79a37df3cbda1ac658e7e89003a42e7536",\n      "native_gt_timestamp_sha256": "560a6a7a725897fc526c3d6536772bb63ab5765a153ff2f3e040d064788c5f92"\n    },\n    {\n      "id": "train_ep24_mars_2_d30_0",\n      "epoch": 24,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 71.25,\n      "start_s": 51.25,\n      "end_s": 111.25,\n      "n_native_gt_outage": 124,\n      "n_native_gt_total": 242,\n      "token_mask_sha256": "63d833f0b713bb4fb84f25c62b86deb8f26260f7a627284500d089600ceaa5cc",\n      "native_gt_timestamp_sha256": "07610ad262b41af29c9bddce5dbad11c18193efdc48c3330c0aad3018ea886e7"\n    },\n    {\n      "id": "train_ep24_mars_2_d60_0",\n      "epoch": 24,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 32.75,\n      "start_s": 12.75,\n      "end_s": 102.75,\n      "n_native_gt_outage": 251,\n      "n_native_gt_total": 361,\n      "token_mask_sha256": "c0c2ec8564a28a400ecdd4fe90d4d6b443981b414ff75cc3098c2fd7dc691a8b",\n      "native_gt_timestamp_sha256": "c885db743c3a595f5d6c5e11a45aeb8b162416c0aed65f9ae10eee7e6a6b4c3a"\n    },\n    {\n      "id": "train_ep24_mars_1_d30_0",\n      "epoch": 24,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 46.2,\n      "start_s": 26.200000000000003,\n      "end_s": 86.2,\n      "n_native_gt_outage": 128,\n      "n_native_gt_total": 246,\n      "token_mask_sha256": "b34423e399ec53c009571b199c4b245f2f0d58ef83150e9bf0611fa732314343",\n      "native_gt_timestamp_sha256": "005bb1b878782882f818f88dcabbdcec632ec59a9991b615ffdd379a12efcf6e"\n    },\n    {\n      "id": "train_ep24_mars_2_d30_1",\n      "epoch": 24,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 55.85,\n      "start_s": 35.85,\n      "end_s": 95.85,\n      "n_native_gt_outage": 119,\n      "n_native_gt_total": 251,\n      "token_mask_sha256": "63d833f0b713bb4fb84f25c62b86deb8f26260f7a627284500d089600ceaa5cc",\n      "native_gt_timestamp_sha256": "21d7671d62b8dc720c6d05eade771bf4eb814fa03f165fda27e2d082a08ec05d"\n    },\n    {\n      "id": "train_ep24_mars_2_d60_1",\n      "epoch": 24,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 47.150000000000006,\n      "start_s": 27.150000000000006,\n      "end_s": 117.15,\n      "n_native_gt_outage": 254,\n      "n_native_gt_total": 359,\n      "token_mask_sha256": "c0c2ec8564a28a400ecdd4fe90d4d6b443981b414ff75cc3098c2fd7dc691a8b",\n      "native_gt_timestamp_sha256": "e5a875a625d795a8c9fa1ec8670e824a3e783c16534c8d1e8dcfd716d31fef46"\n    },\n    {\n      "id": "train_ep24_mars_1_d60_0",\n      "epoch": 24,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 28.400000000000002,\n      "start_s": 8.400000000000002,\n      "end_s": 98.4,\n      "n_native_gt_outage": 238,\n      "n_native_gt_total": 377,\n      "token_mask_sha256": "fed530af485dca69fb59f9b1a865da85e17ffd444c3c0d42d5f5017e0315ab5f",\n      "native_gt_timestamp_sha256": "19781464902d48fe6611f2c8dbc7f78e41b596b3968279b8f04bee1cc3022b4a"\n    },\n    {\n      "id": "train_ep24_mars_1_d30_1",\n      "epoch": 24,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 22.900000000000002,\n      "start_s": 2.900000000000002,\n      "end_s": 62.900000000000006,\n      "n_native_gt_outage": 130,\n      "n_native_gt_total": 242,\n      "token_mask_sha256": "ec337caf619f2fc34accfde4fefc63f5f100805a2dc4a30b788f58ddce2c4bc8",\n      "native_gt_timestamp_sha256": "f0eaf09d7b9ec0e162911be8fd6da935710142be465c767f5c1b164ecc6b9424"\n    },\n    {\n      "id": "train_ep24_mars_2_d10_0",\n      "epoch": 24,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 48.25,\n      "start_s": 28.25,\n      "end_s": 68.25,\n      "n_native_gt_outage": 45,\n      "n_native_gt_total": 159,\n      "token_mask_sha256": "54d1f8b7a9217c3e8206fec44aa90fc97fdb3e89370798f9369d21bcb530fdd3",\n      "native_gt_timestamp_sha256": "ba62e466f495e413eb394b997956520cb8a99a2caedcd86459364ec1ccad0918"\n    },\n    {\n      "id": "train_ep25_mars_1_d60_1",\n      "epoch": 25,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 26.400000000000002,\n      "start_s": 6.400000000000002,\n      "end_s": 96.4,\n      "n_native_gt_outage": 245,\n      "n_native_gt_total": 379,\n      "token_mask_sha256": "fed530af485dca69fb59f9b1a865da85e17ffd444c3c0d42d5f5017e0315ab5f",\n      "native_gt_timestamp_sha256": "fcf471b8cbeda11e1b96ff4c56ad50cf725919636b11848629aea3498a84ce2a"\n    },\n    {\n      "id": "train_ep25_mars_1_d10_0",\n      "epoch": 25,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 57.45,\n      "start_s": 37.45,\n      "end_s": 77.45,\n      "n_native_gt_outage": 37,\n      "n_native_gt_total": 170,\n      "token_mask_sha256": "54d1f8b7a9217c3e8206fec44aa90fc97fdb3e89370798f9369d21bcb530fdd3",\n      "native_gt_timestamp_sha256": "e89ff89f3b13703f37cb378f9e17cd78f0348096442eb5c37c10b05bed711ecd"\n    },\n    {\n      "id": "train_ep25_mars_2_d30_1",\n      "epoch": 25,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 57.0,\n      "start_s": 37.0,\n      "end_s": 97.0,\n      "n_native_gt_outage": 118,\n      "n_native_gt_total": 251,\n      "token_mask_sha256": "b34423e399ec53c009571b199c4b245f2f0d58ef83150e9bf0611fa732314343",\n      "native_gt_timestamp_sha256": "89e62ebac020147d2860ec19fb7247da9f5f0b93b140a0ad7d993eb4724574e6"\n    },\n    {\n      "id": "train_ep25_mars_2_d10_1",\n      "epoch": 25,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 46.1,\n      "start_s": 26.1,\n      "end_s": 66.1,\n      "n_native_gt_outage": 47,\n      "n_native_gt_total": 151,\n      "token_mask_sha256": "9f12542780490d612306b1de6383455d9c904bc5b73f8b9cebd020d927653a95",\n      "native_gt_timestamp_sha256": "36f820016f1df2ab915bc99ce51a105d892e4fe3eea9a25888720a3be252ca64"\n    },\n    {\n      "id": "train_ep25_mars_3_d30_0",\n      "epoch": 25,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 69.25,\n      "start_s": 49.25,\n      "end_s": 109.25,\n      "n_native_gt_outage": 117,\n      "n_native_gt_total": 241,\n      "token_mask_sha256": "b34423e399ec53c009571b199c4b245f2f0d58ef83150e9bf0611fa732314343",\n      "native_gt_timestamp_sha256": "78d7d15e4c379c7d48100be6596e6b8b4c5e6b96ed02a0a4decac09b56b821a4"\n    },\n    {\n      "id": "train_ep25_mars_1_d30_1",\n      "epoch": 25,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 44.5,\n      "start_s": 24.5,\n      "end_s": 84.5,\n      "n_native_gt_outage": 134,\n      "n_native_gt_total": 252,\n      "token_mask_sha256": "ec337caf619f2fc34accfde4fefc63f5f100805a2dc4a30b788f58ddce2c4bc8",\n      "native_gt_timestamp_sha256": "898a3b434969f8f6911f3855542add4a4bde706c8cf9f1011d4bcfaa8817212b"\n    },\n    {\n      "id": "train_ep25_mars_2_d60_1",\n      "epoch": 25,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 34.4,\n      "start_s": 14.399999999999999,\n      "end_s": 104.4,\n      "n_native_gt_outage": 253,\n      "n_native_gt_total": 363,\n      "token_mask_sha256": "fed530af485dca69fb59f9b1a865da85e17ffd444c3c0d42d5f5017e0315ab5f",\n      "native_gt_timestamp_sha256": "ea50a872dd01750ec668584c1b167ad92b9046598bc301bcba0c7b77ff5c7902"\n    },\n    {\n      "id": "train_ep25_mars_3_d60_1",\n      "epoch": 25,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 30.0,\n      "start_s": 10.0,\n      "end_s": 100.0,\n      "n_native_gt_outage": 238,\n      "n_native_gt_total": 352,\n      "token_mask_sha256": "c0c2ec8564a28a400ecdd4fe90d4d6b443981b414ff75cc3098c2fd7dc691a8b",\n      "native_gt_timestamp_sha256": "c9cae4dd8b37d1b9e570fc7014bba03377898ef4cf06584e0bf7fd10fef02530"\n    },\n    {\n      "id": "train_ep25_mars_3_d10_1",\n      "epoch": 25,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 59.550000000000004,\n      "start_s": 39.550000000000004,\n      "end_s": 79.55000000000001,\n      "n_native_gt_outage": 34,\n      "n_native_gt_total": 156,\n      "token_mask_sha256": "21b9e16f0abe800c55f145a18abd0a79a37df3cbda1ac658e7e89003a42e7536",\n      "native_gt_timestamp_sha256": "6b5e03c71dfcd96a7bd2e50f8df6c4a5d6aeb9320ee540536b74e0a651563cab"\n    },\n    {\n      "id": "train_ep25_mars_2_d10_0",\n      "epoch": 25,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 26.1,\n      "start_s": 6.100000000000001,\n      "end_s": 46.1,\n      "n_native_gt_outage": 36,\n      "n_native_gt_total": 145,\n      "token_mask_sha256": "9f12542780490d612306b1de6383455d9c904bc5b73f8b9cebd020d927653a95",\n      "native_gt_timestamp_sha256": "867b23884a94385882ebd26cbd8d5842e0d123fe36b70f84a08fa39c8805e99d"\n    },\n    {\n      "id": "train_ep25_mars_1_d60_0",\n      "epoch": 25,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 23.400000000000002,\n      "start_s": 3.400000000000002,\n      "end_s": 93.4,\n      "n_native_gt_outage": 251,\n      "n_native_gt_total": 374,\n      "token_mask_sha256": "fed530af485dca69fb59f9b1a865da85e17ffd444c3c0d42d5f5017e0315ab5f",\n      "native_gt_timestamp_sha256": "841bfd0dd086a367e5aa8a34de3772697b9b7ca809b015f502e966283021d510"\n    },\n    {\n      "id": "train_ep25_mars_3_d60_0",\n      "epoch": 25,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 57.6,\n      "start_s": 37.6,\n      "end_s": 127.6,\n      "n_native_gt_outage": 236,\n      "n_native_gt_total": 350,\n      "token_mask_sha256": "c0c2ec8564a28a400ecdd4fe90d4d6b443981b414ff75cc3098c2fd7dc691a8b",\n      "native_gt_timestamp_sha256": "d3513404472fb0a10278abf2b9d672aa4efb61f5cc519833995062e1ed25cc3d"\n    },\n    {\n      "id": "train_ep25_mars_2_d30_0",\n      "epoch": 25,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 39.150000000000006,\n      "start_s": 19.150000000000006,\n      "end_s": 79.15,\n      "n_native_gt_outage": 125,\n      "n_native_gt_total": 233,\n      "token_mask_sha256": "b34423e399ec53c009571b199c4b245f2f0d58ef83150e9bf0611fa732314343",\n      "native_gt_timestamp_sha256": "f470588c72cdb18c66e292f8cbe19b79915ee0ed8d66fa91c56da0cabf6fe134"\n    },\n    {\n      "id": "train_ep25_mars_1_d10_1",\n      "epoch": 25,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 78.80000000000001,\n      "start_s": 58.80000000000001,\n      "end_s": 98.80000000000001,\n      "n_native_gt_outage": 34,\n      "n_native_gt_total": 178,\n      "token_mask_sha256": "860d531b40010f5a1ce4a6d8d4c9b3f3cb34a9b01d0a5e57422ce1faf23b77a3",\n      "native_gt_timestamp_sha256": "5aa29f72fe894b7d0691eb25d9d99ed5b99010324824b551bbd248e7555e5fbe"\n    },\n    {\n      "id": "train_ep25_mars_3_d10_0",\n      "epoch": 25,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 100.95,\n      "start_s": 80.95,\n      "end_s": 120.95,\n      "n_native_gt_outage": 49,\n      "n_native_gt_total": 155,\n      "token_mask_sha256": "9f12542780490d612306b1de6383455d9c904bc5b73f8b9cebd020d927653a95",\n      "native_gt_timestamp_sha256": "1dfaad205e9833da8cc96497aed2e3036eeb2ceb615a45330dab1fdfd1c5fedb"\n    },\n    {\n      "id": "train_ep25_mars_2_d60_0",\n      "epoch": 25,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 55.800000000000004,\n      "start_s": 35.800000000000004,\n      "end_s": 125.80000000000001,\n      "n_native_gt_outage": 242,\n      "n_native_gt_total": 376,\n      "token_mask_sha256": "fed530af485dca69fb59f9b1a865da85e17ffd444c3c0d42d5f5017e0315ab5f",\n      "native_gt_timestamp_sha256": "987b9942d1d2203fb8eba29edae8ab7feb10cf05de8b872200235c314094ebe7"\n    },\n    {\n      "id": "train_ep25_mars_3_d30_1",\n      "epoch": 25,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 49.1,\n      "start_s": 29.1,\n      "end_s": 89.1,\n      "n_native_gt_outage": 113,\n      "n_native_gt_total": 238,\n      "token_mask_sha256": "63d833f0b713bb4fb84f25c62b86deb8f26260f7a627284500d089600ceaa5cc",\n      "native_gt_timestamp_sha256": "3a96113b493f89a20e1386b580ed42899ff3fa510bab3d38b03e9db086e9b995"\n    },\n    {\n      "id": "train_ep25_mars_1_d30_0",\n      "epoch": 25,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 51.800000000000004,\n      "start_s": 31.800000000000004,\n      "end_s": 91.80000000000001,\n      "n_native_gt_outage": 121,\n      "n_native_gt_total": 238,\n      "token_mask_sha256": "b34423e399ec53c009571b199c4b245f2f0d58ef83150e9bf0611fa732314343",\n      "native_gt_timestamp_sha256": "47e02129d2512fd6bede88f36187a5a83207047ac7e4f34c9da5ce64e3e1333a"\n    },\n    {\n      "id": "train_ep26_mars_3_d60_0",\n      "epoch": 26,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 65.8,\n      "start_s": 45.8,\n      "end_s": 135.8,\n      "n_native_gt_outage": 235,\n      "n_native_gt_total": 349,\n      "token_mask_sha256": "478ce3b3d3dce9522789ea95341f5ae713e75348c57aff32b390220cd55f9144",\n      "native_gt_timestamp_sha256": "11e05d72061b5f95efc4c366923eb60f41e7852bd4115c37d8617a4b78955f60"\n    },\n    {\n      "id": "train_ep26_mars_3_d60_1",\n      "epoch": 26,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 31.55,\n      "start_s": 11.55,\n      "end_s": 101.55,\n      "n_native_gt_outage": 239,\n      "n_native_gt_total": 347,\n      "token_mask_sha256": "c0c2ec8564a28a400ecdd4fe90d4d6b443981b414ff75cc3098c2fd7dc691a8b",\n      "native_gt_timestamp_sha256": "3ba68f67cebffa01a89b3d239a7c20380d66da43182617df2dfb169fb0339c3f"\n    },\n    {\n      "id": "train_ep26_mars_3_d30_0",\n      "epoch": 26,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 98.30000000000001,\n      "start_s": 78.30000000000001,\n      "end_s": 138.3,\n      "n_native_gt_outage": 117,\n      "n_native_gt_total": 244,\n      "token_mask_sha256": "ec337caf619f2fc34accfde4fefc63f5f100805a2dc4a30b788f58ddce2c4bc8",\n      "native_gt_timestamp_sha256": "2b853a959733e4d0c2965cb24bd78aa52c1a5d76dc3ddee8e581077d91a2274a"\n    },\n    {\n      "id": "train_ep26_mars_3_d10_0",\n      "epoch": 26,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 83.2,\n      "start_s": 63.2,\n      "end_s": 103.2,\n      "n_native_gt_outage": 33,\n      "n_native_gt_total": 153,\n      "token_mask_sha256": "ea945fe6cde424b8878a4a763925347368d146d2fe42a95bb983fd8eb5bdd4c3",\n      "native_gt_timestamp_sha256": "b678f7bdf7b11a48e11971bdc252d68e46ce3984e51e52134dba443333172a8c"\n    },\n    {\n      "id": "train_ep26_mars_1_d10_0",\n      "epoch": 26,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 75.0,\n      "start_s": 55.0,\n      "end_s": 95.0,\n      "n_native_gt_outage": 37,\n      "n_native_gt_total": 169,\n      "token_mask_sha256": "860d531b40010f5a1ce4a6d8d4c9b3f3cb34a9b01d0a5e57422ce1faf23b77a3",\n      "native_gt_timestamp_sha256": "f550ab18cfc3b8491375153c0d534053b8f6dc228086dc0d9c4b165b86b43f96"\n    },\n    {\n      "id": "train_ep26_mars_1_d30_0",\n      "epoch": 26,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 54.2,\n      "start_s": 34.2,\n      "end_s": 94.2,\n      "n_native_gt_outage": 122,\n      "n_native_gt_total": 245,\n      "token_mask_sha256": "b34423e399ec53c009571b199c4b245f2f0d58ef83150e9bf0611fa732314343",\n      "native_gt_timestamp_sha256": "ddc6b1edfc846006236ab62cf65782622aabc29de84d716315615fcfdcefa31c"\n    },\n    {\n      "id": "train_ep26_mars_1_d60_1",\n      "epoch": 26,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 21.3,\n      "start_s": 1.3000000000000007,\n      "end_s": 91.3,\n      "n_native_gt_outage": 252,\n      "n_native_gt_total": 370,\n      "token_mask_sha256": "71da0041c5ffc52dc55131d6c02ef9fd40886bf07a3990e099fe3e7675dd9bab",\n      "native_gt_timestamp_sha256": "1d9fe80b282b719b420c368fce1a64680dd46f688dd310d0478cc9df0a6928ed"\n    },\n    {\n      "id": "train_ep26_mars_2_d60_1",\n      "epoch": 26,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 30.650000000000002,\n      "start_s": 10.650000000000002,\n      "end_s": 100.65,\n      "n_native_gt_outage": 242,\n      "n_native_gt_total": 360,\n      "token_mask_sha256": "944469817cb1d463be9ba8e22d7c037bf6b358337512a5973f0eedd13fdd691e",\n      "native_gt_timestamp_sha256": "e9d5fada8c5ba79602e02897b581f6289fa37883db4e7f31a106471d907c9c04"\n    },\n    {\n      "id": "train_ep26_mars_2_d30_1",\n      "epoch": 26,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 43.400000000000006,\n      "start_s": 23.400000000000006,\n      "end_s": 83.4,\n      "n_native_gt_outage": 129,\n      "n_native_gt_total": 231,\n      "token_mask_sha256": "63d833f0b713bb4fb84f25c62b86deb8f26260f7a627284500d089600ceaa5cc",\n      "native_gt_timestamp_sha256": "b77cca3837a87aa0c95bbe85c7198e260596359b994bcce67d68106f3644d0f9"\n    },\n    {\n      "id": "train_ep26_mars_3_d30_1",\n      "epoch": 26,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 40.75,\n      "start_s": 20.75,\n      "end_s": 80.75,\n      "n_native_gt_outage": 112,\n      "n_native_gt_total": 235,\n      "token_mask_sha256": "ec337caf619f2fc34accfde4fefc63f5f100805a2dc4a30b788f58ddce2c4bc8",\n      "native_gt_timestamp_sha256": "5e1cf2bf0f5e6f52055e865ff2830bcd2e811337c4063a9ba6992acb9cd8515a"\n    },\n    {\n      "id": "train_ep26_mars_1_d10_1",\n      "epoch": 26,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 24.8,\n      "start_s": 4.800000000000001,\n      "end_s": 44.8,\n      "n_native_gt_outage": 46,\n      "n_native_gt_total": 166,\n      "token_mask_sha256": "860d531b40010f5a1ce4a6d8d4c9b3f3cb34a9b01d0a5e57422ce1faf23b77a3",\n      "native_gt_timestamp_sha256": "8a93dc6a3d30f463710e5d27474a6170c886a2ddc07169ebceed0583f71f8601"\n    },\n    {\n      "id": "train_ep26_mars_3_d10_1",\n      "epoch": 26,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 77.30000000000001,\n      "start_s": 57.30000000000001,\n      "end_s": 97.30000000000001,\n      "n_native_gt_outage": 49,\n      "n_native_gt_total": 155,\n      "token_mask_sha256": "54d1f8b7a9217c3e8206fec44aa90fc97fdb3e89370798f9369d21bcb530fdd3",\n      "native_gt_timestamp_sha256": "43cb1ebe64e472f28cec0ee29cb6cdafa03364616e258f539fe6d907cdf8fba5"\n    },\n    {\n      "id": "train_ep26_mars_2_d10_1",\n      "epoch": 26,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 139.3,\n      "start_s": 119.30000000000001,\n      "end_s": 159.3,\n      "n_native_gt_outage": 34,\n      "n_native_gt_total": 171,\n      "token_mask_sha256": "21b9e16f0abe800c55f145a18abd0a79a37df3cbda1ac658e7e89003a42e7536",\n      "native_gt_timestamp_sha256": "3968ac743ab72b49dfce79e5aba880d40a39f48a856349e9101d2ed3da483ac2"\n    },\n    {\n      "id": "train_ep26_mars_1_d60_0",\n      "epoch": 26,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 27.150000000000002,\n      "start_s": 7.150000000000002,\n      "end_s": 97.15,\n      "n_native_gt_outage": 243,\n      "n_native_gt_total": 379,\n      "token_mask_sha256": "c0c2ec8564a28a400ecdd4fe90d4d6b443981b414ff75cc3098c2fd7dc691a8b",\n      "native_gt_timestamp_sha256": "b4fd08d47fd9be54a7f33e50410e4361a817b1463cf4237103aafaf4711a5841"\n    },\n    {\n      "id": "train_ep26_mars_2_d30_0",\n      "epoch": 26,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 55.050000000000004,\n      "start_s": 35.050000000000004,\n      "end_s": 95.05000000000001,\n      "n_native_gt_outage": 117,\n      "n_native_gt_total": 252,\n      "token_mask_sha256": "63d833f0b713bb4fb84f25c62b86deb8f26260f7a627284500d089600ceaa5cc",\n      "native_gt_timestamp_sha256": "5e33b8d082d7b03eec78f83fabf9d9a3f56a97ace5b7f3b33e28cf5674c97f6a"\n    },\n    {\n      "id": "train_ep26_mars_2_d60_0",\n      "epoch": 26,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 48.800000000000004,\n      "start_s": 28.800000000000004,\n      "end_s": 118.80000000000001,\n      "n_native_gt_outage": 251,\n      "n_native_gt_total": 358,\n      "token_mask_sha256": "944469817cb1d463be9ba8e22d7c037bf6b358337512a5973f0eedd13fdd691e",\n      "native_gt_timestamp_sha256": "f8a8426a4606a72d2b5e033640daf636c4f33e59a5624785b71b21537589bfb2"\n    },\n    {\n      "id": "train_ep26_mars_2_d10_0",\n      "epoch": 26,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 84.75,\n      "start_s": 64.75,\n      "end_s": 104.75,\n      "n_native_gt_outage": 53,\n      "n_native_gt_total": 172,\n      "token_mask_sha256": "21b9e16f0abe800c55f145a18abd0a79a37df3cbda1ac658e7e89003a42e7536",\n      "native_gt_timestamp_sha256": "ceec06567a4c47f39ca3a14e9011fe10eda91cacbeb4e45a9545470899a93d1e"\n    },\n    {\n      "id": "train_ep26_mars_1_d30_1",\n      "epoch": 26,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 42.0,\n      "start_s": 22.0,\n      "end_s": 82.0,\n      "n_native_gt_outage": 135,\n      "n_native_gt_total": 250,\n      "token_mask_sha256": "b34423e399ec53c009571b199c4b245f2f0d58ef83150e9bf0611fa732314343",\n      "native_gt_timestamp_sha256": "f96d785ed66227dd7f0d2692fd6f9bef9707b9150dcd354b6a9bd1d8648ce8c3"\n    },\n    {\n      "id": "train_ep27_mars_1_d60_0",\n      "epoch": 27,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 26.75,\n      "start_s": 6.75,\n      "end_s": 96.75,\n      "n_native_gt_outage": 244,\n      "n_native_gt_total": 378,\n      "token_mask_sha256": "c0c2ec8564a28a400ecdd4fe90d4d6b443981b414ff75cc3098c2fd7dc691a8b",\n      "native_gt_timestamp_sha256": "f82a1c0b6818a6abd4060f5ed9166386a4773f421d349bbfa6655ec72ca36ff1"\n    },\n    {\n      "id": "train_ep27_mars_3_d30_1",\n      "epoch": 27,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 91.7,\n      "start_s": 71.7,\n      "end_s": 131.7,\n      "n_native_gt_outage": 118,\n      "n_native_gt_total": 234,\n      "token_mask_sha256": "63d833f0b713bb4fb84f25c62b86deb8f26260f7a627284500d089600ceaa5cc",\n      "native_gt_timestamp_sha256": "f419e55adc0afe671487eb250fa827ef8b3c533506a0c2d5acf7fe80c55cda02"\n    },\n    {\n      "id": "train_ep27_mars_1_d10_0",\n      "epoch": 27,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 37.0,\n      "start_s": 17.0,\n      "end_s": 57.0,\n      "n_native_gt_outage": 43,\n      "n_native_gt_total": 164,\n      "token_mask_sha256": "860d531b40010f5a1ce4a6d8d4c9b3f3cb34a9b01d0a5e57422ce1faf23b77a3",\n      "native_gt_timestamp_sha256": "faa29a8c8c5d4f710085016d0dd70aa7ec2dc7b5b2b159782385caed7d2c2361"\n    },\n    {\n      "id": "train_ep27_mars_2_d60_1",\n      "epoch": 27,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 38.050000000000004,\n      "start_s": 18.050000000000004,\n      "end_s": 108.05000000000001,\n      "n_native_gt_outage": 252,\n      "n_native_gt_total": 358,\n      "token_mask_sha256": "944469817cb1d463be9ba8e22d7c037bf6b358337512a5973f0eedd13fdd691e",\n      "native_gt_timestamp_sha256": "9337a2b8ae988af7bbf274b80add228240af906b432e431b30e9a41a45266a9a"\n    },\n    {\n      "id": "train_ep27_mars_2_d60_0",\n      "epoch": 27,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 77.85000000000001,\n      "start_s": 57.85000000000001,\n      "end_s": 147.85000000000002,\n      "n_native_gt_outage": 250,\n      "n_native_gt_total": 371,\n      "token_mask_sha256": "944469817cb1d463be9ba8e22d7c037bf6b358337512a5973f0eedd13fdd691e",\n      "native_gt_timestamp_sha256": "9d31c9bfedde5b8d5d3b05c0220e0018dabd98f9744c7bcd70479146328f6015"\n    },\n    {\n      "id": "train_ep27_mars_2_d30_1",\n      "epoch": 27,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 24.950000000000003,\n      "start_s": 4.950000000000003,\n      "end_s": 64.95,\n      "n_native_gt_outage": 117,\n      "n_native_gt_total": 230,\n      "token_mask_sha256": "5a4d5eda9c4ff3d08ec925d8e1a633049916ea8630c1944f170e40381baf7176",\n      "native_gt_timestamp_sha256": "b1e7db4e732dfbe9991653071f24e1b5874252e6a7909e9cd2413019509fada9"\n    },\n    {\n      "id": "train_ep27_mars_1_d30_0",\n      "epoch": 27,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 54.2,\n      "start_s": 34.2,\n      "end_s": 94.2,\n      "n_native_gt_outage": 122,\n      "n_native_gt_total": 245,\n      "token_mask_sha256": "b34423e399ec53c009571b199c4b245f2f0d58ef83150e9bf0611fa732314343",\n      "native_gt_timestamp_sha256": "ddc6b1edfc846006236ab62cf65782622aabc29de84d716315615fcfdcefa31c"\n    },\n    {\n      "id": "train_ep27_mars_3_d60_0",\n      "epoch": 27,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 31.950000000000003,\n      "start_s": 11.950000000000003,\n      "end_s": 101.95,\n      "n_native_gt_outage": 236,\n      "n_native_gt_total": 346,\n      "token_mask_sha256": "c0c2ec8564a28a400ecdd4fe90d4d6b443981b414ff75cc3098c2fd7dc691a8b",\n      "native_gt_timestamp_sha256": "72471891fe3c41cb0eedb70e1e000b0b0e5922ce636dcdd374ccd396991eda26"\n    },\n    {\n      "id": "train_ep27_mars_1_d60_1",\n      "epoch": 27,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 25.8,\n      "start_s": 5.800000000000001,\n      "end_s": 95.8,\n      "n_native_gt_outage": 248,\n      "n_native_gt_total": 379,\n      "token_mask_sha256": "fed530af485dca69fb59f9b1a865da85e17ffd444c3c0d42d5f5017e0315ab5f",\n      "native_gt_timestamp_sha256": "1aabd525baeb05fdd16d0d4a2743a73fbaccc8596649059d190869a8f3232377"\n    },\n    {\n      "id": "train_ep27_mars_1_d10_1",\n      "epoch": 27,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 23.700000000000003,\n      "start_s": 3.700000000000003,\n      "end_s": 43.7,\n      "n_native_gt_outage": 47,\n      "n_native_gt_total": 164,\n      "token_mask_sha256": "9f12542780490d612306b1de6383455d9c904bc5b73f8b9cebd020d927653a95",\n      "native_gt_timestamp_sha256": "d570616332d124c37fc235077dbc28797264b3df1a6a28bd4b8298f54afc682e"\n    },\n    {\n      "id": "train_ep27_mars_3_d60_1",\n      "epoch": 27,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 70.35000000000001,\n      "start_s": 50.35000000000001,\n      "end_s": 140.35000000000002,\n      "n_native_gt_outage": 235,\n      "n_native_gt_total": 362,\n      "token_mask_sha256": "71da0041c5ffc52dc55131d6c02ef9fd40886bf07a3990e099fe3e7675dd9bab",\n      "native_gt_timestamp_sha256": "c8de123dec84425ab2c97226693bfac7383d0f1cf138c35947fa43cc43bd6d29"\n    },\n    {\n      "id": "train_ep27_mars_3_d30_0",\n      "epoch": 27,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 101.2,\n      "start_s": 81.2,\n      "end_s": 141.2,\n      "n_native_gt_outage": 118,\n      "n_native_gt_total": 245,\n      "token_mask_sha256": "3dd74d234f7a60145505caa41ec0613e9d52eb9cbdc78e09d68e05c0d586d85a",\n      "native_gt_timestamp_sha256": "5bc8a2ed81bc622cc388f2175f669770d964a9c59325b5e98e4a79ae0da41193"\n    },\n    {\n      "id": "train_ep27_mars_3_d10_0",\n      "epoch": 27,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 70.05,\n      "start_s": 50.05,\n      "end_s": 90.05,\n      "n_native_gt_outage": 45,\n      "n_native_gt_total": 156,\n      "token_mask_sha256": "54d1f8b7a9217c3e8206fec44aa90fc97fdb3e89370798f9369d21bcb530fdd3",\n      "native_gt_timestamp_sha256": "c745ac3848059ab17f925d0f45f7d2a39b7f9b4c5a81267dfaf2af607f67ba32"\n    },\n    {\n      "id": "train_ep27_mars_3_d10_1",\n      "epoch": 27,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 28.950000000000003,\n      "start_s": 8.950000000000003,\n      "end_s": 48.95,\n      "n_native_gt_outage": 39,\n      "n_native_gt_total": 165,\n      "token_mask_sha256": "21b9e16f0abe800c55f145a18abd0a79a37df3cbda1ac658e7e89003a42e7536",\n      "native_gt_timestamp_sha256": "688b4a174df2722af2c172713461413876e6ab31543b657dd20fff6cbf9e2979"\n    },\n    {\n      "id": "train_ep27_mars_1_d30_1",\n      "epoch": 27,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 40.2,\n      "start_s": 20.200000000000003,\n      "end_s": 80.2,\n      "n_native_gt_outage": 127,\n      "n_native_gt_total": 251,\n      "token_mask_sha256": "b34423e399ec53c009571b199c4b245f2f0d58ef83150e9bf0611fa732314343",\n      "native_gt_timestamp_sha256": "5949c7751c99db984fce5e4392be92e892a711562c6d58a3fc582bd2375dc462"\n    },\n    {\n      "id": "train_ep27_mars_2_d10_1",\n      "epoch": 27,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 115.0,\n      "start_s": 95.0,\n      "end_s": 135.0,\n      "n_native_gt_outage": 51,\n      "n_native_gt_total": 162,\n      "token_mask_sha256": "860d531b40010f5a1ce4a6d8d4c9b3f3cb34a9b01d0a5e57422ce1faf23b77a3",\n      "native_gt_timestamp_sha256": "9254c78a3f02d4bb43c0c6559f6a08986b36f03cd0c126db01c765c199985dae"\n    },\n    {\n      "id": "train_ep27_mars_2_d10_0",\n      "epoch": 27,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 42.45,\n      "start_s": 22.450000000000003,\n      "end_s": 62.45,\n      "n_native_gt_outage": 47,\n      "n_native_gt_total": 149,\n      "token_mask_sha256": "54d1f8b7a9217c3e8206fec44aa90fc97fdb3e89370798f9369d21bcb530fdd3",\n      "native_gt_timestamp_sha256": "63f235831e3db410ad9a7eec81cd4064851e3f4df549fcd5fbdaa2686df51161"\n    },\n    {\n      "id": "train_ep27_mars_2_d30_0",\n      "epoch": 27,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 38.6,\n      "start_s": 18.6,\n      "end_s": 78.6,\n      "n_native_gt_outage": 122,\n      "n_native_gt_total": 236,\n      "token_mask_sha256": "63d833f0b713bb4fb84f25c62b86deb8f26260f7a627284500d089600ceaa5cc",\n      "native_gt_timestamp_sha256": "b7e566f6d18037b19a3042d95b1877c41ac9ba964f0baffc132261f6c1c64602"\n    },\n    {\n      "id": "train_ep28_mars_1_d30_1",\n      "epoch": 28,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 34.550000000000004,\n      "start_s": 14.550000000000004,\n      "end_s": 74.55000000000001,\n      "n_native_gt_outage": 113,\n      "n_native_gt_total": 252,\n      "token_mask_sha256": "3dd74d234f7a60145505caa41ec0613e9d52eb9cbdc78e09d68e05c0d586d85a",\n      "native_gt_timestamp_sha256": "229ff4d31d8f57bd964adada77614af3c1732c9eae1b1b5efad4a86c3cb1b9b2"\n    },\n    {\n      "id": "train_ep28_mars_1_d10_1",\n      "epoch": 28,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 68.60000000000001,\n      "start_s": 48.60000000000001,\n      "end_s": 88.60000000000001,\n      "n_native_gt_outage": 48,\n      "n_native_gt_total": 157,\n      "token_mask_sha256": "860d531b40010f5a1ce4a6d8d4c9b3f3cb34a9b01d0a5e57422ce1faf23b77a3",\n      "native_gt_timestamp_sha256": "9c073f5d34fc13c28c91b9f56874d659c9ef5662a45b826a1706d911f2278f6c"\n    },\n    {\n      "id": "train_ep28_mars_3_d30_1",\n      "epoch": 28,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 54.800000000000004,\n      "start_s": 34.800000000000004,\n      "end_s": 94.80000000000001,\n      "n_native_gt_outage": 122,\n      "n_native_gt_total": 236,\n      "token_mask_sha256": "3dd74d234f7a60145505caa41ec0613e9d52eb9cbdc78e09d68e05c0d586d85a",\n      "native_gt_timestamp_sha256": "e45a7f6210141b0503a19d0d0df9f4632d9e40e13c627f10e15b6d066c647d59"\n    },\n    {\n      "id": "train_ep28_mars_3_d10_0",\n      "epoch": 28,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 83.60000000000001,\n      "start_s": 63.60000000000001,\n      "end_s": 103.60000000000001,\n      "n_native_gt_outage": 35,\n      "n_native_gt_total": 154,\n      "token_mask_sha256": "21b9e16f0abe800c55f145a18abd0a79a37df3cbda1ac658e7e89003a42e7536",\n      "native_gt_timestamp_sha256": "db5bf9ff8b84b0a46958a09e6bd9c2cfcca68d01e2e5aeae10f6344184818e7c"\n    },\n    {\n      "id": "train_ep28_mars_1_d60_0",\n      "epoch": 28,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 21.25,\n      "start_s": 1.25,\n      "end_s": 91.25,\n      "n_native_gt_outage": 252,\n      "n_native_gt_total": 370,\n      "token_mask_sha256": "944469817cb1d463be9ba8e22d7c037bf6b358337512a5973f0eedd13fdd691e",\n      "native_gt_timestamp_sha256": "1d9fe80b282b719b420c368fce1a64680dd46f688dd310d0478cc9df0a6928ed"\n    },\n    {\n      "id": "train_ep28_mars_1_d60_1",\n      "epoch": 28,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 27.150000000000002,\n      "start_s": 7.150000000000002,\n      "end_s": 97.15,\n      "n_native_gt_outage": 243,\n      "n_native_gt_total": 379,\n      "token_mask_sha256": "c0c2ec8564a28a400ecdd4fe90d4d6b443981b414ff75cc3098c2fd7dc691a8b",\n      "native_gt_timestamp_sha256": "b4fd08d47fd9be54a7f33e50410e4361a817b1463cf4237103aafaf4711a5841"\n    },\n    {\n      "id": "train_ep28_mars_2_d60_1",\n      "epoch": 28,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 90.2,\n      "start_s": 70.2,\n      "end_s": 160.2,\n      "n_native_gt_outage": 247,\n      "n_native_gt_total": 368,\n      "token_mask_sha256": "944469817cb1d463be9ba8e22d7c037bf6b358337512a5973f0eedd13fdd691e",\n      "native_gt_timestamp_sha256": "bbbd6d113f1d80601bd4848d3b653bb4bf66af6f05f32b2111565f4ef2509ebb"\n    },\n    {\n      "id": "train_ep28_mars_3_d60_1",\n      "epoch": 28,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 54.5,\n      "start_s": 34.5,\n      "end_s": 124.5,\n      "n_native_gt_outage": 237,\n      "n_native_gt_total": 352,\n      "token_mask_sha256": "944469817cb1d463be9ba8e22d7c037bf6b358337512a5973f0eedd13fdd691e",\n      "native_gt_timestamp_sha256": "623b554bda388b514892e6228d82ddb9c656c94041d642153d75a8b15c8d171d"\n    },\n    {\n      "id": "train_ep28_mars_2_d30_0",\n      "epoch": 28,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 30.950000000000003,\n      "start_s": 10.950000000000003,\n      "end_s": 70.95,\n      "n_native_gt_outage": 120,\n      "n_native_gt_total": 237,\n      "token_mask_sha256": "3dd74d234f7a60145505caa41ec0613e9d52eb9cbdc78e09d68e05c0d586d85a",\n      "native_gt_timestamp_sha256": "33dc70df486e868677abb94b396d0353002ef55b9c7ab4e33b3481f21f6a9f93"\n    },\n    {\n      "id": "train_ep28_mars_3_d10_1",\n      "epoch": 28,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 47.7,\n      "start_s": 27.700000000000003,\n      "end_s": 67.7,\n      "n_native_gt_outage": 39,\n      "n_native_gt_total": 152,\n      "token_mask_sha256": "54d1f8b7a9217c3e8206fec44aa90fc97fdb3e89370798f9369d21bcb530fdd3",\n      "native_gt_timestamp_sha256": "2699fef02f9e758c767cecff154be80bd544cd518b8f80b53ca3b6dfe896f8fb"\n    },\n    {\n      "id": "train_ep28_mars_2_d10_0",\n      "epoch": 28,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 74.0,\n      "start_s": 54.0,\n      "end_s": 94.0,\n      "n_native_gt_outage": 34,\n      "n_native_gt_total": 172,\n      "token_mask_sha256": "860d531b40010f5a1ce4a6d8d4c9b3f3cb34a9b01d0a5e57422ce1faf23b77a3",\n      "native_gt_timestamp_sha256": "ada4436054bc5375b2622c3f69be7f3b3127dfdedcb35bc4ffb50509bca2b158"\n    },\n    {\n      "id": "train_ep28_mars_1_d10_0",\n      "epoch": 28,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 35.75,\n      "start_s": 15.75,\n      "end_s": 55.75,\n      "n_native_gt_outage": 37,\n      "n_native_gt_total": 164,\n      "token_mask_sha256": "21b9e16f0abe800c55f145a18abd0a79a37df3cbda1ac658e7e89003a42e7536",\n      "native_gt_timestamp_sha256": "1994b1e1629362bb0079686a9581694cf13d8ade30cb06bf182d19c722354d89"\n    },\n    {\n      "id": "train_ep28_mars_3_d60_0",\n      "epoch": 28,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 68.65,\n      "start_s": 48.650000000000006,\n      "end_s": 138.65,\n      "n_native_gt_outage": 237,\n      "n_native_gt_total": 354,\n      "token_mask_sha256": "fed530af485dca69fb59f9b1a865da85e17ffd444c3c0d42d5f5017e0315ab5f",\n      "native_gt_timestamp_sha256": "95c54f5a54fd04f47884199cf6dce4a8c2eeb88b58c6c4adb8514cc3eee06531"\n    },\n    {\n      "id": "train_ep28_mars_2_d10_1",\n      "epoch": 28,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 50.5,\n      "start_s": 30.5,\n      "end_s": 70.5,\n      "n_native_gt_outage": 37,\n      "n_native_gt_total": 167,\n      "token_mask_sha256": "9f12542780490d612306b1de6383455d9c904bc5b73f8b9cebd020d927653a95",\n      "native_gt_timestamp_sha256": "2a89013e2329be8acdf495a74751632f7b105879b10840ac6ad7037256cbac11"\n    },\n    {\n      "id": "train_ep28_mars_3_d30_0",\n      "epoch": 28,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 76.25,\n      "start_s": 56.25,\n      "end_s": 116.25,\n      "n_native_gt_outage": 129,\n      "n_native_gt_total": 238,\n      "token_mask_sha256": "65e37395e153a7b3efe8de1dd41646a64c5f33d6d16eb16b954eac4a73120341",\n      "native_gt_timestamp_sha256": "2033031525534ee10540e9761726ae15f0e85b0b24b8b937ce68f9d00125a713"\n    },\n    {\n      "id": "train_ep28_mars_2_d30_1",\n      "epoch": 28,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 28.1,\n      "start_s": 8.100000000000001,\n      "end_s": 68.1,\n      "n_native_gt_outage": 117,\n      "n_native_gt_total": 231,\n      "token_mask_sha256": "ec337caf619f2fc34accfde4fefc63f5f100805a2dc4a30b788f58ddce2c4bc8",\n      "native_gt_timestamp_sha256": "1c6b973ad787aa2621661a9f2f1f9db9e8583da134340db8cf19d65dfcdae00d"\n    },\n    {\n      "id": "train_ep28_mars_1_d30_0",\n      "epoch": 28,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 28.05,\n      "start_s": 8.05,\n      "end_s": 68.05,\n      "n_native_gt_outage": 115,\n      "n_native_gt_total": 238,\n      "token_mask_sha256": "63d833f0b713bb4fb84f25c62b86deb8f26260f7a627284500d089600ceaa5cc",\n      "native_gt_timestamp_sha256": "bc31bb8a3be53475ec453d1ed11a778e2a0a4c17512a9246afa851ace7170273"\n    },\n    {\n      "id": "train_ep28_mars_2_d60_0",\n      "epoch": 28,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 55.2,\n      "start_s": 35.2,\n      "end_s": 125.2,\n      "n_native_gt_outage": 241,\n      "n_native_gt_total": 374,\n      "token_mask_sha256": "fed530af485dca69fb59f9b1a865da85e17ffd444c3c0d42d5f5017e0315ab5f",\n      "native_gt_timestamp_sha256": "2739f9dac215b409b756a0750f98b6d9aa58f5429bdd4e5e7830eb1577ab76d6"\n    },\n    {\n      "id": "train_ep29_mars_3_d10_0",\n      "epoch": 29,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 56.25,\n      "start_s": 36.25,\n      "end_s": 76.25,\n      "n_native_gt_outage": 34,\n      "n_native_gt_total": 149,\n      "token_mask_sha256": "860d531b40010f5a1ce4a6d8d4c9b3f3cb34a9b01d0a5e57422ce1faf23b77a3",\n      "native_gt_timestamp_sha256": "d55f1beb8fdc13bc07f5452d21d175fb7303022866abd7a3b09eafafbc74efde"\n    },\n    {\n      "id": "train_ep29_mars_3_d30_0",\n      "epoch": 29,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 60.75,\n      "start_s": 40.75,\n      "end_s": 100.75,\n      "n_native_gt_outage": 120,\n      "n_native_gt_total": 232,\n      "token_mask_sha256": "ec337caf619f2fc34accfde4fefc63f5f100805a2dc4a30b788f58ddce2c4bc8",\n      "native_gt_timestamp_sha256": "1917efd39e270f10c6d27dff8665ce420afc132b73a86d6f42ca2478bdcfdd33"\n    },\n    {\n      "id": "train_ep29_mars_2_d10_0",\n      "epoch": 29,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 58.85,\n      "start_s": 38.85,\n      "end_s": 78.85,\n      "n_native_gt_outage": 43,\n      "n_native_gt_total": 164,\n      "token_mask_sha256": "9f12542780490d612306b1de6383455d9c904bc5b73f8b9cebd020d927653a95",\n      "native_gt_timestamp_sha256": "339937e445eb9d6ae9a3bc800e0a3d94f4bc341b7c1f522ee9d592a6561c5fb2"\n    },\n    {\n      "id": "train_ep29_mars_1_d10_1",\n      "epoch": 29,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 78.4,\n      "start_s": 58.400000000000006,\n      "end_s": 98.4,\n      "n_native_gt_outage": 37,\n      "n_native_gt_total": 179,\n      "token_mask_sha256": "860d531b40010f5a1ce4a6d8d4c9b3f3cb34a9b01d0a5e57422ce1faf23b77a3",\n      "native_gt_timestamp_sha256": "7d21fe2d18a287f84ef5159e48d45245162796daca68074529469d6be0c8caf7"\n    },\n    {\n      "id": "train_ep29_mars_1_d60_1",\n      "epoch": 29,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 23.950000000000003,\n      "start_s": 3.950000000000003,\n      "end_s": 93.95,\n      "n_native_gt_outage": 252,\n      "n_native_gt_total": 374,\n      "token_mask_sha256": "c0c2ec8564a28a400ecdd4fe90d4d6b443981b414ff75cc3098c2fd7dc691a8b",\n      "native_gt_timestamp_sha256": "3798810fabe03a4037a8cb7fd5416d08b8e8feb222862f3fb5d867825f4f7025"\n    },\n    {\n      "id": "train_ep29_mars_1_d30_0",\n      "epoch": 29,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 30.150000000000002,\n      "start_s": 10.150000000000002,\n      "end_s": 70.15,\n      "n_native_gt_outage": 114,\n      "n_native_gt_total": 242,\n      "token_mask_sha256": "3dd74d234f7a60145505caa41ec0613e9d52eb9cbdc78e09d68e05c0d586d85a",\n      "native_gt_timestamp_sha256": "151d204bfce1273c4cbdefea1bbfbaf68b137f59e61c274d3782a044e781b40b"\n    },\n    {\n      "id": "train_ep29_mars_2_d30_0",\n      "epoch": 29,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 33.75,\n      "start_s": 13.75,\n      "end_s": 73.75,\n      "n_native_gt_outage": 115,\n      "n_native_gt_total": 239,\n      "token_mask_sha256": "3dd74d234f7a60145505caa41ec0613e9d52eb9cbdc78e09d68e05c0d586d85a",\n      "native_gt_timestamp_sha256": "0eb20ea526cbdc62f978039312562f65ffd232e83c0c17001939c65d1c153a9b"\n    },\n    {\n      "id": "train_ep29_mars_2_d10_1",\n      "epoch": 29,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 124.30000000000001,\n      "start_s": 104.30000000000001,\n      "end_s": 144.3,\n      "n_native_gt_outage": 42,\n      "n_native_gt_total": 161,\n      "token_mask_sha256": "9f12542780490d612306b1de6383455d9c904bc5b73f8b9cebd020d927653a95",\n      "native_gt_timestamp_sha256": "8026575cb8d48d0df2f791cf7c2540bb5ebcdb70612c38a80fe3cdbf2504e268"\n    },\n    {\n      "id": "train_ep29_mars_2_d60_0",\n      "epoch": 29,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 58.150000000000006,\n      "start_s": 38.150000000000006,\n      "end_s": 128.15,\n      "n_native_gt_outage": 242,\n      "n_native_gt_total": 379,\n      "token_mask_sha256": "c0c2ec8564a28a400ecdd4fe90d4d6b443981b414ff75cc3098c2fd7dc691a8b",\n      "native_gt_timestamp_sha256": "5b282a92a503d258de587f53a837da088419f6ca30d2a5ff072526e1bce2508a"\n    },\n    {\n      "id": "train_ep29_mars_1_d30_1",\n      "epoch": 29,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 44.650000000000006,\n      "start_s": 24.650000000000006,\n      "end_s": 84.65,\n      "n_native_gt_outage": 134,\n      "n_native_gt_total": 252,\n      "token_mask_sha256": "63d833f0b713bb4fb84f25c62b86deb8f26260f7a627284500d089600ceaa5cc",\n      "native_gt_timestamp_sha256": "8eb7f87c263b57bfe765dd7345955fbb7a5726342b0c8b3d4d7747e84e58505d"\n    },\n    {\n      "id": "train_ep29_mars_3_d10_1",\n      "epoch": 29,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 104.9,\n      "start_s": 84.9,\n      "end_s": 124.9,\n      "n_native_gt_outage": 45,\n      "n_native_gt_total": 151,\n      "token_mask_sha256": "54d1f8b7a9217c3e8206fec44aa90fc97fdb3e89370798f9369d21bcb530fdd3",\n      "native_gt_timestamp_sha256": "576b4ab4c79202e4eebc6b9cecce038a46981bcf5a1dda7e5de2cbc1eb9de23d"\n    },\n    {\n      "id": "train_ep29_mars_3_d30_1",\n      "epoch": 29,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 48.150000000000006,\n      "start_s": 28.150000000000006,\n      "end_s": 88.15,\n      "n_native_gt_outage": 109,\n      "n_native_gt_total": 238,\n      "token_mask_sha256": "ec337caf619f2fc34accfde4fefc63f5f100805a2dc4a30b788f58ddce2c4bc8",\n      "native_gt_timestamp_sha256": "021cacfc680501049395a43cf703a23335539adbd5ff74e7a970f939237178f2"\n    },\n    {\n      "id": "train_ep29_mars_2_d30_1",\n      "epoch": 29,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 34.65,\n      "start_s": 14.649999999999999,\n      "end_s": 74.65,\n      "n_native_gt_outage": 117,\n      "n_native_gt_total": 236,\n      "token_mask_sha256": "ec337caf619f2fc34accfde4fefc63f5f100805a2dc4a30b788f58ddce2c4bc8",\n      "native_gt_timestamp_sha256": "465b933b4eb1da0c5d45f351c8d0378fe4b9243f3f2bdf0a90684f4d262fced8"\n    },\n    {\n      "id": "train_ep29_mars_2_d60_1",\n      "epoch": 29,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 57.25,\n      "start_s": 37.25,\n      "end_s": 127.25,\n      "n_native_gt_outage": 240,\n      "n_native_gt_total": 377,\n      "token_mask_sha256": "944469817cb1d463be9ba8e22d7c037bf6b358337512a5973f0eedd13fdd691e",\n      "native_gt_timestamp_sha256": "7c71fe37855455bc54baa94f0d7bb377cfd4ca9c2d6c5a94ce6538452df22a92"\n    },\n    {\n      "id": "train_ep29_mars_3_d60_1",\n      "epoch": 29,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 34.2,\n      "start_s": 14.200000000000003,\n      "end_s": 104.2,\n      "n_native_gt_outage": 233,\n      "n_native_gt_total": 346,\n      "token_mask_sha256": "c0c2ec8564a28a400ecdd4fe90d4d6b443981b414ff75cc3098c2fd7dc691a8b",\n      "native_gt_timestamp_sha256": "21c1270a2f9ad1398fbfa9e7d2853ece89709fa9aef717639dc9cc291afcefac"\n    },\n    {\n      "id": "train_ep29_mars_1_d60_0",\n      "epoch": 29,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 28.75,\n      "start_s": 8.75,\n      "end_s": 98.75,\n      "n_native_gt_outage": 237,\n      "n_native_gt_total": 376,\n      "token_mask_sha256": "c0c2ec8564a28a400ecdd4fe90d4d6b443981b414ff75cc3098c2fd7dc691a8b",\n      "native_gt_timestamp_sha256": "ee03d7db9d13caa7444d83248f407a275edf45269ceb4edfa508deca399b5020"\n    },\n    {\n      "id": "train_ep29_mars_3_d60_0",\n      "epoch": 29,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 29.6,\n      "start_s": 9.600000000000001,\n      "end_s": 99.6,\n      "n_native_gt_outage": 237,\n      "n_native_gt_total": 354,\n      "token_mask_sha256": "c0c2ec8564a28a400ecdd4fe90d4d6b443981b414ff75cc3098c2fd7dc691a8b",\n      "native_gt_timestamp_sha256": "edaa06daa52bfaa5dff1157b0a681412275b9f0f1e47c5092df206a66530955e"\n    },\n    {\n      "id": "train_ep29_mars_1_d10_0",\n      "epoch": 29,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 24.25,\n      "start_s": 4.25,\n      "end_s": 44.25,\n      "n_native_gt_outage": 50,\n      "n_native_gt_total": 164,\n      "token_mask_sha256": "54d1f8b7a9217c3e8206fec44aa90fc97fdb3e89370798f9369d21bcb530fdd3",\n      "native_gt_timestamp_sha256": "04f5682ab0517867dc4c39a20cd7d9f47dfd7f4389c5f8e2ff9b1acae49641d0"\n    },\n    {\n      "id": "train_ep30_mars_1_d60_0",\n      "epoch": 30,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 25.25,\n      "start_s": 5.25,\n      "end_s": 95.25,\n      "n_native_gt_outage": 248,\n      "n_native_gt_total": 379,\n      "token_mask_sha256": "944469817cb1d463be9ba8e22d7c037bf6b358337512a5973f0eedd13fdd691e",\n      "native_gt_timestamp_sha256": "7077fabcb856530bdeb8c835f54efc1474ae82480de46f963f74783b5d90df8a"\n    },\n    {\n      "id": "train_ep30_mars_2_d30_1",\n      "epoch": 30,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 47.300000000000004,\n      "start_s": 27.300000000000004,\n      "end_s": 87.30000000000001,\n      "n_native_gt_outage": 130,\n      "n_native_gt_total": 235,\n      "token_mask_sha256": "ec337caf619f2fc34accfde4fefc63f5f100805a2dc4a30b788f58ddce2c4bc8",\n      "native_gt_timestamp_sha256": "678d1871e6c5ca01218fae0ffc93276e81e5beed66afdcffb182d905e2187efc"\n    },\n    {\n      "id": "train_ep30_mars_1_d60_1",\n      "epoch": 30,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 22.400000000000002,\n      "start_s": 2.400000000000002,\n      "end_s": 92.4,\n      "n_native_gt_outage": 251,\n      "n_native_gt_total": 370,\n      "token_mask_sha256": "fed530af485dca69fb59f9b1a865da85e17ffd444c3c0d42d5f5017e0315ab5f",\n      "native_gt_timestamp_sha256": "1dbbeaebe7589bbdf517ff2634769ffc412b596cc58df8df0ce216a4da86aa75"\n    },\n    {\n      "id": "train_ep30_mars_2_d10_0",\n      "epoch": 30,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 40.400000000000006,\n      "start_s": 20.400000000000006,\n      "end_s": 60.400000000000006,\n      "n_native_gt_outage": 45,\n      "n_native_gt_total": 149,\n      "token_mask_sha256": "860d531b40010f5a1ce4a6d8d4c9b3f3cb34a9b01d0a5e57422ce1faf23b77a3",\n      "native_gt_timestamp_sha256": "c643a2935792ea816a5b2a5dcc1bbeef1c6225b14937c80bb5516c5c3d7401ba"\n    },\n    {\n      "id": "train_ep30_mars_3_d60_1",\n      "epoch": 30,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 59.7,\n      "start_s": 39.7,\n      "end_s": 129.7,\n      "n_native_gt_outage": 237,\n      "n_native_gt_total": 349,\n      "token_mask_sha256": "944469817cb1d463be9ba8e22d7c037bf6b358337512a5973f0eedd13fdd691e",\n      "native_gt_timestamp_sha256": "354bfc64a9e842a2dcefe2943825793bb2ad27973113afe87c8f989a0cc62a9d"\n    },\n    {\n      "id": "train_ep30_mars_2_d60_1",\n      "epoch": 30,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 27.200000000000003,\n      "start_s": 7.200000000000003,\n      "end_s": 97.2,\n      "n_native_gt_outage": 235,\n      "n_native_gt_total": 361,\n      "token_mask_sha256": "fed530af485dca69fb59f9b1a865da85e17ffd444c3c0d42d5f5017e0315ab5f",\n      "native_gt_timestamp_sha256": "dc8982db5ddd43ad6bdfb36f6688d573c59c5e6b5e957481a83b13e57eaabe10"\n    },\n    {\n      "id": "train_ep30_mars_1_d30_0",\n      "epoch": 30,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 53.75,\n      "start_s": 33.75,\n      "end_s": 93.75,\n      "n_native_gt_outage": 124,\n      "n_native_gt_total": 245,\n      "token_mask_sha256": "3dd74d234f7a60145505caa41ec0613e9d52eb9cbdc78e09d68e05c0d586d85a",\n      "native_gt_timestamp_sha256": "efa84fdb6d16c25ab7ec7ecf7e26dd1360d0908786ba0d9cb0248b8136675914"\n    },\n    {\n      "id": "train_ep30_mars_3_d10_0",\n      "epoch": 30,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 69.85000000000001,\n      "start_s": 49.85000000000001,\n      "end_s": 89.85000000000001,\n      "n_native_gt_outage": 45,\n      "n_native_gt_total": 155,\n      "token_mask_sha256": "860d531b40010f5a1ce4a6d8d4c9b3f3cb34a9b01d0a5e57422ce1faf23b77a3",\n      "native_gt_timestamp_sha256": "bb73b8c3edf91065b24925eb78c7e051956f870c644feb7b9ee02af9d69589c3"\n    },\n    {\n      "id": "train_ep30_mars_3_d60_0",\n      "epoch": 30,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 30.700000000000003,\n      "start_s": 10.700000000000003,\n      "end_s": 100.7,\n      "n_native_gt_outage": 238,\n      "n_native_gt_total": 350,\n      "token_mask_sha256": "944469817cb1d463be9ba8e22d7c037bf6b358337512a5973f0eedd13fdd691e",\n      "native_gt_timestamp_sha256": "35b6aa9a2b58fdae8ed82b1803edf40f58ed5821a00c87204e7a4ff45323ec94"\n    },\n    {\n      "id": "train_ep30_mars_1_d30_1",\n      "epoch": 30,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 55.300000000000004,\n      "start_s": 35.300000000000004,\n      "end_s": 95.30000000000001,\n      "n_native_gt_outage": 122,\n      "n_native_gt_total": 249,\n      "token_mask_sha256": "ec337caf619f2fc34accfde4fefc63f5f100805a2dc4a30b788f58ddce2c4bc8",\n      "native_gt_timestamp_sha256": "509c9c60d511b02d0f20ddd2036426ea64303ede0e1eff75c1ea44748eca30eb"\n    },\n    {\n      "id": "train_ep30_mars_1_d10_1",\n      "epoch": 30,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 38.85,\n      "start_s": 18.85,\n      "end_s": 58.85,\n      "n_native_gt_outage": 48,\n      "n_native_gt_total": 163,\n      "token_mask_sha256": "54d1f8b7a9217c3e8206fec44aa90fc97fdb3e89370798f9369d21bcb530fdd3",\n      "native_gt_timestamp_sha256": "fa96b8eb29cdccb1f7aff89629fa4c7783a9f08d9d7225eaf8b4ed780ae61562"\n    },\n    {\n      "id": "train_ep30_mars_2_d10_1",\n      "epoch": 30,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 25.05,\n      "start_s": 5.050000000000001,\n      "end_s": 45.05,\n      "n_native_gt_outage": 36,\n      "n_native_gt_total": 149,\n      "token_mask_sha256": "54d1f8b7a9217c3e8206fec44aa90fc97fdb3e89370798f9369d21bcb530fdd3",\n      "native_gt_timestamp_sha256": "436d53ccf04e24cadabfdddb20c332e859a724432a49e9795a0b30c4eed9515b"\n    },\n    {\n      "id": "train_ep30_mars_1_d10_0",\n      "epoch": 30,\n      "flight": "mars_1",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 36.65,\n      "start_s": 16.65,\n      "end_s": 56.65,\n      "n_native_gt_outage": 40,\n      "n_native_gt_total": 164,\n      "token_mask_sha256": "54d1f8b7a9217c3e8206fec44aa90fc97fdb3e89370798f9369d21bcb530fdd3",\n      "native_gt_timestamp_sha256": "faa29a8c8c5d4f710085016d0dd70aa7ec2dc7b5b2b159782385caed7d2c2361"\n    },\n    {\n      "id": "train_ep30_mars_3_d30_1",\n      "epoch": 30,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 74.35000000000001,\n      "start_s": 54.35000000000001,\n      "end_s": 114.35000000000001,\n      "n_native_gt_outage": 124,\n      "n_native_gt_total": 237,\n      "token_mask_sha256": "ec337caf619f2fc34accfde4fefc63f5f100805a2dc4a30b788f58ddce2c4bc8",\n      "native_gt_timestamp_sha256": "c74a6f55bf2814bc11fd7dbf3c9da6415d594a5df20908615083f4f56d92f520"\n    },\n    {\n      "id": "train_ep30_mars_3_d30_0",\n      "epoch": 30,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 93.05000000000001,\n      "start_s": 73.05000000000001,\n      "end_s": 133.05,\n      "n_native_gt_outage": 119,\n      "n_native_gt_total": 235,\n      "token_mask_sha256": "63d833f0b713bb4fb84f25c62b86deb8f26260f7a627284500d089600ceaa5cc",\n      "native_gt_timestamp_sha256": "0da9167b7a6ff74d2c2df7902f2d90f3fa755f3aa6a3898424827f91937aff0b"\n    },\n    {\n      "id": "train_ep30_mars_3_d10_1",\n      "epoch": 30,\n      "flight": "mars_3",\n      "split": "train",\n      "duration_s": 10,\n      "onset_s": 28.650000000000002,\n      "start_s": 8.650000000000002,\n      "end_s": 48.650000000000006,\n      "n_native_gt_outage": 40,\n      "n_native_gt_total": 166,\n      "token_mask_sha256": "54d1f8b7a9217c3e8206fec44aa90fc97fdb3e89370798f9369d21bcb530fdd3",\n      "native_gt_timestamp_sha256": "472c2881a132653e86a9f2a8c73349b7aea9dd9071f89f7c9636246d2aa5b91b"\n    },\n    {\n      "id": "train_ep30_mars_2_d60_0",\n      "epoch": 30,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 60,\n      "onset_s": 38.35,\n      "start_s": 18.35,\n      "end_s": 108.35,\n      "n_native_gt_outage": 251,\n      "n_native_gt_total": 358,\n      "token_mask_sha256": "c0c2ec8564a28a400ecdd4fe90d4d6b443981b414ff75cc3098c2fd7dc691a8b",\n      "native_gt_timestamp_sha256": "9337a2b8ae988af7bbf274b80add228240af906b432e431b30e9a41a45266a9a"\n    },\n    {\n      "id": "train_ep30_mars_2_d30_0",\n      "epoch": 30,\n      "flight": "mars_2",\n      "split": "train",\n      "duration_s": 30,\n      "onset_s": 78.10000000000001,\n      "start_s": 58.10000000000001,\n      "end_s": 118.10000000000001,\n      "n_native_gt_outage": 126,\n      "n_native_gt_total": 242,\n      "token_mask_sha256": "ec337caf619f2fc34accfde4fefc63f5f100805a2dc4a30b788f58ddce2c4bc8",\n      "native_gt_timestamp_sha256": "d27d67e3fe88711279c38b1065d203a21725ae9b05f00c01fdf2025d9a263382"\n    }\n  ],\n  "same_for_both_models": true\n}\n', 'configs/insane_adaptive/adaptive.json': '{\n  "pilot_id": "fusionnav_insane_adaptive_v1",\n  "methods": [\n    "adaptive_full",\n    "adaptive_gps_only"\n  ],\n  "max_epochs": 30,\n  "patience": 5,\n  "min_delta_m": 0.0,\n  "validation_criterion": "macro mean episode relative-motion RMSE3D on fixed validation outages",\n  "batch_size": 4,\n  "learning_rate": 0.001,\n  "weight_decay": 0.0001,\n  "grad_clip": 1.0,\n  "huber_delta_m": 1.0,\n  "hidden": 32,\n  "token_period_s": 0.05,\n  "imu_gap_limit_s": 0.1,\n  "gps_gap_limit_s": 1.0,\n  "gt_gap_limit_s": 2.0,\n  "history_s": 20.0,\n  "recovery_s": 10.0,\n  "duration_s": [\n    10,\n    30,\n    60\n  ],\n  "validation_onset_stride_s": 60.0,\n  "train_episodes_per_flight_duration_epoch": 2,\n  "cv_history_s": 5.0,\n  "normalization": "IMU dt-weighted mean/std on train flights only; delta/velocity fixed /5; cov standard deviation /10",\n  "imu_frame": "published PX4 MAVROS body axes, ROS FLU; never rotate using GT",\n  "position_frame": "published local ENU, common per-flight first ordinary GNSS origin",\n  "gt_point": "PX4 IMU origin, authors dual RTK plus magnetometer reference",\n  "gnss_point": "ordinary GNSS antenna; its lever arm to PX4 IMU is absent from supplied calibration",\n  "eskf": {\n    "real_data_ready": false,\n    "blockers": [\n      "ordinary GNSS antenna to PX4 IMU lever arm not supplied",\n      "absolute initial yaw unavailable from permitted sensors without validated dynamic alignment"\n    ],\n    "sigma_accel_density": 0.1,\n    "sigma_gyro_density": 0.01,\n    "sigma_accel_bias_walk": 0.001,\n    "sigma_gyro_bias_walk": 0.0001,\n    "noise_status": "engineering defaults for synthetic mechanization checks only; not tuned or asserted flight calibration",\n    "gnss_lever_arm_body_m": null,\n    "parameter_tuning": "none"\n  },\n  "test_used": false,\n  "extra_seeds": false,\n  "seeds": [\n    0,\n    1,\n    2\n  ],\n  "gate": {\n    "type": "sigmoid linear GRU hidden to scalar",\n    "initial_weight": 0.0,\n    "initial_bias": 0.0,\n    "initial_g": 0.5,\n    "velocity": "g * (CV + residual)"\n  },\n  "damping": {\n    "rule": "v(age) = v_CV * exp(-age/tau), age since last legal fix; exact integral",\n    "tau_candidates_s": [\n      1,\n      2,\n      5,\n      10,\n      20,\n      30,\n      60,\n      120\n    ],\n    "selection": "minimum macro relative-motion RMSE across all 540 frozen train episodes; smaller tau on ties; before any validation metric"\n  },\n  "seed_effect": "initial weights only; frozen pilot episode order/augmentation shared across all seeds",\n  "previous_pilot_comparison": "seed0 descriptive only; never pooled with adaptive seeds",\n  "test_loaded": false\n}\n', 'configs/insane_adaptive/previous_pilot_summary.csv': 'scenario,flight,split,method,seed,duration_s,status,n_native_gt,relative_motion_rmse3d_m,absolute_rmse3d_m,absolute_rmse_h_m,absolute_rmse_v_m,final_unavailable_error_m,final_relative_motion_error_m,recovery_first_5s_rmse3d_m,first_recovered_fix_error_m,inference_ms,reason\nmars_4_e1_d10,mars_4,validation,held_gnss,,10,complete,29.0,0.505638754841268,2.693867746558548,1.9935194872448374,1.8118508464891667,3.000195882566154,0.8214462111901781,2.9427343432681736,2.9530275469200427,0.1026890000161984,\nmars_4_e1_d10,mars_4,validation,constant_velocity_gnss,,10,complete,29.0,1.4882019585511368,2.4548701784478797,2.3746778771685886,0.6223283479955023,2.8038903476507726,2.2932269496163564,2.932554368285694,2.9530275469200427,0.1026890000161984,\nmars_4_e1_d10,mars_4,validation,full,0.0,10,complete,29.0,2.0238522566931483,1.476656730966479,1.2255718146063013,0.8237043330899908,1.6012823974078412,3.2034973236064834,2.8820284590534144,2.9530275469200427,1.6757260000304086,\nmars_4_e1_d10,mars_4,validation,gps_only,0.0,10,complete,29.0,1.6593906392168265,1.8049726420715886,1.660593603883064,0.7073578453438886,1.8870966555517803,2.6278667260259985,2.8913901902457306,2.9530275469200427,1.6625339999905009,\nmars_4_e1_d10,mars_4,validation,eskf,,10,not_ready,,,,,,,,,,,ordinary GNSS antenna to PX4 IMU lever arm not supplied; absolute initial yaw unavailable from permitted sensors without validated dynamic alignment\nmars_4_e1_d30,mars_4,validation,held_gnss,,30,complete,115.0,5.346875227228348,6.005079020145959,5.283606637936581,2.8538526475187274,17.189280130341896,16.67429571272127,2.5983106903385833,2.5460928608689124,0.08091300003343349,\nmars_4_e1_d30,mars_4,validation,constant_velocity_gnss,,30,complete,115.0,9.552351248184003,9.45467152278622,6.364850591448302,6.991386883324719,23.816958051159794,23.808220652818186,2.5983106903385833,2.5460928608689124,0.08091300003343349,\nmars_4_e1_d30,mars_4,validation,full,0.0,30,complete,115.0,10.038588404665909,9.048420463081023,3.673649729277456,8.269111835214412,22.427524935882364,23.121021657492392,2.5983106903385833,2.5460928608689124,1.799397000013414,\nmars_4_e1_d30,mars_4,validation,gps_only,0.0,30,complete,115.0,9.466743554095283,8.8333053914744,4.412050347180955,7.652522190297204,22.36221620527203,22.74821250861817,2.5983106903385833,2.5460928608689124,1.7372119999663482,\nmars_4_e1_d30,mars_4,validation,eskf,,30,not_ready,,,,,,,,,,,ordinary GNSS antenna to PX4 IMU lever arm not supplied; absolute initial yaw unavailable from permitted sensors without validated dynamic alignment\nmars_4_e1_d60,mars_4,validation,held_gnss,,60,complete,231.0,24.96708711370028,25.865797774368218,24.548413453111632,8.149533203785351,49.253468783896224,47.865742094996,1.6812522935441816,1.3544766518031564,0.08406999995713704,\nmars_4_e1_d60,mars_4,validation,constant_velocity_gnss,,60,complete,231.0,31.544700059296094,31.99268479443216,26.578196671575792,17.808181884822737,57.93524570357578,57.054658685013436,1.6812522935441816,1.3544766518031564,0.08406999995713704,\nmars_4_e1_d60,mars_4,validation,full,0.0,60,complete,231.0,28.533321828786168,28.44834547965753,19.579294670911843,20.63878825705507,50.888498771942494,50.55960406845913,1.6812522935441816,1.3544766518031564,2.179665999960889,\nmars_4_e1_d60,mars_4,validation,gps_only,0.0,60,complete,231.0,29.467425849449366,29.643436524343624,22.65463249990397,19.118079277665018,54.03075231892688,53.41664660357926,1.6812522935441816,1.3544766518031564,2.17528000001721,\nmars_4_e1_d60,mars_4,validation,eskf,,60,not_ready,,,,,,,,,,,ordinary GNSS antenna to PX4 IMU lever arm not supplied; absolute initial yaw unavailable from permitted sensors without validated dynamic alignment\nmars_4_e2_d10,mars_4,validation,held_gnss,,10,complete,48.0,18.557776345125195,18.68655807598481,18.52740433683505,2.433668273733858,25.511162073570635,25.477702452691997,3.0621519987802133,2.432964321568216,0.07178900000326394,\nmars_4_e2_d10,mars_4,validation,constant_velocity_gnss,,10,complete,48.0,14.789161470726468,15.603277748993603,15.007243632933092,4.2714066832869095,20.1620892399233,19.275967852825115,3.0621519987802133,2.432964321568216,0.07178900000326394,\nmars_4_e2_d10,mars_4,validation,full,0.0,10,complete,48.0,13.39961806744512,14.196455476632927,13.352374427694352,4.822182622285779,18.058969544566427,17.19740883746609,3.0621519987802133,2.432964321568216,1.7116799999712384,\nmars_4_e2_d10,mars_4,validation,gps_only,0.0,10,complete,48.0,14.145136033145484,14.942111987546486,14.23508714675711,4.541916398681393,19.157615042588343,18.294143218137968,3.0621519987802133,2.432964321568216,1.6125500000043758,\nmars_4_e2_d10,mars_4,validation,eskf,,10,not_ready,,,,,,,,,,,ordinary GNSS antenna to PX4 IMU lever arm not supplied; absolute initial yaw unavailable from permitted sensors without validated dynamic alignment\nmars_4_e2_d30,mars_4,validation,held_gnss,,30,complete,120.0,33.256678060435156,33.488176178883535,33.36241627213079,2.899504142608759,58.06512127412136,57.51123988617195,3.250257523327791,2.664977419914867,0.08448900001667425,\nmars_4_e2_d30,mars_4,validation,constant_velocity_gnss,,30,complete,120.0,33.683585786814675,34.617939966117206,33.790171179451505,7.525031505652288,68.97494271491065,67.77462357662496,3.250257523327791,2.664977419914867,0.08448900001667425,\nmars_4_e2_d30,mars_4,validation,full,0.0,30,complete,120.0,30.238074411760813,31.17056136881805,29.884677115065283,8.860585182455244,62.936701451915795,61.72721688368699,3.250257523327791,2.664977419914867,1.8746349999787526,\nmars_4_e2_d30,mars_4,validation,gps_only,0.0,30,complete,120.0,31.80596590938029,32.735760363881056,31.687472880978305,8.21791147547298,65.54258971868325,64.33971070274218,3.250257523327791,2.664977419914867,1.8482240000139427,\nmars_4_e2_d30,mars_4,validation,eskf,,30,not_ready,,,,,,,,,,,ordinary GNSS antenna to PX4 IMU lever arm not supplied; absolute initial yaw unavailable from permitted sensors without validated dynamic alignment\nmars_4_e2_d60,mars_4,validation,held_gnss,,60,complete,244.0,52.24204008053458,52.696970133369696,52.429472881684276,5.302926982902118,30.689360151721104,30.10491393491468,2.9841492785157326,2.7015005999881696,0.09432699999933902,\nmars_4_e2_d60,mars_4,validation,constant_velocity_gnss,,60,complete,244.0,71.2809927581726,72.2908111861054,71.25685365759503,12.182864554989884,106.5635060554521,105.46322901306587,2.9841492785157326,2.7015005999881696,0.09432699999933902,\nmars_4_e2_d60,mars_4,validation,full,0.0,60,complete,244.0,65.64489603526178,66.64267034959336,65.03482155023201,14.550515360427909,100.93268252892693,99.90022632159354,2.9841492785157326,2.7015005999881696,2.225985999984914,\nmars_4_e2_d60,mars_4,validation,gps_only,0.0,60,complete,244.0,67.8477993925995,68.85211184822002,67.5222540734539,13.466941404778495,102.08460827850824,101.01329922254882,2.9841492785157326,2.7015005999881696,2.18691600002785,\nmars_4_e2_d60,mars_4,validation,eskf,,60,not_ready,,,,,,,,,,,ordinary GNSS antenna to PX4 IMU lever arm not supplied; absolute initial yaw unavailable from permitted sensors without validated dynamic alignment\nmars_4_control,mars_4,validation,held_gnss,,0,complete,711.0,,2.9449144346942773,1.5303400098368922,2.5160644828706475,,,,,0.1319879999641671,\nmars_4_control,mars_4,validation,constant_velocity_gnss,,0,complete,711.0,,2.9449144346942773,1.5303400098368922,2.5160644828706475,,,,,0.1319879999641671,\nmars_4_control,mars_4,validation,full,0.0,0,complete,711.0,,2.9449144346942773,1.5303400098368922,2.5160644828706475,,,,,3.8405060000172853,\nmars_4_control,mars_4,validation,gps_only,0.0,0,complete,711.0,,2.9449144346942773,1.5303400098368922,2.5160644828706475,,,,,3.3429759999989983,\nmars_4_control,mars_4,validation,eskf,,0,not_ready,,,,,,,,,,,ordinary GNSS antenna to PX4 IMU lever arm not supplied; absolute initial yaw unavailable from permitted sensors without validated dynamic alignment\nmars_5_e1_d10,mars_5,validation,held_gnss,,10,complete,40.0,0.5871649756165576,6.203632313923567,3.59620380573263,5.054935417390692,6.22060102758166,0.599731512422308,5.608906500915366,5.611468796799512,0.07763000002114495,\nmars_5_e1_d10,mars_5,validation,constant_velocity_gnss,,10,complete,40.0,0.9331397791179162,5.185223131212543,3.4522275161168454,3.8689357835750293,4.3823727914758965,1.6612573624208253,5.515861328902866,5.611468796799512,0.07763000002114495,\nmars_5_e1_d10,mars_5,validation,full,0.0,10,complete,40.0,1.7044380118794815,5.16722825134066,3.7542766273926578,3.5504443111343496,4.5618066756685645,2.904128003820086,5.523827485680941,5.611468796799512,1.9503720000102476,\nmars_5_e1_d10,mars_5,validation,gps_only,0.0,10,complete,40.0,1.350168342052643,5.116049272428045,3.555290953352317,3.678840360075808,4.34268214668204,2.3276938325201946,5.514258774059938,5.611468796799512,1.6082629999800702,\nmars_5_e1_d10,mars_5,validation,eskf,,10,not_ready,,,,,,,,,,,ordinary GNSS antenna to PX4 IMU lever arm not supplied; absolute initial yaw unavailable from permitted sensors without validated dynamic alignment\nmars_5_e1_d30,mars_5,validation,held_gnss,,30,complete,122.0,0.5995465744024691,6.216036417684388,3.594904128276661,5.071072179971212,6.22368437054511,0.6080490760649166,6.377110124028476,6.217555972997001,0.08108400004402938,\nmars_5_e1_d30,mars_5,validation,constant_velocity_gnss,,30,complete,122.0,3.485750097341764,4.052329512529031,3.2005784590078865,2.485492306535765,3.2433212328763834,6.124660065125279,6.217585060158699,6.217555972997001,0.08108400004402938,\nmars_5_e1_d30,mars_5,validation,full,0.0,30,complete,122.0,5.666687075275633,5.4791082786308065,4.838138691575154,2.5715834674464864,7.528580941321103,9.794785809181182,6.479783479664769,6.217555972997001,1.8655690000173308,\nmars_5_e1_d30,mars_5,validation,gps_only,0.0,30,complete,122.0,4.628628327256628,4.587170769135885,3.8801212120880444,2.446792807885275,5.360385685649774,8.039916281230665,6.322453296456052,6.217555972997001,1.8412160000025324,\nmars_5_e1_d30,mars_5,validation,eskf,,30,not_ready,,,,,,,,,,,ordinary GNSS antenna to PX4 IMU lever arm not supplied; absolute initial yaw unavailable from permitted sensors without validated dynamic alignment\nmars_5_e1_d60,mars_5,validation,held_gnss,,60,complete,226.0,6.330690751632288,6.426742214519098,4.980891564253312,4.061247926072975,2.9730255439590274,7.457616398014753,7.593655483507831,7.475861740735473,0.09092900000950976,\nmars_5_e1_d60,mars_5,validation,constant_velocity_gnss,,60,complete,226.0,12.178495665904865,9.214092130750368,4.95394797641064,7.769034254073846,13.750178687862972,19.141591595355408,8.3026071760027,7.475861740735473,0.09092900000950976,\nmars_5_e1_d60,mars_5,validation,full,0.0,60,complete,226.0,16.21858543749416,13.71628160408114,9.552163938618294,9.843401095770451,19.35460550304824,23.96019333813454,8.979110406936158,7.475861740735473,2.194583000004968,\nmars_5_e1_d60,mars_5,validation,gps_only,0.0,60,complete,226.0,14.5598541290255,11.823318927972155,7.740488678521804,8.937320934722017,16.575940803371072,21.672608135016628,8.621471484207243,7.475861740735473,2.1471709999900668,\nmars_5_e1_d60,mars_5,validation,eskf,,60,not_ready,,,,,,,,,,,ordinary GNSS antenna to PX4 IMU lever arm not supplied; absolute initial yaw unavailable from permitted sensors without validated dynamic alignment\nmars_5_control,mars_5,validation,held_gnss,,0,complete,348.0,,6.648175247783689,3.486029714776712,5.6609037222812315,,,,,0.09102799998572664,\nmars_5_control,mars_5,validation,constant_velocity_gnss,,0,complete,348.0,,6.648175247783689,3.486029714776712,5.6609037222812315,,,,,0.09102799998572664,\nmars_5_control,mars_5,validation,full,0.0,0,complete,348.0,,6.648175247783689,3.486029714776712,5.6609037222812315,,,,,2.314011000009941,\nmars_5_control,mars_5,validation,gps_only,0.0,0,complete,348.0,,6.648175247783689,3.486029714776712,5.6609037222812315,,,,,2.1369820000245454,\nmars_5_control,mars_5,validation,eskf,,0,not_ready,,,,,,,,,,,ordinary GNSS antenna to PX4 IMU lever arm not supplied; absolute initial yaw unavailable from permitted sensors without validated dynamic alignment\n', 'configs/insane_adaptive/protocol_lock.json': '{\n  "sha256": {\n    "configs/insane/download_manifest.json": "6723a6bdfc90f5ce608be5cba030509d58a694b976f8b2e675d6f9250b1c029a",\n    "configs/insane/inventory.json": "f370ecb2063be698d8a2397c5f713afa787c3d40110fba4302e4dfaba20d8918",\n    "configs/insane/outages.json": "208ef71a355441c2f795155b448cc29626bca974ecb506960c74757fc07eb474",\n    "configs/insane/pilot.json": "4718d7a7415e89a5ccffe91bcb202db655174113d86186a328761c43de4c1a24",\n    "configs/insane/protocol_lock.json": "e64534f12ce81c092241b1b8ee8c466553175d5484871ece37f4b2f809a73201",\n    "configs/insane/scaler_preflight.json": "7549771e7428907e5a89039540b08e007b7aeb37023ccd6d92cd4d37b94ab6dd",\n    "configs/insane/splits.json": "a2f27d70ec0bae01dd7eb5dabf94994dbedd48898f72b293823549867fa075f4",\n    "configs/insane/training_episodes.json": "84b85d23a0006c2039ec6ee3cb2002a02e11faa1a61e140dde24a95d74e442dd"\n  },\n  "freeze_stage": "before adaptive training or validation metrics",\n  "changes_allowed": [\n    "scalar gate head",\n    "train-selected damped CV",\n    "seeds 0 1 2"\n  ],\n  "adaptive_config_sha256": "fde82dbe224e363979af23c49d8ad2f1f8c36b4f4f724952a0a8481fb107e4b9"\n}\n'}
SOURCE_HASHES = {'src/insane/__init__.py': '5c973c05f41ebe954723a5fb174da4f55a29ce37fc13cb7b389298a808a9c03c', 'src/insane/data.py': '6b163075774a1fcc529091f3ea3c28181063444cae3a3f5839f11542df116a36', 'src/insane/eskf.py': '21ee116adb17133d20870fa8a1eec40a92e13a7215ca5f07e9a472d47590b2fc', 'src/insane/metrics.py': 'bfd5c5b1bbd77e5b428649884b82ed374eec5248558208ac3b586b6b122d0a92', 'src/insane/model.py': 'b0e7cc88af6404f8bafdab7616ff04af9e8e8d646ed3ab777a5ee92b20b9db8c', 'src/insane/report.py': 'b26522beafa1912224c63a762957089fcab9264f7e9d75324cef8ba17d5e25eb', 'src/insane/run.py': '559438510a2a79404695520e496509db2905b8324d438a64111dd3890f5dd81e', 'src/insane_adaptive/__init__.py': '36db52b0c2271401a7cdd7f1365d2b059d01a083d175f036409bd7f43095c061', 'src/insane_adaptive/baselines.py': '7f9a914e3746bb4e1422fa2f5018328b9c2ea19fa37d5d2394052f8f4b54042b', 'src/insane_adaptive/model.py': 'daea4d943eebe0aa4a3eab273b7e084b2cf7a14308e291bcfd6c6b2e6e7367dd', 'src/insane_adaptive/report.py': '54367164506447d5d2d3c77e14836ecce3780d8d61ab3f2b9ae3ec779ddd6b0a', 'src/insane_adaptive/run.py': '5a17a820acf8a64fc5cadc58ebdea26de04601a4833abc53f44cf6ac61284282', 'tests/insane/test_pipeline.py': 'c3e122fb6c6ca3d4e9f555514fae654880c061bb97b36c70ca91bdcb5a6c543a', 'tests/insane_adaptive/test_gate.py': '629d4acd90b196f04009cba50c238bf7c6ab6a1919a56208a6b14074c15327f1', 'configs/insane/download_manifest.json': '6723a6bdfc90f5ce608be5cba030509d58a694b976f8b2e675d6f9250b1c029a', 'configs/insane/inventory.json': 'f370ecb2063be698d8a2397c5f713afa787c3d40110fba4302e4dfaba20d8918', 'configs/insane/outages.json': '208ef71a355441c2f795155b448cc29626bca974ecb506960c74757fc07eb474', 'configs/insane/pilot.json': '4718d7a7415e89a5ccffe91bcb202db655174113d86186a328761c43de4c1a24', 'configs/insane/protocol_lock.json': 'e64534f12ce81c092241b1b8ee8c466553175d5484871ece37f4b2f809a73201', 'configs/insane/scaler_preflight.json': '7549771e7428907e5a89039540b08e007b7aeb37023ccd6d92cd4d37b94ab6dd', 'configs/insane/splits.json': 'a2f27d70ec0bae01dd7eb5dabf94994dbedd48898f72b293823549867fa075f4', 'configs/insane/training_episodes.json': '84b85d23a0006c2039ec6ee3cb2002a02e11faa1a61e140dde24a95d74e442dd', 'configs/insane_adaptive/adaptive.json': 'fde82dbe224e363979af23c49d8ad2f1f8c36b4f4f724952a0a8481fb107e4b9', 'configs/insane_adaptive/previous_pilot_summary.csv': '98f132ce74c1cb8c47976bb67fa3324ccfcc6a5c073fe5a02f46c6242c60086a', 'configs/insane_adaptive/protocol_lock.json': '8356b9370a209d61ba0e7a1c6653e58929f0d30d8b179f0d780cdff1c91e9953'}
for relative,source in SOURCES.items():
    content=source.encode()
    if hashlib.sha256(content).hexdigest()!=SOURCE_HASHES[relative]:raise RuntimeError('Embedded source mismatch')
    dest=CODE_ROOT/relative;dest.parent.mkdir(parents=True,exist_ok=True)
    if dest.exists() and dest.read_bytes()!=content:raise RuntimeError('Use a fresh session; source differs')
    if not dest.exists():dest.write_bytes(content)
sys.path.insert(0,str(CODE_ROOT/'src'))


In [ ]:
import zipfile,json,importlib.util,base64,io
required=['numpy','pandas','torch','scipy','matplotlib']
missing=[n for n in required if importlib.util.find_spec(n) is None]
if missing:raise RuntimeError('Missing dependencies (no install attempted): '+str(missing))
if not DATA_ROOT.exists():raise FileNotFoundError('Attach the private INSANE data package and set DATA_ROOT')
payloads=list(DATA_ROOT.rglob('insane_payload.b64'))
roots=[]
if payloads:
    if len(payloads)!=1:raise RuntimeError('Ambiguous pinned payload')
    payload=base64.b64decode(payloads[0].read_bytes(),validate=True)
    if hashlib.sha256(payload).hexdigest()!='41d462d2f4e0d2f4f41989175c45a9ce63064bd29e3ec42ab0fd499c27b63408':raise RuntimeError('Transport hash mismatch')
    extracted=Path('/kaggle/working/insane_inputs')
    with zipfile.ZipFile(io.BytesIO(payload)) as z:
        if sum(i.file_size for i in z.infolist())>100_000_000:raise RuntimeError('Unexpected bundle size')
        for n in z.namelist():
            if Path(n).is_absolute() or '..' in Path(n).parts:raise RuntimeError('Unsafe archive path')
        z.extractall(extracted)
    roots=[p.parent.parent for p in extracted.rglob('px4_imu.csv') if p.parent.name=='mars_1']
else:
    roots=[p.parent.parent for p in DATA_ROOT.rglob('px4_imu.csv') if p.parent.name=='mars_1']
if not roots:
    archives=list(DATA_ROOT.rglob('insane_data.zip'))
    if len(archives)!=1:raise RuntimeError('Expected one INSANE input archive')
    extracted=Path('/kaggle/working/insane_inputs')
    with zipfile.ZipFile(archives[0]) as z:
        if sum(i.file_size for i in z.infolist())>100_000_000:raise RuntimeError('Unexpected bundle size')
        for n in z.namelist():
            if Path(n).is_absolute() or '..' in Path(n).parts:raise RuntimeError('Unsafe archive path')
        z.extractall(extracted)
    roots=[p.parent.parent for p in extracted.rglob('px4_imu.csv') if p.parent.name=='mars_1']
if len(roots)!=1:raise RuntimeError('Ambiguous data mount')
RAW_ROOT=roots[0]
from insane_adaptive.run import preflight,code_version
checks=preflight(RAW_ROOT,CODE_ROOT)[-1]
(LAUNCH_DIR/'preflight.json').write_text(json.dumps(checks,indent=2)+'\n')
print(checks)


In [ ]:
import subprocess
for suite in ['insane','insane_adaptive']:
    result=subprocess.run([sys.executable,'-B','-m','unittest','discover','-s',str(CODE_ROOT/'tests'/suite),'-p','test_*.py','-v'],cwd=CODE_ROOT,capture_output=True,text=True)
    log=result.stdout+'\n'+result.stderr
    (LAUNCH_DIR/(suite+'_tests.log')).write_text(log)
    print(log)
    if result.returncode:raise RuntimeError('Tests failed; STOP, no training/retry')


In [ ]:
from insane_adaptive.run import execute
if RUN_TRAINING:
    execute(RAW_ROOT,CODE_ROOT,OUT_DIR,run_training=True)
else:
    print('Only mandatory preflight/tests; no training or evaluation.')


Download `outputs/insane_adaptive/` and `insane_adaptive_launch/`.
Includes automatic INSANE_ADAPTIVE_REPORT.md, METHODS_DRAFT.md and every gate trace.
STOP after six scheduled trainings. No test, additional seeds, retries or next version.